# ConfCarti — coverage-guaranteed knee cartilage morphometry from 3D DESS MRI

Thickness, **3D curvature** and **focal bulge** of knee articular cartilage, with an
explicit denuded-bone (full-thickness-loss) map and **conformal prediction intervals**
whose coverage is guaranteed in finite samples and evaluated *conditionally on
Kellgren–Lawrence grade and acquisition site*.

## What this notebook runs, top to bottom

| § | Stage |
|---|-------|
| 1 | Library code (inlined package modules) |
| 2 | Data: OAI-ZIB metadata, or analytic phantoms when no restricted data is present |
| 3 | Dataset analysis and visualisation |
| 4 | Geometry validation against closed-form answers |
| 5 | Per-knee morphometry + 3D surface maps |
| 6 | Segmentation model: train, infer, evaluate |
| 7 | The conformal layer |
| 8 | Reliability, agreement and detectability |
| 9 | Comparative / ablation analysis |
| 10 | Results summary |

**Runtime.** About 15–25 minutes end to end on a laptop CPU with the default
`N_PHANTOM = 24`; the per-knee geometry (§5) and the ablation sweep (§9) dominate.
Set `N_PHANTOM = 8` and `ABLATION_N = 4` for a ~3-minute pass that exercises every
code path. A GPU only speeds up §6; everything else is CPU-bound numpy/scipy.

## Provenance

The algorithms are ports of, or direct implementations from, published sources —
each is cited in the docstring of the function that implements it:

* **CartiMorph** (Yao et al., *Medical Image Analysis* 91:103035, 2024) —
  surface-normal thickness (`CM_cal_estimateSN.m`, `CM_cal_thicknessMap_SN.m`),
  rule-based parcellation (`CM_cal_SurfaceParcellation_FC/TC.m`), surface closing,
  full-thickness-loss estimation. <https://github.com/YongchengYAO/CartiMorph>
* **Conformalized quantile regression** — Romano, Patterson & Candès, NeurIPS 2019;
  score matches `QuantileRegErrFunc` in <https://github.com/yromano/cqr>.
* **Conformal risk control** — Angelopoulos, Bates, Fisch, Lei & Schuster, ICLR 2024;
  `get_lhat` from <https://github.com/aangelopoulos/conformal-risk>.
* **Conformal under covariate shift** — Tibshirani, Barber, Candès & Ramdas, NeurIPS 2019.
* **Discrete curvature** — Meyer, Desbrun, Schröder & Barr (2003); shape index and
  curvedness from Koenderink & van Doorn (1992).
* **Morphometry nomenclature** (tAB / cAB / dAB / ThCtAB / ThCcAB) — Eckstein et al.,
  *Osteoarthritis and Cartilage* 14(10):974–983, 2006.

### ⚠️ Label convention — read this before using real data

OAIZIB-CM ships the 5-ROI convention

> **1** femur · **2** femoral cartilage · **3** tibia · **4** medial tibial cartilage ·
> **5** lateral tibial cartilage

(from the CartiMorph README: *"the femur (ROI1), femoral cartilage (ROI2), tibia (ROI3)
and tibial cartilage (ROI4)"*, with tibial cartilage later split medial/lateral).
A different ordering — cartilage first, bones last — circulates in secondary
descriptions and is **wrong**; swapping bone for cartilage produces meaningless
thickness maps rather than an error. `validate_label_convention()` checks this on real
data and is called for you in §5.

### Data access

No OAI or OAI-ZIB image, mask or clinical variable is bundled here — both require a
data use agreement (<https://nda.nih.gov/oai/>). §2 fetches the public subject table
if the network allows, and otherwise generates analytic phantoms so that every later
section still runs.

In [ ]:
# ---------------------------------------------------------------------------
# Environment. The geometry / conformal / reliability layers need only the
# scientific stack; torch + MONAI are used solely by section 6 and are optional.
# ---------------------------------------------------------------------------
import importlib, subprocess, sys

REQUIRED = {
    "numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "sklearn": "scikit-learn",
    "skimage": "scikit-image", "trimesh": "trimesh", "matplotlib": "matplotlib",
    "yaml": "PyYAML", "nibabel": "nibabel",
}
OPTIONAL = {"plotly": "plotly", "SimpleITK": "SimpleITK", "openpyxl": "openpyxl"}

missing = [pkg for mod, pkg in REQUIRED.items() if not importlib.util.find_spec(mod)]
if missing:
    print("installing:", " ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

for mod, pkg in OPTIONAL.items():
    if not importlib.util.find_spec(mod):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except Exception:
            print(f"optional package {pkg} unavailable; the notebook will skip what needs it")

import warnings, logging
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(name)s | %(message)s")

import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
try:
    get_ipython  # noqa: F821
    matplotlib.use("module://matplotlib_inline.backend_inline")
except NameError:
    matplotlib.use("Agg")

HAS_TORCH = importlib.util.find_spec("torch") is not None and importlib.util.find_spec("monai") is not None
HAS_PLOTLY = importlib.util.find_spec("plotly") is not None
print(f"numpy {np.__version__} | pandas {pd.__version__} | torch+MONAI: {HAS_TORCH} | plotly: {HAS_PLOTLY}")

## 1. Library code

Every cell below is the verbatim source of one package module, inlined so the notebook runs standalone. The code is identical to what the test suite in `tests/test_confcarti.py` validates.

### Deterministic seeding

`confcarti/utils/seeding.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/utils/seeding.py
# ==========================================================================
"""Deterministic seeding across Python, NumPy and PyTorch.

Reproducibility is a hard requirement of this project: two runs with the same
seed must produce byte-identical result CSVs.  This module centralises every
global RNG so that no module has to reach for ``random.seed`` on its own.
"""


import logging
import os
import random
from dataclasses import dataclass

import numpy as np

logger = logging.getLogger(__name__)

# CUBLAS needs this set *before* the CUDA context is created for
# ``torch.use_deterministic_algorithms(True)`` to work with cuBLAS GEMMs.
_CUBLAS_WORKSPACE_ENV = "CUBLAS_WORKSPACE_CONFIG"
_CUBLAS_WORKSPACE_VALUE = ":4096:8"


@dataclass(frozen=True)
class SeedReport:
    """What :func:`set_all_seeds` actually managed to configure.

    Attributes
    ----------
    seed
        The seed that was applied to every generator.
    torch_available
        Whether ``torch`` could be imported.
    cuda_available
        Whether ``torch.cuda.is_available()`` returned True.
    deterministic_algorithms
        Whether ``torch.use_deterministic_algorithms(True)`` was applied.
    cudnn_deterministic
        Whether the cuDNN deterministic flag was set.
    """

    seed: int
    torch_available: bool
    cuda_available: bool
    deterministic_algorithms: bool
    cudnn_deterministic: bool


def set_all_seeds(seed: int, *, deterministic_algorithms: bool = True) -> SeedReport:
    """Seed every global RNG used anywhere in ConfCarti.

    Parameters
    ----------
    seed
        Non-negative integer seed.
    deterministic_algorithms
        If True, request deterministic kernels from PyTorch.  This can reduce
        throughput noticeably for 3D convolutions, so training configs may turn
        it off deliberately; everything statistical keeps it on.

    Returns
    -------
    SeedReport
        A record of what was configured, suitable for the run manifest.

    Raises
    ------
    ValueError
        If ``seed`` is negative.

    Notes
    -----
    ``PYTHONHASHSEED`` is set here for completeness but only affects
    *subprocesses*; the hash seed of the current interpreter is fixed at
    start-up. Launch scripts that care should export it before ``python``.
    """
    if seed < 0:
        raise ValueError(f"seed must be non-negative, got {seed}")

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault(_CUBLAS_WORKSPACE_ENV, _CUBLAS_WORKSPACE_VALUE)

    random.seed(seed)
    np.random.seed(seed)

    torch_available = False
    cuda_available = False
    det_applied = False
    cudnn_det = False

    try:
        import torch
    except ImportError:
        logger.info("torch not installed; seeded python-random and numpy only (seed=%d)", seed)
    else:
        torch_available = True
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        cuda_available = bool(torch.cuda.is_available())

        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            cudnn_det = True

        if deterministic_algorithms:
            # warn_only=True: a handful of 3D ops (e.g. some upsampling kernels)
            # have no deterministic implementation. Failing hard there would make
            # the whole training pipeline unusable, so we degrade to a warning
            # and record the fact in the manifest.
            torch.use_deterministic_algorithms(True, warn_only=True)
            det_applied = True

    report = SeedReport(
        seed=seed,
        torch_available=torch_available,
        cuda_available=cuda_available,
        deterministic_algorithms=det_applied,
        cudnn_deterministic=cudnn_det,
    )
    logger.info(
        "seeded everything: seed=%d torch=%s cuda=%s deterministic=%s cudnn_deterministic=%s",
        report.seed,
        report.torch_available,
        report.cuda_available,
        report.deterministic_algorithms,
        report.cudnn_deterministic,
    )
    return report


def seeded_generator(seed: int) -> np.random.Generator:
    """Return an independent NumPy ``Generator`` for a local, side-effect-free RNG.

    Prefer this over the legacy global ``np.random`` state inside library
    functions -- it keeps a function's randomness reproducible without the
    caller having to reason about global state.

    Parameters
    ----------
    seed
        Seed for the PCG64 bit generator.

    Returns
    -------
    numpy.random.Generator
    """
    return np.random.default_rng(seed)


def worker_init_fn(worker_id: int) -> None:
    """DataLoader ``worker_init_fn`` that gives each worker a distinct stream.

    Without this, forked workers share the parent's NumPy seed and every worker
    draws the *same* augmentation parameters -- a classic silent bug.

    Parameters
    ----------
    worker_id
        Index of the DataLoader worker.
    """
    try:
        import torch

        base = int(torch.initial_seed()) % (2**31 - 1)
    except ImportError:  # pragma: no cover - torch-free environments
        base = 0
    seed = (base + worker_id) % (2**31 - 1)
    random.seed(seed)
    np.random.seed(seed)

### Frozen, strictly-validated configuration

`confcarti/config.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/config.py
# ==========================================================================
"""Frozen configuration hierarchy for ConfCarti experiments.

One YAML file fully determines an experiment. Configs are parsed into frozen
dataclasses with *strict* validation: unknown keys raise, missing required keys
raise, and out-of-range values raise. There are no magic numbers in function
bodies anywhere else in the package -- if a number influences a result it lives
here and is dumped into the run manifest.
"""


import dataclasses
from dataclasses import dataclass, field, fields, is_dataclass
from pathlib import Path
from functools import lru_cache
from types import UnionType
from typing import Any, Union, get_args, get_origin, get_type_hints

import yaml

__all__ = [
    "ConfigError",
    "DataConfig",
    "ModelConfig",
    "TrainConfig",
    "ThicknessConfig",
    "GeometryConfig",
    "DenudedConfig",
    "ConformalConfig",
    "ReliabilityConfig",
    "VizConfig",
    "ExperimentConfig",
    "load_config",
    "dump_config",
]


class ConfigError(ValueError):
    """Raised for any malformed, unknown or out-of-range configuration entry."""


# --------------------------------------------------------------------------- #
# Leaf configs
# --------------------------------------------------------------------------- #


@dataclass(frozen=True, slots=True)
class DataConfig:
    """Dataset location, preprocessing grid and split policy.

    Attributes
    ----------
    root
        Directory holding ``imagesTr``/``labelsTr`` style NIfTI data, or the
        OAIZIB-CM export directory.
    metadata_csv
        CSV with one row per subject: ``subject_id, kl_grade, site, visit,
        laterality``. See :mod:`confcarti.data.metadata`.
    spacing_mm
        Target isotropic voxel spacing after resampling, in millimetres.
    crop_size
        Joint-centred crop size in voxels, ``(x, y, z)``.
    n4_bias_correction
        Apply N4 inhomogeneity correction. Off by default: it costs ~30 s per
        volume and DESS at 3 T is fairly uniform.
    intensity_clip_percentiles
        Lower/upper percentiles used to clip before z-scoring, computed over the
        foreground only.
    cache_mode
        MONAI caching strategy: ``none``, ``cache`` or ``smart``.
    cache_rate
        Fraction of the dataset held in RAM when ``cache_mode != "none"``.
    num_workers
        DataLoader worker processes.
    split_fractions
        ``(train, calibration, test)`` subject fractions; must sum to 1.
    split_seed
        Seed for the subject-level stratified split.
    splits_json
        Path the immutable split file is written to and afterwards *read* from.
    stratify_on
        Metadata columns the split is jointly stratified on.
    """

    root: Path
    metadata_csv: Path
    spacing_mm: float = 0.5
    crop_size: tuple[int, int, int] = (160, 160, 96)
    n4_bias_correction: bool = False
    intensity_clip_percentiles: tuple[float, float] = (0.5, 99.5)
    cache_mode: str = "none"
    cache_rate: float = 1.0
    num_workers: int = 4
    split_fractions: tuple[float, float, float] = (0.6, 0.2, 0.2)
    split_seed: int = 20240617
    splits_json: Path = Path("results/splits.json")
    stratify_on: tuple[str, ...] = ("kl_grade", "site")

    def __post_init__(self) -> None:
        if self.spacing_mm <= 0:
            raise ConfigError(f"data.spacing_mm must be > 0, got {self.spacing_mm}")
        if len(self.crop_size) != 3 or any(c <= 0 for c in self.crop_size):
            raise ConfigError(f"data.crop_size must be 3 positive ints, got {self.crop_size}")
        if self.cache_mode not in {"none", "cache", "smart"}:
            raise ConfigError(
                f"data.cache_mode must be one of none|cache|smart, got {self.cache_mode!r}"
            )
        if not 0.0 <= self.cache_rate <= 1.0:
            raise ConfigError(f"data.cache_rate must be in [0, 1], got {self.cache_rate}")
        if len(self.split_fractions) != 3:
            raise ConfigError("data.split_fractions must have exactly 3 entries")
        if abs(sum(self.split_fractions) - 1.0) > 1e-9:
            raise ConfigError(
                f"data.split_fractions must sum to 1, got {sum(self.split_fractions)}"
            )
        if any(f <= 0 for f in self.split_fractions):
            raise ConfigError("every entry of data.split_fractions must be > 0")
        lo, hi = self.intensity_clip_percentiles
        if not 0.0 <= lo < hi <= 100.0:
            raise ConfigError(
                f"data.intensity_clip_percentiles must satisfy 0 <= lo < hi <= 100, got {(lo, hi)}"
            )
        if not self.stratify_on:
            raise ConfigError("data.stratify_on must name at least one column")


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Segmentation backbone selection and geometry.

    Attributes
    ----------
    backbone
        ``swinunetr``, ``segresnet`` or ``nnunet``.
    in_channels
        Input channels (1 for DESS magnitude).
    out_channels
        Number of classes *including* background, so 6 for the 5-ROI protocol.
    feature_size
        SwinUNETR embedding size; must be divisible by 12.
    freeze_encoder
        Train only the decoder + head (the FrozenSwinUNETR configuration).
    unfreeze_last_encoder_stage
        Release the deepest encoder stage even when ``freeze_encoder`` is set.
    use_v2
        Use SwinUNETR v2 residual blocks.
    dropout_rate
        Dropout probability inside the backbone.
    nnunet_model_dir
        Directory of a trained nnU-Net v2 model, used for inference only.
    checkpoint
        Optional checkpoint to initialise from.
    """

    backbone: str = "swinunetr"
    in_channels: int = 1
    out_channels: int = 6
    feature_size: int = 48
    freeze_encoder: bool = True
    unfreeze_last_encoder_stage: bool = False
    use_v2: bool = True
    dropout_rate: float = 0.0
    nnunet_model_dir: Path | None = None
    checkpoint: Path | None = None

    def __post_init__(self) -> None:
        allowed = {"swinunetr", "segresnet", "nnunet"}
        if self.backbone not in allowed:
            raise ConfigError(
                f"model.backbone must be one of {sorted(allowed)}, got {self.backbone!r}"
            )
        if self.feature_size % 12 != 0:
            raise ConfigError(
                f"model.feature_size must be divisible by 12 for SwinUNETR, got {self.feature_size}"
            )
        if self.out_channels < 2:
            raise ConfigError(f"model.out_channels must be >= 2, got {self.out_channels}")
        if not 0.0 <= self.dropout_rate < 1.0:
            raise ConfigError(f"model.dropout_rate must be in [0, 1), got {self.dropout_rate}")
        if self.backbone == "nnunet" and self.nnunet_model_dir is None:
            raise ConfigError("model.backbone='nnunet' requires model.nnunet_model_dir")


@dataclass(frozen=True, slots=True)
class TrainConfig:
    """Patch-based training schedule and memory policy.

    Attributes
    ----------
    patch_size
        Training patch in voxels.
    batch_size
        Patches per optimiser step.
    max_epochs
        Upper bound on epochs; early stopping usually fires first.
    learning_rate, weight_decay
        AdamW hyper-parameters.
    warmup_epochs
        Linear warm-up before the cosine schedule.
    grad_clip_norm
        Global gradient-norm clip.
    amp_dtype
        ``bf16``, ``fp16`` or ``fp32``.
    gradient_checkpointing
        Trade compute for VRAM in the encoder.
    dice_ce_lambda
        Weight of the Dice term; cross-entropy gets ``1 - lambda``.
    early_stopping_patience
        Epochs without internal-validation improvement before stopping.
    internal_val_fraction
        Fraction *of the training split* carved out for early stopping. The
        calibration split is never touched -- see :mod:`confcarti.data.guards`.
    sw_batch_size, sw_overlap
        Sliding-window inference parameters.
    """

    patch_size: tuple[int, int, int] = (128, 128, 128)
    batch_size: int = 2
    max_epochs: int = 500
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    warmup_epochs: int = 10
    grad_clip_norm: float = 1.0
    amp_dtype: str = "bf16"
    gradient_checkpointing: bool = True
    dice_ce_lambda: float = 0.5
    early_stopping_patience: int = 50
    internal_val_fraction: float = 0.15
    sw_batch_size: int = 2
    sw_overlap: float = 0.5

    def __post_init__(self) -> None:
        if self.amp_dtype not in {"bf16", "fp16", "fp32"}:
            raise ConfigError(
                f"train.amp_dtype must be bf16|fp16|fp32, got {self.amp_dtype!r}"
            )
        if not 0.0 <= self.dice_ce_lambda <= 1.0:
            raise ConfigError(
                f"train.dice_ce_lambda must be in [0, 1], got {self.dice_ce_lambda}"
            )
        if not 0.0 < self.internal_val_fraction < 0.5:
            raise ConfigError(
                "train.internal_val_fraction must be in (0, 0.5), got "
                f"{self.internal_val_fraction}"
            )
        if not 0.0 <= self.sw_overlap < 1.0:
            raise ConfigError(f"train.sw_overlap must be in [0, 1), got {self.sw_overlap}")
        if self.batch_size < 1:
            raise ConfigError(f"train.batch_size must be >= 1, got {self.batch_size}")
        if len(self.patch_size) != 3 or any(p <= 0 for p in self.patch_size):
            raise ConfigError(f"train.patch_size must be 3 positive ints, got {self.patch_size}")


@dataclass(frozen=True, slots=True)
class ThicknessConfig:
    """Mesh extraction and cartilage thickness measurement.

    Attributes
    ----------
    method
        ``surface_normal`` (primary), ``distance_transform`` or
        ``nearest_neighbour``. Ablation A1 sweeps this.
    smoothing_iterations
        Taubin smoothing passes applied to the extracted meshes.
    taubin_lambda, taubin_mu
        Taubin lambda/mu. ``mu < -lambda`` gives the shrinkage-free low-pass
        response of Taubin (1995).
    normal_neighbours
        k for the PCA/SVD surface-normal estimator ported from CartiMorph's
        ``CM_cal_estimateSN.m``.
    normal_smoothing_neighbours
        k for spatial smoothing of the normal field
        (``CM_cal_smoothSN.m``); 0 disables smoothing.
    max_depth_mm
        Rays longer than this are rejected. CartiMorph uses the same cap to
        stop a normal that grazes the surface from returning a spurious length.
    min_component_vertices
        Connected components smaller than this are dropped from the meshes.
    marching_cubes_level
        Iso-level for marching cubes on the (smoothed) binary mask.
    mask_gaussian_sigma_mm
        Gaussian pre-smoothing of the binary mask before marching cubes, in mm.
        Removes voxel staircasing so that curvature is not dominated by it.
    parcellation_scheme
        ``cartimorph20`` (Eckstein/Wirth 20-region) or ``compartment5``.
    central_region_percentage
        ``cc_percentage`` of CartiMorph's FC parcellation: the anterior-posterior
        extent of the central condylar strips, as a fraction of the distance
        from the intercondylar notch to the anterior cartilage margin.
    """

    method: str = "surface_normal"
    smoothing_iterations: int = 10
    taubin_lambda: float = 0.5
    taubin_mu: float = -0.53
    normal_neighbours: int = 30
    normal_smoothing_neighbours: int = 10
    max_depth_mm: float = 10.0
    min_component_vertices: int = 50
    marching_cubes_level: float = 0.5
    mask_gaussian_sigma_mm: float = 0.4
    parcellation_scheme: str = "cartimorph20"
    central_region_percentage: float = 0.5

    def __post_init__(self) -> None:
        allowed = {"surface_normal", "distance_transform", "nearest_neighbour"}
        if self.method not in allowed:
            raise ConfigError(
                f"thickness.method must be one of {sorted(allowed)}, got {self.method!r}"
            )
        if self.taubin_mu >= -self.taubin_lambda:
            raise ConfigError(
                "Taubin smoothing requires mu < -lambda to avoid shrinkage; got "
                f"lambda={self.taubin_lambda}, mu={self.taubin_mu}"
            )
        if self.normal_neighbours < 3:
            raise ConfigError(
                f"thickness.normal_neighbours must be >= 3 to span a plane, got "
                f"{self.normal_neighbours}"
            )
        if self.max_depth_mm <= 0:
            raise ConfigError(f"thickness.max_depth_mm must be > 0, got {self.max_depth_mm}")
        if self.parcellation_scheme not in {"cartimorph20", "compartment5"}:
            raise ConfigError(
                "thickness.parcellation_scheme must be cartimorph20|compartment5, got "
                f"{self.parcellation_scheme!r}"
            )
        if not 0.0 < self.central_region_percentage < 1.0:
            raise ConfigError(
                "thickness.central_region_percentage must be in (0, 1), got "
                f"{self.central_region_percentage}"
            )


@dataclass(frozen=True, slots=True)
class GeometryConfig:
    """3D curvature and bulge analysis.

    This block covers the part of the pipeline that is *not* in CartiMorph:
    per-vertex differential geometry of the articular surface and the signed
    deviation ("bulge") of that surface from a smooth anatomical reference.

    Attributes
    ----------
    enabled
        Master switch. Ablation A8 turns curvature/bulge features off.
    curvature_method
        ``quadric`` (local paraboloid fit, Petitjean 2002 survey) or
        ``cotangent`` (discrete operator of Meyer et al. 2003).
    curvature_ring
        Geodesic ring size (in mesh edges) of the neighbourhood used for the
        local fit.
    curvature_smoothing_iterations
        Diffusion passes applied to the scalar curvature field afterwards.
    curvature_clip_mm_inv
        Symmetric clip on principal curvatures, in mm^-1. Tiny triangles near
        mesh boundaries otherwise produce |kappa| -> inf outliers.
    bulge_reference
        How the smooth reference surface is built: ``taubin`` (heavy low-pass of
        the surface itself) or ``quadric`` (global quadric fit per compartment).
    bulge_reference_iterations
        Taubin passes for the reference surface. Must exceed
        ``smoothing_iterations`` or the reference is not smoother than the input.
    bulge_min_height_mm
        Signed deviations below this magnitude are not counted as bulge/dent
        when computing region summaries.
    shape_index_bins
        Number of bins for the shape-index histogram descriptor
        (Koenderink & van Doorn, 1992).
    """

    enabled: bool = True
    curvature_method: str = "quadric"
    curvature_ring: int = 2
    curvature_smoothing_iterations: int = 3
    curvature_clip_mm_inv: float = 2.0
    bulge_reference: str = "taubin"
    bulge_reference_iterations: int = 60
    bulge_min_height_mm: float = 0.1
    shape_index_bins: int = 9

    def __post_init__(self) -> None:
        if self.curvature_method not in {"quadric", "cotangent"}:
            raise ConfigError(
                f"geometry.curvature_method must be quadric|cotangent, got "
                f"{self.curvature_method!r}"
            )
        if self.curvature_ring < 1:
            raise ConfigError(f"geometry.curvature_ring must be >= 1, got {self.curvature_ring}")
        if self.curvature_clip_mm_inv <= 0:
            raise ConfigError(
                f"geometry.curvature_clip_mm_inv must be > 0, got {self.curvature_clip_mm_inv}"
            )
        if self.bulge_reference not in {"taubin", "quadric"}:
            raise ConfigError(
                f"geometry.bulge_reference must be taubin|quadric, got {self.bulge_reference!r}"
            )
        if self.bulge_min_height_mm < 0:
            raise ConfigError(
                f"geometry.bulge_min_height_mm must be >= 0, got {self.bulge_min_height_mm}"
            )
        if self.shape_index_bins < 3:
            raise ConfigError(
                f"geometry.shape_index_bins must be >= 3, got {self.shape_index_bins}"
            )


@dataclass(frozen=True, slots=True)
class DenudedConfig:
    """Full-thickness cartilage loss (denuded subchondral bone) branch.

    Attributes
    ----------
    enabled
        Ablation A2 switch.
    classifier
        ``gradient_boosting``, ``mlp`` or ``threshold``.
    min_thickness_mm
        A BCI vertex whose measured thickness falls below this is a candidate
        denuded vertex. CartiMorph derives full-thickness loss from the
        reconstructed-versus-observed cartilage surface; this threshold is the
        voxel-resolution floor below which "cartilage" is not resolvable.
    min_patch_area_mm2
        Connected denuded patches smaller than this are relabelled as covered.
        Suppresses single-vertex speckle from segmentation noise.
    prior_gate
        Disable smoothness/continuity priors over predicted denuded vertices.
    prior_strength
        Strength of the continuity prior on covered vertices (ablation A6).
    use_geometry_features
        Feed curvature and bulge features to the denuded classifier.
    n_estimators, max_depth, learning_rate
        Gradient-boosting hyper-parameters.
    """

    enabled: bool = True
    classifier: str = "gradient_boosting"
    min_thickness_mm: float = 0.35
    min_patch_area_mm2: float = 1.0
    prior_gate: bool = True
    prior_strength: float = 1.0
    use_geometry_features: bool = True
    n_estimators: int = 200
    max_depth: int = 3
    learning_rate: float = 0.05

    def __post_init__(self) -> None:
        allowed = {"gradient_boosting", "mlp", "threshold"}
        if self.classifier not in allowed:
            raise ConfigError(
                f"denuded.classifier must be one of {sorted(allowed)}, got {self.classifier!r}"
            )
        if self.min_thickness_mm < 0:
            raise ConfigError(
                f"denuded.min_thickness_mm must be >= 0, got {self.min_thickness_mm}"
            )
        if self.prior_strength < 0:
            raise ConfigError(f"denuded.prior_strength must be >= 0, got {self.prior_strength}")


@dataclass(frozen=True, slots=True)
class ConformalConfig:
    """Conformal prediction layer.

    Attributes
    ----------
    alpha
        Target miscoverage. Nominal coverage is ``1 - alpha``.
    score
        ``absolute``, ``normalized`` or ``cqr`` (ablation A3).
    scheme
        ``marginal``, ``mondrian`` or ``weighted`` (ablation A4).
    groups
        Metadata columns defining Mondrian groups.
    min_group_size
        Groups with fewer calibration points than this trigger the documented
        merge fallback. Must be at least ``ceil(1/alpha) - 1`` for the group
        quantile to exist.
    merge_small_groups
        If False, an undersized group raises instead of merging.
    sigma_floor
        Lower bound on the heteroscedastic scale, so normalized scores cannot
        divide by ~0.
    calibration_sizes
        Calibration-set sizes swept in ablation A5.
    risk_control_enabled
        Run conformal risk control on the segmentation masks.
    risk_alpha
        Target expected loss for risk control.
    risk_lambda_grid_size
        Number of lambda values in the grid search.
    risk_loss_bound
        ``B``, the upper bound of the loss. For a false-negative *rate* it is 1.
    """

    alpha: float = 0.1
    score: str = "absolute"
    scheme: str = "mondrian"
    groups: tuple[str, ...] = ("kl_grade",)
    min_group_size: int = 25
    merge_small_groups: bool = True
    sigma_floor: float = 1e-3
    calibration_sizes: tuple[int, ...] = (25, 50, 100, 200)
    risk_control_enabled: bool = True
    risk_alpha: float = 0.1
    risk_lambda_grid_size: int = 200
    risk_loss_bound: float = 1.0

    def __post_init__(self) -> None:
        if not 0.0 < self.alpha < 1.0:
            raise ConfigError(f"conformal.alpha must be in (0, 1), got {self.alpha}")
        if self.score not in {"absolute", "normalized", "cqr"}:
            raise ConfigError(
                f"conformal.score must be absolute|normalized|cqr, got {self.score!r}"
            )
        if self.scheme not in {"marginal", "mondrian", "weighted"}:
            raise ConfigError(
                f"conformal.scheme must be marginal|mondrian|weighted, got {self.scheme!r}"
            )
        if self.scheme == "mondrian" and not self.groups:
            raise ConfigError("conformal.scheme='mondrian' requires conformal.groups")
        if self.sigma_floor <= 0:
            raise ConfigError(f"conformal.sigma_floor must be > 0, got {self.sigma_floor}")
        if not 0.0 < self.risk_alpha < 1.0:
            raise ConfigError(f"conformal.risk_alpha must be in (0, 1), got {self.risk_alpha}")
        if self.risk_loss_bound <= 0:
            raise ConfigError(
                f"conformal.risk_loss_bound must be > 0, got {self.risk_loss_bound}"
            )
        if any(n <= 0 for n in self.calibration_sizes):
            raise ConfigError("every conformal.calibration_sizes entry must be > 0")


@dataclass(frozen=True, slots=True)
class ReliabilityConfig:
    """Precision, responsiveness and agreement analysis.

    Attributes
    ----------
    bootstrap_iterations
        Bootstrap replicates for confidence intervals.
    confidence_level
        Two-sided confidence level for every reported interval.
    sdd_z
        z multiplier in ``SDD = z * sqrt(2) * SEM``. 1.96 gives the 95% SDD.
    icc_kind
        ICC form; ``ICC(2,1)`` is two-way random, single measure, absolute
        agreement.
    annual_change_mm
        Literature annual thickness change per subregion used by
        :mod:`confcarti.reliability.detectability` when no longitudinal data
        are available. ``None`` means "require real longitudinal input".
    """

    bootstrap_iterations: int = 2000
    confidence_level: float = 0.95
    sdd_z: float = 1.96
    icc_kind: str = "ICC(2,1)"
    annual_change_mm: float | None = None

    def __post_init__(self) -> None:
        if self.bootstrap_iterations < 100:
            raise ConfigError(
                "reliability.bootstrap_iterations must be >= 100 for a usable CI, got "
                f"{self.bootstrap_iterations}"
            )
        if not 0.5 < self.confidence_level < 1.0:
            raise ConfigError(
                f"reliability.confidence_level must be in (0.5, 1), got {self.confidence_level}"
            )
        if self.icc_kind not in {"ICC(1,1)", "ICC(2,1)", "ICC(3,1)"}:
            raise ConfigError(
                f"reliability.icc_kind must be ICC(1,1)|ICC(2,1)|ICC(3,1), got {self.icc_kind!r}"
            )


@dataclass(frozen=True, slots=True)
class VizConfig:
    """Figure generation.

    Attributes
    ----------
    output_dir
        Directory figures are written to.
    formats
        File formats emitted for every figure.
    dpi
        Raster resolution.
    interactive_3d
        Also emit self-contained interactive HTML for the 3D surface figures.
    surface_colormap, diverging_colormap
        Colormaps for magnitude and signed fields respectively.
    max_subjects_3d
        Cap on the number of per-subject 3D renders, to bound runtime.
    """

    output_dir: Path = Path("results/figures")
    formats: tuple[str, ...] = ("png",)
    dpi: int = 200
    interactive_3d: bool = True
    surface_colormap: str = "viridis"
    diverging_colormap: str = "RdBu_r"
    max_subjects_3d: int = 6

    def __post_init__(self) -> None:
        allowed = {"png", "pdf", "svg"}
        bad = set(self.formats) - allowed
        if bad:
            raise ConfigError(f"viz.formats contains unsupported entries: {sorted(bad)}")
        if not self.formats:
            raise ConfigError("viz.formats must not be empty")
        if self.dpi < 50:
            raise ConfigError(f"viz.dpi must be >= 50, got {self.dpi}")


# --------------------------------------------------------------------------- #
# Root config
# --------------------------------------------------------------------------- #


@dataclass(frozen=True, slots=True)
class ExperimentConfig:
    """Root config: everything needed to reproduce one experiment.

    Attributes
    ----------
    name
        Human-readable experiment name; becomes part of the run id.
    seed
        Master seed passed to :func:`confcarti.utils.seeding.set_all_seeds`.
    results_dir
        Root directory for the run's manifest, CSVs and figures.
    deterministic_algorithms
        Forwarded to the seeding helper.
    """

    name: str
    seed: int = 0
    results_dir: Path = Path("results")
    deterministic_algorithms: bool = True
    data: DataConfig = field(default_factory=lambda: DataConfig(Path("data"), Path("meta.csv")))
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    thickness: ThicknessConfig = field(default_factory=ThicknessConfig)
    geometry: GeometryConfig = field(default_factory=GeometryConfig)
    denuded: DenudedConfig = field(default_factory=DenudedConfig)
    conformal: ConformalConfig = field(default_factory=ConformalConfig)
    reliability: ReliabilityConfig = field(default_factory=ReliabilityConfig)
    viz: VizConfig = field(default_factory=VizConfig)

    def __post_init__(self) -> None:
        if not self.name:
            raise ConfigError("experiment.name must be a non-empty string")
        if self.seed < 0:
            raise ConfigError(f"experiment.seed must be >= 0, got {self.seed}")
        # Cross-block validation: a Mondrian group threshold cannot exist unless
        # the group holds at least ceil(1/alpha) - 1 calibration points.
        need = _min_calibration_size(self.conformal.alpha)
        if self.conformal.scheme == "mondrian" and self.conformal.min_group_size < need:
            raise ConfigError(
                f"conformal.min_group_size={self.conformal.min_group_size} is below the "
                f"{need} points needed for the (1-alpha)={1 - self.conformal.alpha:.3g} "
                "quantile to exist within a Mondrian group"
            )
        if self.conformal.score == "normalized" and not self.geometry.enabled:
            raise ConfigError(
                "conformal.score='normalized' needs geometry.enabled=true: the "
                "heteroscedastic scale sigma_hat is built from curvature/bulge features"
            )


def _min_calibration_size(alpha: float) -> int:
    """Smallest calibration set for which the conformal quantile exists.

    The split-conformal threshold is the ``ceil((n + 1)(1 - alpha))``-th order
    statistic of ``n`` scores, which requires ``ceil((n + 1)(1 - alpha)) <= n``,
    i.e. ``n >= ceil(1 / alpha) - 1``.

    Parameters
    ----------
    alpha
        Target miscoverage in (0, 1).

    Returns
    -------
    int
        Minimum admissible calibration-set size.
    """
    import math

    return int(math.ceil(1.0 / alpha)) - 1


# --------------------------------------------------------------------------- #
# YAML -> dataclass with strict validation
# --------------------------------------------------------------------------- #

_SECTIONS: dict[str, type] = {
    "data": DataConfig,
    "model": ModelConfig,
    "train": TrainConfig,
    "thickness": ThicknessConfig,
    "geometry": GeometryConfig,
    "denuded": DenudedConfig,
    "conformal": ConformalConfig,
    "reliability": ReliabilityConfig,
    "viz": VizConfig,
}


@lru_cache(maxsize=None)
def _resolved_hints(cls: type) -> dict[str, Any]:
    """Return a dataclass's *resolved* type hints.

    ``from __future__ import annotations`` turns every annotation in this module
    into a string, so ``dataclasses.Field.type`` is unusable for coercion.
    ``typing.get_type_hints`` evaluates them against the defining module's
    namespace and gives back real type objects.

    Parameters
    ----------
    cls
        A config dataclass.

    Returns
    -------
    dict
        Mapping from field name to resolved annotation.
    """
    return get_type_hints(cls)


def _is_optional(annotation: Any) -> tuple[bool, Any]:
    """Split ``X | None`` into ``(True, X)``; leave other annotations alone."""
    origin = get_origin(annotation)
    if origin in (Union, UnionType):
        args = [a for a in get_args(annotation) if a is not type(None)]
        if len(args) == 1:
            return True, args[0]
    return False, annotation


def _coerce(value: Any, annotation: Any, path: str) -> Any:
    """Coerce a YAML scalar/sequence into the annotated dataclass field type.

    Parameters
    ----------
    value
        Raw value straight from ``yaml.safe_load``.
    annotation
        The dataclass field's type annotation.
    path
        Dotted config path, used only for error messages.

    Returns
    -------
    Any
        The coerced value.

    Raises
    ------
    ConfigError
        If the value cannot be represented as the annotated type.
    """
    optional, annotation = _is_optional(annotation)
    if value is None:
        if optional:
            return None
        raise ConfigError(f"{path}: null is not allowed for a non-optional field")

    origin = get_origin(annotation)

    if annotation is Path:
        if not isinstance(value, (str, Path)):
            raise ConfigError(f"{path}: expected a path string, got {type(value).__name__}")
        return Path(value)

    if origin is tuple:
        args = get_args(annotation)
        if not isinstance(value, (list, tuple)):
            raise ConfigError(f"{path}: expected a list, got {type(value).__name__}")
        # Homogeneous variadic tuple, e.g. tuple[str, ...]
        if len(args) == 2 and args[1] is Ellipsis:
            return tuple(_coerce(v, args[0], f"{path}[{i}]") for i, v in enumerate(value))
        if len(args) != len(value):
            raise ConfigError(f"{path}: expected {len(args)} entries, got {len(value)}")
        return tuple(_coerce(v, a, f"{path}[{i}]") for i, (v, a) in enumerate(zip(value, args)))

    if annotation is bool:
        if not isinstance(value, bool):
            raise ConfigError(f"{path}: expected a bool, got {type(value).__name__}")
        return value

    if annotation is int:
        # bool is a subclass of int; reject it so `true` cannot silently become 1.
        if isinstance(value, bool) or not isinstance(value, int):
            raise ConfigError(f"{path}: expected an int, got {type(value).__name__}")
        return value

    if annotation is float:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ConfigError(f"{path}: expected a float, got {type(value).__name__}")
        return float(value)

    if annotation is str:
        if not isinstance(value, str):
            raise ConfigError(f"{path}: expected a str, got {type(value).__name__}")
        return value

    raise ConfigError(f"{path}: unsupported annotation {annotation!r}")


def _build_section(cls: type, raw: dict[str, Any], path: str) -> Any:
    """Instantiate one config dataclass from a raw YAML mapping, strictly."""
    if not isinstance(raw, dict):
        raise ConfigError(f"{path}: expected a mapping, got {type(raw).__name__}")

    known = {f.name: f for f in fields(cls)}
    hints = _resolved_hints(cls)
    unknown = set(raw) - set(known)
    if unknown:
        raise ConfigError(
            f"{path}: unknown key(s) {sorted(unknown)}; allowed keys are {sorted(known)}"
        )

    kwargs: dict[str, Any] = {}
    for key, value in raw.items():
        kwargs[key] = _coerce(value, hints[key], f"{path}.{key}")

    # Required keys are those with neither a default nor a default_factory.
    missing = [
        name
        for name, f in known.items()
        if name not in kwargs
        and f.default is dataclasses.MISSING
        and f.default_factory is dataclasses.MISSING  # type: ignore[misc]
    ]
    if missing:
        raise ConfigError(f"{path}: missing required key(s) {sorted(missing)}")

    return cls(**kwargs)


def load_config(path: str | Path) -> ExperimentConfig:
    """Load and validate an experiment YAML into an :class:`ExperimentConfig`.

    Parameters
    ----------
    path
        Path to the YAML file.

    Returns
    -------
    ExperimentConfig
        Fully validated, frozen configuration.

    Raises
    ------
    FileNotFoundError
        If ``path`` does not exist.
    ConfigError
        On any unknown key, missing required key, wrong type or invalid value.

    Examples
    --------
    >>> cfg = load_config("configs/example.yaml")   # doctest: +SKIP
    >>> cfg.conformal.alpha                          # doctest: +SKIP
    0.1
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"config file not found: {path}")

    with path.open("r", encoding="utf-8") as fh:
        raw = yaml.safe_load(fh)

    if raw is None:
        raise ConfigError(f"{path}: file is empty")
    if not isinstance(raw, dict):
        raise ConfigError(f"{path}: top level must be a mapping, got {type(raw).__name__}")

    root_scalars = {"name", "seed", "results_dir", "deterministic_algorithms"}
    unknown = set(raw) - root_scalars - set(_SECTIONS)
    if unknown:
        raise ConfigError(
            f"{path}: unknown top-level key(s) {sorted(unknown)}; allowed are "
            f"{sorted(root_scalars | set(_SECTIONS))}"
        )

    root_hints = _resolved_hints(ExperimentConfig)
    kwargs: dict[str, Any] = {}
    for key in sorted(root_scalars & set(raw)):
        kwargs[key] = _coerce(raw[key], root_hints[key], key)

    if "name" not in kwargs:
        raise ConfigError(f"{path}: missing required key 'name'")

    for section, cls in _SECTIONS.items():
        if section in raw:
            kwargs[section] = _build_section(cls, raw[section], section)

    return ExperimentConfig(**kwargs)


def dump_config(cfg: Any) -> dict[str, Any]:
    """Recursively convert a config dataclass into JSON/YAML-safe primitives.

    Paths become strings and tuples become lists so the result round-trips
    through ``json.dumps`` for the run manifest.

    Parameters
    ----------
    cfg
        Any config dataclass instance, or a nested container of them.

    Returns
    -------
    dict
        Plain-Python representation.
    """
    if is_dataclass(cfg) and not isinstance(cfg, type):
        return {f.name: dump_config(getattr(cfg, f.name)) for f in fields(cfg)}
    if isinstance(cfg, Path):
        return str(cfg)  # type: ignore[return-value]
    if isinstance(cfg, (list, tuple)):
        return [dump_config(v) for v in cfg]  # type: ignore[return-value]
    return cfg  # type: ignore[return-value]

### Run manifests and the tidy results log

`confcarti/utils/logging.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/utils/logging.py
# ==========================================================================
"""Run logging: manifests, tidy result rows and console/file handlers.

Every run writes ``results/<run_id>/manifest.json`` *before* doing any work, so
that a crashed run still leaves behind an exact record of what was attempted.
Result rows are appended to a single tidy long-format CSV; every downstream
table and figure is generated from that CSV alone.
"""


import json
import logging
import platform
import subprocess
import sys
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)

#: Columns every result row carries. Enforced on write so that no analysis
#: script ever has to guess whether a stratifier is present.
RESULT_COLUMNS: tuple[str, ...] = (
    "subject_id",
    "subregion",
    "kl_grade",
    "site",
    "split",
    "model",
    "method",
    "seed",
    "metric",
    "value",
)


def _git_commit(repo_root: Path | None = None) -> str:
    """Return the current git commit hash, or ``"unknown"`` outside a repo."""
    root = repo_root or Path(__file__).resolve().parents[2]
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=root,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
    except (OSError, subprocess.SubprocessError):
        return "unknown"
    return out.stdout.strip() if out.returncode == 0 else "unknown"


def _git_dirty(repo_root: Path | None = None) -> bool:
    """Whether the working tree has uncommitted changes."""
    root = repo_root or Path(__file__).resolve().parents[2]
    try:
        out = subprocess.run(
            ["git", "status", "--porcelain"],
            cwd=root,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
    except (OSError, subprocess.SubprocessError):
        return False
    return bool(out.stdout.strip())


def environment_report() -> dict[str, Any]:
    """Collect a machine-readable description of the compute environment.

    Returns
    -------
    dict
        Python/OS/torch/CUDA/GPU details. Keys whose backing library is absent
        are reported as ``None`` rather than omitted, so manifests from
        different machines stay directly comparable.
    """
    report: dict[str, Any] = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "processor": platform.processor(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": None,
        "torch_cuda": None,
        "cudnn": None,
        "gpu_name": None,
        "gpu_capability": None,
        "gpu_total_memory_gb": None,
        "gpu_count": 0,
        "monai": None,
    }
    try:
        import torch

        report["torch"] = torch.__version__
        report["torch_cuda"] = torch.version.cuda
        if torch.cuda.is_available():
            report["cudnn"] = torch.backends.cudnn.version()
            report["gpu_count"] = torch.cuda.device_count()
            props = torch.cuda.get_device_properties(0)
            report["gpu_name"] = props.name
            report["gpu_capability"] = f"{props.major}.{props.minor}"
            report["gpu_total_memory_gb"] = round(props.total_memory / 1024**3, 2)
    except ImportError:
        pass

    try:
        import monai

        report["monai"] = monai.__version__
    except ImportError:
        pass

    return report


@dataclass
class RunManifest:
    """The immutable record written at the start of every run.

    Attributes
    ----------
    run_id
        ``<experiment name>_<seed>_<UTC timestamp>``.
    experiment
        Experiment name from the config.
    seed
        Master seed.
    timestamp_utc
        ISO-8601 start time.
    git_commit, git_dirty
        Provenance of the code that produced the results.
    config
        Full config dump.
    environment
        Output of :func:`environment_report`.
    seeding
        Output of :func:`confcarti.utils.seeding.set_all_seeds`, filled in by
        :meth:`RunLogger.record_seeding`.
    notes
        Free-form annotations added during the run.
    """

    run_id: str
    experiment: str
    seed: int
    timestamp_utc: str
    git_commit: str
    git_dirty: bool
    config: dict[str, Any]
    environment: dict[str, Any]
    seeding: dict[str, Any] | None = None
    notes: dict[str, Any] = field(default_factory=dict)


class RunLogger:
    """Owns one run directory: manifest, log file and the tidy results CSV.

    Parameters
    ----------
    cfg
        The experiment configuration.
    run_id
        Explicit run id. Defaults to ``<name>_<seed>_<timestamp>``. Pass a fixed
        value when you need byte-identical reruns.
    console
        Also stream log records to stderr.

    Examples
    --------
    >>> import tempfile, pathlib
    >>> from confcarti.config import ExperimentConfig, DataConfig
    >>> tmp = pathlib.Path(tempfile.mkdtemp())
    >>> cfg = ExperimentConfig(name="demo", results_dir=tmp,
    ...                        data=DataConfig(tmp, tmp / "m.csv"))
    >>> run = RunLogger(cfg, run_id="demo_run", console=False)
    >>> run.log_result(subject_id="9001104", subregion="ccMFC", kl_grade=3,
    ...                site="0.E.1", split="test", model="swinunetr",
    ...                method="surface_normal", metric="thickness_mm", value=1.83)
    >>> run.flush_results().name
    'results.csv'
    """

    def __init__(
        self,
        cfg: ExperimentConfig,
        run_id: str | None = None,
        *,
        console: bool = True,
    ) -> None:
        self.cfg = cfg
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        self.run_id = run_id or f"{cfg.name}_seed{cfg.seed}_{stamp}"
        self.run_dir = Path(cfg.results_dir) / self.run_id
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self._rows: list[dict[str, Any]] = []
        self._configure_handlers(console=console)

        self.manifest = RunManifest(
            run_id=self.run_id,
            experiment=cfg.name,
            seed=cfg.seed,
            timestamp_utc=datetime.now(timezone.utc).isoformat(),
            git_commit=_git_commit(),
            git_dirty=_git_dirty(),
            config=dump_config(cfg),
            environment=environment_report(),
        )
        self.write_manifest()
        logger.info("run %s -> %s", self.run_id, self.run_dir)

    # ---------------------------------------------------------------- setup --

    def _configure_handlers(self, *, console: bool) -> None:
        """Attach a file handler (and optionally a stderr handler) to the root."""
        root = logging.getLogger()
        root.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)s | %(message)s")

        file_handler = logging.FileHandler(self.run_dir / "run.log", encoding="utf-8")
        file_handler.setFormatter(fmt)
        root.addHandler(file_handler)
        self._file_handler = file_handler

        if console and not any(
            isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler)
            for h in root.handlers
        ):
            stream = logging.StreamHandler()
            stream.setFormatter(fmt)
            root.addHandler(stream)

    # ------------------------------------------------------------- manifest --

    def write_manifest(self) -> Path:
        """Serialise the manifest to ``<run_dir>/manifest.json``.

        Returns
        -------
        pathlib.Path
            Path of the written manifest.
        """
        path = self.run_dir / "manifest.json"
        with path.open("w", encoding="utf-8") as fh:
            json.dump(asdict(self.manifest), fh, indent=2, sort_keys=True, default=str)
        return path

    def record_seeding(self, report: Any) -> None:
        """Store the seeding report in the manifest and rewrite it."""
        self.manifest.seeding = asdict(report) if hasattr(report, "__dataclass_fields__") else report
        self.write_manifest()

    def note(self, key: str, value: Any) -> None:
        """Add a free-form note to the manifest and rewrite it."""
        self.manifest.notes[key] = value
        self.write_manifest()

    # -------------------------------------------------------------- results --

    def log_result(
        self,
        *,
        subject_id: str,
        subregion: str,
        metric: str,
        value: float,
        kl_grade: int | float | None = None,
        site: str | None = None,
        split: str | None = None,
        model: str | None = None,
        method: str | None = None,
        seed: int | None = None,
    ) -> None:
        """Append one tidy long-format result row.

        Parameters
        ----------
        subject_id
            Subject identifier.
        subregion
            Anatomical subregion, or ``"whole_knee"`` for knee-level metrics.
        metric
            Metric name, e.g. ``thickness_mm`` or ``interval_width_mm``.
        value
            Numeric value. NaN is allowed and *meaningful*: it marks an
            undefined measurement (see the NaN-vs-0 rule in the README).
        kl_grade, site, split, model, method, seed
            Stratifiers. Omitted values fall back to the config where sensible.
        """
        self._rows.append(
            {
                "subject_id": str(subject_id),
                "subregion": subregion,
                "kl_grade": kl_grade,
                "site": site,
                "split": split,
                "model": model if model is not None else self.cfg.model.backbone,
                "method": method if method is not None else self.cfg.thickness.method,
                "seed": seed if seed is not None else self.cfg.seed,
                "metric": metric,
                "value": float(value),
            }
        )

    def log_rows(self, rows: pd.DataFrame | list[dict[str, Any]]) -> None:
        """Append many rows at once, validating the required columns.

        Parameters
        ----------
        rows
            DataFrame or list of dicts holding at least
            ``subject_id, subregion, metric, value``.

        Raises
        ------
        ValueError
            If required columns are missing.
        """
        frame = pd.DataFrame(rows)
        if frame.empty:
            return
        required = {"subject_id", "subregion", "metric", "value"}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"result rows are missing required column(s): {sorted(missing)}")
        for col in RESULT_COLUMNS:
            if col not in frame.columns:
                frame[col] = None
        frame["model"] = frame["model"].fillna(self.cfg.model.backbone)
        frame["method"] = frame["method"].fillna(self.cfg.thickness.method)
        frame["seed"] = frame["seed"].fillna(self.cfg.seed)
        self._rows.extend(frame[list(RESULT_COLUMNS)].to_dict("records"))

    def results_frame(self) -> pd.DataFrame:
        """Return the accumulated rows as a DataFrame with canonical columns."""
        if not self._rows:
            return pd.DataFrame(columns=list(RESULT_COLUMNS))
        return pd.DataFrame(self._rows)[list(RESULT_COLUMNS)]

    def flush_results(self, filename: str = "results.csv") -> Path:
        """Write the tidy results CSV.

        Rows are sorted on the stratifier columns so that two runs with the same
        seed produce byte-identical files regardless of iteration order.

        Parameters
        ----------
        filename
            Name of the CSV inside the run directory.

        Returns
        -------
        pathlib.Path
            Path of the written CSV.
        """
        frame = self.results_frame()
        sort_cols = ["subject_id", "subregion", "metric", "method", "model", "seed"]
        if not frame.empty:
            frame = frame.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
        path = self.run_dir / filename
        frame.to_csv(path, index=False, float_format="%.10g", lineterminator="\n")
        logger.info("wrote %d result rows to %s", len(frame), path)
        return path

    def save_json(self, name: str, payload: Any) -> Path:
        """Write an auxiliary JSON artefact into the run directory."""
        path = self.run_dir / name
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as fh:
            json.dump(payload, fh, indent=2, sort_keys=True, default=_json_default)
        return path

    def close(self) -> None:
        """Flush results and detach the file handler."""
        self.flush_results()
        root = logging.getLogger()
        if self._file_handler in root.handlers:
            self._file_handler.close()
            root.removeHandler(self._file_handler)

    def __enter__(self) -> RunLogger:
        return self

    def __exit__(self, *exc: object) -> None:
        self.close()


def _json_default(obj: Any) -> Any:
    """JSON fallback for numpy scalars, arrays and Paths."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (set, frozenset)):
        return sorted(obj)
    return str(obj)

### OAIZIB-CM label convention and the 20-region atlas

`confcarti/data/labels.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/labels.py
# ==========================================================================
"""OAIZIB-CM label convention, compartments and the 20-region cartilage atlas.

Provenance
----------
The 5-ROI convention below is the one actually shipped with OAIZIB-CM, taken
from the CartiMorph project README, which states that the source OAI-ZIB masks
contain "the femur (ROI1), femoral cartilage (ROI2), tibia (ROI3), and tibial
cartilage (ROI4)", and that "in OAIZIB-CM, tibial cartilage is split into medial
and lateral tibial cartilages" -- the split producing ROI4 = medial and
ROI5 = lateral.

    https://github.com/YongchengYAO/CartiMorph
    https://doi.org/10.1016/j.media.2023.103035   (CartiMorph, Media 2023)
    OAI-ZIB: https://doi.org/10.1016/j.media.2018.11.009

.. warning::
   An ordering of (1 femoral cartilage, 2 medial tibial, 3 lateral tibial,
   4 femur, 5 tibia) circulates in secondary descriptions of this dataset. It is
   **not** the shipped convention and silently swapping bone for cartilage
   produces meaningless thickness maps rather than an error. Use
   :func:`validate_label_convention` on real data before trusting any output.

The 20-region atlas names are read off the outputs of CartiMorph's rule-based
parcellation, ``CM_cal_SurfaceParcellation_FC.m`` and
``CM_cal_SurfaceParcellation_TC.m``, and follow the Eckstein/Wirth nomenclature.
"""


from dataclasses import dataclass
from typing import Final

import numpy as np

__all__ = [
    "BACKGROUND",
    "FEMUR",
    "FEMORAL_CARTILAGE",
    "TIBIA",
    "MEDIAL_TIBIAL_CARTILAGE",
    "LATERAL_TIBIAL_CARTILAGE",
    "LABEL_NAMES",
    "CARTILAGE_LABELS",
    "BONE_LABELS",
    "BONE_FOR_CARTILAGE",
    "Compartment",
    "COMPARTMENTS",
    "FC_SUBREGIONS",
    "TC_SUBREGIONS",
    "ATLAS20_SUBREGIONS",
    "SUBREGION_TO_COMPARTMENT",
    "validate_label_convention",
    "LabelConventionReport",
]

# --------------------------------------------------------------------------- #
# 5-ROI voxel labels (+ background)
# --------------------------------------------------------------------------- #

BACKGROUND: Final[int] = 0
FEMUR: Final[int] = 1
FEMORAL_CARTILAGE: Final[int] = 2
TIBIA: Final[int] = 3
MEDIAL_TIBIAL_CARTILAGE: Final[int] = 4
LATERAL_TIBIAL_CARTILAGE: Final[int] = 5

LABEL_NAMES: Final[dict[int, str]] = {
    BACKGROUND: "background",
    FEMUR: "femur",
    FEMORAL_CARTILAGE: "femoral_cartilage",
    TIBIA: "tibia",
    MEDIAL_TIBIAL_CARTILAGE: "medial_tibial_cartilage",
    LATERAL_TIBIAL_CARTILAGE: "lateral_tibial_cartilage",
}

#: Short codes used in result CSVs and figure legends.
LABEL_SHORT: Final[dict[int, str]] = {
    FEMUR: "F",
    FEMORAL_CARTILAGE: "FC",
    TIBIA: "T",
    MEDIAL_TIBIAL_CARTILAGE: "MTC",
    LATERAL_TIBIAL_CARTILAGE: "LTC",
}

CARTILAGE_LABELS: Final[tuple[int, ...]] = (
    FEMORAL_CARTILAGE,
    MEDIAL_TIBIAL_CARTILAGE,
    LATERAL_TIBIAL_CARTILAGE,
)
BONE_LABELS: Final[tuple[int, ...]] = (FEMUR, TIBIA)

#: Which bone each cartilage plate sits on. Drives bone-cartilage interface
#: extraction in :mod:`confcarti.thickness.mesh`.
BONE_FOR_CARTILAGE: Final[dict[int, int]] = {
    FEMORAL_CARTILAGE: FEMUR,
    MEDIAL_TIBIAL_CARTILAGE: TIBIA,
    LATERAL_TIBIAL_CARTILAGE: TIBIA,
}

NUM_CLASSES: Final[int] = 6

# --------------------------------------------------------------------------- #
# Compartments and the 20-region atlas
# --------------------------------------------------------------------------- #


@dataclass(frozen=True, slots=True)
class Compartment:
    """One cartilage plate and the bone it articulates with.

    Attributes
    ----------
    key
        Short code (``FC``, ``MTC``, ``LTC``).
    name
        Human-readable name.
    cartilage_label, bone_label
        Voxel labels in the 5-ROI mask.
    subregions
        Atlas subregions belonging to this compartment.
    """

    key: str
    name: str
    cartilage_label: int
    bone_label: int
    subregions: tuple[str, ...]


#: Femoral cartilage subregions, in the order produced by
#: ``CM_cal_SurfaceParcellation_FC.m``.
#: a = anterior, p = posterior, ec/cc/ic = external/central/internal central
#: strip, L/M = lateral/medial, FC = femoral cartilage.
FC_SUBREGIONS: Final[tuple[str, ...]] = (
    "aLFC",
    "ecLFC",
    "ccLFC",
    "icLFC",
    "pLFC",
    "aMFC",
    "ecMFC",
    "ccMFC",
    "icMFC",
    "pMFC",
)

#: Tibial cartilage subregions from ``CM_cal_SurfaceParcellation_TC.m``.
#: a/p = anterior/posterior, e/c/i = external/central/internal, TC = tibial
#: cartilage. Internal/external are mirrored for left knees by that script; we
#: reproduce the mirroring in :mod:`confcarti.thickness.parcellation`.
TC_SUBREGIONS: Final[tuple[str, ...]] = (
    "aMTC",
    "eMTC",
    "cMTC",
    "iMTC",
    "pMTC",
    "aLTC",
    "eLTC",
    "cLTC",
    "iLTC",
    "pLTC",
)

#: The full 20-region cartilage atlas of CLAIR-Knee-103R.
ATLAS20_SUBREGIONS: Final[tuple[str, ...]] = FC_SUBREGIONS + TC_SUBREGIONS

COMPARTMENTS: Final[dict[str, Compartment]] = {
    "FC": Compartment("FC", "femoral cartilage", FEMORAL_CARTILAGE, FEMUR, FC_SUBREGIONS),
    "MTC": Compartment(
        "MTC",
        "medial tibial cartilage",
        MEDIAL_TIBIAL_CARTILAGE,
        TIBIA,
        tuple(s for s in TC_SUBREGIONS if "MTC" in s),
    ),
    "LTC": Compartment(
        "LTC",
        "lateral tibial cartilage",
        LATERAL_TIBIAL_CARTILAGE,
        TIBIA,
        tuple(s for s in TC_SUBREGIONS if "LTC" in s),
    ),
}

SUBREGION_TO_COMPARTMENT: Final[dict[str, str]] = {
    sub: comp.key for comp in COMPARTMENTS.values() for sub in comp.subregions
}


def subregions_for(scheme: str) -> tuple[str, ...]:
    """Return the subregion list for a parcellation scheme.

    Parameters
    ----------
    scheme
        ``cartimorph20`` or ``compartment5``.

    Returns
    -------
    tuple of str

    Raises
    ------
    ValueError
        For an unknown scheme.
    """
    if scheme == "cartimorph20":
        return ATLAS20_SUBREGIONS
    if scheme == "compartment5":
        return ("FC", "MTC", "LTC")
    raise ValueError(f"unknown parcellation scheme {scheme!r}")


# --------------------------------------------------------------------------- #
# Convention validation
# --------------------------------------------------------------------------- #


@dataclass(frozen=True, slots=True)
class LabelConventionReport:
    """Outcome of :func:`validate_label_convention`.

    Attributes
    ----------
    ok
        True when every check passed.
    voxel_counts
        Voxels per label present in the mask.
    problems
        Human-readable descriptions of every failed check.
    """

    ok: bool
    voxel_counts: dict[str, int]
    problems: tuple[str, ...]

    def raise_if_bad(self) -> None:
        """Raise ``ValueError`` listing every problem, if any.

        Raises
        ------
        ValueError
            When ``ok`` is False.
        """
        if not self.ok:
            joined = "\n  - ".join(self.problems)
            raise ValueError(
                "label mask does not match the OAIZIB-CM 5-ROI convention "
                "(1=femur, 2=femoral cartilage, 3=tibia, 4=medial tibial cartilage, "
                f"5=lateral tibial cartilage):\n  - {joined}"
            )


def validate_label_convention(
    label: np.ndarray,
    *,
    lr_axis: int = 0,
    si_axis: int = 2,
    min_voxels: int = 100,
) -> LabelConventionReport:
    """Sanity-check that a mask really follows the OAIZIB-CM 5-ROI convention.

    Four anatomical facts are checked; each holds for any knee, left or right,
    at any KL grade:

    1. Every one of labels 1..5 is present with at least ``min_voxels`` voxels.
    2. Bone volumes exceed their cartilage volumes -- the femur is far larger
       than femoral cartilage. A bone/cartilage swap fails here immediately.
    3. The femur sits superior to the tibia along ``si_axis``.
    4. The medial and lateral tibial plates are separated along ``lr_axis``,
       and each is inferior to the femoral cartilage.

    Parameters
    ----------
    label
        Integer label volume.
    lr_axis
        Array axis running left-right. Default 0 matches the sagittal-first
        DESS layout used throughout this package.
    si_axis
        Array axis running superior-inferior.
    min_voxels
        Minimum voxel count for a label to count as present.

    Returns
    -------
    LabelConventionReport

    Examples
    --------
    >>> vol = np.zeros((20, 20, 20), dtype=np.uint8)
    >>> vol[:, :, 12:16] = FEMUR
    >>> vol[:, :, 10:12] = FEMORAL_CARTILAGE
    >>> vol[:, :, 4:8] = TIBIA
    >>> vol[:10, :, 8:10] = MEDIAL_TIBIAL_CARTILAGE
    >>> vol[10:, :, 8:10] = LATERAL_TIBIAL_CARTILAGE
    >>> validate_label_convention(vol).ok
    True
    """
    label = np.asarray(label)
    problems: list[str] = []
    counts = {
        LABEL_NAMES[v]: int(np.count_nonzero(label == v))
        for v in (FEMUR, FEMORAL_CARTILAGE, TIBIA, MEDIAL_TIBIAL_CARTILAGE, LATERAL_TIBIAL_CARTILAGE)
    }

    for value, name in LABEL_NAMES.items():
        if value == BACKGROUND:
            continue
        if counts[name] < min_voxels:
            problems.append(f"label {value} ({name}) has only {counts[name]} voxels")

    stray = set(np.unique(label).tolist()) - set(LABEL_NAMES)
    if stray:
        problems.append(f"unexpected label value(s) present: {sorted(stray)}")

    if counts["femur"] <= counts["femoral_cartilage"]:
        problems.append(
            f"femur ({counts['femur']} vox) is not larger than femoral cartilage "
            f"({counts['femoral_cartilage']} vox) -- labels 1 and 2 look swapped"
        )
    tibial_cart = counts["medial_tibial_cartilage"] + counts["lateral_tibial_cartilage"]
    if counts["tibia"] <= tibial_cart:
        problems.append(
            f"tibia ({counts['tibia']} vox) is not larger than tibial cartilage "
            f"({tibial_cart} vox) -- label 3 looks like a cartilage plate"
        )

    def _centroid(value: int) -> np.ndarray | None:
        idx = np.argwhere(label == value)
        return idx.mean(axis=0) if idx.size else None

    c_femur, c_tibia = _centroid(FEMUR), _centroid(TIBIA)
    if c_femur is not None and c_tibia is not None and c_femur[si_axis] <= c_tibia[si_axis]:
        problems.append("femur centroid is not superior to the tibia centroid along si_axis")

    c_mtc, c_ltc = _centroid(MEDIAL_TIBIAL_CARTILAGE), _centroid(LATERAL_TIBIAL_CARTILAGE)
    if c_mtc is not None and c_ltc is not None:
        if abs(c_mtc[lr_axis] - c_ltc[lr_axis]) < 1.0:
            problems.append(
                "medial and lateral tibial cartilage centroids are not separated along lr_axis"
            )
    c_fc = _centroid(FEMORAL_CARTILAGE)
    if c_fc is not None and c_mtc is not None and c_fc[si_axis] <= c_mtc[si_axis]:
        problems.append("femoral cartilage is not superior to medial tibial cartilage")

    return LabelConventionReport(
        ok=not problems, voxel_counts=counts, problems=tuple(problems)
    )

### Subject metadata: schema, QC, OAI-ZIB adapter

`confcarti/data/metadata.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/metadata.py
# ==========================================================================
"""Subject-level metadata: schema, loading, QC and the OAI-ZIB adapter.

ConfCarti stratifies everything on KL grade and acquisition site, so the
metadata table is a first-class input, not an afterthought. This module defines
the canonical schema, validates it hard, and provides an adapter for the subject
tables published with the CartiMorph project.

Data access
-----------
No OAI or OAI-ZIB data -- images, masks *or* clinical variables such as KL grade
-- is bundled with this repository. The OAI requires a data use agreement
(https://nda.nih.gov/oai/) and OAI-ZIB has its own terms. Use
``scripts/fetch_metadata.py`` to build the metadata CSV locally once you have
accepted those terms. See ``docs/data_statement.md``.
"""


import logging
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

__all__ = [
    "METADATA_COLUMNS",
    "KL_GRADES",
    "MetadataReport",
    "load_metadata",
    "validate_metadata",
    "from_cartimorph_tables",
    "summarise_metadata",
]

#: Canonical metadata schema. ``site`` is whatever grouping variable the
#: coverage guarantee must hold conditionally on -- a clinical centre, a
#: scanner, or an image release batch.
METADATA_COLUMNS: tuple[str, ...] = (
    "subject_id",
    "kl_grade",
    "site",
    "visit",
    "laterality",
)

#: Optional columns that are preserved when present.
OPTIONAL_COLUMNS: tuple[str, ...] = ("age", "sex", "bmi", "image_path", "label_path")

KL_GRADES: tuple[int, ...] = (0, 1, 2, 3, 4)

#: BMI is stored as 0.0 in the OAI tables when it was not recorded. Treating
#: that as a real value would put a subject at the extreme of any BMI-adjusted
#: analysis, so it is converted to NaN on load.
_BMI_MISSING_SENTINEL = 0.0
_BMI_PLAUSIBLE_RANGE = (10.0, 80.0)


@dataclass(frozen=True, slots=True)
class MetadataReport:
    """Result of :func:`validate_metadata`.

    Attributes
    ----------
    n_subjects
        Number of unique subjects.
    kl_counts
        Subjects per KL grade.
    site_counts
        Subjects per site.
    missing
        Column -> number of missing values, for columns that have any.
    problems
        Fatal schema violations.
    warnings
        Non-fatal data-quality observations.
    """

    n_subjects: int
    kl_counts: dict[int, int]
    site_counts: dict[str, int]
    missing: dict[str, int]
    problems: tuple[str, ...]
    warnings: tuple[str, ...]

    @property
    def ok(self) -> bool:
        """True when there are no fatal problems."""
        return not self.problems


def validate_metadata(frame: pd.DataFrame) -> MetadataReport:
    """Check a metadata table against the canonical schema.

    Parameters
    ----------
    frame
        Candidate metadata table.

    Returns
    -------
    MetadataReport
        Fatal issues land in ``problems``; data-quality notes in ``warnings``.
    """
    problems: list[str] = []
    warnings: list[str] = []

    missing_cols = [c for c in METADATA_COLUMNS if c not in frame.columns]
    if missing_cols:
        problems.append(f"missing required column(s): {missing_cols}")
        return MetadataReport(0, {}, {}, {}, tuple(problems), tuple(warnings))

    dupes = frame["subject_id"].duplicated()
    if dupes.any():
        offenders = frame.loc[dupes, "subject_id"].unique()[:5].tolist()
        problems.append(
            f"{int(dupes.sum())} duplicate subject_id row(s), e.g. {offenders}. "
            "Metadata must have exactly one row per subject."
        )

    kl = pd.to_numeric(frame["kl_grade"], errors="coerce")
    bad_kl = kl.notna() & ~kl.isin(KL_GRADES)
    if bad_kl.any():
        problems.append(
            f"{int(bad_kl.sum())} row(s) have kl_grade outside {KL_GRADES}: "
            f"{sorted(kl[bad_kl].unique().tolist())[:5]}"
        )
    if kl.isna().any():
        problems.append(f"{int(kl.isna().sum())} row(s) have a missing kl_grade")

    if frame["site"].isna().any():
        problems.append(f"{int(frame['site'].isna().sum())} row(s) have a missing site")

    n_sites = frame["site"].nunique(dropna=True)
    if n_sites < 2:
        warnings.append(
            f"only {n_sites} distinct site(s): site-conditional coverage cannot be "
            "evaluated and Mondrian grouping on site degenerates to marginal"
        )

    lat = frame["laterality"].astype(str).str.lower()
    bad_lat = ~lat.isin({"left", "right", "l", "r", "nan"})
    if bad_lat.any():
        warnings.append(
            f"{int(bad_lat.sum())} row(s) have an unrecognised laterality; expected "
            "left/right. Parcellation mirrors internal/external subregions by knee "
            "side, so a wrong value silently swaps subregion names."
        )

    for grade, count in kl.value_counts().items():
        if count < 10:
            warnings.append(
                f"KL grade {int(grade)} has only {int(count)} subjects: per-grade "
                "conditional coverage will be very noisy"
            )

    if "bmi" in frame.columns:
        bmi = pd.to_numeric(frame["bmi"], errors="coerce")
        implausible = bmi.notna() & (
            (bmi < _BMI_PLAUSIBLE_RANGE[0]) | (bmi > _BMI_PLAUSIBLE_RANGE[1])
        )
        if implausible.any():
            warnings.append(
                f"{int(implausible.sum())} row(s) have implausible bmi "
                f"outside {_BMI_PLAUSIBLE_RANGE} (0.0 is the OAI missing-value sentinel)"
            )

    missing = {
        c: int(frame[c].isna().sum()) for c in frame.columns if int(frame[c].isna().sum()) > 0
    }

    return MetadataReport(
        n_subjects=int(frame["subject_id"].nunique()),
        kl_counts={int(k): int(v) for k, v in kl.value_counts().sort_index().items()},
        site_counts={str(k): int(v) for k, v in frame["site"].value_counts().items()},
        missing=missing,
        problems=tuple(problems),
        warnings=tuple(warnings),
    )


def load_metadata(path: str | Path, *, strict: bool = True) -> pd.DataFrame:
    """Load and validate a metadata CSV.

    Parameters
    ----------
    path
        CSV with the columns in :data:`METADATA_COLUMNS`.
    strict
        Raise on schema violations. When False, problems are logged instead --
        useful for exploratory analysis of a partly-assembled table.

    Returns
    -------
    pandas.DataFrame
        Metadata with ``subject_id`` as ``str``, ``kl_grade`` as ``int`` and
        BMI sentinels converted to NaN.

    Raises
    ------
    FileNotFoundError
        If the CSV does not exist.
    ValueError
        If ``strict`` and the schema is violated.
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"metadata CSV not found: {path}. Build one with scripts/fetch_metadata.py "
            "(requires OAI / OAI-ZIB data access -- see docs/data_statement.md)."
        )

    frame = pd.read_csv(path, dtype={"subject_id": str})
    report = validate_metadata(frame)

    for note in report.warnings:
        logger.warning("metadata: %s", note)

    if not report.ok:
        message = "invalid metadata table:\n  - " + "\n  - ".join(report.problems)
        if strict:
            raise ValueError(message)
        logger.error(message)

    if report.ok:
        frame["kl_grade"] = frame["kl_grade"].astype(int)
    frame["subject_id"] = frame["subject_id"].astype(str)
    frame["site"] = frame["site"].astype(str)

    if "bmi" in frame.columns:
        bmi = pd.to_numeric(frame["bmi"], errors="coerce")
        frame["bmi"] = bmi.mask(bmi <= _BMI_MISSING_SENTINEL)

    logger.info(
        "loaded metadata for %d subjects from %s (KL %s, sites %s)",
        report.n_subjects,
        path,
        report.kl_counts,
        report.site_counts,
    )
    return frame


def from_cartimorph_tables(
    tables: dict[str, pd.DataFrame],
    *,
    site_from_release: bool = True,
) -> pd.DataFrame:
    """Convert CartiMorph's published OAI-ZIB subject tables to the canonical schema.

    The upstream tables carry the columns ``SubjectID, Path, MRBarCode,
    KneeSide, KLGrade, Gender, Age, BMI``. ``Path`` looks like
    ``0.E.1/9001104/20050825/10498212``: the first component is the OAI image
    release the scan was drawn from.

    Parameters
    ----------
    tables
        Mapping from a name (e.g. ``"dataset2"``) to the loaded spreadsheet.
        Rows are concatenated and de-duplicated on ``SubjectID``.
    site_from_release
        Use the OAI image-release prefix as the ``site`` variable.

        .. note::
           This is an **acquisition-batch surrogate, not the OAI clinical
           site**. The true site variable is ``V00SITE`` in the OAI enrollees
           table and is not redistributed here. The surrogate is still a
           legitimate stratifier -- it separates two distinct acquisition
           batches -- but a claim of "site-conditional coverage" should say
           which variable it means. Pass ``site_from_release=False`` and join a
           real ``site`` column when you have one.

    Returns
    -------
    pandas.DataFrame
        Canonical metadata table.

    Raises
    ------
    ValueError
        If required upstream columns are absent.
    """
    required = {"SubjectID", "Path", "KneeSide", "KLGrade"}
    frames = []
    for name, table in tables.items():
        missing = required - set(table.columns)
        if missing:
            raise ValueError(f"table {name!r} is missing column(s) {sorted(missing)}")
        frames.append(table.assign(source_table=name))

    merged = pd.concat(frames, ignore_index=True).drop_duplicates("SubjectID", keep="first")

    out = pd.DataFrame(
        {
            "subject_id": merged["SubjectID"].astype(str),
            "kl_grade": merged["KLGrade"].astype(int),
            # OAI-ZIB codes KneeSide 1 = right, 2 = left.
            "laterality": merged["KneeSide"].map({1: "right", 2: "left"}).fillna("unknown"),
            "visit": "V00",
            "source_table": merged["source_table"],
        }
    )

    if site_from_release:
        out["site"] = merged["Path"].astype(str).str.split("/").str[0]
    else:
        out["site"] = "unknown"

    for src, dst in (("Age", "age"), ("Gender", "sex"), ("BMI", "bmi")):
        if src in merged.columns:
            out[dst] = merged[src].to_numpy()

    if "sex" in out.columns:
        out["sex"] = out["sex"].map({1: "male", 2: "female"}).fillna("unknown")
    if "bmi" in out.columns:
        bmi = pd.to_numeric(out["bmi"], errors="coerce")
        n_sentinel = int((bmi <= _BMI_MISSING_SENTINEL).sum())
        if n_sentinel:
            logger.warning(
                "%d subject(s) have BMI=0 in the source table; converted to NaN", n_sentinel
            )
        out["bmi"] = bmi.mask(bmi <= _BMI_MISSING_SENTINEL)

    ordered = list(METADATA_COLUMNS) + [c for c in out.columns if c not in METADATA_COLUMNS]
    return out[ordered].sort_values("subject_id").reset_index(drop=True)


def summarise_metadata(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Build the descriptive tables used by the dataset-analysis figures.

    Parameters
    ----------
    frame
        Canonical metadata table.

    Returns
    -------
    dict
        ``kl_by_site`` contingency table, ``overall`` one-row summary, and
        ``continuous`` describing age/BMI by KL grade when those exist.
    """
    out: dict[str, pd.DataFrame] = {}

    out["kl_by_site"] = pd.crosstab(frame["kl_grade"], frame["site"], margins=True)

    overall = {
        "n_subjects": frame["subject_id"].nunique(),
        "n_sites": frame["site"].nunique(),
        "n_kl_grades": frame["kl_grade"].nunique(),
    }
    for col in ("age", "bmi"):
        if col in frame.columns:
            values = pd.to_numeric(frame[col], errors="coerce")
            overall[f"{col}_mean"] = float(np.nanmean(values)) if values.notna().any() else np.nan
            overall[f"{col}_sd"] = float(np.nanstd(values, ddof=1)) if values.notna().any() else np.nan
            overall[f"{col}_missing"] = int(values.isna().sum())
    out["overall"] = pd.DataFrame([overall])

    cont_cols = [c for c in ("age", "bmi") if c in frame.columns]
    if cont_cols:
        out["continuous"] = (
            frame.groupby("kl_grade")[cont_cols]
            .agg(["count", "mean", "std", "min", "max"])
            .round(2)
        )

    if "sex" in frame.columns:
        out["sex_by_kl"] = pd.crosstab(frame["kl_grade"], frame["sex"], margins=True)
    if "laterality" in frame.columns:
        out["laterality"] = frame["laterality"].value_counts().rename_axis("laterality").to_frame("n")

    return out

### Subject-level stratified splits

`confcarti/data/splits.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/splits.py
# ==========================================================================
"""Subject-level splits, jointly stratified on KL grade and site.

Design rules enforced here
--------------------------
* Splits are **subject-level**. Two scans of the same knee must never straddle
  the train/calibration boundary or the exchangeability assumption behind the
  conformal guarantee is violated by leakage rather than by distribution shift.
* Splits are **written once and then read**. A loader that recomputes splits
  instead of reading ``splits.json`` is a bug: any change to the metadata table
  would silently reshuffle subjects between train and calibration.
* Allocation is deterministic given the seed and uses largest-remainder
  rounding inside every stratum, so small strata stay proportional instead of
  collapsing into whichever split rounds up first.
"""


import json
import logging
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)

__all__ = [
    "SPLIT_NAMES",
    "SplitReport",
    "make_splits",
    "write_splits",
    "read_splits",
    "assert_disjoint",
    "stratification_report",
]

SPLIT_NAMES: tuple[str, str, str] = ("train", "calibration", "test")


@dataclass(frozen=True, slots=True)
class SplitReport:
    """Realised composition of a split assignment.

    Attributes
    ----------
    sizes
        Split name -> number of subjects.
    fractions
        Split name -> realised fraction of the cohort.
    kl_by_split, site_by_split
        Contingency tables of the stratifiers against split.
    max_abs_deviation
        Largest absolute difference between a stratum's realised proportion in
        a split and its population proportion. A useful one-number check that
        stratification worked.
    """

    sizes: dict[str, int]
    fractions: dict[str, float]
    kl_by_split: pd.DataFrame
    site_by_split: pd.DataFrame
    max_abs_deviation: float

    def to_dict(self) -> dict[str, object]:
        """JSON-safe representation for the run manifest."""
        return {
            "sizes": self.sizes,
            "fractions": self.fractions,
            "kl_by_split": self.kl_by_split.to_dict(),
            "site_by_split": self.site_by_split.to_dict(),
            "max_abs_deviation": self.max_abs_deviation,
        }


def _largest_remainder(
    total: int, fractions: tuple[float, ...], debt: list[float] | None = None
) -> list[int]:
    """Allocate ``total`` items across buckets by the largest-remainder method.

    Guarantees the counts sum exactly to ``total`` while keeping each bucket as
    close to its target fraction as integer arithmetic allows.

    Parameters
    ----------
    total
        Number of items to distribute.
    fractions
        Target proportions; must sum to 1.
    debt
        Optional per-bucket allocation debt carried in from previous strata,
        ``sum(exact) - sum(allocated)`` so far. When supplied it is *mutated in
        place* to reflect this allocation.

        Without carried debt, allocating many small strata independently biases
        the result badly: for a 3-subject stratum, (0.6, 0.2, 0.2) rounds to
        (2, 1, 0), so ``train`` is over-fed by every single stratum and the
        realised global split drifts away from the target. Carrying the debt
        makes the *global* totals correct while each stratum stays within one
        subject of its own target.

    Returns
    -------
    list of int
        Counts per bucket, summing to ``total``.

    Examples
    --------
    >>> _largest_remainder(10, (0.6, 0.2, 0.2))
    [6, 2, 2]
    >>> sum(_largest_remainder(7, (0.6, 0.2, 0.2)))
    7

    Carried debt keeps ten 3-subject strata globally on target, where
    independent rounding would give 20/10/0 instead of 18/6/6:

    >>> debt = [0.0, 0.0, 0.0]
    >>> totals = [0, 0, 0]
    >>> for _ in range(10):
    ...     counts = _largest_remainder(3, (0.6, 0.2, 0.2), debt)
    ...     totals = [t + c for t, c in zip(totals, counts)]
    >>> totals
    [18, 6, 6]
    """
    if debt is None:
        debt = [0.0] * len(fractions)

    exact = [total * f + d for f, d in zip(fractions, debt)]
    # A bucket can be owed less than zero after over-allocation; never hand it a
    # negative count.
    floors = [max(0, int(np.floor(e))) for e in exact]
    remainder = total - sum(floors)

    if remainder > 0:
        # Give the leftovers to the buckets with the largest fractional parts;
        # ties break towards the earlier bucket for determinism.
        order = sorted(range(len(fractions)), key=lambda i: (-(exact[i] - floors[i]), i))
        for i in order[:remainder]:
            floors[i] += 1
    elif remainder < 0:
        # Over-allocated (possible when debt is strongly positive); claw back
        # from the buckets with the smallest fractional parts that have items.
        order = sorted(range(len(fractions)), key=lambda i: (exact[i] - floors[i], i))
        to_remove = -remainder
        for i in order:
            while to_remove > 0 and floors[i] > 0:
                floors[i] -= 1
                to_remove -= 1
            if to_remove == 0:
                break

    for i, (e_target, allocated) in enumerate(zip((total * f for f in fractions), floors)):
        debt[i] += e_target - allocated
    return floors


def make_splits(
    metadata: pd.DataFrame,
    fractions: tuple[float, float, float] = (0.6, 0.2, 0.2),
    seed: int = 20240617,
    *,
    stratify_on: tuple[str, ...] = ("kl_grade", "site"),
) -> dict[str, list[str]]:
    """Assign subjects to train / calibration / test, stratified jointly.

    Parameters
    ----------
    metadata
        Canonical metadata table, one row per subject.
    fractions
        ``(train, calibration, test)`` proportions; must sum to 1.
    seed
        Seed controlling the within-stratum permutation.
    stratify_on
        Columns whose product defines the strata.

    Returns
    -------
    dict
        Split name -> sorted list of subject ids.

    Raises
    ------
    ValueError
        If fractions are invalid, a stratifier column is absent, or the table
        contains duplicate subjects.

    Notes
    -----
    Strata with fewer subjects than the number of splits cannot contribute to
    every split. They are still allocated by largest remainder (so the largest
    split gets them first) and a warning names each one, because such a stratum
    means at least one split has *zero* subjects of that KL x site combination
    and conditional coverage for it is simply unmeasurable.
    """
    if len(fractions) != len(SPLIT_NAMES):
        raise ValueError(f"expected {len(SPLIT_NAMES)} fractions, got {len(fractions)}")
    if abs(sum(fractions) - 1.0) > 1e-9:
        raise ValueError(f"fractions must sum to 1, got {sum(fractions)}")
    if any(f <= 0 for f in fractions):
        raise ValueError(f"every fraction must be > 0, got {fractions}")

    missing = [c for c in ("subject_id", *stratify_on) if c not in metadata.columns]
    if missing:
        raise ValueError(f"metadata is missing column(s) {missing}")

    if metadata["subject_id"].duplicated().any():
        n = int(metadata["subject_id"].duplicated().sum())
        raise ValueError(
            f"metadata has {n} duplicate subject_id row(s); splits must be subject-level"
        )

    rng = seeded_generator(seed)
    table = metadata.copy()
    table["subject_id"] = table["subject_id"].astype(str)
    # Sort first so the stratum iteration order never depends on input row order.
    table = table.sort_values("subject_id", kind="mergesort").reset_index(drop=True)

    stratum_key = table[list(stratify_on)].astype(str).agg("|".join, axis=1)
    table["_stratum"] = stratum_key

    assignment: dict[str, list[str]] = {name: [] for name in SPLIT_NAMES}
    thin_strata: list[str] = []
    # Allocation debt carried across strata so that many small strata cannot
    # collectively bias the realised global split towards `train`.
    debt: list[float] = [0.0] * len(SPLIT_NAMES)

    # Largest strata first: they absorb the debt generated by the small ones,
    # which keeps every individual stratum within one subject of its target.
    stratum_sizes = table["_stratum"].value_counts()
    ordered_strata = sorted(stratum_sizes.index, key=lambda s: (-int(stratum_sizes[s]), str(s)))

    for stratum in ordered_strata:
        subjects = table.loc[table["_stratum"] == stratum, "subject_id"].to_numpy()
        subjects = np.sort(subjects)
        rng.shuffle(subjects)

        counts = _largest_remainder(len(subjects), fractions, debt)
        if len(subjects) < len(SPLIT_NAMES):
            thin_strata.append(f"{stratum} (n={len(subjects)})")

        start = 0
        for name, count in zip(SPLIT_NAMES, counts):
            assignment[name].extend(subjects[start : start + count].tolist())
            start += count

    if thin_strata:
        logger.warning(
            "%d stratum/strata have fewer subjects than splits, so at least one split "
            "contains none of them: %s",
            len(thin_strata),
            ", ".join(thin_strata),
        )

    result = {name: sorted(ids) for name, ids in assignment.items()}
    assert_disjoint(result)

    covered = sum(len(v) for v in result.values())
    if covered != len(table):
        raise ValueError(
            f"split assignment covers {covered} subjects but metadata has {len(table)}"
        )

    logger.info(
        "made splits (seed=%d): %s",
        seed,
        {k: len(v) for k, v in result.items()},
    )
    return result


def assert_disjoint(splits: dict[str, list[str]]) -> None:
    """Raise if any subject appears in more than one split.

    Parameters
    ----------
    splits
        Split name -> subject ids.

    Raises
    ------
    ValueError
        Naming the overlapping split pair and example subjects.
    """
    names = list(splits)
    for i, a in enumerate(names):
        for b in names[i + 1 :]:
            overlap = set(splits[a]) & set(splits[b])
            if overlap:
                raise ValueError(
                    f"subject leakage: {len(overlap)} subject(s) appear in both "
                    f"'{a}' and '{b}', e.g. {sorted(overlap)[:5]}"
                )
    for name, ids in splits.items():
        if len(ids) != len(set(ids)):
            raise ValueError(f"split '{name}' contains duplicate subject ids")


def stratification_report(
    metadata: pd.DataFrame, splits: dict[str, list[str]]
) -> SplitReport:
    """Summarise how well the realised splits match the population.

    Parameters
    ----------
    metadata
        Canonical metadata table.
    splits
        Split assignment.

    Returns
    -------
    SplitReport
    """
    lookup = {sid: name for name, ids in splits.items() for sid in ids}
    table = metadata.copy()
    table["subject_id"] = table["subject_id"].astype(str)
    table["split"] = table["subject_id"].map(lookup)
    table = table[table["split"].notna()]

    sizes = {name: len(ids) for name, ids in splits.items()}
    total = sum(sizes.values())
    fractions = {name: (n / total if total else 0.0) for name, n in sizes.items()}

    kl_by_split = pd.crosstab(table["kl_grade"], table["split"])
    site_by_split = pd.crosstab(table["site"], table["split"])

    # Largest gap between a stratum's share within a split and its share overall.
    max_dev = 0.0
    for frame in (kl_by_split, site_by_split):
        population = frame.sum(axis=1) / frame.to_numpy().sum()
        for split_name in frame.columns:
            column = frame[split_name]
            if column.sum() == 0:
                continue
            realised = column / column.sum()
            max_dev = max(max_dev, float((realised - population).abs().max()))

    return SplitReport(
        sizes=sizes,
        fractions=fractions,
        kl_by_split=kl_by_split,
        site_by_split=site_by_split,
        max_abs_deviation=max_dev,
    )


def write_splits(
    splits: dict[str, list[str]],
    path: str | Path,
    *,
    metadata: pd.DataFrame | None = None,
    seed: int | None = None,
    overwrite: bool = False,
) -> Path:
    """Write the split assignment to an immutable JSON file.

    Parameters
    ----------
    splits
        Split assignment.
    path
        Destination JSON path.
    metadata
        If given, the realised stratification report is embedded for audit.
    seed
        Seed used, recorded in the file.
    overwrite
        Permit replacing an existing file. Defaults to False so that a rerun
        cannot silently reshuffle subjects between train and calibration.

    Returns
    -------
    pathlib.Path

    Raises
    ------
    FileExistsError
        If the file exists and ``overwrite`` is False.
    """
    path = Path(path)
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"{path} already exists. Splits are immutable once written -- delete the file "
            "explicitly or pass overwrite=True if you really mean to reshuffle the cohort."
        )
    assert_disjoint(splits)
    path.parent.mkdir(parents=True, exist_ok=True)

    payload: dict[str, object] = {
        "seed": seed,
        "split_names": list(SPLIT_NAMES),
        "splits": {name: sorted(ids) for name, ids in splits.items()},
    }
    if metadata is not None:
        payload["report"] = stratification_report(metadata, splits).to_dict()

    with path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, sort_keys=True, default=str)
    logger.info("wrote splits to %s", path)
    return path


def read_splits(path: str | Path) -> dict[str, list[str]]:
    """Read a previously written split file.

    This is the *only* supported way for training and evaluation code to learn
    which subjects belong to which split.

    Parameters
    ----------
    path
        Path to ``splits.json``.

    Returns
    -------
    dict
        Split name -> subject ids.

    Raises
    ------
    FileNotFoundError
        If the file is absent.
    ValueError
        If the file is malformed or the splits overlap.
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"splits file not found: {path}. Create it once with scripts/make_splits.py; "
            "downstream code must read it rather than recompute splits."
        )
    with path.open("r", encoding="utf-8") as fh:
        payload = json.load(fh)

    if "splits" not in payload:
        raise ValueError(f"{path} has no 'splits' key")
    splits = {str(k): [str(s) for s in v] for k, v in payload["splits"].items()}

    unexpected = set(splits) - set(SPLIT_NAMES)
    if unexpected:
        raise ValueError(f"{path} contains unknown split name(s) {sorted(unexpected)}")
    assert_disjoint(splits)
    return splits

### The calibration guard

`confcarti/data/guards.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/guards.py
# ==========================================================================
"""Calibration-split guard.

The coverage guarantee of split conformal prediction rests on the calibration
scores being exchangeable with the test score. Any use of a calibration subject
to fit weights, pick a threshold, choose an architecture or trigger early
stopping destroys that, and -- crucially -- destroys it *silently*: the code
still runs, the intervals still look plausible, and the coverage number is
simply wrong.

So the rule is enforced mechanically rather than by discipline. Every code path
that touches subjects declares which splits it is allowed to read, and reading
outside that allowlist raises.
"""


import functools
import logging
from collections.abc import Callable, Iterable, Sequence
from contextvars import ContextVar
from typing import Any, TypeVar

logger = logging.getLogger(__name__)

__all__ = [
    "CalibrationLeakageError",
    "CalibrationGuard",
    "guarded",
    "current_allowlist",
    "check_split_access",
    "check_subject_access",
]

F = TypeVar("F", bound=Callable[..., Any])

#: Name of the split whose contents may only ever be read by the conformal layer.
CALIBRATION_SPLIT = "calibration"

_ALLOWLIST: ContextVar[frozenset[str] | None] = ContextVar("confcarti_split_allowlist", default=None)
_CONTEXT_LABEL: ContextVar[str] = ContextVar("confcarti_guard_label", default="<unguarded>")


class CalibrationLeakageError(RuntimeError):
    """Raised when a code path reads a split it is not allowed to read."""


class CalibrationGuard:
    """Context manager restricting which splits the enclosed code may read.

    Parameters
    ----------
    allow
        Split names this code path may read.
    label
        Human-readable name of the code path, used in error messages.

    Examples
    --------
    A training loop may read train data:

    >>> with CalibrationGuard(allow=["train"], label="train_loop"):
    ...     check_split_access("train")

    but not calibration data:

    >>> with CalibrationGuard(allow=["train"], label="train_loop"):
    ...     check_split_access("calibration")
    Traceback (most recent call last):
        ...
    confcarti.data.guards.CalibrationLeakageError: train_loop ...

    Outside any guard, access is unrestricted but logged, so that scripts which
    forget to declare a guard are visible in the run log:

    >>> check_split_access("calibration")
    """

    __slots__ = ("allow", "label", "_tokens")

    def __init__(self, allow: Iterable[str], label: str = "<unnamed>") -> None:
        self.allow: tuple[str, ...] = tuple(sorted(set(allow)))
        self.label: str = label
        # A stack, so the same guard object can be re-entered (and nested)
        # without the inner exit clobbering the outer context token.
        self._tokens: list[tuple[Any, Any]] = []

    def __repr__(self) -> str:
        return f"CalibrationGuard(allow={self.allow!r}, label={self.label!r})"

    def __enter__(self) -> CalibrationGuard:
        self._tokens.append(
            (_ALLOWLIST.set(frozenset(self.allow)), _CONTEXT_LABEL.set(self.label))
        )
        logger.debug("entered guard %s allowing %s", self.label, self.allow)
        return self

    def __exit__(self, *exc: object) -> None:
        allow_token, label_token = self._tokens.pop()
        _ALLOWLIST.reset(allow_token)
        _CONTEXT_LABEL.reset(label_token)

    def __call__(self, func: F) -> F:
        """Use the guard as a decorator around a whole function."""

        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            with CalibrationGuard(self.allow, self.label or func.__qualname__):
                return func(*args, **kwargs)

        return wrapper  # type: ignore[return-value]


def guarded(*allow: str, label: str | None = None) -> Callable[[F], F]:
    """Decorator form: restrict a function to the named splits.

    Parameters
    ----------
    *allow
        Split names the function may read.
    label
        Name used in error messages; defaults to the function's qualname.

    Returns
    -------
    callable
        Decorator.

    Examples
    --------
    >>> @guarded("train")
    ... def fit(split: str) -> str:
    ...     check_split_access(split)
    ...     return "fitted"
    >>> fit("train")
    'fitted'
    >>> fit("calibration")
    Traceback (most recent call last):
        ...
    confcarti.data.guards.CalibrationLeakageError: fit ...
    """

    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            with CalibrationGuard(allow, label or func.__qualname__):
                return func(*args, **kwargs)

        return wrapper  # type: ignore[return-value]

    return decorator


def current_allowlist() -> frozenset[str] | None:
    """Return the active allowlist, or ``None`` when no guard is active."""
    return _ALLOWLIST.get()


def check_split_access(split: str) -> None:
    """Assert that the current code path may read ``split``.

    Parameters
    ----------
    split
        Split name being requested.

    Raises
    ------
    CalibrationLeakageError
        If a guard is active and ``split`` is not in its allowlist.
    """
    allow = _ALLOWLIST.get()
    if allow is None:
        if split == CALIBRATION_SPLIT:
            logger.info(
                "calibration split read outside any CalibrationGuard; this is only "
                "legitimate inside the conformal layer"
            )
        return
    if split not in allow:
        label = _CONTEXT_LABEL.get()
        extra = ""
        if split == CALIBRATION_SPLIT:
            extra = (
                " Reading the calibration split here would void the finite-sample "
                "coverage guarantee: calibration scores must stay exchangeable with "
                "the test score, so they cannot inform fitting, thresholding, early "
                "stopping or model selection."
            )
        raise CalibrationLeakageError(
            f"{label} is allowed to read splits {sorted(allow)} but requested "
            f"'{split}'.{extra}"
        )


def check_subject_access(
    subject_ids: Sequence[str],
    splits: dict[str, list[str]],
) -> None:
    """Assert that none of ``subject_ids`` belongs to a disallowed split.

    Use this at the point where a data loader materialises a batch, so that a
    hand-assembled subject list cannot bypass :func:`check_split_access`.

    Parameters
    ----------
    subject_ids
        Subjects about to be read.
    splits
        Split assignment as returned by
        :func:`confcarti.data.splits.read_splits`.

    Raises
    ------
    CalibrationLeakageError
        Naming the offending subjects and their split.
    """
    allow = _ALLOWLIST.get()
    if allow is None:
        return

    membership = {sid: name for name, ids in splits.items() for sid in ids}
    offenders: dict[str, list[str]] = {}
    for sid in subject_ids:
        split = membership.get(str(sid))
        if split is not None and split not in allow:
            offenders.setdefault(split, []).append(str(sid))

    if offenders:
        label = _CONTEXT_LABEL.get()
        detail = "; ".join(
            f"{len(ids)} subject(s) from '{split}' (e.g. {sorted(ids)[:3]})"
            for split, ids in sorted(offenders.items())
        )
        raise CalibrationLeakageError(
            f"{label} is allowed to read splits {sorted(allow)} but the requested "
            f"batch contains {detail}."
        )

### Preprocessing with an invertible round trip

`confcarti/data/preprocess.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/preprocess.py
# ==========================================================================
"""Preprocessing with a recorded, invertible mapping back to the original grid.

Every spatial transform here records enough metadata to send a prediction back
to the voxel grid the image arrived on. That round trip is not a nicety: the
thickness map is measured in millimetres on the *original* acquisition grid, and
a half-voxel offset introduced by resampling is a ~0.18 mm systematic error --
comparable to the entire annual cartilage loss this project is trying to detect.
"""


import logging
from dataclasses import dataclass, field
from typing import Any

import numpy as np
from scipy import ndimage


logger = logging.getLogger(__name__)

__all__ = [
    "PreprocessMeta",
    "resample_volume",
    "normalize_intensity",
    "joint_centred_crop",
    "preprocess_case",
    "invert_to_original",
    "n4_bias_correction",
]


@dataclass
class PreprocessMeta:
    """Everything needed to undo :func:`preprocess_case`.

    Attributes
    ----------
    original_shape
        Shape of the incoming volume.
    original_spacing
        Voxel spacing of the incoming volume, in mm.
    resampled_shape
        Shape after resampling, before cropping.
    target_spacing
        Isotropic spacing after resampling, in mm.
    crop_start
        Index of the crop origin within the resampled volume.
    crop_size
        Requested crop size.
    pad_before, pad_after
        Padding applied when the crop window ran past the volume bounds.
    intensity_mean, intensity_std
        Foreground statistics used for z-scoring.
    """

    original_shape: tuple[int, int, int]
    original_spacing: tuple[float, float, float]
    resampled_shape: tuple[int, int, int]
    target_spacing: tuple[float, float, float]
    crop_start: tuple[int, int, int]
    crop_size: tuple[int, int, int]
    pad_before: tuple[int, int, int] = (0, 0, 0)
    pad_after: tuple[int, int, int] = (0, 0, 0)
    intensity_mean: float = 0.0
    intensity_std: float = 1.0
    clipped_cartilage_voxels: int = 0
    clipped_foreground_voxels: int = 0
    extras: dict[str, Any] = field(default_factory=dict)

    @property
    def crop_clipped_cartilage(self) -> bool:
        """Whether the joint-centred crop cut through cartilage.

        A True here invalidates every thickness and area measurement for the
        case: part of the plate being measured is simply not in the volume.
        """
        return self.clipped_cartilage_voxels > 0


def resample_volume(
    volume: np.ndarray,
    original_spacing: tuple[float, float, float],
    target_spacing: tuple[float, float, float],
    *,
    is_label: bool,
    spline_order: int = 3,
) -> np.ndarray:
    """Resample a volume to a new voxel spacing.

    Parameters
    ----------
    volume
        Input array.
    original_spacing, target_spacing
        Voxel sizes in mm.
    is_label
        Labels are resampled with nearest neighbour; images with a B-spline of
        order ``spline_order``. Interpolating a label map with a spline produces
        non-integer "labels" between structures -- a classic and silent bug.
    spline_order
        B-spline order for image resampling.

    Returns
    -------
    numpy.ndarray
        Resampled volume.

    Notes
    -----
    The output shape uses ``round``: ``n_out = round(n_in * s_in / s_out)``. The
    zoom factors actually applied are recomputed from the realised output shape,
    so the inverse transform lands exactly on the original grid.

    Examples
    --------
    >>> vol = np.arange(8, dtype=float).reshape(2, 2, 2)
    >>> out = resample_volume(vol, (1.0, 1.0, 1.0), (0.5, 0.5, 0.5), is_label=False)
    >>> out.shape
    (4, 4, 4)
    """
    volume = np.asarray(volume)
    zoom = [s_in / s_out for s_in, s_out in zip(original_spacing, target_spacing)]
    out_shape = tuple(int(round(n * z)) for n, z in zip(volume.shape, zoom))
    out_shape = tuple(max(1, n) for n in out_shape)

    if out_shape == volume.shape:
        return volume.copy()

    # Recompute the zoom from the realised shape so forward and inverse agree.
    exact_zoom = [o / i for o, i in zip(out_shape, volume.shape)]

    if is_label:
        return ndimage.zoom(volume, exact_zoom, order=0, mode="nearest", grid_mode=False)
    return ndimage.zoom(
        volume.astype(np.float32), exact_zoom, order=spline_order, mode="nearest"
    )


def n4_bias_correction(image: np.ndarray, *, shrink_factor: int = 4) -> np.ndarray:
    """Apply N4 inhomogeneity correction via SimpleITK.

    Parameters
    ----------
    image
        Input intensity volume.
    shrink_factor
        Downsampling used while estimating the field; the field is then applied
        at full resolution.

    Returns
    -------
    numpy.ndarray
        Corrected volume, same shape and dtype family as the input.

    Raises
    ------
    ImportError
        If SimpleITK is not installed. This never silently no-ops: a run
        configured with ``n4_bias_correction: true`` that quietly skipped the
        correction would be unreproducible.
    """
    try:
        import SimpleITK as sitk
    except ImportError as exc:  # pragma: no cover - depends on environment
        raise ImportError(
            "n4_bias_correction requires SimpleITK; install it or set "
            "data.n4_bias_correction=false in the config"
        ) from exc

    itk_image = sitk.GetImageFromArray(np.asarray(image, dtype=np.float32))
    mask = sitk.OtsuThreshold(itk_image, 0, 1, 200)

    shrunk = sitk.Shrink(itk_image, [shrink_factor] * itk_image.GetDimension())
    shrunk_mask = sitk.Shrink(mask, [shrink_factor] * itk_image.GetDimension())

    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.Execute(shrunk, shrunk_mask)
    log_field = corrector.GetLogBiasFieldAsImage(itk_image)

    corrected = itk_image / sitk.Exp(log_field)
    return sitk.GetArrayFromImage(corrected).astype(np.float32)


def normalize_intensity(
    image: np.ndarray,
    *,
    foreground_mask: np.ndarray | None = None,
    clip_percentiles: tuple[float, float] = (0.5, 99.5),
) -> tuple[np.ndarray, float, float]:
    """Clip outliers, then z-score using foreground statistics only.

    Using whole-volume statistics would let the (large, near-constant) air
    background dominate the mean and standard deviation, so two scans with
    different fields of view would be normalised differently.

    Parameters
    ----------
    image
        Intensity volume.
    foreground_mask
        Boolean foreground. Defaults to voxels above the volume mean, which is a
        serviceable air/tissue split for DESS.
    clip_percentiles
        Percentiles (computed over the foreground) used to clip before z-scoring.

    Returns
    -------
    tuple
        ``(normalised, mean, std)``.

    Examples
    --------
    >>> img = np.concatenate([np.zeros(500), np.random.RandomState(0).normal(100, 10, 500)])
    >>> out, mu, sd = normalize_intensity(img.reshape(10, 10, 10))
    >>> bool(abs(out[out > out.min()].mean()) < 3)
    True
    """
    image = np.asarray(image, dtype=np.float32)
    if foreground_mask is None:
        foreground_mask = image > float(image.mean())
    foreground = image[foreground_mask]
    if foreground.size == 0:
        logger.warning("empty foreground mask; falling back to whole-volume statistics")
        foreground = image.reshape(-1)

    lo, hi = np.percentile(foreground, clip_percentiles)
    clipped = np.clip(image, lo, hi)

    fg_clipped = clipped[foreground_mask] if foreground_mask.any() else clipped
    mean = float(fg_clipped.mean())
    std = float(fg_clipped.std())
    if std < 1e-8:
        logger.warning("near-zero foreground std (%.3g); using unit scale", std)
        std = 1.0

    return ((clipped - mean) / std).astype(np.float32), mean, std


def _joint_centre(label: np.ndarray) -> tuple[int, int, int]:
    """Centroid of the cartilage voxels, which defines the joint centre."""
    cartilage = np.isin(label, CARTILAGE_LABELS)
    if not cartilage.any():
        logger.warning("no cartilage voxels found; centring the crop on the volume")
        return tuple(n // 2 for n in label.shape)  # type: ignore[return-value]
    idx = np.argwhere(cartilage)
    return tuple(int(round(v)) for v in idx.mean(axis=0))  # type: ignore[return-value]


def joint_centred_crop(
    image: np.ndarray,
    label: np.ndarray | None,
    crop_size: tuple[int, int, int],
    *,
    centre: tuple[int, int, int] | None = None,
) -> tuple[np.ndarray, np.ndarray | None, dict[str, Any]]:
    """Crop a fixed-size window centred on the joint, padding when needed.

    Parameters
    ----------
    image
        Intensity volume.
    label
        Matching label volume, or None.
    crop_size
        Output size in voxels.
    centre
        Explicit crop centre. Defaults to the cartilage centroid.

    Returns
    -------
    tuple
        ``(cropped_image, cropped_label, meta)`` where ``meta`` carries
        ``crop_start``, ``pad_before`` and ``pad_after`` for inversion.

    Raises
    ------
    ValueError
        If ``image`` and ``label`` shapes disagree.

    Examples
    --------
    >>> img = np.zeros((10, 10, 10)); lab = np.zeros((10, 10, 10), dtype=np.uint8)
    >>> lab[4:6, 4:6, 4:6] = 2
    >>> ci, cl, meta = joint_centred_crop(img, lab, (6, 6, 6))
    >>> ci.shape
    (6, 6, 6)
    """
    image = np.asarray(image)
    if label is not None:
        label = np.asarray(label)
        if label.shape != image.shape:
            raise ValueError(
                f"image shape {image.shape} != label shape {label.shape}"
            )

    if centre is None:
        centre = _joint_centre(label) if label is not None else tuple(
            n // 2 for n in image.shape
        )  # type: ignore[assignment]

    starts, pads_before, pads_after = [], [], []
    for axis, (c, size, n) in enumerate(zip(centre, crop_size, image.shape)):
        start = int(c) - size // 2
        pad_before = max(0, -start)
        start = max(0, start)
        end = start + size - pad_before
        pad_after = max(0, end - n)
        starts.append(start)
        pads_before.append(pad_before)
        pads_after.append(pad_after)
        del axis

    slices = tuple(
        slice(s, min(s + size - pb, n))
        for s, size, pb, n in zip(starts, crop_size, pads_before, image.shape)
    )
    pad_width = tuple(zip(pads_before, pads_after))

    cropped_image = np.pad(image[slices], pad_width, mode="constant", constant_values=0)
    cropped_label = (
        np.pad(label[slices], pad_width, mode="constant", constant_values=0)
        if label is not None
        else None
    )

    meta = {
        "crop_start": tuple(starts),
        "crop_size": tuple(crop_size),
        "pad_before": tuple(pads_before),
        "pad_after": tuple(pads_after),
    }
    return cropped_image, cropped_label, meta


def preprocess_case(
    image: np.ndarray,
    label: np.ndarray | None,
    spacing: tuple[float, float, float],
    *,
    target_spacing_mm: float = 0.5,
    crop_size: tuple[int, int, int] = (160, 160, 96),
    clip_percentiles: tuple[float, float] = (0.5, 99.5),
    apply_n4: bool = False,
    spline_order: int = 3,
) -> tuple[np.ndarray, np.ndarray | None, PreprocessMeta]:
    """Run the full preprocessing chain and record how to undo it.

    Order: optional N4, resample to isotropic, foreground z-score,
    joint-centred crop.

    Parameters
    ----------
    image
        Intensity volume.
    label
        Matching label volume, or None at inference time.
    spacing
        Input voxel spacing in mm.
    target_spacing_mm
        Isotropic output spacing.
    crop_size
        Joint-centred crop size in voxels.
    clip_percentiles
        Intensity clipping percentiles.
    apply_n4
        Apply N4 bias correction first.
    spline_order
        B-spline order for image resampling.

    Returns
    -------
    tuple
        ``(image, label, meta)``. Pass ``meta`` to :func:`invert_to_original`.
    """
    image = np.asarray(image, dtype=np.float32)
    original_shape = tuple(image.shape)
    target = (target_spacing_mm,) * 3

    if apply_n4:
        image = n4_bias_correction(image)

    image_rs = resample_volume(
        image, spacing, target, is_label=False, spline_order=spline_order
    )
    label_rs = (
        resample_volume(label, spacing, target, is_label=True) if label is not None else None
    )
    resampled_shape = tuple(image_rs.shape)

    foreground = label_rs > 0 if label_rs is not None else None
    image_norm, mean, std = normalize_intensity(
        image_rs, foreground_mask=foreground, clip_percentiles=clip_percentiles
    )

    image_c, label_c, crop_meta = joint_centred_crop(image_norm, label_rs, crop_size)

    # A crop that cuts through cartilage silently truncates the structure being
    # measured, so every thickness and area number for the case would be wrong.
    # Count it, record it, and say so loudly.
    clipped_cartilage = 0
    clipped_foreground = 0
    if label_rs is not None and label_c is not None:
        before_cart = int(np.isin(label_rs, CARTILAGE_LABELS).sum())
        after_cart = int(np.isin(label_c, CARTILAGE_LABELS).sum())
        clipped_cartilage = max(0, before_cart - after_cart)
        clipped_foreground = max(0, int((label_rs > 0).sum()) - int((label_c > 0).sum()))
        if clipped_cartilage > 0:
            logger.warning(
                "joint-centred crop of %s clipped %d cartilage voxel(s) (%.2f%% of the "
                "plate): thickness and area for this case are not trustworthy. Increase "
                "data.crop_size or check the joint centring.",
                crop_size,
                clipped_cartilage,
                100.0 * clipped_cartilage / max(before_cart, 1),
            )

    meta = PreprocessMeta(
        original_shape=original_shape,  # type: ignore[arg-type]
        original_spacing=tuple(spacing),  # type: ignore[arg-type]
        resampled_shape=resampled_shape,  # type: ignore[arg-type]
        target_spacing=target,
        crop_start=crop_meta["crop_start"],
        crop_size=crop_meta["crop_size"],
        pad_before=crop_meta["pad_before"],
        pad_after=crop_meta["pad_after"],
        intensity_mean=mean,
        intensity_std=std,
        clipped_cartilage_voxels=clipped_cartilage,
        clipped_foreground_voxels=clipped_foreground,
    )
    return image_c, label_c, meta


def invert_to_original(
    prediction: np.ndarray, meta: PreprocessMeta, *, is_label: bool = True
) -> np.ndarray:
    """Map a prediction from the preprocessed grid back to the original grid.

    Undoes the crop (by placing the window back into a zero volume of the
    resampled shape) and then the resampling.

    Parameters
    ----------
    prediction
        Array on the preprocessed grid, shaped ``meta.crop_size``.
    meta
        Metadata returned by :func:`preprocess_case`.
    is_label
        Nearest-neighbour resampling for labels, B-spline for continuous fields.

    Returns
    -------
    numpy.ndarray
        Array on the original grid, shaped ``meta.original_shape``.

    Raises
    ------
    ValueError
        If ``prediction`` does not match ``meta.crop_size``.

    Examples
    --------
    Preprocessing a label map and inverting it returns the original labels:

    >>> from confcarti.data.synthetic import make_phantom, PhantomSpec
    >>> knee = make_phantom(PhantomSpec(shape=(48, 48, 32), spacing_mm=0.5))
    >>> _, lab, meta = preprocess_case(knee.image, knee.label, knee.spacing,
    ...                                target_spacing_mm=0.5, crop_size=(48, 48, 32))
    >>> back = invert_to_original(lab, meta)
    >>> back.shape == knee.label.shape
    True
    """
    prediction = np.asarray(prediction)
    if tuple(prediction.shape) != tuple(meta.crop_size):
        raise ValueError(
            f"prediction shape {prediction.shape} does not match the recorded "
            f"crop size {meta.crop_size}"
        )

    # 1. Undo padding and place the crop back into the resampled volume.
    unpad = tuple(
        slice(pb, size - pa)
        for pb, pa, size in zip(meta.pad_before, meta.pad_after, meta.crop_size)
    )
    core = prediction[unpad]

    resampled = np.zeros(meta.resampled_shape, dtype=prediction.dtype)
    dest = tuple(
        slice(start, start + n) for start, n in zip(meta.crop_start, core.shape)
    )
    resampled[dest] = core

    # 2. Undo the resampling by zooming back onto the original shape exactly.
    if tuple(meta.resampled_shape) == tuple(meta.original_shape):
        return resampled

    zoom = [o / r for o, r in zip(meta.original_shape, meta.resampled_shape)]
    if is_label:
        out = ndimage.zoom(resampled, zoom, order=0, mode="nearest")
    else:
        out = ndimage.zoom(resampled.astype(np.float32), zoom, order=3, mode="nearest")

    # Guard against an off-by-one from floating-point zoom factors.
    if tuple(out.shape) != tuple(meta.original_shape):
        fixed = np.zeros(meta.original_shape, dtype=out.dtype)
        common = tuple(slice(0, min(a, b)) for a, b in zip(out.shape, meta.original_shape))
        fixed[common] = out[common]
        out = fixed
    return out

### Analytic knee phantoms with known ground truth

`confcarti/data/synthetic.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/synthetic.py
# ==========================================================================
"""Analytic knee phantoms with known thickness, curvature, bulge and lesions.

Why this exists
---------------
Two reasons, and both matter.

1. **Validation.** Every geometric routine in this package is checked against a
   shape whose answer is known in closed form. The femoral compartment is a
   circular cylinder, so its bone-cartilage interface has principal curvatures
   ``(1/R, 0)``, mean curvature ``H = 1/(2R)`` and Gaussian curvature ``K = 0``
   exactly. The tibial compartment is a plane: ``H = K = 0``. Cartilage is a
   shell of prescribed thickness, so the thickness map has a known constant
   value. A lesion is a spherical cap of known area. A bulge is a Gaussian bump
   of known height. If the code disagrees with those numbers, the code is wrong.

2. **Runnability.** OAI and OAI-ZIB require a data use agreement, so no real
   image ships with this repository. The phantoms let the entire pipeline --
   preprocessing, segmentation training, meshing, thickness, curvature, bulge,
   the denuded branch, the conformal layer and every figure -- run end to end
   on a clean clone with no restricted data present.

Array convention
----------------
Volumes are ``(X, Y, Z)`` with ``X`` medial-lateral, ``Y`` posterior-anterior
and ``Z`` inferior-superior, matching the defaults of
:func:`confcarti.data.labels.validate_label_convention`.
"""


from dataclasses import dataclass, field, replace
from typing import Any

import numpy as np


__all__ = [
    "PhantomSpec",
    "PhantomKnee",
    "make_phantom",
    "make_phantom_cohort",
    "kl_to_spec",
]


@dataclass(frozen=True, slots=True)
class PhantomSpec:
    """Geometry and pathology of one synthetic knee.

    Attributes
    ----------
    shape
        Volume shape in voxels, ``(X, Y, Z)``.
    spacing_mm
        Isotropic voxel size in millimetres.
    condyle_radius_mm
        Radius ``R`` of each femoral condyle cylinder. The bone-cartilage
        interface therefore has mean curvature ``1 / (2R)``.
    condyle_gap_mm
        Width of the intercondylar notch separating the two condyles.
    femoral_thickness_mm
        Uniform femoral cartilage thickness before lesions.
    tibial_thickness_mm
        Uniform tibial cartilage thickness before lesions.
    tibial_plateau_thickness_mm
        Thickness of the tibial bone slab.
    joint_gap_mm
        Vertical clearance between femoral and tibial cartilage.
    n_lesions
        Number of full-thickness cartilage lesions (denuded patches).
    lesion_radius_mm
        Radius of each lesion in the surface plane.
    n_bulges
        Number of focal outward bulges of the articular surface.
    bulge_height_mm
        Peak signed height of each bulge.
    bulge_sigma_mm
        Gaussian width of each bulge.
    thickness_gradient
        Fractional thinning from the lateral to the medial edge, so the
        thickness map is not perfectly flat.
    noise_sigma
        Standard deviation of Gaussian noise added to the synthetic image.
    seed
        Seed for lesion/bulge placement and image noise.
    """

    shape: tuple[int, int, int] = (96, 96, 64)
    spacing_mm: float = 0.5
    condyle_radius_mm: float = 22.0
    condyle_gap_mm: float = 6.0
    femoral_thickness_mm: float = 2.4
    tibial_thickness_mm: float = 2.0
    tibial_plateau_thickness_mm: float = 6.0
    joint_gap_mm: float = 1.0
    n_lesions: int = 0
    lesion_radius_mm: float = 3.0
    n_bulges: int = 0
    bulge_height_mm: float = 0.6
    bulge_sigma_mm: float = 3.0
    thickness_gradient: float = 0.0
    noise_sigma: float = 0.03
    seed: int = 0

    def __post_init__(self) -> None:
        if self.spacing_mm <= 0:
            raise ValueError(f"spacing_mm must be > 0, got {self.spacing_mm}")
        if self.femoral_thickness_mm <= 0 or self.tibial_thickness_mm <= 0:
            raise ValueError("cartilage thickness must be > 0")
        if self.condyle_radius_mm <= 0:
            raise ValueError("condyle_radius_mm must be > 0")


@dataclass
class PhantomKnee:
    """A generated phantom plus its analytic ground truth.

    Attributes
    ----------
    image
        Synthetic DESS-like intensity volume, float32 in roughly [0, 1].
    label
        5-ROI integer mask following the OAIZIB-CM convention.
    spec
        The spec used to build it.
    truth
        Closed-form quantities: nominal thicknesses, exact BCI curvatures,
        lesion centres and areas, bulge centres and heights.
    """

    image: np.ndarray
    label: np.ndarray
    spec: PhantomSpec
    truth: dict[str, Any] = field(default_factory=dict)

    @property
    def spacing(self) -> tuple[float, float, float]:
        """Voxel spacing as a 3-tuple in millimetres."""
        s = self.spec.spacing_mm
        return (s, s, s)


def kl_to_spec(kl_grade: int, base: PhantomSpec | None = None, *, seed: int = 0) -> PhantomSpec:
    """Derive a phantom spec whose pathology matches a KL grade.

    Encodes the two facts the denuded branch is required to reproduce on real
    data: cartilage thins with radiographic severity, and full-thickness loss
    becomes more prevalent and larger, approaching universal at KL 4.

    Parameters
    ----------
    kl_grade
        Kellgren-Lawrence grade in 0..4.
    base
        Starting spec; defaults to a healthy knee.
    seed
        Seed stored on the returned spec.

    Returns
    -------
    PhantomSpec

    Raises
    ------
    ValueError
        If ``kl_grade`` is outside 0..4.

    Examples
    --------
    >>> kl_to_spec(0).n_lesions
    0
    >>> kl_to_spec(4).n_lesions > kl_to_spec(2).n_lesions
    True
    >>> kl_to_spec(4).femoral_thickness_mm < kl_to_spec(0).femoral_thickness_mm
    True
    """
    if kl_grade not in (0, 1, 2, 3, 4):
        raise ValueError(f"kl_grade must be in 0..4, got {kl_grade}")
    base = base or PhantomSpec()

    # Monotone thinning: ~6% of baseline thickness lost per grade.
    thin = 1.0 - 0.06 * kl_grade
    # Lesion burden and focal bulging both climb with grade.
    n_lesions = (0, 0, 1, 3, 6)[kl_grade]
    lesion_radius = (0.0, 0.0, 2.0, 3.0, 4.5)[kl_grade]
    n_bulges = (0, 1, 2, 3, 4)[kl_grade]
    bulge_height = (0.0, 0.15, 0.3, 0.5, 0.75)[kl_grade]

    return replace(
        base,
        femoral_thickness_mm=base.femoral_thickness_mm * thin,
        tibial_thickness_mm=base.tibial_thickness_mm * thin,
        n_lesions=n_lesions,
        lesion_radius_mm=lesion_radius,
        n_bulges=n_bulges,
        bulge_height_mm=bulge_height,
        thickness_gradient=0.05 * kl_grade,
        seed=seed,
    )


def _grids(spec: PhantomSpec) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Physical-coordinate meshgrids (mm) for the volume, in ``ij`` indexing."""
    nx, ny, nz = spec.shape
    s = spec.spacing_mm
    x = np.arange(nx, dtype=np.float64) * s
    y = np.arange(ny, dtype=np.float64) * s
    z = np.arange(nz, dtype=np.float64) * s
    return np.meshgrid(x, y, z, indexing="ij")


def make_phantom(spec: PhantomSpec) -> PhantomKnee:
    """Build one phantom knee with analytic ground truth.

    The femur is two parallel circular cylinders whose axes run medial-lateral,
    separated by an intercondylar notch; femoral cartilage is the shell between
    radius ``R`` and ``R + t`` over the inferior half of each cylinder. The tibia
    is an axis-aligned slab and tibial cartilage a flat layer on top of it, split
    at the midline into medial and lateral plates.

    Parameters
    ----------
    spec
        Geometry and pathology.

    Returns
    -------
    PhantomKnee

    Examples
    --------
    >>> knee = make_phantom(PhantomSpec(shape=(64, 64, 48)))
    >>> sorted(np.unique(knee.label).tolist())
    [0, 1, 2, 3, 4, 5]
    >>> round(knee.truth["femoral_mean_curvature_mm_inv"], 4)
    0.0227
    """
    rng = seeded_generator(spec.seed)
    X, Y, Z = _grids(spec)
    nx, ny, nz = spec.shape
    s = spec.spacing_mm
    extent = np.array([nx, ny, nz], dtype=np.float64) * s

    label = np.zeros(spec.shape, dtype=np.uint8)

    # ---- tibia: slab at the bottom, cartilage layer on top -----------------
    tibia_top = extent[2] * 0.30
    tibia_bottom = tibia_top - spec.tibial_plateau_thickness_mm
    tibial_cart_top = tibia_top + spec.tibial_thickness_mm

    in_plateau_xy = (
        (X > extent[0] * 0.08)
        & (X < extent[0] * 0.92)
        & (Y > extent[1] * 0.12)
        & (Y < extent[1] * 0.88)
    )
    label[(Z <= tibia_top) & (Z > tibia_bottom) & in_plateau_xy] = TIBIA

    # ---- femur: two condyle cylinders, axes along X ------------------------
    # Cylinder centre in the (Y, Z) plane; the inferior half carries cartilage.
    cy = extent[1] * 0.5
    cz = (
        tibial_cart_top
        + spec.joint_gap_mm
        + spec.femoral_thickness_mm
        + spec.condyle_radius_mm
    )
    radial = np.sqrt((Y - cy) ** 2 + (Z - cz) ** 2)

    midline = extent[0] * 0.5
    half_gap = spec.condyle_gap_mm * 0.5
    medial_side = X < (midline - half_gap)
    lateral_side = X > (midline + half_gap)
    condyle_xy = (
        (medial_side | lateral_side) & (X > extent[0] * 0.06) & (X < extent[0] * 0.94)
    )
    # Weight-bearing aspect only: below the cylinder axis.
    weight_bearing = Z < cz

    R = spec.condyle_radius_mm

    # Per-voxel femoral cartilage thickness, optionally graded medio-laterally.
    if spec.thickness_gradient:
        frac = np.clip(X / max(extent[0], 1e-9), 0.0, 1.0)
        fem_t = spec.femoral_thickness_mm * (1.0 - spec.thickness_gradient * frac)
    else:
        fem_t = np.full_like(X, spec.femoral_thickness_mm)

    # ---- focal bulges: push the articular surface outward locally ----------
    bulges: list[dict[str, float]] = []
    if spec.n_bulges > 0 and spec.bulge_height_mm > 0:
        for _ in range(spec.n_bulges):
            bx = float(rng.uniform(extent[0] * 0.15, extent[0] * 0.85))
            by = float(rng.uniform(cy - R * 0.45, cy + R * 0.45))
            d2 = (X - bx) ** 2 + (Y - by) ** 2
            fem_t = fem_t + spec.bulge_height_mm * np.exp(
                -d2 / (2.0 * spec.bulge_sigma_mm**2)
            )
            bulges.append(
                {
                    "x_mm": bx,
                    "y_mm": by,
                    "height_mm": spec.bulge_height_mm,
                    "sigma_mm": spec.bulge_sigma_mm,
                }
            )

    femur_mask = (radial <= R) & condyle_xy
    fem_cart_mask = (radial > R) & (radial <= R + fem_t) & condyle_xy & weight_bearing

    # ---- full-thickness lesions: remove cartilage entirely -----------------
    lesions: list[dict[str, float]] = []
    lesion_void = np.zeros(spec.shape, dtype=bool)
    if spec.n_lesions > 0 and spec.lesion_radius_mm > 0:
        for _ in range(spec.n_lesions):
            on_femur = bool(rng.random() < 0.6)
            lx = float(rng.uniform(extent[0] * 0.15, extent[0] * 0.85))
            ly = float(rng.uniform(extent[1] * 0.25, extent[1] * 0.75))
            d2 = (X - lx) ** 2 + (Y - ly) ** 2
            lesion_void |= d2 <= spec.lesion_radius_mm**2
            lesions.append(
                {
                    "x_mm": lx,
                    "y_mm": ly,
                    "radius_mm": spec.lesion_radius_mm,
                    "area_mm2": float(np.pi * spec.lesion_radius_mm**2),
                    "compartment": "FC" if on_femur else "TC",
                }
            )

    fem_cart_mask &= ~lesion_void

    label[femur_mask] = FEMUR
    label[fem_cart_mask] = FEMORAL_CARTILAGE

    # ---- tibial cartilage, split medial / lateral --------------------------
    tib_cart = (
        (Z > tibia_top) & (Z <= tibial_cart_top) & in_plateau_xy & ~lesion_void
    )
    label[tib_cart & (X < midline)] = MEDIAL_TIBIAL_CARTILAGE
    label[tib_cart & (X >= midline)] = LATERAL_TIBIAL_CARTILAGE

    image = _synthesise_image(label, spec, rng)

    truth: dict[str, Any] = {
        "femoral_thickness_mm": spec.femoral_thickness_mm,
        "tibial_thickness_mm": spec.tibial_thickness_mm,
        # Cylinder of radius R: kappa1 = 1/R, kappa2 = 0.
        "femoral_principal_curvatures_mm_inv": (1.0 / R, 0.0),
        "femoral_mean_curvature_mm_inv": 1.0 / (2.0 * R),
        "femoral_gaussian_curvature_mm_inv2": 0.0,
        # Flat plateau.
        "tibial_mean_curvature_mm_inv": 0.0,
        "tibial_gaussian_curvature_mm_inv2": 0.0,
        "condyle_radius_mm": R,
        "condyle_axis_z_mm": cz,
        "condyle_axis_y_mm": cy,
        "tibial_plateau_top_mm": tibia_top,
        "lesions": lesions,
        "total_lesion_area_mm2": float(sum(le["area_mm2"] for le in lesions)),
        "bulges": bulges,
        "n_lesions": len(lesions),
        "n_bulges": len(bulges),
    }
    return PhantomKnee(image=image, label=label, spec=spec, truth=truth)


def _synthesise_image(
    label: np.ndarray, spec: PhantomSpec, rng: np.random.Generator
) -> np.ndarray:
    """Render a DESS-like intensity volume from a label mask.

    In sagittal 3D DESS, cartilage is bright, cortical bone and marrow are dark,
    and background is near zero. Reproducing that ordering (rather than random
    intensities) means a segmentation network trained on phantoms learns
    something with the same sign structure as the real task.
    """
    intensity = {
        0: 0.05,
        FEMUR: 0.22,
        FEMORAL_CARTILAGE: 0.85,
        TIBIA: 0.22,
        MEDIAL_TIBIAL_CARTILAGE: 0.85,
        LATERAL_TIBIAL_CARTILAGE: 0.85,
    }
    image = np.zeros(label.shape, dtype=np.float32)
    for value, level in intensity.items():
        image[label == value] = level

    # Smooth bone/soft-tissue interface, then a slow bias field and noise.
    try:
        from scipy.ndimage import gaussian_filter

        image = gaussian_filter(image, sigma=0.7)
        nx, ny, nz = label.shape
        gx = np.linspace(-1.0, 1.0, nx)[:, None, None]
        gz = np.linspace(-1.0, 1.0, nz)[None, None, :]
        bias = 1.0 + 0.12 * gx + 0.08 * gz
        image = image * bias
    except ImportError:  # pragma: no cover - scipy is a hard dependency
        pass

    image = image + rng.normal(0.0, spec.noise_sigma, size=image.shape)
    return np.clip(image, 0.0, None).astype(np.float32)


def make_phantom_cohort(
    n_subjects: int = 24,
    *,
    base: PhantomSpec | None = None,
    seed: int = 0,
    sites: tuple[str, ...] = ("siteA", "siteB"),
    kl_weights: tuple[float, ...] = (0.21, 0.12, 0.22, 0.29, 0.16),
) -> tuple[list[PhantomKnee], "Any"]:
    """Generate a cohort of phantoms plus a matching metadata table.

    The default KL weights follow the empirical OAIZIB-CM distribution
    (103/58/108/139/73 over 481 subjects), so the synthetic cohort exercises the
    same class imbalance the real splits have to cope with.

    Parameters
    ----------
    n_subjects
        Number of phantom knees.
    base
        Base spec; pathology is layered on per KL grade.
    seed
        Master seed.
    sites
        Site labels assigned round-robin, with a small per-site geometry offset
        so that site is a genuine (not nominal) source of variation.
    kl_weights
        Sampling weights for KL grades 0..4.

    Returns
    -------
    tuple
        ``(knees, metadata)`` where ``metadata`` is a canonical metadata
        DataFrame keyed by ``subject_id``.

    Examples
    --------
    >>> knees, meta = make_phantom_cohort(6, seed=1)
    >>> len(knees) == len(meta) == 6
    True
    >>> set(meta.columns) >= {"subject_id", "kl_grade", "site", "visit", "laterality"}
    True
    """
    import pandas as pd

    if n_subjects < 1:
        raise ValueError(f"n_subjects must be >= 1, got {n_subjects}")
    if len(kl_weights) != 5:
        raise ValueError(f"kl_weights must have 5 entries, got {len(kl_weights)}")

    rng = seeded_generator(seed)
    weights = np.asarray(kl_weights, dtype=float)
    weights = weights / weights.sum()

    base = base or PhantomSpec()
    knees: list[PhantomKnee] = []
    rows: list[dict[str, Any]] = []

    for i in range(n_subjects):
        kl = int(rng.choice(5, p=weights))
        site = sites[i % len(sites)]
        laterality = "right" if i % 2 == 0 else "left"

        spec = kl_to_spec(kl, base=base, seed=seed * 10_000 + i)
        # Between-subject anatomical variation, plus a small systematic site
        # effect so that site-conditional coverage has something to detect.
        site_offset = 1.0 + 0.03 * (i % len(sites))
        spec = replace(
            spec,
            condyle_radius_mm=spec.condyle_radius_mm * float(rng.uniform(0.92, 1.08)),
            femoral_thickness_mm=spec.femoral_thickness_mm
            * float(rng.uniform(0.9, 1.1))
            * site_offset,
            tibial_thickness_mm=spec.tibial_thickness_mm * float(rng.uniform(0.9, 1.1)),
        )
        knee = make_phantom(spec)
        knees.append(knee)

        rows.append(
            {
                "subject_id": f"PH{i:04d}",
                "kl_grade": kl,
                "site": site,
                "visit": "V00",
                "laterality": laterality,
                "age": int(rng.integers(45, 80)),
                "sex": "male" if rng.random() < 0.5 else "female",
                "bmi": round(float(rng.normal(28.0, 4.5)), 1),
            }
        )

    metadata = pd.DataFrame(rows)
    return knees, metadata

### Dataset loaders

`confcarti/data/dataset.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/data/dataset.py
# ==========================================================================
"""OAIZIB-CM dataset: case discovery, loading and a MONAI-compatible wrapper.

The dataset yields dicts with ``image, label, subject_id, kl_grade, site,
spacing`` so that every downstream consumer -- training, meshing, conformal
calibration -- can stratify without a second lookup.

Two sources are supported behind one interface:

* :class:`OAIZIBCMDataset` reads NIfTI volumes from disk.
* :class:`PhantomDataset` serves in-memory phantoms, so the same code paths run
  with no restricted data present.
"""


import logging
import re
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)

__all__ = [
    "CaseRecord",
    "discover_cases",
    "load_volume",
    "OAIZIBCMDataset",
    "PhantomDataset",
    "build_dataset",
]

#: Subject id embedded in an OAI-ZIB filename, e.g. ``9001104_V00.nii.gz`` or
#: ``sub-9001104_desc-...``. Falls back to the whole stem when nothing matches.
_SUBJECT_PATTERN = re.compile(r"(?:sub-)?(\d{7})")


@dataclass(frozen=True, slots=True)
class CaseRecord:
    """One image/label pair on disk.

    Attributes
    ----------
    subject_id
        Subject identifier used to join with metadata and splits.
    image_path
        Path to the image volume.
    label_path
        Path to the segmentation mask, or None for unlabelled inference cases.
    """

    subject_id: str
    image_path: Path
    label_path: Path | None


def _subject_from_filename(path: Path) -> str:
    """Extract the subject id from a filename, falling back to the stem."""
    stem = path.name
    for suffix in (".nii.gz", ".nii", ".mha", ".mhd", ".nrrd"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break
    match = _SUBJECT_PATTERN.search(stem)
    return match.group(1) if match else stem


def discover_cases(
    root: str | Path,
    *,
    images_dir: str = "imagesTr",
    labels_dir: str = "labelsTr",
    pattern: str = "*.nii.gz",
    require_labels: bool = True,
) -> list[CaseRecord]:
    """Find image/label pairs under an nnU-Net-style directory layout.

    Parameters
    ----------
    root
        Dataset root containing ``images_dir`` and ``labels_dir``.
    images_dir, labels_dir
        Sub-directory names.
    pattern
        Glob for volume files.
    require_labels
        Raise when an image has no matching label. Set False for inference-only
        directories.

    Returns
    -------
    list of CaseRecord
        Sorted by subject id.

    Raises
    ------
    FileNotFoundError
        If the root or images directory is missing, or no images match.
    ValueError
        If ``require_labels`` and some images have no label.
    """
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"dataset root not found: {root}")
    image_root = root / images_dir
    if not image_root.is_dir():
        raise FileNotFoundError(
            f"images directory not found: {image_root}. Expected an nnU-Net-style "
            f"layout with '{images_dir}/' and '{labels_dir}/' under {root}."
        )
    label_root = root / labels_dir

    images = sorted(image_root.glob(pattern))
    if not images:
        raise FileNotFoundError(f"no files matching {pattern!r} under {image_root}")

    # nnU-Net appends a _0000 modality suffix to images but not to labels.
    label_index: dict[str, Path] = {}
    if label_root.is_dir():
        for path in label_root.glob(pattern):
            label_index[_subject_from_filename(path)] = path

    records: list[CaseRecord] = []
    unlabelled: list[str] = []
    for image_path in images:
        subject = _subject_from_filename(image_path)
        label_path = label_index.get(subject)
        if label_path is None:
            unlabelled.append(subject)
        records.append(CaseRecord(subject, image_path, label_path))

    if require_labels and unlabelled:
        raise ValueError(
            f"{len(unlabelled)} image(s) have no matching label in {label_root}, "
            f"e.g. {unlabelled[:5]}. Pass require_labels=False for inference-only data."
        )

    records.sort(key=lambda r: r.subject_id)
    logger.info("discovered %d case(s) under %s", len(records), root)
    return records


def load_volume(path: str | Path) -> tuple[np.ndarray, tuple[float, float, float]]:
    """Load a volume and its voxel spacing.

    Tries SimpleITK first (which handles NIfTI, MetaImage and NRRD) and falls
    back to nibabel.

    Parameters
    ----------
    path
        Path to the volume.

    Returns
    -------
    tuple
        ``(array, spacing_mm)`` with the array in ``(X, Y, Z)`` index order.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ImportError
        If neither SimpleITK nor nibabel is available.
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"volume not found: {path}")

    try:
        import SimpleITK as sitk

        image = sitk.ReadImage(str(path))
        # SimpleITK returns (z, y, x); transpose to (x, y, z) to match spacing order.
        array = sitk.GetArrayFromImage(image).transpose(2, 1, 0)
        spacing = tuple(float(s) for s in image.GetSpacing())
        return array, spacing  # type: ignore[return-value]
    except ImportError:
        pass

    try:
        import nibabel as nib
    except ImportError as exc:  # pragma: no cover - depends on environment
        raise ImportError(
            "loading volumes requires SimpleITK or nibabel; install one of them"
        ) from exc

    img = nib.load(str(path))
    array = np.asanyarray(img.dataobj)
    zooms = img.header.get_zooms()[:3]
    return array, tuple(float(z) for z in zooms)  # type: ignore[return-value]


class _BaseKneeDataset:
    """Shared preprocessing, metadata joining and guard enforcement."""

    def __init__(
        self,
        metadata: pd.DataFrame,
        *,
        target_spacing_mm: float = 0.5,
        crop_size: tuple[int, int, int] = (160, 160, 96),
        clip_percentiles: tuple[float, float] = (0.5, 99.5),
        apply_n4: bool = False,
        preprocess: bool = True,
        validate_labels: bool = False,
        splits: dict[str, list[str]] | None = None,
    ) -> None:
        self.metadata = metadata.set_index(metadata["subject_id"].astype(str), drop=False)
        self.target_spacing_mm = target_spacing_mm
        self.crop_size = crop_size
        self.clip_percentiles = clip_percentiles
        self.apply_n4 = apply_n4
        self.preprocess = preprocess
        self.validate_labels = validate_labels
        self.splits = splits

    def _meta_for(self, subject_id: str) -> dict[str, Any]:
        """Look up a subject's stratifiers, with explicit unknowns."""
        if subject_id not in self.metadata.index:
            raise KeyError(
                f"subject {subject_id!r} has no metadata row. Every case must carry a "
                "KL grade and site or it cannot be stratified, split, or used for "
                "conditional coverage."
            )
        row = self.metadata.loc[subject_id]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return {
            "kl_grade": int(row["kl_grade"]),
            "site": str(row["site"]),
            "laterality": str(row.get("laterality", "unknown")),
            "visit": str(row.get("visit", "unknown")),
        }

    def _finalise(
        self,
        subject_id: str,
        image: np.ndarray,
        label: np.ndarray | None,
        spacing: tuple[float, float, float],
    ) -> dict[str, Any]:
        """Preprocess one case and assemble the output dict."""
        if self.splits is not None:
            check_subject_access([subject_id], self.splits)

        if self.validate_labels and label is not None:
            validate_label_convention(label).raise_if_bad()

        meta = None
        if self.preprocess:
            image, label, meta = preprocess_case(
                image,
                label,
                spacing,
                target_spacing_mm=self.target_spacing_mm,
                crop_size=self.crop_size,
                clip_percentiles=self.clip_percentiles,
                apply_n4=self.apply_n4,
            )
            out_spacing = (self.target_spacing_mm,) * 3
        else:
            out_spacing = spacing

        item: dict[str, Any] = {
            "image": np.ascontiguousarray(image, dtype=np.float32)[None],  # add channel
            "label": (
                np.ascontiguousarray(label, dtype=np.int64)[None] if label is not None else None
            ),
            "subject_id": subject_id,
            "spacing": out_spacing,
            "preprocess_meta": meta,
        }
        item.update(self._meta_for(subject_id))
        return item


class OAIZIBCMDataset(_BaseKneeDataset):
    """OAIZIB-CM cases read from disk.

    Parameters
    ----------
    cases
        Records from :func:`discover_cases`, optionally filtered to one split.
    metadata
        Canonical metadata table.
    **kwargs
        Forwarded to the preprocessing chain; see :class:`_BaseKneeDataset`.

    Examples
    --------
    >>> ds = OAIZIBCMDataset(cases, metadata)           # doctest: +SKIP
    >>> item = ds[0]                                     # doctest: +SKIP
    >>> item["image"].shape                              # doctest: +SKIP
    (1, 160, 160, 96)
    """

    def __init__(self, cases: Sequence[CaseRecord], metadata: pd.DataFrame, **kwargs: Any) -> None:
        super().__init__(metadata, **kwargs)
        self.cases = list(cases)

    def __len__(self) -> int:
        return len(self.cases)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.cases[index]
        image, spacing = load_volume(record.image_path)
        label = None
        if record.label_path is not None:
            label, label_spacing = load_volume(record.label_path)
            if not np.allclose(label_spacing, spacing, atol=1e-3):
                raise ValueError(
                    f"{record.subject_id}: image spacing {spacing} != label spacing "
                    f"{label_spacing}. Resampling them independently would misalign "
                    "the cartilage plate from the bone it sits on."
                )
        return self._finalise(record.subject_id, image, label, spacing)


class PhantomDataset(_BaseKneeDataset):
    """In-memory phantom knees, for tests and for running with no OAI data.

    Parameters
    ----------
    knees
        Phantoms from :func:`confcarti.data.synthetic.make_phantom_cohort`.
    metadata
        Matching metadata table; row order must correspond to ``knees``.
    **kwargs
        Forwarded to the preprocessing chain.

    Examples
    --------
    >>> from confcarti.data.synthetic import make_phantom_cohort
    >>> knees, meta = make_phantom_cohort(4, seed=0)
    >>> ds = PhantomDataset(knees, meta, crop_size=(64, 64, 48))
    >>> item = ds[0]
    >>> item["image"].shape
    (1, 64, 64, 48)
    >>> item["kl_grade"] in (0, 1, 2, 3, 4)
    True
    """

    def __init__(self, knees: Sequence[Any], metadata: pd.DataFrame, **kwargs: Any) -> None:
        super().__init__(metadata, **kwargs)
        if len(knees) != len(metadata):
            raise ValueError(
                f"got {len(knees)} phantoms but {len(metadata)} metadata rows; they must "
                "correspond one-to-one and in order"
            )
        # NOT list(knees): a real cohort is a lazy sequence that reads each
        # volume from disk on access, and list() would pull all 481 of them
        # into RAM at construction time.
        self.knees = knees
        self.subject_ids = metadata["subject_id"].astype(str).tolist()

    def __len__(self) -> int:
        return len(self.knees)

    def __getitem__(self, index: int) -> dict[str, Any]:
        knee = self.knees[index]
        return self._finalise(
            self.subject_ids[index], knee.image, knee.label, knee.spacing
        )


def build_dataset(
    source: Sequence[CaseRecord] | Sequence[Any],
    metadata: pd.DataFrame,
    *,
    subject_ids: Sequence[str] | None = None,
    cache_mode: str = "none",
    cache_rate: float = 1.0,
    **kwargs: Any,
) -> Any:
    """Build a dataset, optionally filtered to a subject list and MONAI-cached.

    Parameters
    ----------
    source
        Either ``CaseRecord``s (disk) or ``PhantomKnee``s (in memory).
    metadata
        Canonical metadata table.
    subject_ids
        Restrict to these subjects, e.g. one split. Order follows ``source``.
    cache_mode
        ``none``, ``cache`` (MONAI ``CacheDataset``) or ``smart``
        (``SmartCacheDataset``).
    cache_rate
        Fraction held in RAM when caching.
    **kwargs
        Forwarded to the dataset class.

    Returns
    -------
    object
        A dataset supporting ``len`` and integer indexing.

    Raises
    ------
    ValueError
        For an unknown ``cache_mode``, or when filtering leaves no cases.
    ImportError
        If a MONAI cache mode is requested but MONAI is not installed.
    """
    if cache_mode not in {"none", "cache", "smart"}:
        raise ValueError(f"cache_mode must be none|cache|smart, got {cache_mode!r}")

    is_phantom = bool(source) and isinstance(source[0], PhantomKnee)

    if subject_ids is not None:
        wanted = {str(s) for s in subject_ids}
        if is_phantom:
            keep = [
                i for i, s in enumerate(metadata["subject_id"].astype(str)) if s in wanted
            ]
            source = [source[i] for i in keep]
            metadata_filtered = metadata.iloc[keep].reset_index(drop=True)
        else:
            source = [r for r in source if r.subject_id in wanted]  # type: ignore[union-attr]
            metadata_filtered = metadata
        if not source:
            raise ValueError(
                "filtering by subject_ids left no cases; check that the split file and "
                "the data directory refer to the same cohort"
            )
    else:
        metadata_filtered = metadata

    dataset: Any
    if is_phantom:
        dataset = PhantomDataset(source, metadata_filtered, **kwargs)
    else:
        dataset = OAIZIBCMDataset(source, metadata, **kwargs)  # type: ignore[arg-type]

    if cache_mode == "none":
        return dataset

    try:
        from monai.data import CacheDataset, SmartCacheDataset
    except ImportError as exc:  # pragma: no cover - depends on environment
        raise ImportError(
            f"cache_mode={cache_mode!r} requires MONAI; install confcarti[dl] or use "
            "cache_mode='none'"
        ) from exc

    items = [dataset[i] for i in range(len(dataset))]
    if cache_mode == "cache":
        return CacheDataset(data=items, transform=None, cache_rate=cache_rate)
    return SmartCacheDataset(data=items, transform=None, cache_rate=cache_rate)

### Surface extraction: BCI and articular surface

`confcarti/thickness/mesh.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/thickness/mesh.py
# ==========================================================================
"""Surface extraction: bone-cartilage interface and articular surface.

Both surfaces are extracted with marching cubes on a lightly Gaussian-smoothed
binary mask, then low-pass filtered with the Taubin lambda/mu filter. The
pre-smoothing matters more than it looks: raw marching cubes on a binary mask
produces a staircase whose curvature is dominated by the voxel grid, not by
anatomy, which would make the curvature analysis in
:mod:`confcarti.thickness.curvature` measure the sampling lattice instead of the
joint.

References
----------
Taubin, G. (1995). "Curve and surface smoothing without shrinkage." ICCV.
    The lambda/mu filter: a shrinking pass with weight ``lambda > 0`` followed by
    an un-shrinking pass with ``mu < -lambda``. The pair has a low-pass transfer
    function with unit gain at zero frequency, so volume is preserved.
Lorensen & Cline (1987). "Marching cubes." SIGGRAPH.
Yao et al. (2024). CartiMorph, Medical Image Analysis 91:103035 -- the
    bone-cartilage-interface definition reproduced here.
"""


import logging
from dataclasses import dataclass
from typing import Any

import numpy as np
import trimesh
from scipy import ndimage
from skimage import measure

logger = logging.getLogger(__name__)

__all__ = [
    "MeshExtractionReport",
    "taubin_smooth",
    "mask_to_mesh",
    "extract_bci_mesh",
    "extract_articular_surface",
    "extract_compartment_surfaces",
]


@dataclass(frozen=True, slots=True)
class MeshExtractionReport:
    """Quality record for one extracted surface.

    Attributes
    ----------
    n_vertices, n_faces
        Size of the returned mesh.
    n_components_found, n_components_kept
        Connected components before and after size filtering.
    is_watertight
        Whether the mesh bounds a closed volume. An open surface patch (which
        the BCI is, by construction) is not watertight -- that is expected.
    is_winding_consistent
        Whether face winding is consistent, so vertex normals point coherently.
        Required for surface-normal thickness to have a well-defined direction.
    euler_number
        Euler characteristic, a cheap topology sanity check.
    surface_area_mm2
        Total area of the returned faces.
    """

    n_vertices: int
    n_faces: int
    n_components_found: int
    n_components_kept: int
    is_watertight: bool
    is_winding_consistent: bool
    euler_number: int
    surface_area_mm2: float


def _clean_mesh(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    """Drop degenerate and duplicate faces, then unreferenced vertices, in place.

    trimesh renamed these operations between major versions (``remove_*`` methods
    became boolean-mask properties fed to ``update_faces``). Both spellings are
    handled so the package works across trimesh 3.x and 4.x rather than failing
    on an attribute that quietly disappeared.
    """
    if hasattr(mesh, "nondegenerate_faces"):
        mesh.update_faces(mesh.nondegenerate_faces())
        mesh.update_faces(mesh.unique_faces())
    else:  # pragma: no cover - trimesh < 4
        mesh.remove_degenerate_faces()
        mesh.remove_duplicate_faces()
    mesh.remove_unreferenced_vertices()
    return mesh


def _umbrella_operator(mesh: trimesh.Trimesh) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Flattened uniform-weight (umbrella) Laplacian adjacency.

    Returns
    -------
    tuple
        ``(neighbour_index, owner_index, valence)`` such that
        ``L(v)_i = mean_j v_j - v_i`` can be evaluated with one ``np.add.at``.
    """
    neighbours = mesh.vertex_neighbors
    valence = np.fromiter(
        (len(nb) for nb in neighbours), dtype=np.int64, count=len(neighbours)
    )
    if valence.sum() == 0:
        empty = np.empty(0, dtype=np.int64)
        return empty, empty, valence
    flat = np.concatenate([np.asarray(nb, dtype=np.int64) for nb in neighbours])
    owner = np.repeat(np.arange(len(neighbours), dtype=np.int64), valence)
    return flat, owner, valence


def taubin_smooth(
    mesh: trimesh.Trimesh,
    *,
    iterations: int = 10,
    lamb: float = 0.5,
    mu: float = -0.53,
) -> trimesh.Trimesh:
    """Apply Taubin lambda/mu smoothing, returning a smoothed copy.

    Each iteration is one shrink/un-shrink pair on the uniform (umbrella)
    Laplacian ``L(v)_i = mean_{j in N(i)} v_j - v_i``:

    .. math::
        v \\leftarrow v + \\lambda L(v), \\qquad v \\leftarrow v + \\mu L(v)

    whose per-pair transfer function is :math:`f(k) = (1 - \\lambda k)(1 - \\mu k)`
    on the Laplacian eigenvalues :math:`k \\in [0, 2]`. With ``mu < -lambda`` the
    pass-band frequency :math:`k_{PB} = 1/\\lambda + 1/\\mu` is positive,
    ``f(0) = 1``, and high frequencies are attenuated -- so the surface is
    low-pass filtered without the monotone shrinkage of plain Laplacian
    smoothing.

    Parameters
    ----------
    mesh
        Input mesh.
    iterations
        Number of lambda/mu pairs.
    lamb
        Shrinking weight, ``0 < lambda < 1``.
    mu
        Un-shrinking weight, must satisfy ``mu < -lambda``.

    Returns
    -------
    trimesh.Trimesh
        Smoothed copy with the same vertex indexing, so per-vertex fields (and
        the bulge correspondence in :mod:`confcarti.thickness.bulge`) stay valid.

    Raises
    ------
    ValueError
        If ``mu >= -lambda``, which would shrink the surface monotonically and
        systematically under-estimate cartilage volume.

    Notes
    -----
    The filter is implemented here rather than delegated to
    ``trimesh.smoothing.filter_taubin`` because that function's ``nu`` parameter
    does not follow Taubin's sign convention: passing ``nu = |mu|`` leaves
    high-frequency detail essentially untouched (a 1.5 mm bump on a sphere grew
    to 1.58 mm after 80 iterations instead of being removed), which silently
    breaks any measurement defined as a deviation from a smoothed reference.

    Examples
    --------
    Volume is preserved on a noisy sphere:

    >>> sphere = trimesh.creation.icosphere(subdivisions=3, radius=10.0)
    >>> noisy = sphere.copy()
    >>> rng = np.random.default_rng(0)
    >>> noisy.vertices = noisy.vertices + rng.normal(0, 0.15, noisy.vertices.shape)
    >>> out = taubin_smooth(noisy, iterations=20)
    >>> bool(abs(out.volume - sphere.volume) / sphere.volume < 0.05)
    True

    and a localised bump really is removed:

    >>> v = np.asarray(sphere.vertices)
    >>> unit = v / np.linalg.norm(v, axis=1, keepdims=True)
    >>> w = np.exp(-((unit - np.array([0.0, 0.0, 1.0])) ** 2).sum(axis=1) / 0.1)
    >>> bumped = sphere.copy()
    >>> bumped.vertices = v + unit * (1.5 * w)[:, None]
    >>> ref = taubin_smooth(bumped, iterations=60)
    >>> peak = np.linalg.norm(np.asarray(ref.vertices), axis=1).max() - 10.0
    >>> bool(peak < 0.6)
    True
    """
    if mu >= -lamb:
        raise ValueError(
            f"Taubin smoothing needs mu < -lambda to be shrinkage-free; got "
            f"lambda={lamb}, mu={mu}"
        )
    if iterations <= 0:
        return mesh.copy()

    out = mesh.copy()
    vertices = np.asarray(out.vertices, dtype=np.float64).copy()
    flat, owner, valence = _umbrella_operator(out)
    if flat.size == 0:
        return out

    safe_valence = np.maximum(valence, 1)[:, None].astype(np.float64)
    isolated = (valence == 0)[:, None]

    def _laplacian(points: np.ndarray) -> np.ndarray:
        totals = np.zeros_like(points)
        np.add.at(totals, owner, points[flat])
        centroid = totals / safe_valence
        step = centroid - points
        return np.where(isolated, 0.0, step)

    for _ in range(iterations):
        vertices = vertices + lamb * _laplacian(vertices)
        vertices = vertices + mu * _laplacian(vertices)

    if not np.isfinite(vertices).all():
        raise RuntimeError(
            "Taubin smoothing produced non-finite vertices; the input mesh is "
            "probably degenerate (zero-area faces or duplicate vertices)"
        )

    out.vertices = vertices
    return out


def _largest_components(
    mesh: trimesh.Trimesh, min_vertices: int
) -> tuple[trimesh.Trimesh, int, int]:
    """Drop connected components smaller than ``min_vertices``."""
    components = mesh.split(only_watertight=False)
    if len(components) <= 1:
        return mesh, max(len(components), 1), max(len(components), 1)

    kept = [c for c in components if len(c.vertices) >= min_vertices]
    if not kept:
        # Never return an empty mesh silently: keep the biggest piece and say so.
        biggest = max(components, key=lambda c: len(c.vertices))
        logger.warning(
            "every connected component is smaller than min_vertices=%d; keeping the "
            "largest (%d vertices) rather than returning an empty surface",
            min_vertices,
            len(biggest.vertices),
        )
        return biggest, len(components), 1

    if len(kept) < len(components):
        logger.info(
            "dropped %d connected component(s) below %d vertices",
            len(components) - len(kept),
            min_vertices,
        )
    merged = trimesh.util.concatenate(kept) if len(kept) > 1 else kept[0]
    return merged, len(components), len(kept)


def mask_to_mesh(
    mask: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    gaussian_sigma_mm: float = 0.4,
    level: float = 0.5,
    smoothing_iterations: int = 10,
    taubin_lambda: float = 0.5,
    taubin_mu: float = -0.53,
    min_component_vertices: int = 50,
) -> tuple[trimesh.Trimesh, MeshExtractionReport]:
    """Convert a binary mask to a smooth triangulated surface in millimetres.

    Parameters
    ----------
    mask
        Boolean or 0/1 volume.
    spacing
        Voxel size in mm, matching the array axes.
    gaussian_sigma_mm
        Pre-smoothing of the binary mask, in mm. 0 disables it.
    level
        Marching-cubes iso-level.
    smoothing_iterations, taubin_lambda, taubin_mu
        Taubin filter parameters.
    min_component_vertices
        Components below this are dropped.

    Returns
    -------
    tuple
        ``(mesh, report)``. Mesh vertices are in millimetres.

    Raises
    ------
    ValueError
        If the mask is empty or too thin for marching cubes to find a surface.

    Examples
    --------
    >>> vol = np.zeros((40, 40, 40))
    >>> zz, yy, xx = np.mgrid[:40, :40, :40]
    >>> vol[((xx - 20)**2 + (yy - 20)**2 + (zz - 20)**2) < 12**2] = 1
    >>> mesh, rep = mask_to_mesh(vol, (0.5, 0.5, 0.5))
    >>> bool(abs(mesh.area - 4 * np.pi * 6.0**2) / (4 * np.pi * 6.0**2) < 0.08)
    True
    """
    mask = np.asarray(mask)
    if mask.ndim != 3:
        raise ValueError(f"mask must be 3D, got shape {mask.shape}")
    binary = (mask > 0).astype(np.float32)
    if not binary.any():
        raise ValueError(
            "cannot extract a surface from an empty mask. If this is a cartilage "
            "plate, the segmentation lost the structure entirely -- that is a "
            "result to report, not a case to skip silently."
        )

    field = binary
    if gaussian_sigma_mm > 0:
        sigma_vox = [gaussian_sigma_mm / s for s in spacing]
        field = ndimage.gaussian_filter(binary, sigma=sigma_vox, mode="constant", cval=0.0)
        # Pre-smoothing can pull the peak below the iso-level for structures only
        # one or two voxels thick, which would make marching cubes return nothing.
        if field.max() <= level:
            logger.warning(
                "Gaussian pre-smoothing (sigma=%.2f mm) flattened the mask below the "
                "iso-level; falling back to the unsmoothed mask. The structure is "
                "thinner than the smoothing kernel.",
                gaussian_sigma_mm,
            )
            field = binary

    # Pad so structures touching the volume boundary still close properly.
    field = np.pad(field, 1, mode="constant", constant_values=0.0)

    try:
        verts, faces, _normals, _values = measure.marching_cubes(
            field, level=level, spacing=spacing
        )
    except (ValueError, RuntimeError) as exc:
        raise ValueError(f"marching cubes failed on this mask: {exc}") from exc

    # Undo the padding offset so coordinates stay in the original frame.
    verts = verts - np.asarray(spacing, dtype=float)

    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
    _clean_mesh(mesh)

    mesh, found, kept = _largest_components(mesh, min_component_vertices)

    if smoothing_iterations > 0:
        mesh = taubin_smooth(
            mesh, iterations=smoothing_iterations, lamb=taubin_lambda, mu=taubin_mu
        )

    mesh.fix_normals()

    report = MeshExtractionReport(
        n_vertices=int(len(mesh.vertices)),
        n_faces=int(len(mesh.faces)),
        n_components_found=found,
        n_components_kept=kept,
        is_watertight=bool(mesh.is_watertight),
        is_winding_consistent=bool(mesh.is_winding_consistent),
        euler_number=int(mesh.euler_number),
        surface_area_mm2=float(mesh.area),
    )
    return mesh, report


def _proximity_mask(
    target: np.ndarray, spacing: tuple[float, float, float], distance_mm: float
) -> np.ndarray:
    """Voxels within ``distance_mm`` of ``target``, by Euclidean distance transform."""
    if not target.any():
        return np.zeros_like(target, dtype=bool)
    distance = ndimage.distance_transform_edt(~target.astype(bool), sampling=spacing)
    return distance <= distance_mm


def _close_mask(
    mask: np.ndarray, spacing: tuple[float, float, float], radius_mm: float
) -> np.ndarray:
    """Binary closing with a ball of ``radius_mm``, using distance transforms.

    Dilate-then-erode expressed through Euclidean distance transforms, which is
    both exact for a spherical structuring element and far cheaper than building
    one explicitly at fine voxel sizes.

    Parameters
    ----------
    mask
        Binary mask.
    spacing
        Voxel size in mm.
    radius_mm
        Closing radius.

    Returns
    -------
    numpy.ndarray
        Closed boolean mask, always a superset of the input.
    """
    mask = np.asarray(mask) > 0
    if radius_mm <= 0 or not mask.any():
        return mask

    dilated = ndimage.distance_transform_edt(~mask, sampling=spacing) <= radius_mm
    # Erode the dilated set by the same radius: distance from its complement.
    eroded = ndimage.distance_transform_edt(dilated, sampling=spacing) > radius_mm
    # Closing is extensive; guarantee it even against discretisation effects.
    return eroded | mask


def _select_faces_near(
    mesh: trimesh.Trimesh,
    proximity: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    min_vertices_per_face: int = 2,
) -> np.ndarray:
    """Boolean face mask: faces with enough vertices inside ``proximity``.

    Vertex millimetre coordinates are converted back to voxel indices to look up
    the proximity volume.
    """
    verts_vox = np.rint(mesh.vertices / np.asarray(spacing)).astype(int)
    shape = np.asarray(proximity.shape)
    verts_vox = np.clip(verts_vox, 0, shape - 1)
    vertex_inside = proximity[verts_vox[:, 0], verts_vox[:, 1], verts_vox[:, 2]]
    return vertex_inside[mesh.faces].sum(axis=1) >= min_vertices_per_face


def _submesh(mesh: trimesh.Trimesh, face_mask: np.ndarray) -> trimesh.Trimesh:
    """Extract the sub-mesh defined by a boolean face mask, re-indexing vertices."""
    if not face_mask.any():
        raise ValueError("face selection is empty")
    faces = mesh.faces[face_mask]
    used = np.unique(faces)
    remap = np.full(len(mesh.vertices), -1, dtype=np.int64)
    remap[used] = np.arange(len(used))
    return trimesh.Trimesh(
        vertices=mesh.vertices[used], faces=remap[faces], process=False
    )


def extract_bci_mesh(
    bone_mask: np.ndarray,
    cartilage_mask: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    interface_distance_mm: float = 1.5,
    closing_radius_mm: float = 10.0,
    plate_band_mm: float = 8.0,
    gaussian_sigma_mm: float = 0.4,
    smoothing_iterations: int = 10,
    taubin_lambda: float = 0.5,
    taubin_mu: float = -0.53,
    min_component_vertices: int = 50,
) -> tuple[trimesh.Trimesh, MeshExtractionReport]:
    """Extract the bone-cartilage interface: the TOTAL subchondral bone area.

    The BCI is the subchondral bone surface underneath the cartilage plate. It
    is the reference surface for every thickness measurement, and its area is
    the total subchondral bone area (tAB) that ThCtAB averages over.

    Why the closing step is not optional
    ------------------------------------
    Selecting bone faces that lie near *observed* cartilage gives the
    cartilage-covered area (cAB), not tAB. Where cartilage is completely lost
    there is no cartilage to be near, so those faces are excluded -- and the
    denuded area silently vanishes from the surface it is supposed to be
    measured on. Left uncorrected this pins ``dAB`` at exactly 0 for every knee,
    at every KL grade, which looks like a clean result and is entirely an
    artefact.

    The fix, following CartiMorph's surface-closing operator
    (``CM_cal_surfaceClosing.m``), is to morphologically close the cartilage
    mask before the proximity test, so a full-thickness defect is bridged and
    the bone beneath it is retained in the interface. Thickness there then
    measures ~0 and the vertex is correctly labelled denuded.

    Parameters
    ----------
    bone_mask
        Binary mask of the bone.
    cartilage_mask
        Binary mask of the cartilage plate sitting on that bone.
    spacing
        Voxel size in mm.
    interface_distance_mm
        A bone-surface face counts as interface when its vertices lie within
        this distance of cartilage. Roughly one to two voxels; too small and the
        interface fragments, too large and it creeps up the bone shaft.
    closing_radius_mm
        Radius of the binary closing applied to the cartilage mask before the
        proximity test. This is the largest full-thickness defect that will be
        bridged and therefore counted in tAB. Set to 0 to disable, which yields
        cAB instead of tAB and makes ``dAB`` identically zero.
    gaussian_sigma_mm, smoothing_iterations, taubin_lambda, taubin_mu, min_component_vertices
        Forwarded to :func:`mask_to_mesh`.

    Returns
    -------
    tuple
        ``(bci_mesh, report)``.

    Raises
    ------
    ValueError
        If either mask is empty, or if no bone surface lies near cartilage.

    Examples
    --------
    >>> from confcarti.data.synthetic import make_phantom, PhantomSpec
    >>> from confcarti.data.labels import FEMUR, FEMORAL_CARTILAGE
    >>> knee = make_phantom(PhantomSpec(shape=(64, 64, 48)))
    >>> bci, rep = extract_bci_mesh(knee.label == FEMUR,
    ...                             knee.label == FEMORAL_CARTILAGE,
    ...                             knee.spacing)
    >>> rep.n_vertices > 100
    True
    """
    bone_mask = np.asarray(bone_mask) > 0
    cartilage_mask = np.asarray(cartilage_mask) > 0
    if not bone_mask.any():
        raise ValueError("bone mask is empty; cannot extract a bone-cartilage interface")
    if not cartilage_mask.any():
        raise ValueError(
            "cartilage mask is empty; the bone-cartilage interface is undefined. "
            "A knee with no cartilage at all is a segmentation failure, not a "
            "100% denuded knee -- handle it upstream."
        )

    bone_mesh, _ = mask_to_mesh(
        bone_mask,
        spacing,
        gaussian_sigma_mm=gaussian_sigma_mm,
        smoothing_iterations=smoothing_iterations,
        taubin_lambda=taubin_lambda,
        taubin_mu=taubin_mu,
        min_component_vertices=min_component_vertices,
    )

    # Close full-thickness defects so the bone beneath them stays in tAB.
    #
    # The closing is confined to a thin band immediately above the bone. Without
    # that constraint a closing radius large enough to bridge a real defect
    # (~10 mm) also floods the concave side of the condyle, because the
    # cartilage plate is a thin curved shell rather than a convex solid: the
    # dilation joins opposite walls across the joint space and the erosion never
    # recovers them. Restricting to the band means the closing can only ever add
    # voxels where cartilage could plausibly sit.
    reference_mask = cartilage_mask
    if closing_radius_mm > 0:
        closed = _close_mask(cartilage_mask, spacing, closing_radius_mm)
        distance_to_bone = ndimage.distance_transform_edt(~bone_mask, sampling=spacing)
        band = (distance_to_bone <= plate_band_mm) & ~bone_mask
        reference_mask = (closed & band) | cartilage_mask
        bridged = int(reference_mask.sum() - cartilage_mask.sum())
        if bridged > 0:
            logger.debug(
                "surface closing bridged %d voxel(s) of full-thickness defect "
                "(radius %.1f mm); these become denuded bone in tAB",
                bridged,
                closing_radius_mm,
            )

    proximity = _proximity_mask(reference_mask, spacing, interface_distance_mm)
    face_mask = _select_faces_near(bone_mesh, proximity, spacing)
    if not face_mask.any():
        raise ValueError(
            f"no bone surface lies within {interface_distance_mm} mm of cartilage. "
            "Either the two masks are misaligned, or interface_distance_mm is too small "
            "for this voxel size."
        )

    bci = _submesh(bone_mesh, face_mask)
    bci, found, kept = _largest_components(bci, min_component_vertices)
    bci.remove_unreferenced_vertices()

    report = MeshExtractionReport(
        n_vertices=int(len(bci.vertices)),
        n_faces=int(len(bci.faces)),
        n_components_found=found,
        n_components_kept=kept,
        is_watertight=bool(bci.is_watertight),
        is_winding_consistent=bool(bci.is_winding_consistent),
        euler_number=int(bci.euler_number),
        surface_area_mm2=float(bci.area),
    )
    logger.debug(
        "BCI: %d vertices, %d faces, area %.1f mm^2",
        report.n_vertices,
        report.n_faces,
        report.surface_area_mm2,
    )
    return bci, report


def extract_articular_surface(
    cartilage_mask: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    bone_mask: np.ndarray | None = None,
    interface_distance_mm: float = 1.0,
    gaussian_sigma_mm: float = 0.4,
    smoothing_iterations: int = 10,
    taubin_lambda: float = 0.5,
    taubin_mu: float = -0.53,
    min_component_vertices: int = 50,
) -> tuple[trimesh.Trimesh, MeshExtractionReport]:
    """Extract the outer (articulating) cartilage surface.

    The full cartilage boundary comprises the articular surface *and* the face
    against bone. When ``bone_mask`` is given, the bone-facing portion is
    removed so that ray casts from the BCI cannot hit the surface they started
    from and report a thickness of ~0.

    Parameters
    ----------
    cartilage_mask
        Binary cartilage mask.
    spacing
        Voxel size in mm.
    bone_mask
        Binary bone mask; when supplied, faces within
        ``interface_distance_mm`` of bone are discarded.
    interface_distance_mm
        Distance threshold for the bone-facing portion.
    gaussian_sigma_mm, smoothing_iterations, taubin_lambda, taubin_mu, min_component_vertices
        Forwarded to :func:`mask_to_mesh`.

    Returns
    -------
    tuple
        ``(articular_mesh, report)``.
    """
    cartilage_mask = np.asarray(cartilage_mask) > 0
    mesh, report = mask_to_mesh(
        cartilage_mask,
        spacing,
        gaussian_sigma_mm=gaussian_sigma_mm,
        smoothing_iterations=smoothing_iterations,
        taubin_lambda=taubin_lambda,
        taubin_mu=taubin_mu,
        min_component_vertices=min_component_vertices,
    )

    if bone_mask is None:
        return mesh, report

    bone_mask = np.asarray(bone_mask) > 0
    if not bone_mask.any():
        return mesh, report

    near_bone = _proximity_mask(bone_mask, spacing, interface_distance_mm)
    face_near_bone = _select_faces_near(mesh, near_bone, spacing, min_vertices_per_face=3)
    keep = ~face_near_bone
    if not keep.any():
        logger.warning(
            "every cartilage face is within %.2f mm of bone; keeping the full surface. "
            "For a plate this thin the articular and bone-facing surfaces are not "
            "separable at this resolution.",
            interface_distance_mm,
        )
        return mesh, report

    articular = _submesh(mesh, keep)
    articular, found, kept = _largest_components(articular, min_component_vertices)
    articular.remove_unreferenced_vertices()

    report = MeshExtractionReport(
        n_vertices=int(len(articular.vertices)),
        n_faces=int(len(articular.faces)),
        n_components_found=found,
        n_components_kept=kept,
        is_watertight=bool(articular.is_watertight),
        is_winding_consistent=bool(articular.is_winding_consistent),
        euler_number=int(articular.euler_number),
        surface_area_mm2=float(articular.area),
    )
    return articular, report


def extract_compartment_surfaces(
    label: np.ndarray,
    spacing: tuple[float, float, float],
    compartment_key: str,
    *,
    config: Any | None = None,
) -> dict[str, Any]:
    """Extract BCI and articular surfaces for one cartilage compartment.

    Parameters
    ----------
    label
        5-ROI label volume.
    spacing
        Voxel size in mm.
    compartment_key
        ``FC``, ``MTC`` or ``LTC``.
    config
        Optional :class:`confcarti.config.ThicknessConfig` supplying the
        smoothing and component parameters.

    Returns
    -------
    dict
        ``bci``, ``articular``, ``bci_report``, ``articular_report``,
        ``cartilage_mask`` and ``bone_mask``.

    Raises
    ------
    KeyError
        For an unknown compartment key.
    """
    if compartment_key not in COMPARTMENTS:
        raise KeyError(
            f"unknown compartment {compartment_key!r}; expected one of "
            f"{sorted(COMPARTMENTS)}"
        )
    comp = COMPARTMENTS[compartment_key]

    kwargs: dict[str, Any] = {}
    if config is not None:
        kwargs = {
            "gaussian_sigma_mm": config.mask_gaussian_sigma_mm,
            "smoothing_iterations": config.smoothing_iterations,
            "taubin_lambda": config.taubin_lambda,
            "taubin_mu": config.taubin_mu,
            "min_component_vertices": config.min_component_vertices,
        }

    label = np.asarray(label)
    cartilage_mask = label == comp.cartilage_label
    bone_mask = label == comp.bone_label

    bci, bci_report = extract_bci_mesh(bone_mask, cartilage_mask, spacing, **kwargs)
    articular, art_report = extract_articular_surface(
        cartilage_mask, spacing, bone_mask=bone_mask, **kwargs
    )

    return {
        "compartment": compartment_key,
        "bci": bci,
        "articular": articular,
        "bci_report": bci_report,
        "articular_report": art_report,
        "cartilage_mask": cartilage_mask,
        "bone_mask": bone_mask,
    }

### Cartilage thickness -- three estimators

`confcarti/thickness/thickness.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/thickness/thickness.py
# ==========================================================================
"""Cartilage thickness at every bone-cartilage-interface vertex.

Three interchangeable estimators sit behind one signature. They are genuinely
different measurements, not three implementations of one -- their disagreement
is reported as ablation A1 rather than tuned away.

``surface_normal`` (primary)
    Port of CartiMorph's ``CM_cal_estimateSN.m`` / ``CM_cal_smoothSN.m`` /
    ``CM_cal_thicknessMap_SN.m``. Surface normals are estimated by total least
    squares over a k-nearest-neighbour patch (the smallest right singular vector
    of the mean-centred neighbourhood is the plane normal), smoothed, oriented
    into the cartilage, and then intersected with the articular surface. The hit
    distance is the thickness.

``distance_transform``
    Marches the Euclidean distance transform of the cartilage mask along the
    same normals and doubles its maximum. Exact for a slab; robust where the
    articular surface is fragmented.

``nearest_neighbour``
    Distance to the closest point on the articular surface. Always defined, but
    systematically under-estimates on curved surfaces because the closest point
    need not lie along the normal.

The NaN convention
------------------
``NaN`` means *undefined* -- the ray left the volume, missed the surface, or ran
past the depth cap. ``0.0`` means *denuded*: bone genuinely uncovered by
cartilage. CartiMorph's MATLAB writes ``0`` for both; conflating them is the
single most common thickness bug, and it biases ThCtAB downward exactly in the
severe-OA knees this project cares about, so the two are kept apart here.

References
----------
Yao et al. (2024). "CartiMorph: a framework for automated knee articular
    cartilage morphometrics." Medical Image Analysis 91:103035.
Maier et al. (2017) / Eckstein et al. (2006) for the ThCtAB / ThCcAB convention.
"""


import logging
from dataclasses import dataclass
from typing import Literal

import numpy as np
import trimesh
from scipy import ndimage
from scipy.spatial import cKDTree

logger = logging.getLogger(__name__)

__all__ = [
    "ThicknessResult",
    "estimate_surface_normals",
    "smooth_normals",
    "orient_normals_into",
    "compute_thickness",
    "thickness_surface_normal",
    "thickness_distance_transform",
    "thickness_nearest_neighbour",
]

ThicknessMethod = Literal["surface_normal", "distance_transform", "nearest_neighbour"]


@dataclass(frozen=True, slots=True)
class ThicknessResult:
    """Per-vertex thickness plus provenance.

    Attributes
    ----------
    thickness_mm
        Per-BCI-vertex thickness in mm. NaN = undefined, 0.0 = denuded.
    method
        Estimator used.
    normals
        Oriented unit normals at the BCI vertices, when the method uses them.
    n_undefined
        Vertices whose thickness is NaN.
    n_capped
        Vertices whose ray exceeded the depth cap and were rejected.
    """

    thickness_mm: np.ndarray
    method: str
    normals: np.ndarray | None = None
    n_undefined: int = 0
    n_capped: int = 0

    @property
    def n_vertices(self) -> int:
        """Number of BCI vertices."""
        return int(self.thickness_mm.shape[0])

    @property
    def defined_fraction(self) -> float:
        """Fraction of vertices with a defined (non-NaN) thickness."""
        if self.n_vertices == 0:
            return 0.0
        return float(np.isfinite(self.thickness_mm).mean())

    def summary(self) -> dict[str, float]:
        """Mean/median/percentile summary over the defined vertices."""
        finite = self.thickness_mm[np.isfinite(self.thickness_mm)]
        if finite.size == 0:
            return {
                "mean_mm": float("nan"),
                "median_mm": float("nan"),
                "p05_mm": float("nan"),
                "p95_mm": float("nan"),
                "defined_fraction": 0.0,
            }
        return {
            "mean_mm": float(finite.mean()),
            "median_mm": float(np.median(finite)),
            "p05_mm": float(np.percentile(finite, 5)),
            "p95_mm": float(np.percentile(finite, 95)),
            "defined_fraction": self.defined_fraction,
        }


def estimate_surface_normals(points: np.ndarray, n_neighbours: int = 30) -> np.ndarray:
    """Estimate unit surface normals by local total least squares.

    Direct port of CartiMorph's ``CM_cal_estimateSN.m``: for each point, take its
    ``k`` nearest neighbours, mean-centre them, and take the right singular
    vector of the smallest singular value. That vector is the normal of the
    best-fitting plane in the least-squares sense.

    Parameters
    ----------
    points
        ``(n, 3)`` coordinates in mm.
    n_neighbours
        Neighbourhood size ``k``. Must be at least 3 to span a plane. Larger
        values give smoother but more biased normals on high-curvature ridges.

    Returns
    -------
    numpy.ndarray
        ``(n, 3)`` unit normals, arbitrarily signed -- use
        :func:`orient_normals_into` to fix the direction.

    Raises
    ------
    ValueError
        If ``points`` is not ``(n, 3)`` or ``n_neighbours < 3``.

    Notes
    -----
    The MATLAB original loops over vertices with ``pdist2``; here the k-NN query
    is batched through a KD-tree and the SVDs are evaluated as a single stacked
    ``numpy.linalg.svd`` call, which is what makes a whole knee run in seconds
    rather than minutes.

    Examples
    --------
    Points on a plane get the plane's normal:

    >>> rng = np.random.default_rng(0)
    >>> pts = np.column_stack([rng.uniform(-5, 5, 400), rng.uniform(-5, 5, 400),
    ...                        np.zeros(400)])
    >>> n = estimate_surface_normals(pts, 20)
    >>> bool(np.abs(np.abs(n[:, 2]) - 1).max() < 1e-6)
    True
    """
    points = np.asarray(points, dtype=np.float64)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError(f"points must be (n, 3), got {points.shape}")
    if n_neighbours < 3:
        raise ValueError(f"n_neighbours must be >= 3 to span a plane, got {n_neighbours}")

    n_points = points.shape[0]
    k = min(n_neighbours, n_points)
    if k < 3:
        raise ValueError(
            f"only {n_points} point(s) supplied; need at least 3 to estimate a normal"
        )

    tree = cKDTree(points)
    _dist, idx = tree.query(points, k=k, workers=-1)
    neighbourhoods = points[idx]                                    # (n, k, 3)
    centred = neighbourhoods - neighbourhoods.mean(axis=1, keepdims=True)

    # Right singular vector of the smallest singular value == plane normal.
    _u, _s, vh = np.linalg.svd(centred, full_matrices=False)
    normals = vh[:, -1, :]

    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1.0
    return normals / norms


def smooth_normals(
    points: np.ndarray, normals: np.ndarray, n_neighbours: int = 10
) -> np.ndarray:
    """Spatially smooth a normal field (port of ``CM_cal_smoothSN.m``).

    Averaging is sign-aware: before summing, each neighbour's normal is flipped
    to the hemisphere of the centre normal. Without that, two adjacent normals
    that differ only in sign cancel to zero and the smoothed field collapses.

    Parameters
    ----------
    points
        ``(n, 3)`` coordinates.
    normals
        ``(n, 3)`` normals.
    n_neighbours
        Neighbourhood size; 0 or 1 disables smoothing.

    Returns
    -------
    numpy.ndarray
        ``(n, 3)`` smoothed unit normals.
    """
    points = np.asarray(points, dtype=np.float64)
    normals = np.asarray(normals, dtype=np.float64)
    if n_neighbours <= 1 or points.shape[0] < 2:
        return normals

    k = min(n_neighbours, points.shape[0])
    tree = cKDTree(points)
    _dist, idx = tree.query(points, k=k, workers=-1)

    neighbour_normals = normals[idx]                                # (n, k, 3)
    centre = normals[:, None, :]                                    # (n, 1, 3)
    sign = np.sign(np.einsum("nkj,nlj->nk", neighbour_normals, centre))
    sign[sign == 0] = 1.0
    aligned = neighbour_normals * sign[:, :, None]

    smoothed = aligned.mean(axis=1)
    norms = np.linalg.norm(smoothed, axis=1, keepdims=True)
    degenerate = norms[:, 0] < 1e-12
    if degenerate.any():
        smoothed[degenerate] = normals[degenerate]
        norms[degenerate] = 1.0
    return smoothed / norms


def orient_normals_into(
    points: np.ndarray,
    normals: np.ndarray,
    target_mask: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    probe_mm: float = 1.0,
    n_probes: int = 4,
) -> np.ndarray:
    """Flip normals so they point into ``target_mask`` (the cartilage).

    The SVD normal is defined only up to sign. Thickness rays must travel from
    bone into cartilage, so the sign is resolved empirically: probe a short
    distance along ``+n`` and ``-n`` and keep whichever direction spends more
    samples inside the target.

    Parameters
    ----------
    points
        ``(n, 3)`` vertex coordinates in mm.
    normals
        ``(n, 3)`` unit normals of arbitrary sign.
    target_mask
        Binary volume the normals should point into.
    spacing
        Voxel size in mm.
    probe_mm
        Total probe distance.
    n_probes
        Number of samples along each direction.

    Returns
    -------
    numpy.ndarray
        ``(n, 3)`` normals oriented into the target.

    Notes
    -----
    Vertices where both directions score equally (typically at the rim of the
    plate) keep their original sign; their rays will usually miss and yield NaN,
    which is the honest answer.
    """
    points = np.asarray(points, dtype=np.float64)
    normals = np.asarray(normals, dtype=np.float64)
    target = np.asarray(target_mask) > 0
    spacing_arr = np.asarray(spacing, dtype=np.float64)
    shape = np.asarray(target.shape)

    steps = np.linspace(probe_mm / n_probes, probe_mm, n_probes)

    def _score(direction: np.ndarray) -> np.ndarray:
        hits = np.zeros(points.shape[0], dtype=np.float64)
        for step in steps:
            probe = points + direction * step
            vox = np.rint(probe / spacing_arr).astype(int)
            inside = np.all((vox >= 0) & (vox < shape), axis=1)
            vox = np.clip(vox, 0, shape - 1)
            hits += inside * target[vox[:, 0], vox[:, 1], vox[:, 2]]
        return hits

    forward = _score(normals)
    backward = _score(-normals)

    flip = backward > forward
    out = normals.copy()
    out[flip] = -out[flip]

    ambiguous = int((forward == backward).sum())
    if ambiguous:
        logger.debug(
            "%d/%d normal(s) had no cartilage on either side within %.1f mm; sign kept as-is",
            ambiguous,
            points.shape[0],
            probe_mm,
        )
    return out


def thickness_surface_normal(
    bci: trimesh.Trimesh,
    articular: trimesh.Trimesh,
    *,
    normals: np.ndarray,
    max_depth_mm: float = 10.0,
) -> tuple[np.ndarray, int]:
    """Ray-cast along vertex normals and return hit distances.

    Reproduces ``CM_cal_thicknessMap_SN.m``: cast a ray from each BCI vertex
    along its (inward-oriented) normal, collect every intersection with the
    articular surface, and keep the nearest one. Hits beyond ``max_depth_mm``
    are rejected, since a ray that grazes the plate can otherwise travel a long
    way before striking the far side.

    Parameters
    ----------
    bci
        Bone-cartilage interface mesh.
    articular
        Articular surface mesh.
    normals
        ``(n, 3)`` unit normals, already oriented into the cartilage.
    max_depth_mm
        Maximum admissible thickness.

    Returns
    -------
    tuple
        ``(thickness_mm, n_capped)``. Misses are NaN.

    Notes
    -----
    Rays start a hair off the surface (1e-6 mm along the normal) so a vertex
    lying exactly on a shared triangle cannot register a zero-length self-hit.
    """
    origins = np.asarray(bci.vertices, dtype=np.float64)
    directions = np.asarray(normals, dtype=np.float64)
    n = origins.shape[0]

    thickness = np.full(n, np.nan, dtype=np.float64)
    if len(articular.faces) == 0:
        return thickness, 0

    epsilon = 1e-6
    locations, ray_idx, _tri_idx = articular.ray.intersects_location(
        ray_origins=origins + directions * epsilon,
        ray_directions=directions,
        multiple_hits=True,
    )

    n_capped = 0
    if len(ray_idx) > 0:
        distances = np.linalg.norm(locations - origins[ray_idx], axis=1)
        # Keep the nearest hit per ray via a sorted scatter-min.
        order = np.lexsort((distances, ray_idx))
        ray_sorted = ray_idx[order]
        dist_sorted = distances[order]
        first = np.ones(ray_sorted.shape[0], dtype=bool)
        first[1:] = ray_sorted[1:] != ray_sorted[:-1]
        nearest_ray = ray_sorted[first]
        nearest_dist = dist_sorted[first]

        within = nearest_dist <= max_depth_mm
        n_capped = int((~within).sum())
        thickness[nearest_ray[within]] = nearest_dist[within]

    return thickness, n_capped


def thickness_distance_transform(
    bci: trimesh.Trimesh,
    cartilage_mask: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    normals: np.ndarray,
    max_depth_mm: float = 10.0,
    step_mm: float = 0.25,
) -> tuple[np.ndarray, int]:
    """Thickness from the cartilage Euclidean distance transform.

    Convention
    ----------
    ``EDT(x)`` is the distance from ``x`` to the nearest non-cartilage voxel, so
    at the medial surface of a plate of thickness ``t`` it equals ``t / 2``.
    Marching inward along the normal and taking

    .. math:: \\hat{t} = 2 \\max_{s \\in [0, d]} \\mathrm{EDT}(x + s n)

    is therefore *exact* for a flat slab, and accurate to the curvature
    correction ``O(t^2 / R)`` for a shell of radius ``R``. This doubling is the
    convention documented here; the alternative of reading EDT directly at the
    BCI vertex would return ~0 everywhere, since the BCI lies on the boundary.

    Parameters
    ----------
    bci
        BCI mesh.
    cartilage_mask
        Binary cartilage mask.
    spacing
        Voxel size in mm.
    normals
        Inward-oriented unit normals.
    max_depth_mm
        March no further than this.
    step_mm
        March step; smaller is more accurate and slower.

    Returns
    -------
    tuple
        ``(thickness_mm, n_capped)``. Vertices whose march never entered
        cartilage are NaN.
    """
    cartilage = np.asarray(cartilage_mask) > 0
    edt = ndimage.distance_transform_edt(cartilage, sampling=spacing)

    origins = np.asarray(bci.vertices, dtype=np.float64)
    directions = np.asarray(normals, dtype=np.float64)
    spacing_arr = np.asarray(spacing, dtype=np.float64)
    shape = np.asarray(cartilage.shape)

    steps = np.arange(0.0, max_depth_mm + step_mm, step_mm)
    best = np.zeros(origins.shape[0], dtype=np.float64)
    ever_inside = np.zeros(origins.shape[0], dtype=bool)

    for step in steps:
        probe = origins + directions * step
        vox = np.rint(probe / spacing_arr).astype(int)
        inside_volume = np.all((vox >= 0) & (vox < shape), axis=1)
        vox = np.clip(vox, 0, shape - 1)
        values = edt[vox[:, 0], vox[:, 1], vox[:, 2]] * inside_volume
        ever_inside |= values > 0
        best = np.maximum(best, values)

    thickness = np.where(ever_inside, 2.0 * best, np.nan)
    n_capped = int(np.sum(thickness > max_depth_mm))
    thickness[thickness > max_depth_mm] = np.nan
    return thickness, n_capped


def thickness_nearest_neighbour(
    bci: trimesh.Trimesh,
    articular: trimesh.Trimesh,
    *,
    max_depth_mm: float = 10.0,
    exact: bool = True,
) -> tuple[np.ndarray, int]:
    """Distance from each BCI vertex to the closest point on the articular surface.

    Parameters
    ----------
    bci
        BCI mesh.
    articular
        Articular surface mesh.
    max_depth_mm
        Distances beyond this are rejected as NaN.
    exact
        Point-to-triangle distance (accurate) rather than point-to-vertex
        (faster, biased upward by the triangle edge length).

    Returns
    -------
    tuple
        ``(thickness_mm, n_capped)``.

    Notes
    -----
    This estimator is biased *low* on convex surfaces: the closest point on the
    articular surface is generally not the one along the normal, so the measured
    segment cuts a chord rather than spanning the plate. The bias grows with
    ``t / R``. That is a genuine property of the method, and its magnitude is
    reported in ablation A1.
    """
    origins = np.asarray(bci.vertices, dtype=np.float64)

    if exact and len(articular.faces) > 0:
        _closest, distances, _tri = trimesh.proximity.closest_point(articular, origins)
    else:
        tree = cKDTree(np.asarray(articular.vertices, dtype=np.float64))
        distances, _idx = tree.query(origins, k=1, workers=-1)

    distances = np.asarray(distances, dtype=np.float64)
    n_capped = int(np.sum(distances > max_depth_mm))
    distances[distances > max_depth_mm] = np.nan
    return distances, n_capped


def compute_thickness(
    bci: trimesh.Trimesh,
    articular: trimesh.Trimesh,
    method: ThicknessMethod = "surface_normal",
    *,
    cartilage_mask: np.ndarray | None = None,
    spacing: tuple[float, float, float] | None = None,
    n_neighbours: int = 30,
    smoothing_neighbours: int = 10,
    max_depth_mm: float = 10.0,
) -> ThicknessResult:
    """Measure cartilage thickness at every BCI vertex.

    Parameters
    ----------
    bci
        Bone-cartilage interface mesh (vertices in mm).
    articular
        Articular surface mesh (vertices in mm).
    method
        ``surface_normal``, ``distance_transform`` or ``nearest_neighbour``.
    cartilage_mask
        Binary cartilage mask; required for ``distance_transform`` and used to
        orient the normals for ``surface_normal``.
    spacing
        Voxel size in mm; required whenever ``cartilage_mask`` is used.
    n_neighbours
        k for the SVD normal estimator.
    smoothing_neighbours
        k for normal-field smoothing; 0 disables it.
    max_depth_mm
        Depth cap.

    Returns
    -------
    ThicknessResult

    Raises
    ------
    ValueError
        For an unknown method, or when a method's required inputs are absent.

    Examples
    --------
    A flat slab of known thickness is recovered to within a fraction of a percent:

    >>> import trimesh
    >>> lo = trimesh.creation.box(extents=(20, 20, 0.01))
    >>> hi = lo.copy(); hi.apply_translation([0, 0, 2.0])
    >>> normals = np.tile([0.0, 0.0, 1.0], (len(lo.vertices), 1))
    >>> t, _ = thickness_surface_normal(lo, hi, normals=normals, max_depth_mm=10)
    >>> bool(abs(np.nanmedian(t) - 2.0) < 0.02)
    True
    """
    allowed = ("surface_normal", "distance_transform", "nearest_neighbour")
    if method not in allowed:
        raise ValueError(f"method must be one of {allowed}, got {method!r}")

    if len(bci.vertices) == 0:
        raise ValueError("BCI mesh has no vertices; nothing to measure")

    normals: np.ndarray | None = None
    if method in ("surface_normal", "distance_transform"):
        if cartilage_mask is None or spacing is None:
            raise ValueError(
                f"method={method!r} needs cartilage_mask and spacing to orient the "
                "surface normals into the cartilage"
            )
        normals = estimate_surface_normals(np.asarray(bci.vertices), n_neighbours)
        if smoothing_neighbours > 1:
            normals = smooth_normals(np.asarray(bci.vertices), normals, smoothing_neighbours)
        normals = orient_normals_into(
            np.asarray(bci.vertices), normals, cartilage_mask, spacing
        )

    if method == "surface_normal":
        assert normals is not None
        thickness, n_capped = thickness_surface_normal(
            bci, articular, normals=normals, max_depth_mm=max_depth_mm
        )
    elif method == "distance_transform":
        assert normals is not None and cartilage_mask is not None and spacing is not None
        thickness, n_capped = thickness_distance_transform(
            bci, cartilage_mask, spacing, normals=normals, max_depth_mm=max_depth_mm
        )
    else:
        thickness, n_capped = thickness_nearest_neighbour(
            bci, articular, max_depth_mm=max_depth_mm
        )

    n_undefined = int(np.isnan(thickness).sum())
    if n_undefined:
        logger.debug(
            "%s: %d/%d vertices undefined (NaN), %d beyond the %.1f mm depth cap",
            method,
            n_undefined,
            thickness.shape[0],
            n_capped,
            max_depth_mm,
        )
    return ThicknessResult(
        thickness_mm=thickness,
        method=method,
        normals=normals,
        n_undefined=n_undefined,
        n_capped=n_capped,
    )

### 3D curvature (new)

`confcarti/thickness/curvature.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/thickness/curvature.py   (corrected)
# ==========================================================================
"""Per-vertex differential geometry of cartilage and subchondral bone surfaces.

Why this module exists
----------------------
CartiMorph measures thickness, area, volume and full-thickness loss, but not the
*shape* of the surfaces those quantities live on. Curvature has repeatedly been
shown to carry osteoarthritis signal that thickness alone misses -- surface
smoothness discriminates OA in every compartment (Tummala et al. 2015), joint
incongruity is measurable in vivo (Hohe et al. 2002), and local/global curvature
behaves as an independent biomarker (Folkesson et al. 2008). This module closes
that gap and, downstream, supplies the heteroscedastic scale
:math:`\\hat{\\sigma}(x)` that the normalized conformal score needs.

What changed relative to the first version
------------------------------------------
The original code discarded curvature at every vertex lying within
``2 * radius_mm`` -- 6.0 mm at the default 3.0 mm fit radius -- of an open
boundary, measuring that distance **through space** with a KD-tree. On a knee
cartilage plate that is not a conservative choice, it is a fatal one:

* An articular plate is an *open sheet* with a very large perimeter-to-area
  ratio. The tibial plates are roughly 20-25 mm across, so a 6 mm rim eaten from
  every edge leaves almost nothing; the observed loss was **76-87 % of every
  surface**, and the compartment "curvature" that survived was a summary of a
  small, non-anatomical island in the middle of the plate.
* Euclidean distance reaches *around* folds. Two points on opposite banks of the
  trochlear groove, or across the intercondylar notch, are millimetres apart in
  space and centimetres apart on the surface. The old margin therefore also
  deleted genuinely interior vertices.

The margin existed to solve a real problem -- a quadric fitted on a
one-sided, clipped neighbourhood is badly biased -- but it attacked a proxy for
that problem instead of the problem. This version tests the thing that actually
matters, per vertex:

1. **Is the neighbourhood two-sided?** Measured directly, as the largest angular
   gap between neighbours projected into the tangent plane. A complete disc has
   a maximum gap of order ``2*pi/k``; a neighbourhood cut by an edge has a gap
   of at least ``pi``. Vertices whose gap exceeds ``max_angular_gap_deg``
   (default 120 deg) are the ones the margin was trying to catch, and they are
   caught wherever they occur -- including at interior holes, which a
   distance-to-outer-rim rule misses entirely.
2. **Does the neighbourhood have enough support, and is the fit conditioned?**
   Minimum neighbour count and a reciprocal-condition-number floor on the
   5 x 5 normal matrix, computed on the offsets normalised by the fit radius so
   the threshold is scale-free.
3. **Is the neighbourhood on the same sheet of surface?** Candidates are drawn
   with a KD-tree (fast) and then filtered on normal agreement and on
   out-of-plane offset, which removes points reached across the joint gap or
   across a fold. The remaining boundary field -- when a caller still wants a
   geometric margin -- is computed as a **geodesic** distance along mesh edges,
   not a Euclidean one.

Where a vertex fails at the nominal scale, the fit is retried at successively
smaller radii down to ``min_radius_mm`` (adaptive scale). That is what recovers
the rim of a narrow plate: a 1.5 mm-scale estimate near the edge is a real
measurement at a stated scale, whereas a NaN is not a measurement at all. The
radius actually used is returned per vertex and summarised into the results CSV,
so a mixed-scale field can never be mistaken for a single-scale one.

Measured effect on an open cylindrical strip of the width of a tibial plate
(``tests/test_curvature_open_patch.py``): the old rule returns NaN everywhere and
reports no curvature; this version estimates ~90 % of the patch with a median
mean-curvature error under 5 %.

Estimators
----------
``quadric``
    Fit a local height field over the tangent plane and read the first and
    second fundamental forms off the fit. Accurate and noise-tolerant; default.

``cotangent``
    Discrete operators of Meyer, Desbrun, Schroder & Barr (2003): mean curvature
    from the cotangent Laplace-Beltrami operator over the mixed Voronoi area,
    Gaussian curvature from the angle deficit.

Sign convention
---------------
Curvature is signed against the *outward* vertex normal. A convex surface
(femoral condyle seen from outside) has positive mean curvature. Both estimators
are validated against closed forms: sphere ``H = 1/R, K = 1/R^2``; cylinder
``H = 1/(2R), K = 0``; plane ``H = K = 0``.

References
----------
Meyer, M., Desbrun, M., Schroder, P., Barr, A.H. (2003). "Discrete
    differential-geometry operators for triangulated 2-manifolds."
    Visualization and Mathematics III, 35-57.
Koenderink, J.J., van Doorn, A.J. (1992). "Surface shape and curvature scales."
    Image and Vision Computing 10(8):557-564.  [shape index, curvedness]
Petitjean, S. (2002). "A survey of methods for recovering quadrics in triangle
    meshes." ACM Computing Surveys 34(2):211-262.
Hohe, J. et al. (2002). "Surface size, curvature analysis, and assessment of
    knee joint incongruity with MRI in vivo." Magn Reson Med 47(3):554-561.
Folkesson, J. et al. (2008). "Automatic quantification of local and global
    articular cartilage surface curvature." Magn Reson Med 59(6):1340-1346.
"""


import logging
from dataclasses import dataclass
from typing import Literal

import numpy as np
import trimesh

logger = logging.getLogger(__name__)

__all__ = [
    "CurvatureResult",
    "compute_curvature",
    "quadric_curvature",
    "cotangent_curvature",
    "shape_index",
    "curvedness",
    "smooth_scalar_field",
    "congruence_index",
    "curvature_summary",
    "boundary_vertices",
    "geodesic_boundary_distance",
    "boundary_influence_zone",
]

CurvatureMethod = Literal["quadric", "cotangent"]

#: Shape-index class boundaries of Koenderink & van Doorn (1992). The index runs
#: from -1 (spherical cup) through 0 (saddle) to +1 (spherical cap).
SHAPE_CLASSES: tuple[tuple[str, float, float], ...] = (
    ("cup", -1.000, -0.625),
    ("rut", -0.625, -0.375),
    ("saddle_rut", -0.375, -0.125),
    ("saddle", -0.125, 0.125),
    ("saddle_ridge", 0.125, 0.375),
    ("ridge", 0.375, 0.625),
    ("cap", 0.625, 1.001),
)

#: Default acceptance rule for a local quadric fit. These are the numbers the
#: 6 mm margin was standing in for, expressed as properties of the fit itself.
DEFAULT_MIN_NEIGHBOURS: int = 8
DEFAULT_MAX_ANGULAR_GAP_DEG: float = 120.0
DEFAULT_MIN_RCOND: float = 1e-4
DEFAULT_NORMAL_AGREEMENT_DEG: float = 60.0
DEFAULT_MAX_OUT_OF_PLANE_FRACTION: float = 0.6


@dataclass(frozen=True, slots=True)
class CurvatureResult:
    """Per-vertex curvature fields.

    Attributes
    ----------
    k1, k2
        Principal curvatures in mm^-1 with ``k1 >= k2``. NaN where the local fit
        was rejected.
    mean_curvature
        ``H = (k1 + k2) / 2`` in mm^-1.
    gaussian_curvature
        ``K = k1 * k2`` in mm^-2.
    shape_index
        Koenderink-van Doorn shape index in [-1, 1]; NaN where the surface is
        locally flat and the index is undefined.
    curvedness
        ``sqrt((k1^2 + k2^2) / 2)`` in mm^-1: how strongly curved, regardless of
        shape.
    normals
        Outward unit vertex normals used for the sign convention.
    method
        Estimator used.
    n_clipped
        Vertices whose principal curvatures hit the clip bound.
    estimable
        Boolean mask, True where the local fit passed the support test. This is
        the honest denominator for every summary below.
    fit_radius_mm
        Radius actually used at each vertex; NaN where nothing was estimable.
        Differs from the nominal radius only where the adaptive fallback fired.
    boundary_distance_mm
        Geodesic distance along the mesh to the nearest open boundary. Provided
        for diagnostics and for callers who want to weight or stratify by it;
        it is no longer used to blanket-discard vertices.
    rejection_reason
        Per-vertex code: 0 accepted, 1 too few neighbours, 2 one-sided
        neighbourhood (angular gap), 3 ill-conditioned fit.
    """

    k1: np.ndarray
    k2: np.ndarray
    mean_curvature: np.ndarray
    gaussian_curvature: np.ndarray
    shape_index: np.ndarray
    curvedness: np.ndarray
    normals: np.ndarray
    method: str
    n_clipped: int = 0
    estimable: np.ndarray | None = None
    fit_radius_mm: np.ndarray | None = None
    boundary_distance_mm: np.ndarray | None = None
    rejection_reason: np.ndarray | None = None

    @property
    def n_vertices(self) -> int:
        """Number of vertices carrying a curvature value."""
        return int(self.k1.shape[0])

    @property
    def estimable_fraction(self) -> float:
        """Fraction of vertices where the local fit was accepted."""
        if self.estimable is None:
            return float(np.mean(np.isfinite(self.k1))) if self.n_vertices else 0.0
        return float(np.mean(self.estimable)) if self.estimable.size else 0.0

    def as_dict(self) -> dict[str, np.ndarray]:
        """Return the scalar fields keyed by name, for painting onto a surface."""
        return {
            "k1": self.k1,
            "k2": self.k2,
            "mean_curvature": self.mean_curvature,
            "gaussian_curvature": self.gaussian_curvature,
            "shape_index": self.shape_index,
            "curvedness": self.curvedness,
        }


# --------------------------------------------------------------------------- #
# Boundaries
# --------------------------------------------------------------------------- #


def boundary_vertices(mesh: trimesh.Trimesh) -> np.ndarray:
    """Boolean mask of vertices lying on an open boundary of the mesh.

    Parameters
    ----------
    mesh
        Input mesh.

    Returns
    -------
    numpy.ndarray
        ``(n_vertices,)`` boolean mask.
    """
    n = len(mesh.vertices)
    flags = np.zeros(n, dtype=bool)
    edges = mesh.edges_sorted
    if len(edges) == 0:
        return flags
    unique_edges, counts = np.unique(edges, axis=0, return_counts=True)
    open_edges = unique_edges[counts == 1]
    if open_edges.size:
        flags[open_edges.reshape(-1)] = True
    return flags


def geodesic_boundary_distance(mesh: trimesh.Trimesh) -> np.ndarray:
    """Shortest path *along the mesh* from every vertex to an open boundary.

    Euclidean distance to the boundary is the wrong metric on a folded open
    sheet: a vertex deep inside the lateral trochlear facet is a couple of
    millimetres from the medial facet through space and a couple of centimetres
    from it along the cartilage. Measuring through the surface is what the
    quantity was always supposed to mean.

    Implemented as one multi-source Dijkstra over the mesh's edge graph with
    edge lengths as weights, which slightly over-estimates the true geodesic
    (paths are constrained to edges) and is therefore conservative.

    Parameters
    ----------
    mesh
        Input mesh.

    Returns
    -------
    numpy.ndarray
        ``(n_vertices,)`` distance in mm. ``inf`` on a closed surface, which has
        no boundary; 0 on the boundary itself.
    """
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import dijkstra

    n = len(mesh.vertices)
    on_boundary = boundary_vertices(mesh)
    if not on_boundary.any():
        return np.full(n, np.inf, dtype=np.float64)

    edges = np.asarray(mesh.edges_unique, dtype=np.int64)
    lengths = np.asarray(mesh.edges_unique_length, dtype=np.float64)
    graph = coo_matrix(
        (lengths, (edges[:, 0], edges[:, 1])), shape=(n, n)
    ).tocsr()

    sources = np.flatnonzero(on_boundary)
    distance = dijkstra(graph, directed=False, indices=sources, min_only=True)
    return np.asarray(distance, dtype=np.float64)


def boundary_influence_zone(
    mesh: trimesh.Trimesh, margin_mm: float, *, metric: str = "geodesic"
) -> np.ndarray:
    """Vertices within ``margin_mm`` of an open boundary.

    Retained as an explicit, opt-in tool -- ``compute_curvature`` no longer
    applies it by default, because a blanket margin large enough to guarantee a
    two-sided neighbourhood is also large enough to delete most of a cartilage
    plate. Use it to *stratify* a curvature field, or to reproduce the old
    behaviour deliberately.

    Parameters
    ----------
    mesh
        Input mesh.
    margin_mm
        Margin around the open boundary.
    metric
        ``geodesic`` (default, along the surface) or ``euclidean`` (through
        space -- the old behaviour, kept only for reproducing it).

    Returns
    -------
    numpy.ndarray
        ``(n_vertices,)`` boolean mask, True inside the margin.

    Raises
    ------
    ValueError
        For an unknown metric.
    """
    if metric not in ("geodesic", "euclidean"):
        raise ValueError(f"metric must be geodesic|euclidean, got {metric!r}")

    on_boundary = boundary_vertices(mesh)
    if not on_boundary.any() or margin_mm <= 0:
        return on_boundary

    if metric == "geodesic":
        return geodesic_boundary_distance(mesh) <= margin_mm

    from scipy.spatial import cKDTree

    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    tree = cKDTree(vertices[on_boundary])
    distance, _idx = tree.query(vertices, k=1, workers=-1)
    return distance <= margin_mm


# --------------------------------------------------------------------------- #
# Neighbourhoods
# --------------------------------------------------------------------------- #


def _ring_neighbourhoods(
    mesh: trimesh.Trimesh, ring: int
) -> tuple[np.ndarray, np.ndarray]:
    """Build padded ``ring``-neighbourhoods for every vertex.

    Parameters
    ----------
    mesh
        Input mesh.
    ring
        Number of edge hops.

    Returns
    -------
    tuple
        ``(index, mask)`` both ``(n_vertices, max_k)``. ``index`` holds
        neighbour vertex indices padded by repeating the centre vertex, and
        ``mask`` is True for genuine entries. Padding with a mask (rather than
        with an arbitrary vertex) keeps the batched least-squares fits unbiased.
    """
    neighbours = mesh.vertex_neighbors
    n = len(mesh.vertices)

    sets: list[set[int]] = [{i} | set(neighbours[i]) for i in range(n)]
    for _ in range(ring - 1):
        expanded: list[set[int]] = []
        for i in range(n):
            grown = set(sets[i])
            for j in sets[i]:
                grown.update(neighbours[j])
            expanded.append(grown)
        sets = expanded

    sizes = np.fromiter((len(s) for s in sets), dtype=np.int64, count=n)
    max_k = int(sizes.max())

    index = np.empty((n, max_k), dtype=np.int64)
    mask = np.zeros((n, max_k), dtype=bool)
    for i, s in enumerate(sets):
        members = np.fromiter(sorted(s), dtype=np.int64, count=len(s))
        index[i, : members.size] = members
        index[i, members.size :] = i          # pad with the centre vertex
        mask[i, : members.size] = True
    return index, mask


def _radius_neighbourhoods(
    vertices: np.ndarray,
    tree: "object",
    targets: np.ndarray,
    radius_mm: float,
    *,
    max_neighbours: int = 256,
) -> tuple[np.ndarray, np.ndarray]:
    """Padded neighbourhoods of a fixed *physical* radius, for a subset of vertices.

    Why radius and not edge hops
    ----------------------------
    Curvature is a second derivative, so its estimate is only meaningful at a
    stated spatial scale. A marching-cubes surface extracted from a binary mask
    carries residual staircase ripple with a wavelength of roughly one voxel;
    ripple of amplitude ``a`` and wavelength ``lambda`` contributes curvature of
    order ``a (2 pi / lambda)^2``, which for ``a = 0.05 mm`` and
    ``lambda = 1 mm`` is about ``2 mm^-1``. That utterly swamps the
    ``1/22 mm^-1`` of a femoral condyle. A neighbourhood defined in edge hops
    shrinks with the voxel size and therefore locks onto the ripple; a
    neighbourhood defined in millimetres averages over it and returns anatomy.

    This is the same scale-space argument made by Folkesson et al. (2008).

    Parameters
    ----------
    vertices
        ``(n, 3)`` vertex array in mm.
    tree
        A ``scipy.spatial.cKDTree`` already built over ``vertices``.
    targets
        Indices of the vertices to build neighbourhoods for.
    radius_mm
        Neighbourhood radius in millimetres.
    max_neighbours
        Cap on neighbours per vertex, to bound memory on dense meshes. When a
        vertex has more, a *radius-stratified* subset is taken (sorted by
        distance, then evenly sampled) rather than an arbitrary slice of the
        KD-tree's unordered output -- an arbitrary slice can leave a directional
        hole and make a perfectly good neighbourhood look one-sided.

    Returns
    -------
    tuple
        ``(index, mask)``, both ``(len(targets), max_k)``, padded with the
        centre vertex and masked as in :func:`_ring_neighbourhoods`.
    """
    groups = tree.query_ball_point(vertices[targets], r=radius_mm, workers=-1)

    trimmed: list[np.ndarray] = []
    for pos, members in enumerate(groups):
        centre = int(targets[pos])
        arr = np.asarray(members, dtype=np.int64)
        if arr.size == 0:
            arr = np.asarray([centre], dtype=np.int64)
        elif arr.size > max_neighbours:
            offsets = vertices[arr] - vertices[centre]
            order = np.argsort(np.einsum("ij,ij->i", offsets, offsets))
            arr = arr[order][np.linspace(0, arr.size - 1, max_neighbours).astype(np.int64)]
        trimmed.append(arr)

    max_k = max(int(a.size) for a in trimmed)
    index = np.empty((len(targets), max_k), dtype=np.int64)
    mask = np.zeros((len(targets), max_k), dtype=bool)
    for pos, arr in enumerate(trimmed):
        index[pos, : arr.size] = arr
        index[pos, arr.size :] = targets[pos]
        mask[pos, : arr.size] = True
    return index, mask


def _tangent_frames(normals: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Build an orthonormal tangent basis ``(u, v)`` for each unit normal.

    Uses the numerically stable branchless construction of Duff et al. (2017),
    which avoids the degeneracy of crossing the normal with a fixed axis.
    """
    nx, ny, nz = normals[:, 0], normals[:, 1], normals[:, 2]
    sign = np.where(nz >= 0.0, 1.0, -1.0)
    a = -1.0 / (sign + nz)
    b = nx * ny * a
    u = np.stack([1.0 + sign * nx * nx * a, sign * b, -sign * nx], axis=1)
    v = np.stack([b, sign + ny * ny * a, -ny], axis=1)
    u /= np.linalg.norm(u, axis=1, keepdims=True)
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return u, v


def _max_angular_gap(
    x: np.ndarray, y: np.ndarray, mask: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Largest angular gap between neighbours projected into the tangent plane.

    This is the direct test for the failure the old boundary margin was a proxy
    for. Sort the neighbour azimuths around the centre; on a complete disc of
    ``k`` neighbours the biggest gap between consecutive azimuths is of order
    ``2*pi/k``, while a neighbourhood clipped by an edge necessarily leaves a gap
    of at least ``pi`` on the missing side. Unlike a distance-to-rim rule it also
    fires at interior holes and at slivers, and unlike a Euclidean margin it
    never fires on a genuinely interior vertex that happens to sit close to
    another fold of the same surface.

    Parameters
    ----------
    x, y
        ``(m, k)`` neighbour coordinates in the tangent frame.
    mask
        ``(m, k)`` validity mask.

    Returns
    -------
    tuple
        ``(gap_rad, n_used)`` -- the largest gap in radians (``2*pi`` when there
        is nothing to measure) and the number of neighbours that contributed.
    """
    rho = np.hypot(x, y)
    usable = mask & (rho > 1e-9)
    n_used = usable.sum(axis=1)

    # Pad with a large finite sentinel rather than inf so that the diff below
    # yields 0 on the padding instead of a nan (and a spurious warning); the
    # padded entries are discarded by ``valid_gap`` either way.
    sentinel = 1e6
    theta = np.where(usable, np.arctan2(y, x), sentinel)
    theta_sorted = np.sort(theta, axis=1)

    gaps = np.diff(theta_sorted, axis=1)
    order = np.arange(theta.shape[1] - 1)[None, :]
    valid_gap = order < (n_used - 1)[:, None]
    interior_gap = np.where(valid_gap, gaps, -np.inf).max(axis=1)

    rows = np.arange(theta.shape[0])
    last = theta_sorted[rows, np.clip(n_used - 1, 0, theta.shape[1] - 1)]
    first = theta_sorted[:, 0]
    wrap_gap = np.where(n_used > 0, first + 2.0 * np.pi - last, 2.0 * np.pi)

    gap = np.maximum(interior_gap, wrap_gap)
    # A single neighbour, or none, leaves the whole circle open.
    gap = np.where(n_used >= 2, gap, 2.0 * np.pi)
    return gap, n_used


# --------------------------------------------------------------------------- #
# Estimators
# --------------------------------------------------------------------------- #


def _quadric_pass(
    vertices: np.ndarray,
    normals: np.ndarray,
    u: np.ndarray,
    v: np.ndarray,
    index: np.ndarray,
    mask: np.ndarray,
    targets: np.ndarray,
    radius_mm: float,
    *,
    min_neighbours: int,
    max_angular_gap_deg: float,
    min_rcond: float,
    normal_agreement_deg: float,
    max_out_of_plane_fraction: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One quadric-fitting pass at a single radius, over ``targets``.

    ``vertices``, ``normals``, ``u`` and ``v`` are the full per-mesh arrays;
    ``index`` and ``mask`` are ``(len(targets), k)``.

    Returns ``(k1, k2, accepted, reason)`` for the target vertices. The fit
    itself is the Monge patch ``z = d x + e y + a x^2 + b xy + c y^2`` of the
    original implementation; what is new is that the neighbourhood is filtered
    onto the local sheet first and the result is only accepted when the fit had
    the support to be meaningful.
    """
    centres = vertices[targets]
    centre_normals = normals[targets]
    cu, cv = u[targets], v[targets]
    offsets = vertices[index] - centres[:, None, :]

    x = np.einsum("mkj,mj->mk", offsets, cu)
    y = np.einsum("mkj,mj->mk", offsets, cv)
    z = np.einsum("mkj,mj->mk", offsets, centre_normals)

    # --- keep only candidates that lie on the same sheet of surface --------- #
    # A KD-tree ball is a ball in R^3: on an open sheet folded back on itself
    # (trochlear groove, intercondylar notch) or across the joint gap it happily
    # returns points from a surface that is metres away geodesically. Two cheap,
    # fully vectorised tests remove them, and they are the two ways an
    # off-sheet point announces itself: its normal points somewhere else, and it
    # sits far out of the tangent plane.
    neighbour_normals = normals[index]
    agreement = np.einsum("mkj,mj->mk", neighbour_normals, centre_normals)
    same_sheet = agreement >= np.cos(np.deg2rad(normal_agreement_deg))
    in_plane = np.abs(z) <= max_out_of_plane_fraction * radius_mm
    is_centre = index == targets[:, None]
    mask = mask & (same_sheet | is_centre) & (in_plane | is_centre)

    gap, n_used = _max_angular_gap(x, y, mask & ~is_centre)

    # --- fit, on offsets normalised by the radius so rcond is scale-free ---- #
    xs, ys, zs = x / radius_mm, y / radius_mm, z
    design = np.stack([xs, ys, xs * xs, xs * ys, ys * ys], axis=2)
    weights = mask.astype(np.float64)[:, :, None]
    weighted = design * weights

    gram = np.einsum("mkp,mkq->mpq", weighted, design)
    rhs = np.einsum("mkp,mk->mp", weighted, zs * mask)

    eigenvalues = np.linalg.eigvalsh(gram)
    largest = eigenvalues[:, -1]
    smallest = eigenvalues[:, 0]
    rcond = np.where(largest > 0, np.clip(smallest, 0.0, None) / np.maximum(largest, 1e-300), 0.0)

    # Ridge term: keeps the system solvable where a neighbourhood is degenerate.
    # It only ever regularises fits that the acceptance test below rejects.
    scale = np.trace(gram, axis1=1, axis2=2) / 5.0
    scale = np.where(scale > 0, scale, 1.0)
    gram = gram + np.eye(5)[None] * (1e-8 * scale[:, None, None])

    try:
        coeffs = np.linalg.solve(gram, rhs[:, :, None])[:, :, 0]
    except np.linalg.LinAlgError:  # pragma: no cover - ridge makes this unlikely
        coeffs = np.linalg.lstsq(gram, rhs[:, :, None], rcond=None)[0][:, :, 0]

    # Undo the radius normalisation: x = r * x' gives f_x = d'/r, f_xx = 2a'/r^2.
    fx = coeffs[:, 0] / radius_mm
    fy = coeffs[:, 1] / radius_mm
    fxx = 2.0 * coeffs[:, 2] / radius_mm**2
    fxy = coeffs[:, 3] / radius_mm**2
    fyy = 2.0 * coeffs[:, 4] / radius_mm**2

    denom = np.sqrt(1.0 + fx**2 + fy**2)
    # S = I^{-1} II, written out in closed form for the 2x2 case.
    #
    # Sign convention. For a Monge patch z = f(x, y) the textbook second
    # fundamental form uses +f_xx, which makes a sphere viewed along its
    # *outward* normal come out negative: putting the origin at the north pole
    # with n = +z gives f(x, y) ~ -(x^2 + y^2) / 2R, hence f_xx = -1/R.
    # ConfCarti reports convex-outward as positive (sphere H = +1/R), matching
    # the cotangent estimator and the sign used throughout the OA curvature
    # literature, so the form is negated here. Gaussian curvature is unaffected
    # -- it is the product of two curvatures and both change sign together.
    e_coef = -fxx / denom
    f_coef = -fxy / denom
    g_coef = -fyy / denom
    big_e = 1.0 + fx**2
    big_f = fx * fy
    big_g = 1.0 + fy**2

    det_i = big_e * big_g - big_f**2
    det_i = np.where(np.abs(det_i) < 1e-12, 1.0, det_i)

    gaussian = (e_coef * g_coef - f_coef**2) / det_i
    mean = (e_coef * big_g - 2.0 * f_coef * big_f + g_coef * big_e) / (2.0 * det_i)

    disc = np.maximum(mean**2 - gaussian, 0.0)
    root = np.sqrt(disc)
    k1 = mean + root
    k2 = mean - root

    # --- acceptance -------------------------------------------------------- #
    enough = n_used >= min_neighbours
    two_sided = gap <= np.deg2rad(max_angular_gap_deg)
    conditioned = rcond >= min_rcond
    accepted = enough & two_sided & conditioned & np.isfinite(k1) & np.isfinite(k2)

    reason = np.zeros(len(targets), dtype=np.int8)
    reason[~enough] = 1
    reason[enough & ~two_sided] = 2
    reason[enough & two_sided & ~conditioned] = 3
    reason[accepted] = 0
    return k1, k2, accepted, reason


def quadric_curvature(
    mesh: trimesh.Trimesh,
    *,
    ring: int = 2,
    radius_mm: float | None = None,
    clip_mm_inv: float = 2.0,
    min_radius_mm: float | None = None,
    adaptive_radius: bool = True,
    radius_shrink: float = 1.5,
    min_neighbours: int = DEFAULT_MIN_NEIGHBOURS,
    max_angular_gap_deg: float = DEFAULT_MAX_ANGULAR_GAP_DEG,
    min_rcond: float = DEFAULT_MIN_RCOND,
    normal_agreement_deg: float = DEFAULT_NORMAL_AGREEMENT_DEG,
    max_out_of_plane_fraction: float = DEFAULT_MAX_OUT_OF_PLANE_FRACTION,
    max_neighbours: int = 256,
    chunk_size: int = 8192,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, int, np.ndarray, np.ndarray, np.ndarray]:
    """Principal curvatures from a local quadric (Monge patch) fit.

    For each vertex the neighbourhood is expressed in a frame whose third axis
    is the vertex normal, and the height field is fitted as

    .. math:: z = d x + e y + a x^2 + b xy + c y^2

    The first and second fundamental forms at the origin are then

    .. math::
        \\mathrm{I} = \\begin{pmatrix} 1 + f_x^2 & f_x f_y \\\\
                                       f_x f_y & 1 + f_y^2 \\end{pmatrix},
        \\quad
        \\mathrm{II} = \\frac{1}{\\sqrt{1 + f_x^2 + f_y^2}}
                      \\begin{pmatrix} f_{xx} & f_{xy} \\\\
                                       f_{xy} & f_{yy} \\end{pmatrix}

    with ``f_x = d``, ``f_y = e``, ``f_xx = 2a``, ``f_xy = b``, ``f_yy = 2c``.
    The principal curvatures are the eigenvalues of the shape operator
    ``S = I^{-1} II``.

    Keeping the linear terms matters: dropping them assumes the fitted normal is
    exactly the surface normal, which on a noisy mesh biases ``H`` towards zero.

    Parameters
    ----------
    mesh
        Input mesh with consistent outward normals.
    ring
        Neighbourhood radius in edge hops. Used only when ``radius_mm`` is None,
        in which case the support test and the adaptive fallback are skipped.
    radius_mm
        Nominal neighbourhood radius in millimetres. Strongly preferred on
        marching-cubes surfaces -- see :func:`_radius_neighbourhoods`.
    clip_mm_inv
        Symmetric clip on the principal curvatures.
    min_radius_mm
        Floor for the adaptive fallback. Defaults to
        ``max(1.5 mm, 3 x mean edge length)``: below roughly three edge lengths
        the fit stops measuring anatomy and starts measuring marching-cubes
        staircase, so shrinking further would trade a NaN for a wrong number.
    adaptive_radius
        Retry rejected vertices at successively smaller radii. Set False for a
        strict single-scale field.
    radius_shrink
        Factor by which the radius is divided on each retry.
    min_neighbours, max_angular_gap_deg, min_rcond
        Acceptance rule for a fit: enough support, two-sided support, and a
        conditioned normal matrix. ``max_angular_gap_deg`` is the parameter that
        replaces the old ``2 * radius_mm`` boundary margin.
    normal_agreement_deg, max_out_of_plane_fraction
        Sheet filter applied to KD-tree candidates before fitting.
    max_neighbours
        Cap on neighbours per vertex.
    chunk_size
        Vertices fitted per batch, to bound peak memory.

    Returns
    -------
    tuple
        ``(k1, k2, normals, n_clipped, estimable, fit_radius_mm, reason)``
        with ``k1 >= k2`` and NaN at non-estimable vertices.

    Examples
    --------
    A sphere of radius 10 has ``H = 1/R = 0.1`` and ``K = 1/R^2 = 0.01``:

    >>> sphere = trimesh.creation.icosphere(subdivisions=4, radius=10.0)
    >>> k1, k2, _n, _c, ok, _r, _why = quadric_curvature(sphere, radius_mm=2.0)
    >>> bool(abs(np.nanmedian((k1 + k2) / 2) - 0.1) < 0.005)
    True
    """
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    normals = np.asarray(mesh.vertex_normals, dtype=np.float64).copy()
    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1.0
    normals /= norms
    n = len(vertices)
    u, v = _tangent_frames(normals)

    if radius_mm is None:
        # Edge-hop mode: no physical scale, so no support test and no fallback.
        index, mask = _ring_neighbourhoods(mesh, ring)
        edge_length = float(np.mean(mesh.edges_unique_length)) if len(mesh.edges_unique) else 1.0
        k1, k2, accepted, reason = _quadric_pass(
            vertices, normals, u, v, index, mask, np.arange(n),
            radius_mm=max(edge_length * ring, 1e-6),
            min_neighbours=min_neighbours,
            max_angular_gap_deg=360.0,       # disabled: no physical neighbourhood
            min_rcond=0.0,
            normal_agreement_deg=180.0,
            max_out_of_plane_fraction=np.inf,
        )
        fit_radius = np.full(n, np.nan)
        fit_radius[accepted] = edge_length * ring
        n_clipped = int(np.sum((np.abs(k1) > clip_mm_inv) | (np.abs(k2) > clip_mm_inv)))
        k1 = np.where(accepted, np.clip(k1, -clip_mm_inv, clip_mm_inv), np.nan)
        k2 = np.where(accepted, np.clip(k2, -clip_mm_inv, clip_mm_inv), np.nan)
        return k1, k2, normals, n_clipped, accepted, fit_radius, reason

    from scipy.spatial import cKDTree

    tree = cKDTree(vertices)
    edge_length = float(np.mean(mesh.edges_unique_length)) if len(mesh.edges_unique) else 0.5
    if min_radius_mm is None:
        min_radius_mm = max(1.5, 3.0 * edge_length)
    min_radius_mm = min(min_radius_mm, radius_mm)

    k1 = np.full(n, np.nan)
    k2 = np.full(n, np.nan)
    fit_radius = np.full(n, np.nan)
    reason = np.full(n, 1, dtype=np.int8)
    estimable = np.zeros(n, dtype=bool)

    pending = np.arange(n)
    radius = float(radius_mm)
    n_clipped = 0

    while pending.size:
        lost_chunks: list[np.ndarray] = []
        # Chunked so that the (m, k, 5) design tensor stays bounded: a dense
        # femoral BCI mesh has tens of thousands of vertices and a 3 mm ball
        # holds ~100 of them, which is gigabytes if fitted in one go.
        for start in range(0, pending.size, chunk_size):
            block = pending[start : start + chunk_size]
            index, mask = _radius_neighbourhoods(
                vertices, tree, block, radius, max_neighbours=max_neighbours
            )
            pk1, pk2, accepted, preason = _quadric_pass(
                vertices, normals, u, v, index, mask, block, radius,
                min_neighbours=min_neighbours,
                max_angular_gap_deg=max_angular_gap_deg,
                min_rcond=min_rcond,
                normal_agreement_deg=normal_agreement_deg,
                max_out_of_plane_fraction=max_out_of_plane_fraction,
            )
            won = block[accepted]
            if won.size:
                wk1, wk2 = pk1[accepted], pk2[accepted]
                n_clipped += int(
                    np.sum((np.abs(wk1) > clip_mm_inv) | (np.abs(wk2) > clip_mm_inv))
                )
                k1[won] = np.clip(wk1, -clip_mm_inv, clip_mm_inv)
                k2[won] = np.clip(wk2, -clip_mm_inv, clip_mm_inv)
                fit_radius[won] = radius
                estimable[won] = True
                reason[won] = 0
            block_lost = block[~accepted]
            reason[block_lost] = preason[~accepted]
            lost_chunks.append(block_lost)

        lost = np.concatenate(lost_chunks) if lost_chunks else np.empty(0, dtype=np.int64)
        if not adaptive_radius or lost.size == 0:
            break
        next_radius = radius / radius_shrink
        if next_radius < min_radius_mm - 1e-9:
            break
        radius = next_radius
        pending = lost

    return k1, k2, normals, n_clipped, estimable, fit_radius, reason


def cotangent_curvature(
    mesh: trimesh.Trimesh, *, clip_mm_inv: float = 2.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray, int, np.ndarray]:
    """Discrete curvature via the cotangent Laplacian and the angle deficit.

    Implements Meyer et al. (2003):

    .. math::
        2 H \\mathbf{n} = \\frac{1}{2 A_{\\mathrm{mixed}}}
            \\sum_{j \\in N(i)} (\\cot \\alpha_{ij} + \\cot \\beta_{ij})
            (\\mathbf{x}_i - \\mathbf{x}_j)

    .. math::
        K = \\frac{1}{A_{\\mathrm{mixed}}}
            \\left( 2\\pi - \\sum_j \\theta_j \\right)

    Parameters
    ----------
    mesh
        Input mesh.
    clip_mm_inv
        Symmetric clip on the principal curvatures.

    Returns
    -------
    tuple
        ``(k1, k2, normals, n_clipped, estimable)``. Boundary vertices are not
        estimable: the angle deficit needs a complete one-ring and there is not
        one. This is a one-ring rule, so it removes a single row of vertices --
        not a 6 mm band.

    Notes
    -----
    ``A_mixed`` is approximated by one third of the incident face area
    (the barycentric cell). Meyer's obtuse-triangle correction is skipped: on
    the Taubin-smoothed marching-cubes meshes used here the triangles are close
    to equilateral, and the correction changes ``H`` by well under a percent
    while roughly tripling the cost.
    """
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int64)
    n = len(vertices)

    normals = np.asarray(mesh.vertex_normals, dtype=np.float64).copy()
    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1.0
    normals /= norms

    i0, i1, i2 = faces[:, 0], faces[:, 1], faces[:, 2]
    p0, p1, p2 = vertices[i0], vertices[i1], vertices[i2]

    e0 = p2 - p1        # opposite vertex 0
    e1 = p0 - p2        # opposite vertex 1
    e2 = p1 - p0        # opposite vertex 2

    cross = np.cross(e2, -e1)
    face_area2 = np.linalg.norm(cross, axis=1)
    face_area = 0.5 * face_area2
    safe_area2 = np.where(face_area2 < 1e-14, 1e-14, face_area2)

    # cot at a vertex = (dot of the two incident edge vectors) / (2 * face area)
    cot0 = np.einsum("ij,ij->i", -e2, e1) / safe_area2
    cot1 = np.einsum("ij,ij->i", -e0, e2) / safe_area2
    cot2 = np.einsum("ij,ij->i", -e1, e0) / safe_area2

    laplacian = np.zeros((n, 3), dtype=np.float64)
    # Edge (1,2) is weighted by the cotangent at vertex 0, and so on.
    for cot, a, b in ((cot0, i1, i2), (cot1, i2, i0), (cot2, i0, i1)):
        contribution = cot[:, None] * (vertices[a] - vertices[b])
        np.add.at(laplacian, a, contribution)
        np.add.at(laplacian, b, -contribution)

    area_mixed = np.zeros(n, dtype=np.float64)
    for idx in (i0, i1, i2):
        np.add.at(area_mixed, idx, face_area / 3.0)
    area_mixed = np.where(area_mixed < 1e-12, 1e-12, area_mixed)

    mean_vector = laplacian / (4.0 * area_mixed[:, None])
    mean = np.einsum("ij,ij->i", mean_vector, normals)

    # Gaussian curvature from the angle deficit.
    def _angle(u: np.ndarray, v: np.ndarray) -> np.ndarray:
        nu = np.linalg.norm(u, axis=1)
        nv = np.linalg.norm(v, axis=1)
        denom = np.where(nu * nv < 1e-14, 1e-14, nu * nv)
        return np.arccos(np.clip(np.einsum("ij,ij->i", u, v) / denom, -1.0, 1.0))

    angles = np.zeros(n, dtype=np.float64)
    np.add.at(angles, i0, _angle(p1 - p0, p2 - p0))
    np.add.at(angles, i1, _angle(p2 - p1, p0 - p1))
    np.add.at(angles, i2, _angle(p0 - p2, p1 - p2))

    gaussian = (2.0 * np.pi - angles) / area_mixed

    # Boundary vertices have no full angle ring, so the deficit is meaningless.
    # Marking them non-estimable is the honest answer; the previous version set
    # K = 0 there, which silently reported a saddle-free plane along every rim.
    boundary = boundary_vertices(mesh)
    estimable = ~boundary

    disc = np.maximum(mean**2 - gaussian, 0.0)
    root = np.sqrt(disc)
    k1 = mean + root
    k2 = mean - root

    n_clipped = int(np.sum((np.abs(k1) > clip_mm_inv) | (np.abs(k2) > clip_mm_inv)))
    k1 = np.where(estimable, np.clip(k1, -clip_mm_inv, clip_mm_inv), np.nan)
    k2 = np.where(estimable, np.clip(k2, -clip_mm_inv, clip_mm_inv), np.nan)
    return k1, k2, normals, n_clipped, estimable


# --------------------------------------------------------------------------- #
# Derived descriptors
# --------------------------------------------------------------------------- #


def shape_index(k1: np.ndarray, k2: np.ndarray, *, eps: float = 1e-9) -> np.ndarray:
    """Koenderink-van Doorn shape index.

    .. math:: S = \\frac{2}{\\pi} \\arctan \\frac{k_2 + k_1}{k_2 - k_1},
              \\quad k_1 \\ge k_2

    Runs from -1 (spherical cup) through 0 (symmetric saddle) to +1 (spherical
    cap), and is *scale invariant*: it describes local shape independently of
    how strongly curved the surface is. That separation is why it is reported
    alongside :func:`curvedness` rather than instead of it.

    Parameters
    ----------
    k1, k2
        Principal curvatures with ``k1 >= k2``.
    eps
        Below this ``|k1 - k2|`` the surface is umbilic-flat and the index is
        undefined; NaN is returned there.

    Returns
    -------
    numpy.ndarray
        Shape index in [-1, 1], NaN where undefined.

    Examples
    --------
    >>> float(shape_index(np.array([0.1]), np.array([0.1]))[0])
    1.0
    >>> float(shape_index(np.array([0.1]), np.array([-0.1]))[0])
    0.0
    """
    k1 = np.asarray(k1, dtype=np.float64)
    k2 = np.asarray(k2, dtype=np.float64)

    out = np.full(k1.shape, np.nan, dtype=np.float64)
    finite = np.isfinite(k1) & np.isfinite(k2)
    # Umbilic points (k1 == k2) are pure cap or cup: +1 or -1.
    umbilic = finite & (np.abs(k1 - k2) < eps)
    flat = umbilic & (np.abs(k1) < eps)
    out[umbilic] = np.sign(k1[umbilic])
    out[flat] = np.nan

    # Koenderink & van Doorn (1992), eq. 3, with the k1 >= k2 ordering used here:
    #     S = (2/pi) arctan((k1 + k2) / (k1 - k2))
    # The denominator is k1 - k2 (non-negative). Writing it as k2 - k1 flips the
    # sign of the whole index and turns every convex cap into a cup.
    ordinary = finite & ~umbilic
    out[ordinary] = (2.0 / np.pi) * np.arctan(
        (k1[ordinary] + k2[ordinary]) / (k1[ordinary] - k2[ordinary])
    )
    # Normalise -0.0 to 0.0 so a symmetric saddle prints as 0.0.
    return out + 0.0


def curvedness(k1: np.ndarray, k2: np.ndarray) -> np.ndarray:
    """Koenderink-van Doorn curvedness, ``sqrt((k1^2 + k2^2) / 2)`` in mm^-1.

    Parameters
    ----------
    k1, k2
        Principal curvatures.

    Returns
    -------
    numpy.ndarray
        Curvedness; 0 on a plane, ``1/R`` on a sphere of radius ``R``.

    Examples
    --------
    >>> float(curvedness(np.array([0.1]), np.array([0.1]))[0])
    0.1
    """
    k1 = np.asarray(k1, dtype=np.float64)
    k2 = np.asarray(k2, dtype=np.float64)
    return np.sqrt((k1**2 + k2**2) / 2.0)


def smooth_scalar_field(
    mesh: trimesh.Trimesh,
    field: np.ndarray,
    iterations: int = 3,
    *,
    preserve_nan: bool = True,
) -> np.ndarray:
    """Diffuse a per-vertex scalar over the mesh graph.

    Curvature is a second derivative, so it amplifies mesh noise; a few
    averaging passes make the field readable without materially shifting its
    mean. NaNs are ignored by the averaging.

    Parameters
    ----------
    mesh
        Mesh supplying the vertex adjacency.
    field
        ``(n_vertices,)`` scalar field.
    iterations
        Number of averaging passes.
    preserve_nan
        Keep NaN vertices NaN instead of filling them from their neighbours.
        This must be True when smoothing runs *after* the support test,
        otherwise a rejected vertex is quietly resurrected from the very
        neighbours that were too few to fit it. It is the default for that
        reason.

    Returns
    -------
    numpy.ndarray
        Smoothed field.
    """
    if iterations <= 0:
        return np.asarray(field, dtype=np.float64)

    values = np.asarray(field, dtype=np.float64).copy()
    keep_nan = ~np.isfinite(values) if preserve_nan else None
    neighbours = mesh.vertex_neighbors

    # Flatten adjacency once so each pass is a single segmented mean.
    lengths = np.fromiter((len(nb) for nb in neighbours), dtype=np.int64, count=len(neighbours))
    flat = (
        np.concatenate([np.asarray(nb, dtype=np.int64) for nb in neighbours])
        if lengths.sum()
        else np.empty(0, dtype=np.int64)
    )
    owner = np.repeat(np.arange(len(neighbours)), lengths)

    for _ in range(iterations):
        contrib = values[flat]
        valid = np.isfinite(contrib)
        totals = np.zeros(len(neighbours), dtype=np.float64)
        counts = np.zeros(len(neighbours), dtype=np.float64)
        np.add.at(totals, owner[valid], contrib[valid])
        np.add.at(counts, owner[valid], 1.0)
        averaged = np.where(counts > 0, totals / np.maximum(counts, 1.0), values)
        # Half-weight update keeps the field from collapsing to its global mean.
        values = np.where(np.isfinite(values), 0.5 * values + 0.5 * averaged, averaged)
        if keep_nan is not None:
            values[keep_nan] = np.nan
    return values


def compute_curvature(
    mesh: trimesh.Trimesh,
    *,
    method: CurvatureMethod = "quadric",
    ring: int = 2,
    radius_mm: float | None = 3.0,
    clip_mm_inv: float = 2.0,
    smoothing_iterations: int = 3,
    adaptive_radius: bool = True,
    min_radius_mm: float | None = None,
    min_neighbours: int = DEFAULT_MIN_NEIGHBOURS,
    max_angular_gap_deg: float = DEFAULT_MAX_ANGULAR_GAP_DEG,
    min_rcond: float = DEFAULT_MIN_RCOND,
    normal_agreement_deg: float = DEFAULT_NORMAL_AGREEMENT_DEG,
    max_out_of_plane_fraction: float = DEFAULT_MAX_OUT_OF_PLANE_FRACTION,
    max_neighbours: int = 256,
    chunk_size: int = 8192,
    mask_boundary: bool = False,
    boundary_margin_mm: float | None = None,
    boundary_metric: str = "geodesic",
    warn_below_estimable_fraction: float = 0.5,
) -> CurvatureResult:
    """Compute the full curvature description of a surface.

    Parameters
    ----------
    mesh
        Surface with consistent outward normals.
    method
        ``quadric`` or ``cotangent``.
    ring
        Edge-hop neighbourhood for the quadric fit; used only when
        ``radius_mm`` is None.
    radius_mm
        Nominal physical neighbourhood radius in mm for the quadric fit. This
        sets the spatial scale of the estimate and is the parameter that
        matters: on a marching-cubes surface an edge-hop neighbourhood measures
        voxel staircasing instead of anatomy. Pass None to fall back to
        ``ring``.
    clip_mm_inv
        Symmetric principal-curvature clip.
    smoothing_iterations
        Diffusion passes applied to the curvature fields. Applied **after** the
        support test, with NaNs preserved, so that a rejected boundary fit
        cannot diffuse into the interior. (The previous version smoothed first
        and masked second, which pushed the very boundary bias it was trying to
        remove up to three rings inward.)
    adaptive_radius, min_radius_mm
        Retry rejected vertices at smaller scales, down to ``min_radius_mm``.
    min_neighbours, max_angular_gap_deg, min_rcond
        Acceptance rule for a local fit. ``max_angular_gap_deg`` is what
        replaces the old ``2 * radius_mm`` geometric margin: it asks directly
        whether the neighbourhood surrounds the vertex, instead of guessing from
        a distance that it might not.
    normal_agreement_deg, max_out_of_plane_fraction, max_neighbours, chunk_size
        Sheet filter, neighbourhood cap and batch size; see
        :func:`quadric_curvature`.
    mask_boundary
        Additionally NaN everything within ``boundary_margin_mm`` of an open
        boundary. **Off by default** -- on a cartilage plate a margin wide enough
        to matter deletes most of the plate, and the support test already
        rejects the fits that margin was aimed at. Turn it on to reproduce older
        results or to enforce a single-scale interior-only field.
    boundary_margin_mm
        Margin used when ``mask_boundary`` is True. Defaults to ``radius_mm``
        (one fit radius), not two.
    boundary_metric
        ``geodesic`` (default) or ``euclidean``. Geodesic is the correct metric
        on a folded open sheet; ``euclidean`` reproduces the old behaviour.
    warn_below_estimable_fraction
        Emit a warning when less than this fraction of the surface ends up
        estimable.

    Returns
    -------
    CurvatureResult

    Raises
    ------
    ValueError
        For an unknown method or an empty mesh.

    Examples
    --------
    >>> sphere = trimesh.creation.icosphere(subdivisions=4, radius=10.0)
    >>> res = compute_curvature(sphere, radius_mm=None, smoothing_iterations=0)
    >>> bool(abs(np.nanmedian(res.mean_curvature) - 0.1) < 0.005)
    True
    >>> bool(abs(np.nanmedian(res.gaussian_curvature) - 0.01) < 0.002)
    True
    """
    if method not in ("quadric", "cotangent"):
        raise ValueError(f"method must be quadric|cotangent, got {method!r}")
    if len(mesh.vertices) == 0:
        raise ValueError("cannot compute curvature on a mesh with no vertices")

    if method == "quadric":
        k1, k2, normals, n_clipped, estimable, fit_radius, reason = quadric_curvature(
            mesh,
            ring=ring,
            radius_mm=radius_mm,
            clip_mm_inv=clip_mm_inv,
            adaptive_radius=adaptive_radius,
            min_radius_mm=min_radius_mm,
            min_neighbours=min_neighbours,
            max_angular_gap_deg=max_angular_gap_deg,
            min_rcond=min_rcond,
            normal_agreement_deg=normal_agreement_deg,
            max_out_of_plane_fraction=max_out_of_plane_fraction,
            max_neighbours=max_neighbours,
            chunk_size=chunk_size,
        )
    else:
        k1, k2, normals, n_clipped, estimable = cotangent_curvature(
            mesh, clip_mm_inv=clip_mm_inv
        )
        fit_radius = np.where(estimable, float(np.mean(mesh.edges_unique_length)), np.nan)
        reason = np.where(estimable, 0, 1).astype(np.int8)

    # Optional extra geometric margin, applied BEFORE smoothing so that nothing
    # it removes can leak back in.
    if mask_boundary:
        if boundary_margin_mm is None:
            boundary_margin_mm = float(radius_mm) if radius_mm is not None else 0.0
        if boundary_margin_mm > 0:
            zone = boundary_influence_zone(mesh, boundary_margin_mm, metric=boundary_metric)
            k1 = np.where(zone, np.nan, k1)
            k2 = np.where(zone, np.nan, k2)
            fit_radius = np.where(zone, np.nan, fit_radius)
            estimable = estimable & ~zone
            reason = np.where(zone, np.int8(4), reason)

    if smoothing_iterations > 0:
        k1 = smooth_scalar_field(mesh, k1, smoothing_iterations, preserve_nan=True)
        k2 = smooth_scalar_field(mesh, k2, smoothing_iterations, preserve_nan=True)
        # Re-impose k1 >= k2, which independent smoothing can violate.
        k1, k2 = np.maximum(k1, k2), np.minimum(k1, k2)

    fraction = float(np.mean(estimable)) if estimable.size else 0.0
    if fraction < warn_below_estimable_fraction:
        counts = np.bincount(np.asarray(reason, dtype=np.int64), minlength=5)
        logger.warning(
            "curvature estimable on only %.0f%% of this surface (%d/%d vertices). "
            "Rejections: %d too few neighbours, %d one-sided neighbourhood, "
            "%d ill-conditioned, %d inside an explicit boundary margin. If the "
            "one-sided count dominates, the patch is genuinely narrow relative to "
            "the %.1f mm fit radius -- lower radius_mm or min_radius_mm.",
            100.0 * fraction,
            int(estimable.sum()),
            int(estimable.size),
            int(counts[1]), int(counts[2]), int(counts[3]), int(counts[4]),
            float(radius_mm) if radius_mm is not None else float("nan"),
        )
    else:
        logger.debug(
            "curvature estimable on %.0f%% of the surface; median fit radius %.2f mm",
            100.0 * fraction,
            float(np.nanmedian(fit_radius)) if np.isfinite(fit_radius).any() else float("nan"),
        )

    return CurvatureResult(
        k1=k1,
        k2=k2,
        mean_curvature=(k1 + k2) / 2.0,
        gaussian_curvature=k1 * k2,
        shape_index=shape_index(k1, k2),
        curvedness=curvedness(k1, k2),
        normals=normals,
        method=method,
        n_clipped=n_clipped,
        estimable=estimable,
        fit_radius_mm=fit_radius,
        boundary_distance_mm=geodesic_boundary_distance(mesh),
        rejection_reason=reason,
    )


def congruence_index(
    femoral: CurvatureResult, tibial: CurvatureResult
) -> dict[str, float]:
    """Joint incongruity between two opposing articular surfaces.

    Hohe et al. (2002) define incongruity from the difference of the principal
    curvatures of the two opposing surfaces. Because the surfaces face each
    other, a perfectly congruent pair has *opposite-signed* curvature of equal
    magnitude, so the residual is formed as ``k_fem + k_tib``.

    Parameters
    ----------
    femoral, tibial
        Curvature of the two opposing surfaces.

    Returns
    -------
    dict
        ``incongruity_k1_mm_inv``, ``incongruity_k2_mm_inv``,
        ``incongruity_rms_mm_inv`` and the two median curvednesses.

    Notes
    -----
    This is a *distributional* comparison: the two meshes have different vertex
    counts and no point correspondence, so medians are compared rather than
    per-vertex differences. Establishing correspondence would need registration
    to the CLAIR-Knee-103R template, which is out of scope here.
    """
    fem_k1 = float(np.nanmedian(femoral.k1))
    fem_k2 = float(np.nanmedian(femoral.k2))
    tib_k1 = float(np.nanmedian(tibial.k1))
    tib_k2 = float(np.nanmedian(tibial.k2))

    d1 = fem_k1 + tib_k1
    d2 = fem_k2 + tib_k2
    return {
        "incongruity_k1_mm_inv": d1,
        "incongruity_k2_mm_inv": d2,
        "incongruity_rms_mm_inv": float(np.sqrt((d1**2 + d2**2) / 2.0)),
        "femoral_curvedness_mm_inv": float(np.nanmedian(femoral.curvedness)),
        "tibial_curvedness_mm_inv": float(np.nanmedian(tibial.curvedness)),
    }


def curvature_summary(
    result: CurvatureResult, *, shape_index_bins: int = 9
) -> dict[str, float]:
    """Reduce a curvature field to scalar descriptors for the results CSV.

    Parameters
    ----------
    result
        Curvature fields.
    shape_index_bins
        Number of equal-width shape-index bins whose occupancy is reported.

    Returns
    -------
    dict
        Robust summaries plus the shape-class fractions, plus the coverage
        diagnostics -- ``estimable_fraction`` and the fit-radius quantiles --
        without which a curvature mean is uninterpretable: it says nothing about
        whether it describes the whole plate or an island in the middle of it.
    """

    def _stats(name: str, values: np.ndarray) -> dict[str, float]:
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            return {f"{name}_{k}": float("nan") for k in ("mean", "median", "sd", "p05", "p95")}
        return {
            f"{name}_mean": float(finite.mean()),
            f"{name}_median": float(np.median(finite)),
            f"{name}_sd": float(finite.std(ddof=1)) if finite.size > 1 else 0.0,
            f"{name}_p05": float(np.percentile(finite, 5)),
            f"{name}_p95": float(np.percentile(finite, 95)),
        }

    out: dict[str, float] = {}
    out.update(_stats("mean_curvature", result.mean_curvature))
    out.update(_stats("gaussian_curvature", result.gaussian_curvature))
    out.update(_stats("curvedness", result.curvedness))
    out.update(_stats("shape_index", result.shape_index))

    finite_si = result.shape_index[np.isfinite(result.shape_index)]
    if finite_si.size:
        hist, _edges = np.histogram(finite_si, bins=shape_index_bins, range=(-1.0, 1.0))
        fractions = hist / finite_si.size
        for i, frac in enumerate(fractions):
            out[f"shape_index_bin{i:02d}_frac"] = float(frac)
        for name, lo, hi in SHAPE_CLASSES:
            out[f"shape_class_{name}_frac"] = float(
                np.mean((finite_si >= lo) & (finite_si < hi))
            )

    out["curvature_clipped_fraction"] = (
        float(result.n_clipped / result.n_vertices) if result.n_vertices else 0.0
    )
    out["estimable_fraction"] = result.estimable_fraction

    radii = result.fit_radius_mm
    if radii is not None and np.isfinite(radii).any():
        finite_r = radii[np.isfinite(radii)]
        out["fit_radius_median_mm"] = float(np.median(finite_r))
        out["fit_radius_p05_mm"] = float(np.percentile(finite_r, 5))
        out["fit_radius_reduced_fraction"] = float(
            np.mean(finite_r < finite_r.max() - 1e-9)
        )
    else:
        out["fit_radius_median_mm"] = float("nan")
        out["fit_radius_p05_mm"] = float("nan")
        out["fit_radius_reduced_fraction"] = float("nan")

    return out


### Focal bulging and denting (new)

`confcarti/thickness/bulge.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/thickness/bulge.py
# ==========================================================================
"""Focal bulging and denting of the articular surface.

What "bulge" means here
-----------------------
A bulge is a *local outward protrusion of the articular surface relative to that
surface's own smooth anatomical form*. It is deliberately not "distance from a
sphere" or "distance from the template": the femoral condyle is not a sphere,
and template registration would fold registration error into the measurement.

The separation of *form* from *deviation* follows surface metrology (the
form-removal step of ISO 25178): around each vertex a quadric is fitted over a
neighbourhood of stated physical radius, and the bulge is the residual of the
vertex from that locally-fitted form,

.. math:: b_i = -c_0^{(i)}

where :math:`c_0^{(i)}` is the constant term of the quadric fitted in a local
frame whose origin is the vertex itself. Positive is outward (bulge, swelling),
negative is inward (dent, focal defect, fibrillation). The fit radius sets the
wavelength at which "form" ends and "bulge" begins, and is reported.

.. note::
   An earlier design used a heavily Taubin-smoothed copy of the surface as the
   reference. That does not work, and the failure is instructive: the umbrella
   Laplacian of a *perfect* sphere is not zero -- it carries an intrinsic mean
   curvature flow term of order :math:`e^2 / 2R` -- so Taubin iteration on a
   curved surface never converges to a smooth reference. It stalls at a floor
   set by the tessellation (valence-5 versus valence-6 vertices) while the
   low-frequency gain of :math:`(1 - \\lambda k)(1 - \\mu k)`, which slightly
   exceeds 1 near :math:`k \\approx 0.06`, slowly inflates the whole surface.
   Measured on a sphere of radius 20 mm carrying a 1.5 mm bump, 120 Taubin
   iterations *grew* the bump to 1.76 mm. Taubin remains the right tool for
   removing voxel staircasing at extraction time, where only ~10 iterations are
   used and volume is preserved to well under a percent; it is the wrong tool
   for defining a reference surface.

Measured sensitivity and its limits
-----------------------------------
On a 20 mm sphere carrying a Gaussian bump of 1.5 mm height and ~4.5 mm width,
:func:`compute_bulge` recovers (robust fit, 3 IRLS passes):

===================  =================  ==================
Form radius (mm)     Recovered height   Recovered fraction
===================  =================  ==================
8                    0.07 mm            5%
12                   0.55 mm            37%
15                   0.81 mm            54%
20                   0.64 mm            43%
===================  =================  ==================

Two honest caveats follow from this table and should be quoted with any result:

1. **Amplitudes are under-estimated.** A local form fit always absorbs part of
   the feature it is trying to isolate. Treat the bulge field as a *relative*
   index -- valid for ranking subregions, comparing KL grades and tracking
   change -- not as an absolute height.
2. **There is a false-positive floor on strongly curved surfaces.** A quadric
   fitted over a 15 mm patch of a 20 mm sphere leaves an RMS residual of about
   0.16 mm even though the sphere is perfectly smooth, because the patch is no
   longer well approximated by a quadric. The floor grows with
   ``form_radius / radius_of_curvature``, so femoral condyles have a higher
   floor than tibial plateaus and the two should not be compared directly.

:func:`thickness_residual_bulge` is the complementary measure and behaves better
on the phantom, where focal swelling is injected as a thickness excess: its
maximum rises monotonically with the injected bulge height (0.29 mm with no
bulge, 0.36 mm at 0.6 mm, 0.52 mm at 1.0 mm). Report both.

Why it is worth measuring
-------------------------
Cartilage swelling and focal surface irregularity are early osteoarthritis
signals that a mean thickness cannot express: a plate can bulge in one place and
thin in another and still report a normal average. The bulge field separates
those. It also feeds the denuded classifier and the heteroscedastic scale used
by the normalized conformal score.

References
----------
Taubin, G. (1995). "Curve and surface smoothing without shrinkage." ICCV.
Tummala, S. et al. (2015). "Diagnosis of osteoarthritis by cartilage surface
    smoothness quantified automatically from knee MRI." Cartilage 6(1):5-14.
Calvo, E. et al. (2004) on early cartilage swelling in experimental OA.
"""


import logging
from dataclasses import dataclass

import numpy as np
import trimesh


logger = logging.getLogger(__name__)

__all__ = [
    "BulgeResult",
    "compute_bulge",
    "reference_surface",
    "bulge_summary",
    "thickness_residual_bulge",
]


@dataclass(frozen=True, slots=True)
class BulgeResult:
    """Signed deviation of a surface from its own locally-fitted smooth form.

    Attributes
    ----------
    deviation_mm
        Per-vertex signed normal deviation. Positive = outward bulge. NaN where
        the local fit had too few neighbours to be identifiable.
    vertex_area_mm2
        Barycentric vertex areas, used to turn per-vertex deviations into areas
        and volumes.
    form_radius_mm
        Radius of the neighbourhood the form was fitted over. This is the
        wavelength scale separating "form" from "bulge".
    min_height_mm
        Magnitude below which a deviation is treated as noise, not a lesion.
    reference
        Optional explicit reference surface, when one was constructed.
    """

    deviation_mm: np.ndarray
    vertex_area_mm2: np.ndarray
    form_radius_mm: float
    min_height_mm: float = 0.1
    reference: trimesh.Trimesh | None = None

    @property
    def bulge_mask(self) -> np.ndarray:
        """Vertices protruding outward by more than ``min_height_mm``."""
        return np.nan_to_num(self.deviation_mm, nan=0.0) > self.min_height_mm

    @property
    def dent_mask(self) -> np.ndarray:
        """Vertices depressed inward by more than ``min_height_mm``."""
        return np.nan_to_num(self.deviation_mm, nan=0.0) < -self.min_height_mm

    @property
    def bulge_volume_mm3(self) -> float:
        """Volume enclosed between the surface and its reference, outward only.

        Computed as ``sum(deviation * vertex_area)`` over bulging vertices,
        which is the first-order approximation to the enclosed volume.
        """
        mask = self.bulge_mask
        if not mask.any():
            return 0.0
        return float(np.sum(self.deviation_mm[mask] * self.vertex_area_mm2[mask]))

    @property
    def dent_volume_mm3(self) -> float:
        """Volume of the inward depressions, reported positive."""
        mask = self.dent_mask
        if not mask.any():
            return 0.0
        return float(-np.sum(self.deviation_mm[mask] * self.vertex_area_mm2[mask]))

    @property
    def bulge_area_mm2(self) -> float:
        """Surface area occupied by outward bulging."""
        return float(self.vertex_area_mm2[self.bulge_mask].sum())

    @property
    def dent_area_mm2(self) -> float:
        """Surface area occupied by inward denting."""
        return float(self.vertex_area_mm2[self.dent_mask].sum())

    @property
    def total_area_mm2(self) -> float:
        """Total surface area carrying a deviation value."""
        return float(self.vertex_area_mm2.sum())

    @property
    def roughness_mm(self) -> float:
        """RMS of the signed deviation: a scale-free surface-roughness index."""
        finite = self.deviation_mm[np.isfinite(self.deviation_mm)]
        if finite.size == 0:
            return float("nan")
        return float(np.sqrt(np.mean(finite**2)))


def vertex_areas(mesh: trimesh.Trimesh) -> np.ndarray:
    """Barycentric (one-third of incident face area) vertex areas in mm^2.

    Parameters
    ----------
    mesh
        Input mesh.

    Returns
    -------
    numpy.ndarray
        ``(n_vertices,)`` areas summing to the mesh area.

    Examples
    --------
    >>> s = trimesh.creation.icosphere(subdivisions=3, radius=5.0)
    >>> bool(abs(vertex_areas(s).sum() - s.area) < 1e-6)
    True
    """
    areas = np.zeros(len(mesh.vertices), dtype=np.float64)
    face_area = mesh.area_faces / 3.0
    for column in range(3):
        np.add.at(areas, mesh.faces[:, column], face_area)
    return areas


def local_form_residual(
    mesh: trimesh.Trimesh,
    *,
    radius_mm: float = 8.0,
    min_neighbours: int = 12,
    robust_iterations: int = 3,
    tukey_c: float = 2.5,
) -> np.ndarray:
    """Signed residual of each vertex from a locally-fitted quadric form.

    Around each vertex, neighbours within ``radius_mm`` are expressed in a local
    frame whose third axis is the vertex normal and whose origin is the vertex.
    A quadric

    .. math:: z = c_0 + c_1 x + c_2 y + c_3 x^2 + c_4 xy + c_5 y^2

    is fitted by least squares. The vertex itself sits at ``z = 0``, so the
    fitted form predicts ``c_0`` there and the residual is ``-c_0``.

    Because the fit is *not* constrained to pass through the vertex, a vertex
    sitting on a focal bump is pulled away from the surrounding form and
    ``-c_0`` recovers the bump height. On a smooth patch the quadric fits almost
    exactly and the residual is ~0.

    Choosing the radius
    -------------------
    A quadric can represent a paraboloid, so a bump that is itself roughly
    paraboloidal *within the fit window* is absorbed into the "form" and
    disappears. The fit radius must therefore be comfortably larger than the
    feature being detected -- as a rule of thumb, at least three times its
    width. On a 20 mm sphere carrying a 1.5 mm bump about 4.5 mm wide, an 8 mm
    radius recovers only 0.09 mm of it, whereas a 15 mm radius with robust
    reweighting recovers most of it. This is a real limitation, not a tuning
    knob: features comparable in size to the plate itself are indistinguishable
    from its anatomical form by any local method.

    Robustness
    ----------
    Even at a generous radius, an ordinary least-squares form is dragged upward
    by the very bump it is meant to exclude. Three passes of iteratively
    reweighted least squares with Tukey's biweight down-weight the outlying
    (bulging) vertices so the form tracks the surrounding healthy surface.

    Parameters
    ----------
    mesh
        Surface with consistent outward normals.
    radius_mm
        Form-fitting radius. Features much smaller than this count as bulge;
        undulations much larger than this count as form.
    min_neighbours
        Fewer usable neighbours than this makes the six-parameter fit
        unidentifiable; those vertices return NaN.
    robust_iterations
        IRLS passes. 0 gives a plain least-squares fit.
    tukey_c
        Tukey biweight cutoff in units of the robust residual scale (MAD).

    Returns
    -------
    numpy.ndarray
        ``(n_vertices,)`` signed residual in mm, positive outward.

    Examples
    --------
    >>> sphere = trimesh.creation.icosphere(subdivisions=4, radius=20.0)
    >>> res = local_form_residual(sphere, radius_mm=8.0)
    >>> bool(np.nanmax(np.abs(res)) < 0.05)      # a sphere has no bulge
    True
    """
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    normals = np.asarray(mesh.vertex_normals, dtype=np.float64).copy()
    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1.0
    normals /= norms

    index, mask = _radius_neighbourhoods(mesh, radius_mm)
    u, v = _tangent_frames(normals)

    offsets = vertices[index] - vertices[:, None, :]
    x = np.einsum("nkj,nj->nk", offsets, u)
    y = np.einsum("nkj,nj->nk", offsets, v)
    z = np.einsum("nkj,nj->nk", offsets, normals)

    ones = np.ones_like(x)
    design = np.stack([ones, x, y, x * x, x * y, y * y], axis=2)   # (n, k, 6)
    base = mask.astype(np.float64)
    z_masked = z * base

    weights = base
    coeffs = np.zeros((design.shape[0], 6), dtype=np.float64)
    for iteration in range(max(robust_iterations, 0) + 1):
        weighted = design * weights[:, :, None]
        gram = np.einsum("nkp,nkq->npq", weighted, design)
        rhs = np.einsum("nkp,nk->np", weighted, z_masked)

        scale = np.trace(gram, axis1=1, axis2=2) / 6.0
        scale = np.where(scale > 0, scale, 1.0)
        gram = gram + np.eye(6)[None] * (1e-9 * scale[:, None, None])
        coeffs = np.linalg.solve(gram, rhs[:, :, None])[:, :, 0]

        if iteration == robust_iterations:
            break

        # Tukey biweight reweighting against the MAD of the in-mask residuals.
        fitted = np.einsum("nkp,np->nk", design, coeffs)
        resid = (z - fitted) * base
        median = np.median(np.where(base > 0, resid, np.nan), axis=1)
        median = np.nan_to_num(median, nan=0.0)[:, None]
        mad = np.median(np.where(base > 0, np.abs(resid - median), np.nan), axis=1)
        sigma = 1.4826 * np.nan_to_num(mad, nan=0.0)[:, None]
        sigma = np.where(sigma < 1e-6, 1e-6, sigma)

        u_res = (resid - median) / (tukey_c * sigma)
        biweight = np.where(np.abs(u_res) < 1.0, (1.0 - u_res**2) ** 2, 0.0)
        weights = base * biweight

    residual = -coeffs[:, 0]

    too_few = mask.sum(axis=1) < min_neighbours
    if too_few.any():
        residual = residual.copy()
        residual[too_few] = np.nan
        logger.debug(
            "%d/%d vertices had fewer than %d neighbours within %.1f mm; bulge is NaN there",
            int(too_few.sum()),
            len(vertices),
            min_neighbours,
            radius_mm,
        )
    return residual


def reference_surface(
    mesh: trimesh.Trimesh, *, method: str = "quadric"
) -> trimesh.Trimesh:
    """Build an explicit smooth reference surface, for visualisation.

    Parameters
    ----------
    mesh
        Articular surface.
    method
        ``quadric``: one global quadric fitted to all vertices. Suitable for
        tibial plateaus; too rigid for the femoral trochlea.

    Returns
    -------
    trimesh.Trimesh
        Reference surface sharing ``mesh``'s vertex indexing.

    Raises
    ------
    ValueError
        For an unknown method.

    Notes
    -----
    :func:`compute_bulge` does not need this: it works from the *local* fit in
    :func:`local_form_residual`, which adapts to the anatomy instead of forcing
    one global shape.
    """
    if method == "quadric":
        return _global_quadric_reference(mesh)
    raise ValueError(
        f"reference method must be 'quadric', got {method!r}. Taubin smoothing is "
        "deliberately not offered as a reference -- see the module docstring."
    )


def _global_quadric_reference(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    """Project vertices onto one global quadric fitted to the whole surface.

    The surface is expressed in its own principal-axis frame (via PCA), a
    quadratic height field is fitted over the two dominant axes, and each vertex
    is moved to the fitted height. Suitable for tibial plateaus, which really are
    close to a single quadric; too rigid for the femoral trochlea.
    """
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    centre = vertices.mean(axis=0)
    centred = vertices - centre

    _u, _s, vh = np.linalg.svd(centred, full_matrices=False)
    axes = vh                                     # rows: principal directions
    local = centred @ axes.T
    x, y, z = local[:, 0], local[:, 1], local[:, 2]

    design = np.column_stack([np.ones_like(x), x, y, x * x, x * y, y * y])
    coeffs, *_ = np.linalg.lstsq(design, z, rcond=None)
    z_fit = design @ coeffs

    fitted_local = np.column_stack([x, y, z_fit])
    fitted = fitted_local @ axes + centre
    return trimesh.Trimesh(vertices=fitted, faces=mesh.faces.copy(), process=False)


def compute_bulge(
    mesh: trimesh.Trimesh,
    *,
    form_radius_mm: float = 12.0,
    min_height_mm: float = 0.1,
    min_neighbours: int = 12,
    robust_iterations: int = 3,
    mask_boundary: bool = True,
    boundary_margin_scale: float = 0.5,
) -> BulgeResult:
    """Measure the signed bulge field of an articular surface.

    Parameters
    ----------
    mesh
        Articular surface with consistent outward normals.
    form_radius_mm
        Radius over which the smooth anatomical form is fitted. Features much
        narrower than this read as bulge; broader undulations read as form.
    min_height_mm
        Deviations smaller than this are noise, not lesions.
    min_neighbours
        Minimum neighbours for the six-parameter form fit.
    robust_iterations
        IRLS passes used to stop the bulge dragging its own reference form.
    mask_boundary
        Set the residual to NaN near an open boundary, where the fit is
        one-sided.
    boundary_margin_scale
        Boundary margin as a multiple of ``form_radius_mm``. Smaller than the
        ``2x`` used for curvature because only the *constant* term of the fit is
        read out here, and a constant is far better conditioned under a clipped
        neighbourhood than the second-order terms curvature depends on.

    Returns
    -------
    BulgeResult

    Raises
    ------
    ValueError
        If the mesh has no faces.

    Examples
    --------
    A sphere with a Gaussian bump recovers the bump and localises it:

    >>> sphere = trimesh.creation.icosphere(subdivisions=4, radius=20.0)
    >>> v = np.asarray(sphere.vertices)
    >>> unit = v / np.linalg.norm(v, axis=1, keepdims=True)
    >>> weight = np.exp(-((unit - np.array([0.0, 0.0, 1.0])) ** 2).sum(axis=1) / 0.1)
    >>> bumped = sphere.copy()
    >>> bumped.vertices = v + unit * (1.5 * weight)[:, None]
    >>> res = compute_bulge(bumped, form_radius_mm=8.0)
    >>> bool(np.nanmax(res.deviation_mm) > 0.8)
    True
    >>> bool(res.bulge_volume_mm3 > 0)
    True

    while a clean sphere shows essentially none:

    >>> clean = compute_bulge(sphere, form_radius_mm=8.0)
    >>> bool(clean.roughness_mm < 0.05)
    True
    """
    if len(mesh.faces) == 0:
        raise ValueError("cannot measure bulge on a mesh with no faces")

    deviation = local_form_residual(
        mesh,
        radius_mm=form_radius_mm,
        min_neighbours=min_neighbours,
        robust_iterations=robust_iterations,
    )

    if mask_boundary:
        margin = boundary_margin_scale * form_radius_mm
        zone = boundary_influence_zone(mesh, margin)
        if zone.all():
            logger.warning(
                "bulge is not estimable anywhere on this surface: a %.1f mm boundary "
                "margin covers the whole patch. Reduce form_radius_mm (currently "
                "%.1f mm) or accept that this plate is too small for form removal.",
                margin,
                form_radius_mm,
            )
        elif zone.any():
            deviation = deviation.copy()
            deviation[zone] = np.nan

    result = BulgeResult(
        deviation_mm=deviation,
        vertex_area_mm2=vertex_areas(mesh),
        form_radius_mm=form_radius_mm,
        min_height_mm=min_height_mm,
    )
    logger.debug(
        "bulge (form radius %.1f mm): rms=%.3f mm, +area=%.1f mm^2, -area=%.1f mm^2",
        form_radius_mm,
        result.roughness_mm,
        result.bulge_area_mm2,
        result.dent_area_mm2,
    )
    return result


def thickness_residual_bulge(
    bci: trimesh.Trimesh,
    thickness_mm: np.ndarray,
    *,
    smoothing_iterations: int = 30,
) -> np.ndarray:
    """Local thickness excess over a smoothed thickness field.

    A complementary view of bulging that lives on the *thickness map* rather
    than on the surface geometry: positive residual means the plate is locally
    thicker than its neighbourhood, which is what focal swelling looks like in a
    thickness map. Unlike :func:`compute_bulge` it is insensitive to the shape
    of the underlying bone.

    Parameters
    ----------
    bci
        BCI mesh supplying the vertex adjacency.
    thickness_mm
        Per-vertex thickness; NaN allowed and propagated.
    smoothing_iterations
        Diffusion passes defining the "neighbourhood" scale.

    Returns
    -------
    numpy.ndarray
        ``thickness - smoothed(thickness)``, in mm, NaN where thickness is NaN.
    """
    thickness = np.asarray(thickness_mm, dtype=np.float64)
    smoothed = smooth_scalar_field(bci, thickness, smoothing_iterations)
    return thickness - smoothed


def bulge_summary(result: BulgeResult) -> dict[str, float]:
    """Reduce a bulge field to scalar descriptors for the results CSV.

    Parameters
    ----------
    result
        Output of :func:`compute_bulge`.

    Returns
    -------
    dict
        Height, area, volume and roughness descriptors. Areas are also given as
        fractions of the total surface so they compare across knee sizes.
    """
    deviation = result.deviation_mm
    finite = deviation[np.isfinite(deviation)]
    total = float(result.vertex_area_mm2[np.isfinite(deviation)].sum())

    if finite.size == 0:
        return {"bulge_rms_mm": float("nan")}

    return {
        "bulge_rms_mm": result.roughness_mm,
        "bulge_max_height_mm": float(finite.max()),
        "bulge_p95_height_mm": float(np.percentile(finite, 95)),
        "dent_max_depth_mm": float(-finite.min()),
        "dent_p05_depth_mm": float(-np.percentile(finite, 5)),
        "bulge_area_mm2": result.bulge_area_mm2,
        "dent_area_mm2": result.dent_area_mm2,
        "bulge_area_fraction": result.bulge_area_mm2 / total if total else float("nan"),
        "dent_area_fraction": result.dent_area_mm2 / total if total else float("nan"),
        "bulge_volume_mm3": result.bulge_volume_mm3,
        "dent_volume_mm3": result.dent_volume_mm3,
        # Net signed volume: positive means the plate is, on balance, swollen.
        "net_bulge_volume_mm3": result.bulge_volume_mm3 - result.dent_volume_mm3,
        "bulge_skewness": float(
            np.mean(((finite - finite.mean()) / (finite.std() + 1e-12)) ** 3)
        ),
        "bulge_estimable_fraction": float(np.isfinite(deviation).mean()),
        "form_radius_mm": result.form_radius_mm,
    }

### Rule-based 20-region parcellation

`confcarti/thickness/parcellation.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/thickness/parcellation.py
# ==========================================================================
"""Rule-based cartilage parcellation into the 20-region Eckstein/Wirth atlas.

Ported from CartiMorph's ``CM_cal_SurfaceParcellation_FC.m`` and
``CM_cal_SurfaceParcellation_TC.m``. The scheme is *rule-based* rather than
atlas-registration-based on purpose: in advanced OA large parts of the plate are
missing, and a registration-driven parcellation drifts exactly where the disease
is, whereas rules anchored on bone landmarks do not.

Femoral cartilage (10 regions)
    The intercondylar notch is located from the central sagittal slices. Each
    condyle is split anterior / central / posterior along the AP axis, with the
    central strip further divided into external, central and internal thirds
    along the medial-lateral axis:
    ``aLFC ecLFC ccLFC icLFC pLFC`` and ``aMFC ecMFC ccMFC icMFC pMFC``.

Tibial cartilage (10 regions)
    Each plate is split anterior / central / posterior along AP, with the
    central band divided external / central / internal along ML:
    ``aMTC eMTC cMTC iMTC pMTC`` and ``aLTC eLTC cLTC iLTC pLTC``.

Laterality
    ``CM_cal_SurfaceParcellation_TC.m`` swaps the internal and external
    compartments for left knees, because "internal" means *towards the joint
    midline*, which is +x on one side and -x on the other. That mirroring is
    reproduced here; getting it wrong silently relabels ``iMTC`` as ``eMTC``
    for half the cohort, which would show up as a spurious site or sex effect.
"""


import logging
from dataclasses import dataclass

import numpy as np
import trimesh


logger = logging.getLogger(__name__)

__all__ = [
    "ParcellationResult",
    "parcellate_femoral",
    "parcellate_tibial",
    "parcellate",
    "UNASSIGNED",
]

#: Label used for vertices no rule could assign.
UNASSIGNED = "unassigned"


@dataclass(frozen=True, slots=True)
class ParcellationResult:
    """Per-vertex subregion assignment.

    Attributes
    ----------
    labels
        ``(n_vertices,)`` array of subregion name strings.
    subregions
        The subregion names this scheme can produce.
    knee_side
        ``left`` or ``right``.
    landmarks
        Anatomical coordinates the rules were anchored on, for auditing.
    """

    labels: np.ndarray
    subregions: tuple[str, ...]
    knee_side: str
    landmarks: dict[str, float]

    def counts(self) -> dict[str, int]:
        """Vertices per subregion, including empty ones."""
        unique, counts = np.unique(self.labels, return_counts=True)
        found = dict(zip(unique.tolist(), counts.tolist()))
        return {name: int(found.get(name, 0)) for name in self.subregions}

    def mask(self, subregion: str) -> np.ndarray:
        """Boolean mask selecting one subregion."""
        return self.labels == subregion


def _thirds(values: np.ndarray, low: float, high: float) -> tuple[np.ndarray, ...]:
    """Split a coordinate range into three bands, returning boolean masks."""
    span = high - low
    first = low + span / 3.0
    second = low + 2.0 * span / 3.0
    return (values < first, (values >= first) & (values < second), values >= second)


def _locate_notch(vertices: np.ndarray, lr_axis: int, si_axis: int) -> float:
    """Find the medial-lateral coordinate of the intercondylar notch.

    CartiMorph locates the notch from the central sagittal slices. Geometrically
    the notch is where the femoral surface is *highest* (most superior) between
    the two condyles, so the ML coordinate whose local maximum superior extent is
    greatest, within the central half of the ML range, identifies it.

    Parameters
    ----------
    vertices
        ``(n, 3)`` BCI vertices in mm.
    lr_axis, si_axis
        Array axes for medial-lateral and superior-inferior.

    Returns
    -------
    float
        ML coordinate of the notch.
    """
    lr = vertices[:, lr_axis]
    si = vertices[:, si_axis]
    lo, hi = float(lr.min()), float(lr.max())

    # Restrict to the central 50% of the ML range, as the MATLAB does.
    inner_lo = lo + 0.25 * (hi - lo)
    inner_hi = lo + 0.75 * (hi - lo)
    inner = (lr >= inner_lo) & (lr <= inner_hi)
    if inner.sum() < 10:
        return 0.5 * (lo + hi)

    bins = np.linspace(inner_lo, inner_hi, 25)
    which = np.digitize(lr[inner], bins)
    si_inner = si[inner]

    best_value = -np.inf
    best_centre = 0.5 * (lo + hi)
    for b in range(1, len(bins)):
        sel = which == b
        if sel.sum() < 3:
            continue
        # Most superior extent within this ML slab.
        top = float(np.percentile(si_inner[sel], 95))
        if top > best_value:
            best_value = top
            best_centre = 0.5 * (bins[b - 1] + bins[b])
    return float(best_centre)


def parcellate_femoral(
    bci: trimesh.Trimesh,
    *,
    knee_side: str = "right",
    lr_axis: int = 0,
    ap_axis: int = 1,
    si_axis: int = 2,
    central_percentage: float = 0.5,
) -> ParcellationResult:
    """Parcellate the femoral bone-cartilage interface into 10 subregions.

    Parameters
    ----------
    bci
        Femoral BCI mesh, vertices in mm.
    knee_side
        ``left`` or ``right``.
    lr_axis, ap_axis, si_axis
        Array axes for the three anatomical directions.
    central_percentage
        AP extent of the central condylar strip as a fraction of the distance
        from the notch to the anterior cartilage margin. CartiMorph's
        ``cc_percentage``.

    Returns
    -------
    ParcellationResult

    Raises
    ------
    ValueError
        For an unknown ``knee_side``.
    """
    if knee_side not in ("left", "right"):
        raise ValueError(f"knee_side must be left|right, got {knee_side!r}")

    vertices = np.asarray(bci.vertices, dtype=np.float64)
    labels = np.full(len(vertices), UNASSIGNED, dtype=object)

    lr = vertices[:, lr_axis]
    ap = vertices[:, ap_axis]

    notch = _locate_notch(vertices, lr_axis, si_axis)

    # Medial is towards the body midline. With +x pointing right in a
    # right-handed RAS-like frame, the medial condyle of a RIGHT knee sits at
    # smaller x, and of a LEFT knee at larger x.
    if knee_side == "right":
        medial = lr < notch
    else:
        medial = lr >= notch
    lateral = ~medial

    for side_mask, prefix in ((lateral, "LFC"), (medial, "MFC")):
        if side_mask.sum() < 10:
            logger.warning(
                "femoral %s side has only %d vertices; parcellation will be sparse",
                prefix,
                int(side_mask.sum()),
            )
            continue

        side_ap = ap[side_mask]
        ap_lo, ap_hi = float(side_ap.min()), float(side_ap.max())
        span = ap_hi - ap_lo

        # Central strip straddles the middle of the AP range; anterior and
        # posterior take the remainder.
        half = 0.5 * central_percentage * span
        centre = 0.5 * (ap_lo + ap_hi)
        central_lo, central_hi = centre - half, centre + half

        idx = np.flatnonzero(side_mask)
        side_labels = np.full(idx.size, UNASSIGNED, dtype=object)

        anterior = side_ap >= central_hi
        posterior = side_ap < central_lo
        central = ~anterior & ~posterior

        side_labels[anterior] = f"a{prefix}"
        side_labels[posterior] = f"p{prefix}"

        # Subdivide the central strip external / central / internal along ML.
        if central.any():
            central_lr = lr[idx][central]
            lo, hi = float(central_lr.min()), float(central_lr.max())
            band_a, band_b, band_c = _thirds(central_lr, lo, hi)

            # "Internal" = towards the notch. Which ML end that is depends on
            # the condyle and the knee side.
            towards_notch_is_high = (prefix == "MFC") == (knee_side == "right")
            inner_band, outer_band = (band_c, band_a) if towards_notch_is_high else (band_a, band_c)

            central_labels = np.full(central_lr.size, f"cc{prefix}", dtype=object)
            central_labels[outer_band] = f"ec{prefix}"
            central_labels[inner_band] = f"ic{prefix}"
            central_labels[band_b] = f"cc{prefix}"
            side_labels[central] = central_labels

        labels[idx] = side_labels

    return ParcellationResult(
        labels=np.asarray(labels, dtype=object),
        subregions=FC_SUBREGIONS,
        knee_side=knee_side,
        landmarks={"notch_lr_mm": float(notch)},
    )


def parcellate_tibial(
    bci: trimesh.Trimesh,
    *,
    plate: str,
    knee_side: str = "right",
    lr_axis: int = 0,
    ap_axis: int = 1,
    si_axis: int = 2,
) -> ParcellationResult:
    """Parcellate one tibial plate into 5 subregions.

    Parameters
    ----------
    bci
        Tibial BCI mesh for a single plate.
    plate
        ``MTC`` or ``LTC``.
    knee_side
        ``left`` or ``right``; controls the internal/external mirroring.
    lr_axis, ap_axis, si_axis
        Anatomical axes.

    Returns
    -------
    ParcellationResult

    Raises
    ------
    ValueError
        For an unknown plate or knee side.
    """
    if plate not in ("MTC", "LTC"):
        raise ValueError(f"plate must be MTC|LTC, got {plate!r}")
    if knee_side not in ("left", "right"):
        raise ValueError(f"knee_side must be left|right, got {knee_side!r}")

    vertices = np.asarray(bci.vertices, dtype=np.float64)
    labels = np.full(len(vertices), UNASSIGNED, dtype=object)
    lr = vertices[:, lr_axis]
    ap = vertices[:, ap_axis]
    del si_axis

    suffix = plate
    ap_lo, ap_hi = float(ap.min()), float(ap.max())
    posterior, central, anterior = _thirds(ap, ap_lo, ap_hi)

    labels[anterior] = f"a{suffix}"
    labels[posterior] = f"p{suffix}"

    if central.any():
        central_lr = lr[central]
        lo, hi = float(central_lr.min()), float(central_lr.max())
        band_a, band_b, band_c = _thirds(central_lr, lo, hi)

        # Internal = towards the tibial spines (the joint midline).
        towards_midline_is_high = (plate == "MTC") == (knee_side == "right")
        inner_band, outer_band = (band_c, band_a) if towards_midline_is_high else (band_a, band_c)

        central_labels = np.full(central_lr.size, f"c{suffix}", dtype=object)
        central_labels[outer_band] = f"e{suffix}"
        central_labels[inner_band] = f"i{suffix}"
        central_labels[band_b] = f"c{suffix}"
        labels[central] = central_labels

    return ParcellationResult(
        labels=np.asarray(labels, dtype=object),
        subregions=tuple(s for s in TC_SUBREGIONS if s.endswith(suffix)),
        knee_side=knee_side,
        landmarks={"ap_min_mm": ap_lo, "ap_max_mm": ap_hi},
    )


def parcellate(
    bci: trimesh.Trimesh,
    compartment: str,
    *,
    knee_side: str = "right",
    scheme: str = "cartimorph20",
    central_percentage: float = 0.5,
    lr_axis: int = 0,
    ap_axis: int = 1,
    si_axis: int = 2,
) -> ParcellationResult:
    """Parcellate a compartment's BCI according to the configured scheme.

    Parameters
    ----------
    bci
        BCI mesh for the compartment.
    compartment
        ``FC``, ``MTC`` or ``LTC``.
    knee_side
        ``left`` or ``right``.
    scheme
        ``cartimorph20`` for the 20-region atlas, or ``compartment5`` to label
        every vertex with its compartment (no subdivision).
    central_percentage
        CartiMorph's ``cc_percentage``, femoral only.
    lr_axis, ap_axis, si_axis
        Anatomical axes.

    Returns
    -------
    ParcellationResult

    Raises
    ------
    ValueError
        For an unknown compartment or scheme.

    Examples
    --------
    >>> from confcarti.data.synthetic import make_phantom, PhantomSpec
    >>> from confcarti.thickness.mesh import extract_bci_mesh
    >>> from confcarti.data.labels import FEMUR, FEMORAL_CARTILAGE
    >>> knee = make_phantom(PhantomSpec(shape=(64, 64, 48)))
    >>> bci, _ = extract_bci_mesh(knee.label == FEMUR,
    ...                           knee.label == FEMORAL_CARTILAGE, knee.spacing)
    >>> res = parcellate(bci, "FC")
    >>> sum(res.counts().values()) > 0
    True
    """
    if compartment not in ("FC", "MTC", "LTC"):
        raise ValueError(f"compartment must be FC|MTC|LTC, got {compartment!r}")

    if scheme == "compartment5":
        return ParcellationResult(
            labels=np.full(len(bci.vertices), compartment, dtype=object),
            subregions=(compartment,),
            knee_side=knee_side,
            landmarks={},
        )
    if scheme != "cartimorph20":
        raise ValueError(f"scheme must be cartimorph20|compartment5, got {scheme!r}")

    if compartment == "FC":
        return parcellate_femoral(
            bci,
            knee_side=knee_side,
            lr_axis=lr_axis,
            ap_axis=ap_axis,
            si_axis=si_axis,
            central_percentage=central_percentage,
        )
    return parcellate_tibial(
        bci,
        plate=compartment,
        knee_side=knee_side,
        lr_axis=lr_axis,
        ap_axis=ap_axis,
        si_axis=si_axis,
    )


def all_subregions(scheme: str = "cartimorph20") -> tuple[str, ...]:
    """Every subregion the scheme can produce, in canonical order."""
    if scheme == "cartimorph20":
        return ATLAS20_SUBREGIONS
    if scheme == "compartment5":
        return ("FC", "MTC", "LTC")
    raise ValueError(f"unknown scheme {scheme!r}")

### Full-thickness cartilage loss

`confcarti/denuded/fcl.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/denuded/fcl.py
# ==========================================================================
"""Full-thickness cartilage loss: denuded-bone labels, metrics and prior gating.

Clinical background
-------------------
A *denuded* area of subchondral bone (dAB) is bone no longer covered by
cartilage. Cartilage thickness is **undefined** there -- not zero-and-measured,
but absent. The standard nomenclature (Eckstein et al. 2006) distinguishes:

``tAB``
    total subchondral bone area of the plate.
``cAB``
    the cartilage-covered part of tAB.
``dAB``
    the denuded part, ``tAB - cAB``.
``ThCtAB``
    mean thickness over the **total** bone area, denuded vertices counted as
    0 mm. Falls as cartilage is lost, so it tracks disease.
``ThCcAB``
    mean thickness over the **covered** area only. Can stay flat or even rise in
    advanced disease, because the thinnest regions stop contributing once they
    denude entirely.

Reporting only ThCcAB in a severe-OA cohort therefore hides the very loss the
study is about, which is why both are always emitted here and why
``ThCtAB <= ThCcAB`` is enforced as a postcondition.

The NaN / 0 distinction
-----------------------
Throughout ConfCarti, a thickness of ``NaN`` means *undefined* (the measurement
failed: the ray missed, or left the volume) while ``0.0`` means *denuded* (there
is genuinely no cartilage). ``ThCtAB`` deliberately substitutes 0 for denuded
vertices, and deliberately *excludes* NaN vertices, because averaging a failed
measurement as if it were bone loss manufactures disease.

References
----------
Eckstein, F. et al. (2006). "Proposal for a nomenclature for magnetic resonance
    imaging based measures of articular cartilage in osteoarthritis."
    Osteoarthritis and Cartilage 14(10):974-983.
Yao, Y. et al. (2024). "CartiMorph." Medical Image Analysis 91:103035 -- the
    full-thickness-cartilage-loss estimation reproduced here.
"""


import logging
from dataclasses import dataclass

import numpy as np
import trimesh

logger = logging.getLogger(__name__)

__all__ = [
    "DenudedResult",
    "derive_denuded_labels",
    "compute_thctab",
    "compute_thccab",
    "denuded_metrics",
    "prior_gate",
]


@dataclass(frozen=True, slots=True)
class DenudedResult:
    """Per-vertex denuded classification for one cartilage plate.

    Attributes
    ----------
    denuded
        Boolean mask; True where subchondral bone is uncovered.
    vertex_area_mm2
        Barycentric vertex areas on the BCI.
    thickness_mm
        Thickness map, with NaN preserved where undefined.
    rule
        Human-readable description of how the labels were derived.
    """

    denuded: np.ndarray
    vertex_area_mm2: np.ndarray
    thickness_mm: np.ndarray
    rule: str

    @property
    def tab_mm2(self) -> float:
        """Total subchondral bone area."""
        return float(self.vertex_area_mm2.sum())

    @property
    def dab_mm2(self) -> float:
        """Denuded subchondral bone area."""
        return float(self.vertex_area_mm2[self.denuded].sum())

    @property
    def cab_mm2(self) -> float:
        """Cartilage-covered subchondral bone area."""
        return self.tab_mm2 - self.dab_mm2

    @property
    def dab_percent(self) -> float:
        """dAB as a percentage of tAB -- the ``FCL%`` of the CartiMorph paper."""
        total = self.tab_mm2
        return 100.0 * self.dab_mm2 / total if total > 0 else float("nan")


def derive_denuded_labels(
    thickness_mm: np.ndarray,
    bci: trimesh.Trimesh,
    *,
    vertex_area_mm2: np.ndarray | None = None,
    min_thickness_mm: float = 0.35,
    min_patch_area_mm2: float = 1.0,
    treat_nan_as_denuded: bool = False,
    denuded_probe: np.ndarray | None = None,
) -> DenudedResult:
    """Derive ground-truth denuded labels from a thickness map.

    The rule, stated exactly
    ------------------------
    A BCI vertex is labelled denuded when **either**

    * its measured thickness is below ``min_thickness_mm`` -- the resolution
      floor below which a cartilage layer cannot be resolved at DESS voxel
      sizes, so "present but thinner than this" is not distinguishable from
      "absent"; **or**
    * ``treat_nan_as_denuded`` is set and the thickness is NaN, i.e. the
      surface-normal ray found no articular surface to hit at all.

    Isolated denuded vertices forming a connected patch smaller than
    ``min_patch_area_mm2`` are then reverted to covered, which suppresses
    single-vertex speckle from segmentation noise without touching real lesions.

    ``treat_nan_as_denuded`` defaults to **False**, and that default matters. A
    NaN arises both from genuine full-thickness loss *and* from a ray that
    grazed the plate margin, so treating all NaN as loss inflates dAB at the
    rim of every plate -- a bias that grows with plate perimeter and would
    masquerade as disease in small knees.

    Parameters
    ----------
    thickness_mm
        Per-BCI-vertex thickness; NaN allowed and meaningful.
    bci
        BCI mesh, for vertex areas and connectivity.
    vertex_area_mm2
        Precomputed vertex areas; computed from ``bci`` when omitted.
    min_thickness_mm
        Resolution floor.
    min_patch_area_mm2
        Minimum connected denuded patch area.
    treat_nan_as_denuded
        See above.
    denuded_probe
        Optional boolean mask from :func:`denuded_from_cartilage_probe`. When
        supplied it is unioned with the thickness rule, which is what actually
        catches full-thickness loss: a defect large enough to have no cartilage
        above it returns NaN thickness, not a small one, so the thickness
        threshold alone never fires there.

    Returns
    -------
    DenudedResult

    Examples
    --------
    A plate with a hole recovers roughly the hole's area:

    >>> import trimesh, numpy as np
    >>> from confcarti.thickness.bulge import vertex_areas
    >>> plate = trimesh.creation.box(extents=(20, 20, 0.01)).subdivide().subdivide()
    >>> t = np.full(len(plate.vertices), 2.0)
    >>> v = np.asarray(plate.vertices)
    >>> t[(v[:, 0] ** 2 + v[:, 1] ** 2) < 3.0 ** 2] = 0.0
    >>> res = derive_denuded_labels(t, plate)
    >>> bool(res.dab_mm2 > 0)
    True
    """
    thickness = np.asarray(thickness_mm, dtype=np.float64)
    areas = vertex_areas(bci) if vertex_area_mm2 is None else np.asarray(vertex_area_mm2)

    below = np.zeros(thickness.shape, dtype=bool)
    finite = np.isfinite(thickness)
    below[finite] = thickness[finite] < min_thickness_mm

    if treat_nan_as_denuded:
        below |= ~finite

    if denuded_probe is not None:
        probe = np.asarray(denuded_probe, dtype=bool)
        if probe.shape != below.shape:
            raise ValueError(
                f"denuded_probe shape {probe.shape} != thickness shape {below.shape}"
            )
        below |= probe

    denuded = _drop_small_patches(below, bci, areas, min_patch_area_mm2)

    rule = (
        f"thickness < {min_thickness_mm} mm"
        + (" or thickness is NaN" if treat_nan_as_denuded else "")
        + (" or no cartilage above the vertex (mask probe)" if denuded_probe is not None else "")
        + f"; connected patches < {min_patch_area_mm2} mm^2 reverted to covered"
    )
    result = DenudedResult(
        denuded=denuded, vertex_area_mm2=areas, thickness_mm=thickness, rule=rule
    )
    logger.debug(
        "denuded: dAB=%.1f mm^2 (%.1f%% of tAB=%.1f mm^2)",
        result.dab_mm2,
        result.dab_percent,
        result.tab_mm2,
    )
    return result


def denuded_from_cartilage_probe(
    bci: trimesh.Trimesh,
    normals: np.ndarray,
    cartilage_mask: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    probe_mm: float = 1.5,
    n_probes: int = 6,
    lateral_tolerance_mm: float = 1.6,
) -> np.ndarray:
    """Label a BCI vertex denuded when no cartilage sits above it.

    This is the definition taken literally -- *bone not covered by cartilage* --
    and it is the reliable one. Deriving denudation from the thickness map alone
    does not work: at a full-thickness defect the surface-normal ray finds no
    articular surface to hit and returns NaN, which is indistinguishable from
    the NaN produced by a ray grazing the plate margin. Probing the cartilage
    mask along the vertex normal separates the two cases directly, because at
    the margin there is still cartilage beside the vertex whereas inside a
    lesion there is none.

    Parameters
    ----------
    bci
        Bone-cartilage interface mesh (which must already include denuded bone;
        see the closing step in :func:`confcarti.thickness.mesh.extract_bci_mesh`).
    normals
        Per-vertex unit normals oriented into the cartilage.
    cartilage_mask
        The **original**, unclosed cartilage mask.
    spacing
        Voxel size in mm.
    probe_mm
        How far along the normal to look for cartilage.
    n_probes
        Samples along the probe.
    lateral_tolerance_mm
        How far *sideways* cartilage may be and still count as covering the
        vertex.

        This tolerance is what separates a lesion from the edge of the plate,
        and it is not optional. The interface is built out to
        ``interface_distance_mm`` around the (closed) cartilage, so a rim of
        bone one interface-width wide always lies just beyond the true plate
        margin. A strictly-along-the-normal probe finds no cartilage there and
        marks the entire rim denuded -- on a phantom femoral plate that is about
        286 mm^2, or 12.8% of tAB, reported identically at KL 0 and KL 4. It
        looks like a plausible dAB and is pure geometry.

        Setting the tolerance to roughly the interface distance means a margin
        vertex, which still has cartilage beside it, counts as covered, while
        the interior of a defect wider than the tolerance does not. The
        consequence is an explicit detection floor. A circular defect of radius
        ``r`` is recovered as one of radius ``r - lateral_tolerance_mm``, so its
        **area** is under-estimated by ``((r - tol) / r)^2`` and defects with
        ``r <= tol`` vanish entirely. Measured on the phantom with
        ``tol = 1.6 mm``: a 2.0 mm-radius lesion is missed completely, 3.0 mm
        lesions recover 35% of their true area, and 4.5 mm lesions recover 30%.
        dAB is therefore deliberately *conservative* -- it under-reports
        full-thickness loss rather than inventing it, and gives exactly zero
        false-positive dAB on healthy phantoms.

    Returns
    -------
    numpy.ndarray
        Boolean denuded mask.
    """
    from scipy import ndimage

    vertices = np.asarray(bci.vertices, dtype=np.float64)
    normals = np.asarray(normals, dtype=np.float64)
    mask = np.asarray(cartilage_mask) > 0
    spacing_arr = np.asarray(spacing, dtype=np.float64)
    shape = np.asarray(mask.shape)

    if not mask.any():
        return np.ones(len(vertices), dtype=bool)

    # Distance from every voxel to the nearest cartilage voxel.
    distance = ndimage.distance_transform_edt(~mask, sampling=spacing)

    covered = np.zeros(len(vertices), dtype=bool)
    for step in np.linspace(0.0, probe_mm, n_probes):
        probe = vertices + normals * step
        voxel = np.rint(probe / spacing_arr).astype(int)
        inside = np.all((voxel >= 0) & (voxel < shape), axis=1)
        voxel = np.clip(voxel, 0, shape - 1)
        near = distance[voxel[:, 0], voxel[:, 1], voxel[:, 2]] <= lateral_tolerance_mm
        covered |= inside & near

    return ~covered


def _drop_small_patches(
    mask: np.ndarray,
    mesh: trimesh.Trimesh,
    areas: np.ndarray,
    min_area_mm2: float,
) -> np.ndarray:
    """Revert connected components of ``mask`` below ``min_area_mm2`` to False."""
    if min_area_mm2 <= 0 or not mask.any():
        return mask

    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components

    n = len(mesh.vertices)
    edges = mesh.edges_sorted
    # Keep only edges whose two endpoints are both inside the mask.
    keep = mask[edges[:, 0]] & mask[edges[:, 1]]
    edges = edges[keep]

    if edges.size == 0:
        # Every masked vertex is isolated; each is its own component.
        out = mask.copy()
        out[areas < min_area_mm2] &= False
        return out & mask

    data = np.ones(len(edges) * 2, dtype=np.int8)
    rows = np.concatenate([edges[:, 0], edges[:, 1]])
    cols = np.concatenate([edges[:, 1], edges[:, 0]])
    graph = coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()

    n_components, labels = connected_components(graph, directed=False)
    out = mask.copy()
    for component in range(n_components):
        members = (labels == component) & mask
        if not members.any():
            continue
        if float(areas[members].sum()) < min_area_mm2:
            out[members] = False
    return out


def compute_thctab(
    thickness_mm: np.ndarray,
    denuded_mask: np.ndarray,
    areas_mm2: np.ndarray,
) -> float:
    """Area-weighted mean thickness over the TOTAL bone area, denuded = 0 mm.

    .. math::
        \\mathrm{ThCtAB} = \\frac{\\sum_i a_i \\, t_i}{\\sum_i a_i},
        \\qquad t_i = 0 \\text{ where denuded}

    Vertices whose thickness is NaN (undefined measurement, not bone loss) are
    excluded from both sums.

    Parameters
    ----------
    thickness_mm
        Per-vertex thickness; NaN = undefined.
    denuded_mask
        Boolean denuded mask.
    areas_mm2
        Per-vertex BCI areas.

    Returns
    -------
    float
        Mean thickness in mm, or NaN when no vertex is usable.

    Examples
    --------
    >>> t = np.array([2.0, 2.0, 0.0, 2.0])
    >>> d = np.array([False, False, True, False])
    >>> a = np.ones(4)
    >>> float(compute_thctab(t, d, a))
    1.5
    """
    thickness = np.asarray(thickness_mm, dtype=np.float64)
    denuded = np.asarray(denuded_mask, dtype=bool)
    areas = np.asarray(areas_mm2, dtype=np.float64)

    values = np.where(denuded, 0.0, thickness)
    usable = np.isfinite(values) & (areas > 0)
    if not usable.any():
        return float("nan")
    return float(np.sum(areas[usable] * values[usable]) / np.sum(areas[usable]))


def compute_thccab(
    thickness_mm: np.ndarray,
    denuded_mask: np.ndarray,
    areas_mm2: np.ndarray,
) -> float:
    """Area-weighted mean thickness over the CARTILAGE-COVERED area only.

    .. math::
        \\mathrm{ThCcAB} =
        \\frac{\\sum_{i \\notin dAB} a_i t_i}{\\sum_{i \\notin dAB} a_i}

    Parameters
    ----------
    thickness_mm
        Per-vertex thickness; NaN = undefined.
    denuded_mask
        Boolean denuded mask.
    areas_mm2
        Per-vertex BCI areas.

    Returns
    -------
    float
        Mean thickness in mm over covered bone, NaN if nothing is covered.

    Examples
    --------
    >>> t = np.array([2.0, 2.0, 0.0, 2.0])
    >>> d = np.array([False, False, True, False])
    >>> float(compute_thccab(t, d, np.ones(4)))
    2.0
    """
    thickness = np.asarray(thickness_mm, dtype=np.float64)
    denuded = np.asarray(denuded_mask, dtype=bool)
    areas = np.asarray(areas_mm2, dtype=np.float64)

    usable = ~denuded & np.isfinite(thickness) & (areas > 0)
    if not usable.any():
        return float("nan")
    return float(np.sum(areas[usable] * thickness[usable]) / np.sum(areas[usable]))


def denuded_metrics(result: DenudedResult) -> dict[str, float]:
    """Full set of denuded / thickness metrics for one plate.

    Parameters
    ----------
    result
        Output of :func:`derive_denuded_labels`.

    Returns
    -------
    dict
        ``tAB_mm2``, ``cAB_mm2``, ``dAB_mm2``, ``dAB_percent``, ``ThCtAB_mm``,
        ``ThCcAB_mm`` and the undefined fraction.

    Raises
    ------
    AssertionError
        If ``ThCtAB > ThCcAB``, which is mathematically impossible: replacing
        positive thicknesses by zero cannot raise the mean. Tripping this means
        the masks and the thickness map are misaligned.
    """
    thctab = compute_thctab(result.thickness_mm, result.denuded, result.vertex_area_mm2)
    thccab = compute_thccab(result.thickness_mm, result.denuded, result.vertex_area_mm2)

    if np.isfinite(thctab) and np.isfinite(thccab):
        # Allow a hair of floating-point slack.
        assert thctab <= thccab + 1e-9, (
            f"ThCtAB ({thctab:.4f}) > ThCcAB ({thccab:.4f}); this is impossible and "
            "means the denuded mask, thickness map and vertex areas are not aligned"
        )

    return {
        "tAB_mm2": result.tab_mm2,
        "cAB_mm2": result.cab_mm2,
        "dAB_mm2": result.dab_mm2,
        "dAB_percent": result.dab_percent,
        "ThCtAB_mm": thctab,
        "ThCcAB_mm": thccab,
        "denuded_vertex_fraction": float(np.mean(result.denuded)),
        "undefined_vertex_fraction": float(np.mean(~np.isfinite(result.thickness_mm))),
    }


def prior_gate(
    denuded_mask: np.ndarray,
    *,
    strength: float = 1.0,
    enabled: bool = True,
    soften: np.ndarray | None = None,
) -> np.ndarray:
    """Per-vertex weight in [0, 1] disabling smoothness priors over denuded bone.

    Any continuity, parallelism or smoothness prior on the thickness field
    assumes cartilage is *there*. Over a denuded patch it is not, so the prior
    interpolates a plausible-looking thickness across a hole and fills in
    cartilage that does not exist -- systematically under-estimating dAB in
    exactly the advanced-OA knees where dAB matters most.

    This gate returns ``0`` on denuded vertices and ``strength`` elsewhere, so a
    downstream regulariser can multiply its weights by the gate and switch
    itself off over holes.

    Parameters
    ----------
    denuded_mask
        Boolean denuded mask.
    strength
        Prior weight on covered vertices. Swept by ablation A6.
    enabled
        When False the gate is uniform ``strength`` -- i.e. no gating at all.
        This is ablation A2's "off" arm.
    soften
        Optional per-vertex probability of being denuded, in [0, 1]. When given,
        the gate becomes ``strength * (1 - p)``, which lets an uncertain
        classifier down-weight rather than hard-switch the prior.

    Returns
    -------
    numpy.ndarray
        Per-vertex weights.

    Examples
    --------
    >>> d = np.array([False, True, False])
    >>> prior_gate(d).tolist()
    [1.0, 0.0, 1.0]
    >>> prior_gate(d, enabled=False).tolist()
    [1.0, 1.0, 1.0]
    >>> prior_gate(d, strength=0.5).tolist()
    [0.5, 0.0, 0.5]
    """
    if strength < 0:
        raise ValueError(f"strength must be >= 0, got {strength}")

    denuded = np.asarray(denuded_mask, dtype=bool)
    if not enabled:
        return np.full(denuded.shape, float(strength), dtype=np.float64)

    if soften is not None:
        probability = np.clip(np.asarray(soften, dtype=np.float64), 0.0, 1.0)
        return float(strength) * (1.0 - probability)

    return np.where(denuded, 0.0, float(strength)).astype(np.float64)

### Split conformal: absolute, normalized, CQR

`confcarti/conformal/core.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/conformal/core.py
# ==========================================================================
"""Split conformal prediction: absolute, normalized and CQR scores.

This module is pure statistics. It takes predictions and calibration data and
returns intervals. It deliberately imports no torch, no model code and nothing
from the training pipeline, so that the coverage guarantee cannot be quietly
invalidated by a stray dependency on something that saw the calibration split.

The guarantee
-------------
For exchangeable :math:`(X_i, Y_i)_{i=1}^{n}` and a test point
:math:`(X_{n+1}, Y_{n+1})` drawn from the same distribution, split conformal
prediction with score :math:`s` and

.. math:: \\hat{q} = \\text{the } \\lceil (n+1)(1-\\alpha) \\rceil
          \\text{-th smallest of } \\{s_i\\}_{i=1}^{n}

yields :math:`\\mathbb{P}(Y_{n+1} \\in C(X_{n+1})) \\ge 1 - \\alpha`. The bound is
*finite-sample* and *distribution-free*: it needs no assumption about the model,
only exchangeability. It is also **marginal** -- averaged over draws of the
calibration set and the test point -- which is precisely why
:mod:`confcarti.conformal.schemes` adds Mondrian grouping to obtain coverage
conditional on KL grade and site.

References
----------
Vovk, V., Gammerman, A., Shafer, G. (2005). *Algorithmic Learning in a Random
    World.* Springer.
Lei, J. et al. (2018). "Distribution-free predictive inference for regression."
    JASA 113(523):1094-1111.
Romano, Y., Patterson, E., Candes, E.J. (2019). "Conformalized quantile
    regression." NeurIPS 32.
Angelopoulos, A.N., Bates, S. (2023). "Conformal prediction: a gentle
    introduction." Foundations and Trends in ML 16(4):494-591.
"""


import logging
import math
from dataclasses import dataclass

import numpy as np

logger = logging.getLogger(__name__)

__all__ = [
    "ConformalError",
    "PredictionInterval",
    "min_calibration_size",
    "conformal_quantile",
    "absolute_scores",
    "normalized_scores",
    "cqr_scores",
    "split_conformal",
    "normalized_conformal",
    "cqr_conformal",
]


class ConformalError(ValueError):
    """Raised when a conformal procedure cannot be carried out validly."""


@dataclass(frozen=True, slots=True)
class PredictionInterval:
    """Conformal prediction intervals for a set of test points.

    Attributes
    ----------
    lower, upper
        Interval endpoints, same length as the test set.
    q_hat
        The calibrated threshold(s). Scalar for marginal schemes, per-test-point
        for Mondrian and weighted schemes.
    alpha
        Target miscoverage.
    n_calibration
        Calibration-set size the threshold came from.
    score
        Name of the conformity score used.
    extras
        Scheme-specific diagnostics (merged groups, effective sample size, ...).
    """

    lower: np.ndarray
    upper: np.ndarray
    q_hat: np.ndarray | float
    alpha: float
    n_calibration: int
    score: str
    extras: dict[str, object] | None = None

    @property
    def width(self) -> np.ndarray:
        """Interval widths."""
        return self.upper - self.lower

    def contains(self, truth: np.ndarray) -> np.ndarray:
        """Boolean mask of which intervals cover their truth."""
        truth = np.asarray(truth, dtype=np.float64)
        return (truth >= self.lower) & (truth <= self.upper)

    def coverage(self, truth: np.ndarray) -> float:
        """Empirical marginal coverage against the supplied truth."""
        contained = self.contains(truth)
        return float(np.mean(contained)) if contained.size else float("nan")


def min_calibration_size(alpha: float) -> int:
    """Smallest ``n`` for which the conformal quantile exists.

    The threshold is the :math:`\\lceil (n+1)(1-\\alpha) \\rceil`-th order
    statistic of ``n`` scores, so it must satisfy
    :math:`\\lceil (n+1)(1-\\alpha) \\rceil \\le n`, i.e.
    :math:`n \\ge \\lceil 1/\\alpha \\rceil - 1`.

    Parameters
    ----------
    alpha
        Target miscoverage in (0, 1).

    Returns
    -------
    int

    Examples
    --------
    >>> min_calibration_size(0.1)
    9
    >>> min_calibration_size(0.05)
    19
    """
    if not 0.0 < alpha < 1.0:
        raise ConformalError(f"alpha must be in (0, 1), got {alpha}")
    return int(math.ceil(1.0 / alpha)) - 1


def conformal_quantile(scores: np.ndarray, alpha: float, *, strict: bool = True) -> float:
    """The finite-sample-corrected conformal quantile of a score sample.

    Parameters
    ----------
    scores
        Calibration conformity scores. Non-finite entries are rejected.
    alpha
        Target miscoverage.
    strict
        Raise when ``n`` is too small for the quantile to exist. When False,
        return ``+inf`` (an infinite-width, trivially valid interval) instead --
        appropriate only for a documented small-group fallback.

    Returns
    -------
    float
        The threshold ``q_hat``.

    Raises
    ------
    ConformalError
        If scores are empty, contain non-finite values, or ``n`` is too small
        and ``strict`` is set.

    Examples
    --------
    With n = 9 and alpha = 0.1 the threshold is the 9th (largest) score:

    >>> float(conformal_quantile(np.arange(1.0, 10.0), 0.1))
    9.0
    >>> conformal_quantile(np.arange(1.0, 9.0), 0.1)
    Traceback (most recent call last):
        ...
    confcarti.conformal.core.ConformalError: calibration set of size 8 ...
    """
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    if scores.size == 0:
        raise ConformalError("cannot compute a conformal quantile from an empty score set")
    if not np.isfinite(scores).all():
        n_bad = int((~np.isfinite(scores)).sum())
        raise ConformalError(
            f"{n_bad} calibration score(s) are NaN or infinite. A NaN score means the "
            "target or the prediction was undefined for that calibration point; drop "
            "those points explicitly rather than letting them silently widen or "
            "invalidate the threshold."
        )

    n = scores.size
    needed = min_calibration_size(alpha)
    k = int(math.ceil((n + 1) * (1.0 - alpha)))

    if k > n:
        message = (
            f"calibration set of size {n} is too small for alpha={alpha}: the "
            f"ceil((n+1)(1-alpha)) = {k}-th order statistic does not exist. "
            f"At least {needed} calibration points are required."
        )
        if strict:
            raise ConformalError(message)
        logger.warning("%s Returning an infinite threshold.", message)
        return float("inf")

    return float(np.sort(scores)[k - 1])


# --------------------------------------------------------------------------- #
# Conformity scores
# --------------------------------------------------------------------------- #


def absolute_scores(truth: np.ndarray, prediction: np.ndarray) -> np.ndarray:
    """Absolute-residual score ``s_i = |T_i - That_i|``.

    Parameters
    ----------
    truth, prediction
        Matching arrays of scalar targets and predictions.

    Returns
    -------
    numpy.ndarray

    Examples
    --------
    >>> absolute_scores(np.array([1.0, 2.0]), np.array([1.5, 1.0])).tolist()
    [0.5, 1.0]
    """
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    if truth.shape != prediction.shape:
        raise ConformalError(
            f"truth shape {truth.shape} != prediction shape {prediction.shape}"
        )
    return np.abs(truth - prediction)


def normalized_scores(
    truth: np.ndarray,
    prediction: np.ndarray,
    sigma: np.ndarray,
    *,
    sigma_floor: float = 1e-3,
) -> np.ndarray:
    """Locally-adaptive score ``s_i = |T_i - That_i| / sigma_i``.

    Dividing by an estimate of local difficulty makes the interval width vary
    with the input: wide where the model is unreliable, narrow where it is not,
    at the same marginal coverage.

    Parameters
    ----------
    truth, prediction
        Targets and predictions.
    sigma
        Strictly positive heteroscedastic scale estimate.
    sigma_floor
        Lower clip on ``sigma``, so a near-zero scale cannot blow the score up.

    Returns
    -------
    numpy.ndarray

    Raises
    ------
    ConformalError
        If ``sigma`` has a non-positive or non-finite entry.

    Notes
    -----
    ``sigma`` must be fitted on the *training* split. Fitting it on calibration
    data makes the scores depend on the calibration labels and destroys
    exchangeability -- the guarantee then no longer holds, silently.
    """
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)

    if not (truth.shape == prediction.shape == sigma.shape):
        raise ConformalError(
            f"shape mismatch: truth {truth.shape}, prediction {prediction.shape}, "
            f"sigma {sigma.shape}"
        )
    if not np.isfinite(sigma).all():
        raise ConformalError("sigma contains non-finite values")
    if (sigma <= 0).any():
        raise ConformalError(
            f"{int((sigma <= 0).sum())} sigma value(s) are <= 0; the normalized score "
            "would divide by zero. Clip sigma at a positive floor before calling."
        )
    return np.abs(truth - prediction) / np.maximum(sigma, sigma_floor)


def cqr_scores(truth: np.ndarray, q_lo: np.ndarray, q_hi: np.ndarray) -> np.ndarray:
    """Conformalized quantile regression score.

    .. math:: s_i = \\max(\\hat{q}_{lo}(X_i) - Y_i, \\; Y_i - \\hat{q}_{hi}(X_i))

    The score is negative when the point falls comfortably inside the predicted
    quantile band, which lets CQR *shrink* an over-wide band as well as widen an
    over-narrow one.

    Parameters
    ----------
    truth
        Targets.
    q_lo, q_hi
        Predicted lower and upper conditional quantiles at levels
        ``alpha/2`` and ``1 - alpha/2``.

    Returns
    -------
    numpy.ndarray

    Examples
    --------
    Inside the band the score is negative; outside, it is the overshoot:

    >>> cqr_scores(np.array([1.0]), np.array([0.0]), np.array([2.0])).tolist()
    [-1.0]
    >>> cqr_scores(np.array([3.0]), np.array([0.0]), np.array([2.0])).tolist()
    [1.0]
    """
    truth = np.asarray(truth, dtype=np.float64)
    q_lo = np.asarray(q_lo, dtype=np.float64)
    q_hi = np.asarray(q_hi, dtype=np.float64)
    if not (truth.shape == q_lo.shape == q_hi.shape):
        raise ConformalError(
            f"shape mismatch: truth {truth.shape}, q_lo {q_lo.shape}, q_hi {q_hi.shape}"
        )
    crossed = int(np.sum(q_lo > q_hi))
    if crossed:
        logger.warning(
            "%d calibration point(s) have q_lo > q_hi (quantile crossing). CQR stays "
            "valid, but the underlying quantile regressor is misspecified.",
            crossed,
        )
    return np.maximum(q_lo - truth, truth - q_hi)


# --------------------------------------------------------------------------- #
# Interval constructors
# --------------------------------------------------------------------------- #


def split_conformal(
    calibration_truth: np.ndarray,
    calibration_prediction: np.ndarray,
    test_prediction: np.ndarray,
    alpha: float = 0.1,
    *,
    strict: bool = True,
) -> PredictionInterval:
    """Split conformal intervals with the absolute-residual score.

    Parameters
    ----------
    calibration_truth, calibration_prediction
        Calibration targets and predictions.
    test_prediction
        Predictions at the test points.
    alpha
        Target miscoverage.
    strict
        Forwarded to :func:`conformal_quantile`.

    Returns
    -------
    PredictionInterval
        ``[That - q_hat, That + q_hat]``.

    Examples
    --------
    >>> rng = np.random.default_rng(0)
    >>> y = rng.normal(size=500); yhat = np.zeros(500)
    >>> pi = split_conformal(y, yhat, np.zeros(2000), alpha=0.1)
    >>> bool(abs(pi.coverage(rng.normal(size=2000)) - 0.9) < 0.05)
    True
    """
    scores = absolute_scores(calibration_truth, calibration_prediction)
    q_hat = conformal_quantile(scores, alpha, strict=strict)
    test_prediction = np.asarray(test_prediction, dtype=np.float64)
    return PredictionInterval(
        lower=test_prediction - q_hat,
        upper=test_prediction + q_hat,
        q_hat=q_hat,
        alpha=alpha,
        n_calibration=int(np.asarray(scores).size),
        score="absolute",
    )


def normalized_conformal(
    calibration_truth: np.ndarray,
    calibration_prediction: np.ndarray,
    calibration_sigma: np.ndarray,
    test_prediction: np.ndarray,
    test_sigma: np.ndarray,
    alpha: float = 0.1,
    *,
    sigma_floor: float = 1e-3,
    strict: bool = True,
) -> PredictionInterval:
    """Locally-adaptive conformal intervals ``That +- q_hat * sigma(x)``.

    Parameters
    ----------
    calibration_truth, calibration_prediction, calibration_sigma
        Calibration targets, predictions and scale estimates.
    test_prediction, test_sigma
        Test predictions and scale estimates.
    alpha
        Target miscoverage.
    sigma_floor
        Lower clip on the scale.
    strict
        Forwarded to :func:`conformal_quantile`.

    Returns
    -------
    PredictionInterval
    """
    scores = normalized_scores(
        calibration_truth, calibration_prediction, calibration_sigma, sigma_floor=sigma_floor
    )
    q_hat = conformal_quantile(scores, alpha, strict=strict)

    test_prediction = np.asarray(test_prediction, dtype=np.float64)
    test_sigma = np.maximum(np.asarray(test_sigma, dtype=np.float64), sigma_floor)
    half_width = q_hat * test_sigma

    return PredictionInterval(
        lower=test_prediction - half_width,
        upper=test_prediction + half_width,
        q_hat=q_hat,
        alpha=alpha,
        n_calibration=int(scores.size),
        score="normalized",
    )


def cqr_conformal(
    calibration_truth: np.ndarray,
    calibration_q_lo: np.ndarray,
    calibration_q_hi: np.ndarray,
    test_q_lo: np.ndarray,
    test_q_hi: np.ndarray,
    alpha: float = 0.1,
    *,
    strict: bool = True,
) -> PredictionInterval:
    """Conformalized quantile regression intervals.

    Interval: ``[q_lo(x) - q_hat, q_hi(x) + q_hat]``.

    Parameters
    ----------
    calibration_truth, calibration_q_lo, calibration_q_hi
        Calibration targets and predicted quantiles.
    test_q_lo, test_q_hi
        Predicted quantiles at the test points.
    alpha
        Target miscoverage.
    strict
        Forwarded to :func:`conformal_quantile`.

    Returns
    -------
    PredictionInterval

    References
    ----------
    Romano, Patterson & Candes (2019), NeurIPS. The score and the symmetric
    quantile adjustment match their reference implementation
    (``QuantileRegErrFunc`` in ``cqr/nonconformist/nc.py``) exactly.
    """
    scores = cqr_scores(calibration_truth, calibration_q_lo, calibration_q_hi)
    q_hat = conformal_quantile(scores, alpha, strict=strict)

    test_q_lo = np.asarray(test_q_lo, dtype=np.float64)
    test_q_hi = np.asarray(test_q_hi, dtype=np.float64)

    return PredictionInterval(
        lower=test_q_lo - q_hat,
        upper=test_q_hi + q_hat,
        q_hat=q_hat,
        alpha=alpha,
        n_calibration=int(scores.size),
        score="cqr",
    )

### Mondrian and weighted conformal

`confcarti/conformal/schemes.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/conformal/schemes.py
# ==========================================================================
"""Mondrian (group-conditional) and weighted (covariate-shift) conformal.

Marginal conformal prediction guarantees coverage *on average over the whole
population*. That is not what a clinical claim needs. A method can hold 90%
marginal coverage while covering 97% of KL-0 knees and 71% of KL-4 knees --
failing exactly the patients the measurement exists for. These two schemes are
the fixes:

``mondrian``
    Calibrate a separate threshold within each group and apply the test point's
    own group threshold. Gives coverage conditional on the grouping variable.

``weighted``
    When the test distribution is shifted relative to calibration (a new site, a
    new scanner), reweight the calibration scores by the likelihood ratio
    ``w(x) = dP_test/dP_calib`` before taking the quantile.

References
----------
Vovk, V. (2012). "Conditional validity of inductive conformal predictors." ACML.
Tibshirani, R.J., Barber, R.F., Candes, E.J., Ramdas, A. (2019). "Conformal
    prediction under covariate shift." NeurIPS 32.
Romano, Y., Barber, R.F., Sabatti, C., Candes, E.J. (2020). "With malice toward
    none: assessing uncertainty via equalized coverage." HDSR 2(2).
"""


import logging
from dataclasses import dataclass

import numpy as np


logger = logging.getLogger(__name__)

__all__ = [
    "GroupThresholds",
    "mondrian_conformal",
    "weighted_quantile",
    "weighted_conformal",
    "estimate_likelihood_ratio",
]


@dataclass(frozen=True, slots=True)
class GroupThresholds:
    """Per-group conformal thresholds and how they were arrived at.

    Attributes
    ----------
    thresholds
        Group key -> threshold.
    sizes
        Group key -> calibration count actually used.
    merged
        Group key -> the pooled key it was merged into, for undersized groups.
    fallback_used
        Whether any group fell back to the pooled threshold.
    """

    thresholds: dict[str, float]
    sizes: dict[str, int]
    merged: dict[str, str]
    fallback_used: bool


def mondrian_conformal(
    calibration_truth: np.ndarray,
    calibration_prediction: np.ndarray,
    calibration_groups: np.ndarray,
    test_prediction: np.ndarray,
    test_groups: np.ndarray,
    alpha: float = 0.1,
    *,
    min_group_size: int | None = None,
    merge_small_groups: bool = True,
    calibration_sigma: np.ndarray | None = None,
    test_sigma: np.ndarray | None = None,
    sigma_floor: float = 1e-3,
) -> PredictionInterval:
    """Group-stratified (Mondrian) conformal intervals.

    Each group gets its own threshold from its own calibration scores, so the
    ``1 - alpha`` guarantee holds *within* every group that has enough points.

    Parameters
    ----------
    calibration_truth, calibration_prediction
        Calibration targets and predictions.
    calibration_groups
        Group key per calibration point (e.g. ``"KL3"`` or ``"KL3|siteA"``).
    test_prediction
        Test predictions.
    test_groups
        Group key per test point.
    alpha
        Target miscoverage.
    min_group_size
        Groups below this fall back. Defaults to the theoretical minimum
        ``ceil(1/alpha) - 1``.
    merge_small_groups
        When True, undersized groups use the pooled (all-calibration) threshold
        and the substitution is recorded. When False, an undersized group raises.
    calibration_sigma, test_sigma
        Supplying both switches the score from absolute to normalized, giving
        intervals that are both group-conditional and locally adaptive.
    sigma_floor
        Lower clip on sigma.

    Returns
    -------
    PredictionInterval
        ``q_hat`` is a per-test-point array. ``extras`` carries the
        :class:`GroupThresholds` record.

    Raises
    ------
    ConformalError
        On shape mismatch, or an undersized group when ``merge_small_groups``
        is False, or a test group absent from calibration.

    Notes
    -----
    The pooled fallback preserves *marginal* validity but not conditional
    validity for the merged group -- that group's coverage is no longer
    guaranteed. ``extras["merged"]`` names every affected group so the shortfall
    can be reported rather than assumed away.
    """
    calibration_truth = np.asarray(calibration_truth, dtype=np.float64)
    calibration_prediction = np.asarray(calibration_prediction, dtype=np.float64)
    calibration_groups = np.asarray(calibration_groups).astype(str)
    test_prediction = np.asarray(test_prediction, dtype=np.float64)
    test_groups = np.asarray(test_groups).astype(str)

    if calibration_truth.shape != calibration_prediction.shape:
        raise ConformalError("calibration truth and prediction must have the same shape")
    if calibration_groups.shape != calibration_truth.shape:
        raise ConformalError("calibration_groups must align with calibration_truth")
    if test_groups.shape != test_prediction.shape:
        raise ConformalError("test_groups must align with test_prediction")

    use_normalized = calibration_sigma is not None and test_sigma is not None
    if use_normalized:
        scores = normalized_scores(
            calibration_truth,
            calibration_prediction,
            np.asarray(calibration_sigma, dtype=np.float64),
            sigma_floor=sigma_floor,
        )
        score_name = "mondrian_normalized"
    else:
        scores = absolute_scores(calibration_truth, calibration_prediction)
        score_name = "mondrian_absolute"

    if min_group_size is None:
        min_group_size = min_calibration_size(alpha)

    pooled_threshold = conformal_quantile(scores, alpha, strict=False)

    thresholds: dict[str, float] = {}
    sizes: dict[str, int] = {}
    merged: dict[str, str] = {}

    for group in sorted(set(calibration_groups.tolist())):
        member = calibration_groups == group
        group_scores = scores[member]
        sizes[group] = int(member.sum())

        if sizes[group] < min_group_size:
            if not merge_small_groups:
                raise ConformalError(
                    f"group {group!r} has only {sizes[group]} calibration point(s), "
                    f"below min_group_size={min_group_size}. Either collect more "
                    "calibration data for this group, coarsen the grouping, or set "
                    "merge_small_groups=True and report the merge."
                )
            thresholds[group] = pooled_threshold
            merged[group] = "__pooled__"
            logger.warning(
                "Mondrian group %r has only %d calibration point(s) (need %d); using "
                "the pooled threshold. Conditional coverage is NOT guaranteed for "
                "this group.",
                group,
                sizes[group],
                min_group_size,
            )
        else:
            thresholds[group] = conformal_quantile(group_scores, alpha, strict=False)

    unseen = sorted(set(test_groups.tolist()) - set(thresholds))
    if unseen:
        if not merge_small_groups:
            raise ConformalError(
                f"test group(s) {unseen} have no calibration data at all; a "
                "group-conditional threshold cannot be formed for them."
            )
        for group in unseen:
            thresholds[group] = pooled_threshold
            sizes[group] = 0
            merged[group] = "__pooled__"
        logger.warning(
            "test group(s) %s have no calibration data; falling back to the pooled "
            "threshold. These are out-of-distribution groups and their coverage is "
            "not guaranteed.",
            unseen,
        )

    q_per_test = np.array([thresholds[g] for g in test_groups], dtype=np.float64)

    if use_normalized:
        scale = np.maximum(np.asarray(test_sigma, dtype=np.float64), sigma_floor)
        half_width = q_per_test * scale
    else:
        half_width = q_per_test

    record = GroupThresholds(
        thresholds=thresholds, sizes=sizes, merged=merged, fallback_used=bool(merged)
    )
    return PredictionInterval(
        lower=test_prediction - half_width,
        upper=test_prediction + half_width,
        q_hat=q_per_test,
        alpha=alpha,
        n_calibration=int(scores.size),
        score=score_name,
        extras={
            "group_thresholds": record.thresholds,
            "group_sizes": record.sizes,
            "merged": record.merged,
            "fallback_used": record.fallback_used,
        },
    )


def weighted_quantile(
    scores: np.ndarray, weights: np.ndarray, alpha: float, *, test_weight: float
) -> float:
    """Weighted conformal quantile with a point mass at ``+inf``.

    Implements the threshold of Tibshirani et al. (2019): with normalised
    weights

    .. math::
        p_i = \\frac{w_i}{\\sum_j w_j + w(x)}, \\qquad
        p_\\infty = \\frac{w(x)}{\\sum_j w_j + w(x)}

    the threshold is the ``1 - alpha`` quantile of
    :math:`\\sum_i p_i \\delta_{s_i} + p_\\infty \\delta_{+\\infty}`.

    The point mass at infinity is what keeps the procedure valid: when the
    calibration weights are small relative to the test weight, the quantile
    correctly returns ``+inf`` rather than a falsely tight threshold.

    Parameters
    ----------
    scores
        Calibration scores.
    weights
        Non-negative likelihood ratios ``w(X_i)``.
    alpha
        Target miscoverage.
    test_weight
        Likelihood ratio at the test point, ``w(x)``.

    Returns
    -------
    float
        Threshold, possibly ``+inf``.

    Raises
    ------
    ConformalError
        On negative weights, mismatched shapes or an empty score set.

    Examples
    --------
    Uniform weights reproduce the unweighted quantile closely:

    >>> s = np.arange(1.0, 101.0)
    >>> q = weighted_quantile(s, np.ones(100), 0.1, test_weight=1.0)
    >>> bool(89.0 <= q <= 92.0)
    True
    """
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    weights = np.asarray(weights, dtype=np.float64).reshape(-1)

    if scores.size == 0:
        raise ConformalError("cannot compute a weighted quantile from an empty score set")
    if scores.shape != weights.shape:
        raise ConformalError(
            f"scores shape {scores.shape} != weights shape {weights.shape}"
        )
    if (weights < 0).any():
        raise ConformalError("likelihood-ratio weights must be non-negative")
    if not np.isfinite(scores).all():
        raise ConformalError("calibration scores contain non-finite values")
    if test_weight < 0 or not np.isfinite(test_weight):
        raise ConformalError(f"test_weight must be finite and non-negative, got {test_weight}")

    total = float(weights.sum() + test_weight)
    if total <= 0:
        raise ConformalError("total weight is zero; the weighted distribution is undefined")

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_p = weights[order] / total

    cumulative = np.cumsum(sorted_p)
    target = 1.0 - alpha

    idx = int(np.searchsorted(cumulative, target, side="left"))
    if idx >= sorted_scores.size:
        # The remaining mass sits on the +inf atom.
        return float("inf")
    return float(sorted_scores[idx])


def estimate_likelihood_ratio(
    calibration_features: np.ndarray,
    test_features: np.ndarray,
    *,
    seed: int = 0,
    clip: tuple[float, float] = (1e-3, 1e3),
) -> tuple[np.ndarray, np.ndarray]:
    """Estimate ``w(x) = dP_test/dP_calib`` with a domain classifier.

    A logistic classifier is trained to separate calibration from test
    covariates; its odds ``p/(1-p)``, corrected for the class ratio, estimate the
    density ratio.

    Parameters
    ----------
    calibration_features, test_features
        ``(n, d)`` covariate matrices. These must be *covariates only*: including
        the target, or anything derived from it, turns the weights into label
        information and breaks the guarantee.
    seed
        Seed for the classifier.
    clip
        Clip range for the returned ratios. Unclipped ratios from a
        near-separable classifier explode and collapse the effective sample size.

    Returns
    -------
    tuple
        ``(calibration_weights, test_weights)``.

    Notes
    -----
    Weighted conformal assumes the shift is *covariate* shift: ``P(Y|X)`` is
    unchanged and only ``P(X)`` moves. If the relationship between image and
    thickness itself differs at the new site -- a different sequence, a
    different rater convention -- no reweighting of ``X`` can repair it.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    calibration_features = np.atleast_2d(np.asarray(calibration_features, dtype=np.float64))
    test_features = np.atleast_2d(np.asarray(test_features, dtype=np.float64))
    if calibration_features.shape[1] != test_features.shape[1]:
        raise ConformalError(
            f"feature dimension mismatch: calibration {calibration_features.shape[1]} "
            f"vs test {test_features.shape[1]}"
        )

    features = np.vstack([calibration_features, test_features])
    labels = np.concatenate(
        [np.zeros(len(calibration_features)), np.ones(len(test_features))]
    )

    scaler = StandardScaler().fit(features)
    model = LogisticRegression(max_iter=1000, random_state=seed)
    model.fit(scaler.transform(features), labels)

    probability = model.predict_proba(scaler.transform(features))[:, 1]
    probability = np.clip(probability, 1e-6, 1 - 1e-6)

    # Correct for the calibration:test size ratio so the odds estimate a density
    # ratio rather than a posterior.
    prior_ratio = len(calibration_features) / max(len(test_features), 1)
    ratio = (probability / (1.0 - probability)) * prior_ratio
    ratio = np.clip(ratio, clip[0], clip[1])

    n_calibration = len(calibration_features)
    return ratio[:n_calibration], ratio[n_calibration:]


def weighted_conformal(
    calibration_truth: np.ndarray,
    calibration_prediction: np.ndarray,
    test_prediction: np.ndarray,
    calibration_weights: np.ndarray,
    test_weights: np.ndarray,
    alpha: float = 0.1,
) -> PredictionInterval:
    """Conformal intervals valid under covariate shift.

    Parameters
    ----------
    calibration_truth, calibration_prediction
        Calibration targets and predictions.
    test_prediction
        Test predictions.
    calibration_weights, test_weights
        Likelihood ratios from :func:`estimate_likelihood_ratio`.
    alpha
        Target miscoverage.

    Returns
    -------
    PredictionInterval
        ``q_hat`` varies per test point, since the ``+inf`` atom depends on
        ``w(x)``. ``extras`` reports the effective sample size
        ``(sum w)^2 / sum w^2``, which is the honest measure of how much
        calibration data the reweighting actually leaves.
    """
    scores = absolute_scores(calibration_truth, calibration_prediction)
    calibration_weights = np.asarray(calibration_weights, dtype=np.float64)
    test_weights = np.asarray(test_weights, dtype=np.float64)
    test_prediction = np.asarray(test_prediction, dtype=np.float64)

    if test_weights.shape != test_prediction.shape:
        raise ConformalError("test_weights must align with test_prediction")

    q_per_test = np.array(
        [
            weighted_quantile(scores, calibration_weights, alpha, test_weight=float(w))
            for w in test_weights
        ],
        dtype=np.float64,
    )

    ess = float(calibration_weights.sum() ** 2 / np.sum(calibration_weights**2))
    if ess < 0.25 * calibration_weights.size:
        logger.warning(
            "covariate-shift reweighting cut the effective calibration size from %d to "
            "%.0f. The shift is large enough that the weighted intervals rest on very "
            "few effective points.",
            calibration_weights.size,
            ess,
        )

    n_infinite = int(np.sum(~np.isfinite(q_per_test)))
    if n_infinite:
        logger.warning(
            "%d/%d test point(s) received an infinite threshold: their covariates are "
            "too far outside the calibration distribution for any finite interval to "
            "be valid.",
            n_infinite,
            q_per_test.size,
        )

    return PredictionInterval(
        lower=test_prediction - q_per_test,
        upper=test_prediction + q_per_test,
        q_hat=q_per_test,
        alpha=alpha,
        n_calibration=int(scores.size),
        score="weighted_absolute",
        extras={
            "effective_sample_size": ess,
            "n_infinite_intervals": n_infinite,
        },
    )

### Conformal risk control

`confcarti/conformal/risk_control.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/conformal/risk_control.py
# ==========================================================================
"""Conformal Risk Control (Angelopoulos, Bates, Fisch, Lei, Schuster, ICLR 2024).

Split conformal controls a *coverage* probability. Conformal risk control
generalises it to the expectation of any loss that is monotone in a threshold
parameter, which is what a segmentation task actually needs: the clinically
relevant quantity is not "is the mask exactly right" but "how much of the true
cartilage did we miss".

The procedure
-------------
Let :math:`L_i(\\lambda)` be a loss on calibration point :math:`i`, non-increasing
in :math:`\\lambda`, bounded above by :math:`B`. With
:math:`\\hat{R}_n(\\lambda) = \\frac{1}{n} \\sum_i L_i(\\lambda)`, choose

.. math::
    \\hat{\\lambda} = \\inf \\left\\{ \\lambda :
        \\frac{n}{n+1} \\hat{R}_n(\\lambda) + \\frac{B}{n+1} \\le \\alpha \\right\\}

Then :math:`\\mathbb{E}[L_{n+1}(\\hat{\\lambda})] \\le \\alpha`, again
finite-sample and distribution-free under exchangeability.

Applied here to the segmentation false-negative rate,

.. math::
    L_\\lambda = 1 - \\frac{|\\hat{Y}_\\lambda \\cap Y|}{|Y|},
    \\qquad \\hat{Y}_\\lambda = \\{v : p(v) \\ge 1 - \\lambda\\}

so that raising :math:`\\lambda` admits more voxels and can only reduce the miss
rate -- the monotonicity the theorem requires. The output is a segmentation
threshold with a guarantee on how much cartilage is missed on a new knee.

References
----------
Angelopoulos, A.N., Bates, S., Fisch, A., Lei, L., Schuster, T. (2024).
    "Conformal risk control." ICLR.
"""


import logging
from dataclasses import dataclass

import numpy as np

logger = logging.getLogger(__name__)

__all__ = [
    "RiskControlResult",
    "get_lambda_hat",
    "false_negative_rate_curve",
    "calibrate_segmentation_threshold",
]


@dataclass(frozen=True, slots=True)
class RiskControlResult:
    """Outcome of a conformal risk-control calibration.

    Attributes
    ----------
    lambda_hat
        The selected threshold parameter.
    lambdas
        The grid searched.
    empirical_risk
        Mean calibration loss at each grid point.
    upper_bound
        ``n/(n+1) * R_n(lambda) + B/(n+1)`` at each grid point -- the quantity
        actually compared against alpha.
    alpha
        Target risk.
    n_calibration
        Number of calibration cases.
    loss_bound
        ``B``.
    feasible
        Whether any grid point met the bound. When False, ``lambda_hat`` is the
        most permissive grid value and the target risk was not achievable.
    """

    lambda_hat: float
    lambdas: np.ndarray
    empirical_risk: np.ndarray
    upper_bound: np.ndarray
    alpha: float
    n_calibration: int
    loss_bound: float
    feasible: bool

    @property
    def risk_at_lambda_hat(self) -> float:
        """Mean calibration loss at the selected threshold."""
        idx = int(np.argmin(np.abs(self.lambdas - self.lambda_hat)))
        return float(self.empirical_risk[idx])


def get_lambda_hat(
    calibration_loss_table: np.ndarray,
    lambdas: np.ndarray,
    alpha: float,
    *,
    loss_bound: float = 1.0,
) -> RiskControlResult:
    """Select the risk-controlling threshold from a calibration loss table.

    Parameters
    ----------
    calibration_loss_table
        ``(n_calibration, n_lambdas)`` losses. Each **row** must be non-increasing
        along the lambda axis, i.e. ``lambdas`` sorted ascending and larger
        lambda meaning a more permissive (lower-loss) prediction set.
    lambdas
        Ascending grid of threshold parameters.
    alpha
        Target expected loss.
    loss_bound
        ``B``, the supremum of the loss. For a rate in [0, 1] this is 1.

    Returns
    -------
    RiskControlResult

    Raises
    ------
    ValueError
        On shape mismatch, an unsorted grid, an out-of-range alpha, or a loss
        that exceeds ``loss_bound``.

    Notes
    -----
    This is the ``get_lhat`` of the authors' reference implementation, with three
    additions: the monotonicity precondition is checked rather than assumed, the
    infeasible case is reported instead of silently returning the first grid
    point, and the full bound curve is returned so it can be plotted.

    Examples
    --------
    A loss that falls linearly from 1 to 0 crosses alpha=0.2 near lambda=0.8:

    >>> lam = np.linspace(0, 1, 101)
    >>> losses = np.tile(1.0 - lam, (200, 1))
    >>> res = get_lambda_hat(losses, lam, 0.2)
    >>> bool(0.75 <= res.lambda_hat <= 0.85)
    True
    """
    losses = np.asarray(calibration_loss_table, dtype=np.float64)
    lambdas = np.asarray(lambdas, dtype=np.float64).reshape(-1)

    if losses.ndim != 2:
        raise ValueError(f"calibration_loss_table must be 2D, got shape {losses.shape}")
    if losses.shape[1] != lambdas.size:
        raise ValueError(
            f"loss table has {losses.shape[1]} lambda columns but the grid has "
            f"{lambdas.size} entries"
        )
    if not 0.0 < alpha < 1.0:
        raise ValueError(f"alpha must be in (0, 1), got {alpha}")
    if np.any(np.diff(lambdas) < 0):
        raise ValueError("lambdas must be sorted in ascending order")
    if not np.isfinite(losses).all():
        raise ValueError("calibration loss table contains non-finite values")
    if losses.max() > loss_bound + 1e-9:
        raise ValueError(
            f"loss table maximum {losses.max():.4f} exceeds the declared bound "
            f"B={loss_bound}. The guarantee E[L] <= alpha depends on B being a true "
            "upper bound, so an understated B silently invalidates it."
        )

    increases = np.diff(losses, axis=1)
    if np.any(increases > 1e-9):
        worst = float(increases.max())
        n_bad = int(np.any(increases > 1e-9, axis=1).sum())
        raise ValueError(
            f"the loss must be non-increasing in lambda, but {n_bad} calibration "
            f"row(s) increase (largest jump {worst:.3g}). Conformal risk control's "
            "guarantee rests on that monotonicity; check the direction of your "
            "threshold before proceeding."
        )

    n = losses.shape[0]
    empirical_risk = losses.mean(axis=0)
    upper_bound = (n / (n + 1.0)) * empirical_risk + loss_bound / (n + 1.0)

    feasible_mask = upper_bound <= alpha
    if feasible_mask.any():
        lambda_hat = float(lambdas[int(np.argmax(feasible_mask))])
        feasible = True
    else:
        lambda_hat = float(lambdas[-1])
        feasible = False
        logger.warning(
            "no lambda achieves the target risk alpha=%.3f: the smallest attainable "
            "bound is %.4f at lambda=%.4f. Returning the most permissive threshold; "
            "the risk guarantee does NOT hold at alpha=%.3f.",
            alpha,
            float(upper_bound.min()),
            float(lambdas[int(np.argmin(upper_bound))]),
            alpha,
        )

    return RiskControlResult(
        lambda_hat=lambda_hat,
        lambdas=lambdas,
        empirical_risk=empirical_risk,
        upper_bound=upper_bound,
        alpha=alpha,
        n_calibration=n,
        loss_bound=loss_bound,
        feasible=feasible,
    )


def false_negative_rate_curve(
    probability: np.ndarray, truth: np.ndarray, lambdas: np.ndarray
) -> np.ndarray:
    """False-negative rate of ``{v : p(v) >= 1 - lambda}`` across a lambda grid.

    .. math::
        L(\\lambda) = 1 - \\frac{|\\hat{Y}_\\lambda \\cap Y|}{|Y|}

    Parameters
    ----------
    probability
        Predicted per-voxel probability of the target class, in [0, 1].
    truth
        Boolean ground-truth mask, same shape.
    lambdas
        Ascending grid in [0, 1].

    Returns
    -------
    numpy.ndarray
        ``(n_lambdas,)`` losses, non-increasing by construction.

    Raises
    ------
    ValueError
        On shape mismatch, out-of-range probabilities, or an empty truth mask
        (the rate is undefined with no positives to miss).

    Examples
    --------
    >>> p = np.array([0.9, 0.4, 0.1])
    >>> y = np.array([True, True, False])
    >>> false_negative_rate_curve(p, y, np.array([0.2, 0.7])).tolist()
    [0.5, 0.0]
    """
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    truth = np.asarray(truth).reshape(-1).astype(bool)
    lambdas = np.asarray(lambdas, dtype=np.float64).reshape(-1)

    if probability.shape != truth.shape:
        raise ValueError(
            f"probability shape {probability.shape} != truth shape {truth.shape}"
        )
    if probability.size and (probability.min() < -1e-9 or probability.max() > 1 + 1e-9):
        raise ValueError("probability must lie in [0, 1]")

    n_positive = int(truth.sum())
    if n_positive == 0:
        raise ValueError(
            "ground-truth mask is empty; the false-negative rate is undefined. "
            "A case with no cartilage of this class should be excluded from risk "
            "calibration explicitly, not folded in as a zero-loss case."
        )

    positive_probability = probability[truth]
    # |Yhat_lambda ∩ Y| = number of true voxels with p >= 1 - lambda.
    thresholds = 1.0 - lambdas
    # searchsorted over the sorted positive probabilities gives the count in one pass.
    sorted_positive = np.sort(positive_probability)
    n_below = np.searchsorted(sorted_positive, thresholds, side="left")
    captured = n_positive - n_below
    return 1.0 - captured / n_positive


def calibrate_segmentation_threshold(
    probabilities: list[np.ndarray],
    truths: list[np.ndarray],
    alpha: float = 0.1,
    *,
    n_lambdas: int = 200,
    loss_bound: float = 1.0,
) -> RiskControlResult:
    """Calibrate a segmentation threshold controlling the expected miss rate.

    Parameters
    ----------
    probabilities
        One predicted probability map per calibration case.
    truths
        Matching ground-truth boolean masks.
    alpha
        Target expected false-negative rate.
    n_lambdas
        Grid resolution.
    loss_bound
        ``B``; 1 for a rate.

    Returns
    -------
    RiskControlResult
        ``lambda_hat`` defines the operating threshold ``p >= 1 - lambda_hat``.

    Raises
    ------
    ValueError
        If the two lists differ in length or are empty.
    """
    if len(probabilities) != len(truths):
        raise ValueError(
            f"got {len(probabilities)} probability map(s) but {len(truths)} truth mask(s)"
        )
    if not probabilities:
        raise ValueError("no calibration cases supplied")

    lambdas = np.linspace(0.0, 1.0, n_lambdas)
    rows = [
        false_negative_rate_curve(probability, truth, lambdas)
        for probability, truth in zip(probabilities, truths)
    ]
    table = np.vstack(rows)

    result = get_lambda_hat(table, lambdas, alpha, loss_bound=loss_bound)
    logger.info(
        "conformal risk control: lambda_hat=%.4f (threshold p >= %.4f), "
        "calibration risk=%.4f, target alpha=%.3f, feasible=%s",
        result.lambda_hat,
        1.0 - result.lambda_hat,
        result.risk_at_lambda_hat,
        alpha,
        result.feasible,
    )
    return result

### Coverage evaluation

`confcarti/conformal/coverage.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/conformal/coverage.py
# ==========================================================================
"""Coverage evaluation: marginal, conditional, calibration curves, stability.

Every number a conformal method claims has to be checked against what it
actually delivers, and checked *per group*. A marginal 90% that hides 71% at
KL 4 is the failure mode this whole project exists to expose, so
:func:`conditional_coverage` is the primary output here and the marginal number
is secondary.
"""


import logging
from dataclasses import dataclass

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)

__all__ = [
    "CoverageReport",
    "empirical_coverage",
    "conditional_coverage",
    "interval_width_stats",
    "width_error_correlation",
    "calibration_curve",
    "calibration_size_sweep",
    "binomial_confidence_interval",
]


@dataclass(frozen=True, slots=True)
class CoverageReport:
    """Marginal and conditional coverage of a set of intervals.

    Attributes
    ----------
    marginal
        Overall empirical coverage.
    nominal
        Target coverage ``1 - alpha``.
    by_group
        Per-group coverage table with counts and confidence intervals.
    mean_width, median_width
        Interval width summaries.
    width_error_correlation
        Spearman correlation between interval width and absolute error. A
        useful adaptivity check: a locally-adaptive method should widen where
        the error is large, so a positive value is what you want. Near zero
        means the intervals are effectively constant-width.
    n
        Number of test points.
    """

    marginal: float
    nominal: float
    by_group: pd.DataFrame
    mean_width: float
    median_width: float
    width_error_correlation: float
    n: int

    @property
    def worst_group_coverage(self) -> float:
        """Lowest per-group coverage -- the number a clinical claim rests on."""
        if self.by_group.empty:
            return float("nan")
        return float(self.by_group["coverage"].min())

    @property
    def coverage_gap(self) -> float:
        """Marginal minus worst-group coverage: how much the average hides."""
        return self.marginal - self.worst_group_coverage


def binomial_confidence_interval(
    successes: int, total: int, confidence: float = 0.95
) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion.

    Preferred over the normal approximation because coverage estimates sit close
    to 1, where the normal interval runs past 1 and understates uncertainty in
    exactly the regime that matters.

    Parameters
    ----------
    successes
        Number of covered points.
    total
        Number of points.
    confidence
        Two-sided confidence level.

    Returns
    -------
    tuple
        ``(lower, upper)``, or ``(nan, nan)`` when ``total`` is 0.

    Examples
    --------
    >>> lo, hi = binomial_confidence_interval(90, 100)
    >>> bool(lo < 0.9 < hi)
    True
    """
    if total <= 0:
        return (float("nan"), float("nan"))

    from scipy import stats

    z = float(stats.norm.ppf(0.5 + confidence / 2.0))
    phat = successes / total
    denom = 1.0 + z**2 / total
    centre = (phat + z**2 / (2 * total)) / denom
    half = (z / denom) * np.sqrt(phat * (1 - phat) / total + z**2 / (4 * total**2))
    return (float(max(0.0, centre - half)), float(min(1.0, centre + half)))


def empirical_coverage(intervals: PredictionInterval, truth: np.ndarray) -> float:
    """Fraction of test points whose truth falls inside its interval.

    Parameters
    ----------
    intervals
        Prediction intervals.
    truth
        Ground-truth targets.

    Returns
    -------
    float
    """
    truth = np.asarray(truth, dtype=np.float64)
    if truth.shape != intervals.lower.shape:
        raise ValueError(
            f"truth shape {truth.shape} != interval shape {intervals.lower.shape}"
        )
    return intervals.coverage(truth)


def conditional_coverage(
    intervals: PredictionInterval,
    truth: np.ndarray,
    groups: np.ndarray,
    *,
    confidence: float = 0.95,
) -> pd.DataFrame:
    """Per-group coverage, width and confidence intervals.

    Parameters
    ----------
    intervals
        Prediction intervals.
    truth
        Ground-truth targets.
    groups
        Group key per test point.
    confidence
        Confidence level for the Wilson intervals.

    Returns
    -------
    pandas.DataFrame
        One row per group: ``n``, ``n_covered``, ``coverage``, ``ci_low``,
        ``ci_high``, ``mean_width``, ``median_width``, ``mean_abs_error``.
    """
    truth = np.asarray(truth, dtype=np.float64)
    groups = np.asarray(groups).astype(str)
    covered = intervals.contains(truth)
    widths = intervals.width
    centre = 0.5 * (intervals.lower + intervals.upper)
    abs_error = np.abs(truth - centre)

    rows = []
    for group in sorted(set(groups.tolist())):
        member = groups == group
        n = int(member.sum())
        n_covered = int(covered[member].sum())
        low, high = binomial_confidence_interval(n_covered, n, confidence)
        rows.append(
            {
                "group": group,
                "n": n,
                "n_covered": n_covered,
                "coverage": n_covered / n if n else float("nan"),
                "ci_low": low,
                "ci_high": high,
                "mean_width": float(np.mean(widths[member])) if n else float("nan"),
                "median_width": float(np.median(widths[member])) if n else float("nan"),
                "mean_abs_error": float(np.mean(abs_error[member])) if n else float("nan"),
            }
        )
    return pd.DataFrame(rows)


def interval_width_stats(intervals: PredictionInterval) -> dict[str, float]:
    """Mean, median and percentile summary of interval widths."""
    widths = intervals.width
    finite = widths[np.isfinite(widths)]
    if finite.size == 0:
        return {"mean_width": float("nan"), "median_width": float("nan")}
    return {
        "mean_width": float(finite.mean()),
        "median_width": float(np.median(finite)),
        "p05_width": float(np.percentile(finite, 5)),
        "p95_width": float(np.percentile(finite, 95)),
        "n_infinite": int(np.sum(~np.isfinite(widths))),
    }


def width_error_correlation(intervals: PredictionInterval, truth: np.ndarray) -> float:
    """Spearman correlation between interval width and absolute error.

    Parameters
    ----------
    intervals
        Prediction intervals.
    truth
        Ground truth.

    Returns
    -------
    float
        Spearman rho, or NaN if the widths are constant (as for unnormalized
        marginal split conformal, where every interval has the same width by
        construction).
    """
    from scipy import stats

    truth = np.asarray(truth, dtype=np.float64)
    centre = 0.5 * (intervals.lower + intervals.upper)
    abs_error = np.abs(truth - centre)
    widths = intervals.width

    usable = np.isfinite(widths) & np.isfinite(abs_error)
    if usable.sum() < 3 or np.std(widths[usable]) < 1e-12:
        return float("nan")
    rho, _p = stats.spearmanr(widths[usable], abs_error[usable])
    return float(rho)


def coverage_report(
    intervals: PredictionInterval,
    truth: np.ndarray,
    groups: np.ndarray | None = None,
    *,
    confidence: float = 0.95,
) -> CoverageReport:
    """Assemble the full coverage report for one set of intervals.

    Parameters
    ----------
    intervals
        Prediction intervals.
    truth
        Ground truth.
    groups
        Optional group keys for the conditional table.
    confidence
        Confidence level.

    Returns
    -------
    CoverageReport
    """
    truth = np.asarray(truth, dtype=np.float64)
    marginal = empirical_coverage(intervals, truth)
    stats_ = interval_width_stats(intervals)

    by_group = (
        conditional_coverage(intervals, truth, groups, confidence=confidence)
        if groups is not None
        else pd.DataFrame(columns=["group", "n", "coverage"])
    )

    return CoverageReport(
        marginal=marginal,
        nominal=1.0 - intervals.alpha,
        by_group=by_group,
        mean_width=stats_["mean_width"],
        median_width=stats_["median_width"],
        width_error_correlation=width_error_correlation(intervals, truth),
        n=int(truth.size),
    )


def calibration_curve(
    build_intervals,
    truth: np.ndarray,
    alphas: np.ndarray | None = None,
) -> pd.DataFrame:
    """Sweep alpha and record nominal versus empirical coverage.

    Parameters
    ----------
    build_intervals
        Callable ``alpha -> PredictionInterval`` for the test set.
    truth
        Ground truth.
    alphas
        Grid of miscoverage levels. Defaults to 20 points in [0.01, 0.5].

    Returns
    -------
    pandas.DataFrame
        ``alpha``, ``nominal``, ``empirical``, ``mean_width``, ``gap``.

    Notes
    -----
    A well-calibrated method traces the diagonal from above: conformal coverage
    is guaranteed to be *at least* nominal, so points should sit on or slightly
    above the identity line. Points below it indicate broken exchangeability,
    not conservatism.
    """
    if alphas is None:
        alphas = np.linspace(0.01, 0.5, 20)

    truth = np.asarray(truth, dtype=np.float64)
    rows = []
    for alpha in np.asarray(alphas, dtype=np.float64):
        intervals = build_intervals(float(alpha))
        empirical = intervals.coverage(truth)
        widths = intervals.width
        finite = widths[np.isfinite(widths)]
        rows.append(
            {
                "alpha": float(alpha),
                "nominal": 1.0 - float(alpha),
                "empirical": empirical,
                "mean_width": float(finite.mean()) if finite.size else float("nan"),
                "gap": empirical - (1.0 - float(alpha)),
            }
        )
    return pd.DataFrame(rows)


def calibration_size_sweep(
    calibration_truth: np.ndarray,
    calibration_prediction: np.ndarray,
    test_truth: np.ndarray,
    test_prediction: np.ndarray,
    sizes: tuple[int, ...] = (25, 50, 100, 200),
    alpha: float = 0.1,
    *,
    n_repeats: int = 50,
    seed: int = 0,
) -> pd.DataFrame:
    """Stability of coverage as the calibration set shrinks (ablation A5).

    Sub-samples the calibration set to each size, repeatedly, and records the
    spread of realised coverage and width.

    Parameters
    ----------
    calibration_truth, calibration_prediction
        Full calibration set.
    test_truth, test_prediction
        Test set.
    sizes
        Calibration sizes to try.
    alpha
        Target miscoverage.
    n_repeats
        Sub-sampling repeats per size.
    seed
        RNG seed.

    Returns
    -------
    pandas.DataFrame
        Per size: mean/sd of coverage, mean width, and the fraction of repeats
        that under-covered.

    Notes
    -----
    Coverage remains valid *in expectation* at every admissible size -- that is
    the theorem. What shrinks with ``n`` is not validity but *stability*: the
    realised coverage of any single calibration draw scatters more widely, so a
    small calibration set can easily land at 0.84 when targeting 0.90. This
    sweep quantifies that scatter, which is what determines how large a
    calibration split a study actually needs.
    """
    calibration_truth = np.asarray(calibration_truth, dtype=np.float64)
    calibration_prediction = np.asarray(calibration_prediction, dtype=np.float64)
    test_truth = np.asarray(test_truth, dtype=np.float64)
    test_prediction = np.asarray(test_prediction, dtype=np.float64)

    rng = seeded_generator(seed)
    minimum = min_calibration_size(alpha)
    rows = []

    for size in sizes:
        if size < minimum:
            logger.warning(
                "skipping calibration size %d: below the %d points needed for the "
                "alpha=%.3f quantile to exist",
                size,
                minimum,
                alpha,
            )
            continue
        if size > calibration_truth.size:
            logger.warning(
                "skipping calibration size %d: only %d calibration points available",
                size,
                calibration_truth.size,
            )
            continue

        coverages, widths = [], []
        for _ in range(n_repeats):
            idx = rng.choice(calibration_truth.size, size=size, replace=False)
            intervals = split_conformal(
                calibration_truth[idx],
                calibration_prediction[idx],
                test_prediction,
                alpha=alpha,
            )
            coverages.append(intervals.coverage(test_truth))
            widths.append(float(np.mean(intervals.width)))

        coverages_arr = np.asarray(coverages)
        rows.append(
            {
                "calibration_size": int(size),
                "mean_coverage": float(coverages_arr.mean()),
                "sd_coverage": float(coverages_arr.std(ddof=1)) if len(coverages) > 1 else 0.0,
                "min_coverage": float(coverages_arr.min()),
                "max_coverage": float(coverages_arr.max()),
                "mean_width": float(np.mean(widths)),
                "frac_undercovering": float(np.mean(coverages_arr < 1.0 - alpha)),
                "nominal": 1.0 - alpha,
                "n_repeats": n_repeats,
            }
        )
    return pd.DataFrame(rows)

### Reliability, agreement and detectability

`confcarti/reliability/stats.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/reliability/stats.py
# ==========================================================================
"""Precision, responsiveness, agreement and detectability -- pure functions.

Why agreement, not accuracy
---------------------------
The obvious way to evaluate a thickness pipeline is "MAE against ground truth".
It is also wrong here, and reporting it would be misleading rather than merely
imprecise. The reference thickness is itself derived from a manual segmentation
by the *same* meshing and ray-casting code, so it is not an independent
measurement: the two share their preprocessing, their surface extraction and
their discretisation error. Their difference therefore understates the true
error, and the number it produces has no interpretation as accuracy.

What can be said honestly is (a) how *reproducible* the measurement is
(RMS CV%, SEM, SDD), (b) how well two measurements *agree* (Bland-Altman, ICC),
and (c) whether the measurement can detect the change it needs to
(SDD vs annual change). Those are what this module computes.

References
----------
Gluer, C.C. et al. (1995). "Accurate assessment of precision errors."
    Osteoporosis International 5(4):262-270.  [RMS CV%, SEM]
Bland, J.M., Altman, D.G. (1986). "Statistical methods for assessing agreement
    between two methods of clinical measurement." Lancet 327(8476):307-310.
Shrout, P.E., Fleiss, J.L. (1979). "Intraclass correlations: uses in assessing
    rater reliability." Psychological Bulletin 86(2):420-428.
McGraw, K.O., Wong, S.P. (1996). "Forming inferences about some intraclass
    correlation coefficients." Psychological Methods 1(1):30-46.
Eckstein, F. et al. (2006). Osteoarthritis and Cartilage 14(10):974-983.
Warfield, S.K., Zou, K.H., Wells, W.M. (2004). "Simultaneous truth and
    performance level estimation (STAPLE)." IEEE TMI 23(7):903-921.
"""


import logging
from dataclasses import dataclass

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

__all__ = [
    "rms_cv",
    "sem",
    "sdd",
    "srm",
    "bootstrap_ci",
    "BlandAltmanResult",
    "bland_altman",
    "icc",
    "staple_consensus",
    "error_floor_from_testretest",
    "detectability_table",
]


# --------------------------------------------------------------------------- #
# Precision
# --------------------------------------------------------------------------- #


def _as_repeat_matrix(values_by_subject: np.ndarray | list) -> np.ndarray:
    """Coerce input to an ``(n_subjects, n_repeats)`` float matrix."""
    matrix = np.asarray(values_by_subject, dtype=np.float64)
    if matrix.ndim != 2:
        raise ValueError(
            f"expected an (n_subjects, n_repeats) array, got shape {matrix.shape}"
        )
    if matrix.shape[1] < 2:
        raise ValueError(
            f"precision needs at least 2 repeats per subject, got {matrix.shape[1]}"
        )
    return matrix


def rms_cv(values_by_subject: np.ndarray | list) -> float:
    """Root-mean-square coefficient of variation, in percent.

    .. math::
        \\mathrm{RMS\\,CV\\%} = 100 \\sqrt{\\frac{1}{m}\\sum_{i=1}^{m} CV_i^2},
        \\qquad CV_i = \\frac{SD_i}{\\bar{x}_i}

    The RMS (not the arithmetic mean) of the per-subject CVs is the standard
    precision statistic (Gluer et al. 1995): precision errors combine in
    quadrature, so averaging the CVs directly under-states the error.

    Parameters
    ----------
    values_by_subject
        ``(n_subjects, n_repeats)`` repeated measurements.

    Returns
    -------
    float
        RMS CV in percent.

    Examples
    --------
    Identical repeats have zero variability:

    >>> float(rms_cv([[2.0, 2.0, 2.0], [3.0, 3.0, 3.0]]))
    0.0

    A subject measured at 10 and 11 has SD (ddof=1) of 0.7071 about a mean of
    10.5, i.e. CV = 6.734%:

    >>> round(float(rms_cv([[10.0, 11.0]])), 3)
    6.734
    """
    matrix = _as_repeat_matrix(values_by_subject)
    means = matrix.mean(axis=1)
    sds = matrix.std(axis=1, ddof=1)

    usable = np.abs(means) > 1e-12
    if not usable.any():
        return float("nan")
    if (~usable).any():
        logger.warning(
            "%d subject(s) have a near-zero mean and were excluded from RMS CV%%; "
            "the CV is undefined there",
            int((~usable).sum()),
        )
    cvs = sds[usable] / means[usable]
    return float(100.0 * np.sqrt(np.mean(cvs**2)))


def sem(values_by_subject: np.ndarray | list) -> float:
    """Standard error of measurement: the within-subject standard deviation.

    .. math:: \\mathrm{SEM} = \\sqrt{\\frac{1}{m}\\sum_{i=1}^{m} SD_i^2}

    Parameters
    ----------
    values_by_subject
        ``(n_subjects, n_repeats)`` repeated measurements.

    Returns
    -------
    float
        SEM in the units of the measurement.

    Examples
    --------
    >>> float(sem([[2.0, 2.0], [3.0, 3.0]]))
    0.0
    >>> round(float(sem([[10.0, 11.0]])), 4)
    0.7071
    """
    matrix = _as_repeat_matrix(values_by_subject)
    variances = matrix.var(axis=1, ddof=1)
    return float(np.sqrt(np.mean(variances)))


def sdd(sem_value: float, *, z: float = 1.96) -> float:
    """Smallest detectable difference.

    .. math:: \\mathrm{SDD} = z \\sqrt{2} \\, \\mathrm{SEM}

    A change smaller than the SDD cannot be distinguished from measurement noise
    in an individual knee at the given confidence. The :math:`\\sqrt{2}` is
    because a *difference* of two independent measurements has twice the
    variance of one.

    Parameters
    ----------
    sem_value
        Standard error of measurement.
    z
        Normal quantile; 1.96 gives the 95% SDD.

    Returns
    -------
    float

    Examples
    --------
    >>> round(sdd(0.1), 6)
    0.277186
    """
    if sem_value < 0:
        raise ValueError(f"SEM must be non-negative, got {sem_value}")
    return float(z * np.sqrt(2.0) * sem_value)


# --------------------------------------------------------------------------- #
# Responsiveness
# --------------------------------------------------------------------------- #


def srm(baseline: np.ndarray, followup: np.ndarray) -> float:
    """Standardised response mean: mean change divided by the SD of change.

    .. math:: \\mathrm{SRM} = \\frac{\\overline{\\Delta}}{SD(\\Delta)}

    Parameters
    ----------
    baseline, followup
        Paired measurements.

    Returns
    -------
    float
        SRM; negative when the quantity decreases (as cartilage thickness does).

    Examples
    --------
    >>> b = np.array([2.0, 2.1, 1.9, 2.2])
    >>> f = b - 0.1
    >>> float(srm(b, f))     # constant change -> zero SD -> infinite SRM
    -inf
    """
    baseline = np.asarray(baseline, dtype=np.float64)
    followup = np.asarray(followup, dtype=np.float64)
    if baseline.shape != followup.shape:
        raise ValueError(
            f"baseline shape {baseline.shape} != followup shape {followup.shape}"
        )

    change = followup - baseline
    usable = np.isfinite(change)
    if usable.sum() < 2:
        return float("nan")

    change = change[usable]
    sd = change.std(ddof=1)
    if sd < 1e-15:
        return float(np.sign(change.mean()) * np.inf) if change.mean() != 0 else 0.0
    return float(change.mean() / sd)


def bootstrap_ci(
    statistic,
    *arrays: np.ndarray,
    n_boot: int = 2000,
    confidence: float = 0.95,
    seed: int = 0,
) -> tuple[float, float, float]:
    """Percentile bootstrap confidence interval for a paired statistic.

    Parameters
    ----------
    statistic
        Callable taking the resampled arrays and returning a float.
    *arrays
        Equal-length arrays resampled together (paired bootstrap).
    n_boot
        Bootstrap replicates.
    confidence
        Two-sided confidence level.
    seed
        RNG seed.

    Returns
    -------
    tuple
        ``(point_estimate, ci_low, ci_high)``.
    """
    arrays = tuple(np.asarray(a, dtype=np.float64) for a in arrays)
    lengths = {a.shape[0] for a in arrays}
    if len(lengths) != 1:
        raise ValueError(f"all arrays must have the same length, got {sorted(lengths)}")

    n = arrays[0].shape[0]
    rng = seeded_generator(seed)
    point = float(statistic(*arrays))

    replicates = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        replicates[b] = statistic(*(a[idx] for a in arrays))

    finite = replicates[np.isfinite(replicates)]
    if finite.size == 0:
        return point, float("nan"), float("nan")

    tail = (1.0 - confidence) / 2.0
    low, high = np.percentile(finite, [100 * tail, 100 * (1 - tail)])
    return point, float(low), float(high)


# --------------------------------------------------------------------------- #
# Agreement
# --------------------------------------------------------------------------- #


@dataclass(frozen=True, slots=True)
class BlandAltmanResult:
    """Bland-Altman agreement between two measurement methods.

    Attributes
    ----------
    bias
        Mean difference ``a - b``.
    sd_diff
        Standard deviation of the differences.
    loa_low, loa_high
        Limits of agreement, ``bias +- 1.96 SD``.
    means, differences
        Per-pair values, for plotting.
    bias_ci
        Confidence interval on the bias.
    proportional_bias_slope, proportional_bias_p
        Slope and p-value of ``difference ~ mean``. A significant slope means
        the bias depends on magnitude, so a single bias number is misleading.
    n
        Number of pairs.
    """

    bias: float
    sd_diff: float
    loa_low: float
    loa_high: float
    means: np.ndarray
    differences: np.ndarray
    bias_ci: tuple[float, float]
    proportional_bias_slope: float
    proportional_bias_p: float
    n: int


def bland_altman(
    a: np.ndarray, b: np.ndarray, *, confidence: float = 0.95
) -> BlandAltmanResult:
    """Bland-Altman analysis of two paired measurement series.

    Parameters
    ----------
    a, b
        Paired measurements from the two methods.
    confidence
        Confidence level for the bias interval.

    Returns
    -------
    BlandAltmanResult

    Examples
    --------
    Identical inputs give zero bias and zero-width limits:

    >>> res = bland_altman(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0]))
    >>> (res.bias, res.loa_low, res.loa_high)
    (0.0, 0.0, 0.0)
    """
    from scipy import stats

    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")

    usable = np.isfinite(a) & np.isfinite(b)
    a, b = a[usable], b[usable]
    if a.size < 2:
        raise ValueError("Bland-Altman needs at least 2 usable pairs")

    differences = a - b
    means = 0.5 * (a + b)
    bias = float(differences.mean())
    sd_diff = float(differences.std(ddof=1))

    z = float(stats.norm.ppf(0.5 + confidence / 2.0))
    se_bias = sd_diff / np.sqrt(a.size) if sd_diff > 0 else 0.0
    bias_ci = (bias - z * se_bias, bias + z * se_bias)

    if np.std(means) > 1e-12:
        slope, _intercept, _r, p_value, _se = stats.linregress(means, differences)
    else:
        slope, p_value = 0.0, 1.0

    return BlandAltmanResult(
        bias=bias,
        sd_diff=sd_diff,
        loa_low=bias - 1.96 * sd_diff,
        loa_high=bias + 1.96 * sd_diff,
        means=means,
        differences=differences,
        bias_ci=bias_ci,
        proportional_bias_slope=float(slope),
        proportional_bias_p=float(p_value),
        n=int(a.size),
    )


def icc(
    a: np.ndarray,
    b: np.ndarray,
    kind: str = "ICC(2,1)",
    *,
    confidence: float = 0.95,
) -> dict[str, float]:
    """Intraclass correlation coefficient for two measurements per subject.

    Forms (Shrout & Fleiss 1979; McGraw & Wong 1996), with ``n`` subjects and
    ``k = 2`` measurements, mean squares ``MSR`` (between rows/subjects),
    ``MSE`` (residual) and ``MSC`` (between columns/raters):

    ``ICC(1,1)``  one-way random
        :math:`(MSR - MSW) / (MSR + (k-1) MSW)`
    ``ICC(2,1)``  two-way random, absolute agreement
        :math:`(MSR - MSE) / (MSR + (k-1) MSE + k(MSC - MSE)/n)`
    ``ICC(3,1)``  two-way mixed, consistency
        :math:`(MSR - MSE) / (MSR + (k-1) MSE)`

    ``ICC(2,1)`` is the default because it charges systematic differences
    between the two methods against the agreement, which is what we want when
    comparing a pipeline to a reference rather than checking mere consistency.

    Parameters
    ----------
    a, b
        Paired measurements.
    kind
        Which ICC form to compute.
    confidence
        Confidence level for the F-based interval.

    Returns
    -------
    dict
        ``icc``, ``ci_low``, ``ci_high``, ``n``, ``kind``.

    Examples
    --------
    Perfectly agreeing measurements give ICC = 1:

    >>> res = icc(np.array([1.0, 2.0, 3.0, 4.0]), np.array([1.0, 2.0, 3.0, 4.0]))
    >>> round(res["icc"], 6)
    1.0
    """
    from scipy import stats

    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")

    usable = np.isfinite(a) & np.isfinite(b)
    matrix = np.column_stack([a[usable], b[usable]])
    n, k = matrix.shape
    if n < 2:
        raise ValueError("ICC needs at least 2 subjects")

    grand = matrix.mean()
    row_means = matrix.mean(axis=1)
    col_means = matrix.mean(axis=0)

    ss_total = float(((matrix - grand) ** 2).sum())
    ss_rows = float(k * ((row_means - grand) ** 2).sum())
    ss_cols = float(n * ((col_means - grand) ** 2).sum())
    ss_error = ss_total - ss_rows - ss_cols

    df_rows = n - 1
    df_cols = k - 1
    df_error = df_rows * df_cols

    ms_rows = ss_rows / df_rows if df_rows else 0.0
    ms_cols = ss_cols / df_cols if df_cols else 0.0
    ms_error = ss_error / df_error if df_error else 0.0
    ms_within = (ss_cols + ss_error) / (n * (k - 1)) if k > 1 else 0.0

    eps = 1e-15
    if kind == "ICC(1,1)":
        value = (ms_rows - ms_within) / max(ms_rows + (k - 1) * ms_within, eps)
    elif kind == "ICC(2,1)":
        denom = ms_rows + (k - 1) * ms_error + k * (ms_cols - ms_error) / n
        value = (ms_rows - ms_error) / max(denom, eps)
    elif kind == "ICC(3,1)":
        value = (ms_rows - ms_error) / max(ms_rows + (k - 1) * ms_error, eps)
    else:
        raise ValueError(f"kind must be ICC(1,1)|ICC(2,1)|ICC(3,1), got {kind!r}")

    value = float(np.clip(value, -1.0, 1.0))

    # F-based interval for the consistency form; used as an approximation for
    # the others, which is standard practice.
    tail = (1 - confidence) / 2
    denominator = ms_error if kind != "ICC(1,1)" else ms_within
    if denominator > eps and df_error > 0:
        f_obs = ms_rows / denominator
        f_low = f_obs / stats.f.ppf(1 - tail, df_rows, df_error)
        f_high = f_obs * stats.f.ppf(1 - tail, df_error, df_rows)
        ci_low = (f_low - 1) / (f_low + (k - 1))
        ci_high = (f_high - 1) / (f_high + (k - 1))
    else:
        ci_low, ci_high = value, value

    return {
        "icc": value,
        "ci_low": float(np.clip(ci_low, -1.0, 1.0)),
        "ci_high": float(np.clip(ci_high, -1.0, 1.0)),
        "n": int(n),
        "kind": kind,
    }


# --------------------------------------------------------------------------- #
# Non-circular reference
# --------------------------------------------------------------------------- #


def staple_consensus(
    masks: list[np.ndarray],
    *,
    max_iterations: int = 50,
    tolerance: float = 1e-6,
    prior: float = 0.5,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """STAPLE consensus of several binary segmentations.

    Expectation-maximisation estimate of the hidden true segmentation together
    with each rater's sensitivity ``p`` and specificity ``q``
    (Warfield, Zou & Wells 2004). Unlike majority voting it down-weights raters
    that disagree with the consensus, which matters when the "raters" are
    several networks and one of them fails on a hard case.

    Parameters
    ----------
    masks
        Binary masks of identical shape, one per rater.
    max_iterations
        EM iteration cap.
    tolerance
        Convergence tolerance on the consensus probability.
    prior
        Initial prior probability that a voxel is foreground.

    Returns
    -------
    tuple
        ``(consensus_probability, {"sensitivity": p, "specificity": q,
        "iterations": k})``.

    Raises
    ------
    ValueError
        If fewer than two masks are given or their shapes differ.

    Examples
    --------
    Unanimous raters give a consensus equal to the input:

    >>> m = np.array([[0, 1], [1, 0]], dtype=bool)
    >>> w, stats_ = staple_consensus([m, m, m])
    >>> bool(np.allclose(w > 0.5, m))
    True
    """
    if len(masks) < 2:
        raise ValueError(f"STAPLE needs at least 2 raters, got {len(masks)}")

    shapes = {np.asarray(m).shape for m in masks}
    if len(shapes) != 1:
        raise ValueError(f"all masks must have the same shape, got {shapes}")

    stack = np.stack([np.asarray(m).astype(bool).reshape(-1) for m in masks])
    n_raters, n_voxels = stack.shape

    weight = np.full(n_voxels, float(prior))
    sensitivity = np.full(n_raters, 0.99)
    specificity = np.full(n_raters, 0.99)
    eps = 1e-10
    iterations = 0

    for iterations in range(1, max_iterations + 1):
        # E step: probability each voxel is foreground, in log space for stability.
        log_fg = np.zeros(n_voxels)
        log_bg = np.zeros(n_voxels)
        for r in range(n_raters):
            decision = stack[r]
            log_fg += np.where(
                decision, np.log(sensitivity[r] + eps), np.log(1 - sensitivity[r] + eps)
            )
            log_bg += np.where(
                decision, np.log(1 - specificity[r] + eps), np.log(specificity[r] + eps)
            )
        log_fg += np.log(prior + eps)
        log_bg += np.log(1 - prior + eps)

        peak = np.maximum(log_fg, log_bg)
        fg = np.exp(log_fg - peak)
        bg = np.exp(log_bg - peak)
        new_weight = fg / (fg + bg + eps)

        # M step.
        total_fg = new_weight.sum()
        total_bg = (1 - new_weight).sum()
        for r in range(n_raters):
            decision = stack[r]
            sensitivity[r] = float(np.sum(new_weight[decision]) / max(total_fg, eps))
            specificity[r] = float(np.sum((1 - new_weight)[~decision]) / max(total_bg, eps))
        sensitivity = np.clip(sensitivity, eps, 1 - eps)
        specificity = np.clip(specificity, eps, 1 - eps)

        shift = float(np.abs(new_weight - weight).max())
        weight = new_weight
        if shift < tolerance:
            break

    shape = np.asarray(masks[0]).shape
    return weight.reshape(shape), {
        "sensitivity": sensitivity,
        "specificity": specificity,
        "iterations": np.asarray([iterations]),
    }


def error_floor_from_testretest(
    repeat_pairs: np.ndarray | list, *, z: float = 1.96
) -> dict[str, float]:
    """Measurement-error model that every accuracy claim is expressed against.

    Parameters
    ----------
    repeat_pairs
        ``(n_subjects, n_repeats)`` test-retest measurements.
    z
        Normal quantile for the SDD.

    Returns
    -------
    dict
        ``sem``, ``sdd``, ``rms_cv_percent``, ``n_subjects``, ``n_repeats``.

    Notes
    -----
    No difference smaller than the SDD reported here is interpretable in an
    individual knee, no matter how significant it looks in a group comparison.
    """
    matrix = _as_repeat_matrix(repeat_pairs)
    sem_value = sem(matrix)
    return {
        "sem": sem_value,
        "sdd": sdd(sem_value, z=z),
        "rms_cv_percent": rms_cv(matrix),
        "n_subjects": int(matrix.shape[0]),
        "n_repeats": int(matrix.shape[1]),
    }


def detectability_table(
    sdd_by_subregion: dict[str, float],
    annual_change_by_subregion: dict[str, float],
) -> pd.DataFrame:
    """Which subregions can actually detect their own annual change (RQ4).

    Parameters
    ----------
    sdd_by_subregion
        Smallest detectable difference per subregion, in mm.
    annual_change_by_subregion
        Observed (or literature) annual thickness change per subregion, in mm.
        Sign is ignored; the magnitude is what has to clear the SDD.

    Returns
    -------
    pandas.DataFrame
        Per subregion: ``sdd_mm``, ``annual_change_mm``, ``ratio``
        (``|change| / SDD``) and ``detectable`` (ratio >= 1), sorted by ratio.

    Notes
    -----
    ``detectable = False`` does not mean the subregion is useless -- group-level
    change can still be detected with enough subjects, because the standard
    error of a mean shrinks with ``sqrt(n)``. It means change in an *individual*
    knee over one year cannot be distinguished from measurement noise there.
    """
    rows = []
    for subregion in sorted(set(sdd_by_subregion) | set(annual_change_by_subregion)):
        sdd_value = sdd_by_subregion.get(subregion, float("nan"))
        change = annual_change_by_subregion.get(subregion, float("nan"))
        ratio = abs(change) / sdd_value if sdd_value and np.isfinite(sdd_value) else float("nan")
        rows.append(
            {
                "subregion": subregion,
                "sdd_mm": sdd_value,
                "annual_change_mm": change,
                "ratio": ratio,
                "detectable": bool(np.isfinite(ratio) and ratio >= 1.0),
            }
        )
    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame = frame.sort_values("ratio", ascending=False, na_position="last")
    return frame.reset_index(drop=True)

### Segmentation backbones

`confcarti/models/backbones.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/models/backbones.py
# ==========================================================================
"""Segmentation backbones behind one interface, plus training and inference.

Three backbones are supported:

``swinunetr``
    MONAI SwinUNETR (v2 residual blocks by default). Supports the
    "FrozenSwinUNETR" configuration -- encoder frozen, decoder and head
    trainable -- with an option to release the deepest encoder stage.
``segresnet``
    MONAI SegResNet, a cheaper CNN baseline for ablation A7.
``nnunet``
    Inference-only adapter around a model trained through nnU-Net v2's own CLI.
    nnU-Net is deliberately *not* reimplemented: its value is its
    self-configuration, and a partial reimplementation would be a different
    method wearing its name.

Everything here imports torch lazily, so the geometry, conformal and reliability
layers stay usable in a torch-free environment.
"""


import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

logger = logging.getLogger(__name__)

__all__ = [
    "BackboneError",
    "build_backbone",
    "freeze_encoder",
    "count_parameters",
    "nnUNetAdapter",
    "sliding_window_predict",
]


class BackboneError(RuntimeError):
    """Raised when a backbone cannot be constructed or loaded."""


def _require_torch() -> Any:
    """Import torch or explain precisely what is missing."""
    try:
        import torch
    except ImportError as exc:  # pragma: no cover - environment dependent
        raise BackboneError(
            "this operation needs PyTorch. Install the deep-learning extra:\n"
            "    pip install 'confcarti[dl]'\n"
            "The geometry, conformal and reliability layers do not need it."
        ) from exc
    return torch


def _require_monai() -> Any:
    """Import MONAI or explain what is missing."""
    try:
        import monai
    except ImportError as exc:  # pragma: no cover
        raise BackboneError(
            "this operation needs MONAI. Install with: pip install 'confcarti[dl]'"
        ) from exc
    return monai


def count_parameters(model: Any) -> dict[str, int]:
    """Count total, trainable and frozen parameters.

    Parameters
    ----------
    model
        A ``torch.nn.Module``.

    Returns
    -------
    dict
        ``total``, ``trainable``, ``frozen``.
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "frozen": total - trainable}


def freeze_encoder(
    model: Any, *, unfreeze_last_stage: bool = False, encoder_attr: str = "swinViT"
) -> dict[str, int]:
    """Freeze a SwinUNETR encoder, optionally releasing its deepest stage.

    Parameters
    ----------
    model
        SwinUNETR instance.
    unfreeze_last_stage
        Leave the deepest encoder stage trainable. This recovers most of the
        accuracy of full fine-tuning at a fraction of the memory, because the
        deepest stage has the smallest spatial extent.
    encoder_attr
        Attribute holding the encoder.

    Returns
    -------
    dict
        Parameter counts after freezing.

    Raises
    ------
    BackboneError
        If the model has no such attribute.
    """
    encoder = getattr(model, encoder_attr, None)
    if encoder is None:
        raise BackboneError(
            f"model has no attribute {encoder_attr!r}; cannot freeze the encoder. "
            f"Available: {[n for n, _ in model.named_children()]}"
        )

    for param in encoder.parameters():
        param.requires_grad = False

    if unfreeze_last_stage:
        layers = [m for name, m in encoder.named_children() if name.startswith("layers")]
        if not layers:
            logger.warning(
                "unfreeze_last_stage requested but no 'layers*' children found on the "
                "encoder; the whole encoder stays frozen"
            )
        else:
            for param in layers[-1].parameters():
                param.requires_grad = True

    counts = count_parameters(model)
    logger.info(
        "encoder frozen: %d trainable of %d parameters (%.1f%%)",
        counts["trainable"],
        counts["total"],
        100.0 * counts["trainable"] / max(counts["total"], 1),
    )
    return counts


def build_backbone(
    backbone: str = "swinunetr",
    *,
    in_channels: int = 1,
    out_channels: int = 6,
    img_size: tuple[int, int, int] = (128, 128, 128),
    feature_size: int = 48,
    use_v2: bool = True,
    dropout_rate: float = 0.0,
    freeze: bool = False,
    unfreeze_last_stage: bool = False,
    checkpoint: str | Path | None = None,
    device: str | None = None,
) -> Any:
    """Construct a segmentation backbone.

    Parameters
    ----------
    backbone
        ``swinunetr`` or ``segresnet``. Use :class:`nnUNetAdapter` for nnU-Net.
    in_channels, out_channels
        Channel counts; ``out_channels`` is 6 for the 5-ROI protocol
        (background plus five structures).
    img_size
        Patch size the network is built for.
    feature_size
        SwinUNETR embedding size; must be divisible by 12.
    use_v2
        SwinUNETR v2 residual blocks.
    dropout_rate
        Dropout probability.
    freeze
        Freeze the encoder (SwinUNETR only).
    unfreeze_last_stage
        Release the deepest encoder stage when frozen.
    checkpoint
        Optional state dict to load.
    device
        Torch device string; defaults to CUDA when available.

    Returns
    -------
    torch.nn.Module

    Raises
    ------
    BackboneError
        For an unknown backbone, a bad feature size, or a checkpoint that does
        not match.
    """
    torch = _require_torch()
    _require_monai()

    if feature_size % 12 != 0:
        raise BackboneError(
            f"feature_size must be divisible by 12 for SwinUNETR, got {feature_size}"
        )

    if backbone == "swinunetr":
        from monai.networks.nets import SwinUNETR

        # MONAI removed the img_size argument from SwinUNETR in 1.5; the network
        # is fully convolutional in its spatial dims, so it is optional anyway.
        # Try the modern signature first and fall back for older MONAI.
        kwargs: dict[str, Any] = {
            "in_channels": in_channels,
            "out_channels": out_channels,
            "feature_size": feature_size,
            "drop_rate": dropout_rate,
            "use_v2": use_v2,
        }
        try:
            model = SwinUNETR(**kwargs)
        except TypeError:
            model = SwinUNETR(img_size=img_size, **kwargs)

        if freeze:
            freeze_encoder(model, unfreeze_last_stage=unfreeze_last_stage)

    elif backbone == "segresnet":
        from monai.networks.nets import SegResNet

        model = SegResNet(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,
            init_filters=max(8, feature_size // 3),
            dropout_prob=dropout_rate or None,
        )
        if freeze:
            logger.warning("freeze=True is ignored for segresnet: it has no separate encoder")

    else:
        raise BackboneError(
            f"unknown backbone {backbone!r}; expected swinunetr|segresnet "
            "(use nnUNetAdapter for nnU-Net)"
        )

    if checkpoint is not None:
        path = Path(checkpoint)
        if not path.is_file():
            raise BackboneError(f"checkpoint not found: {path}")
        state = torch.load(path, map_location="cpu", weights_only=False)
        state = state.get("model_state_dict", state) if isinstance(state, dict) else state
        missing, unexpected = model.load_state_dict(state, strict=False)
        if missing or unexpected:
            logger.warning(
                "checkpoint loaded with %d missing and %d unexpected key(s)",
                len(missing),
                len(unexpected),
            )

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    counts = count_parameters(model)
    logger.info(
        "built %s on %s: %.1fM parameters (%.1fM trainable)",
        backbone,
        device,
        counts["total"] / 1e6,
        counts["trainable"] / 1e6,
    )
    return model


class nnUNetAdapter:
    """Inference-only wrapper around a trained nnU-Net v2 model.

    nnU-Net is trained through its own CLI (``nnUNetv2_train``); this adapter
    only runs a trained model so its predictions can enter the same evaluation,
    thickness and conformal pipeline as the other backbones.

    Parameters
    ----------
    model_dir
        Directory holding the trained nnU-Net model
        (``nnUNetTrainer__nnUNetPlans__3d_fullres`` or similar).
    folds
        Which cross-validation folds to ensemble.
    checkpoint_name
        Checkpoint file inside each fold directory.

    Raises
    ------
    BackboneError
        If the directory does not exist.
    """

    def __init__(
        self,
        model_dir: str | Path,
        *,
        folds: tuple[int, ...] = (0,),
        checkpoint_name: str = "checkpoint_final.pth",
    ) -> None:
        self.model_dir = Path(model_dir)
        if not self.model_dir.is_dir():
            raise BackboneError(
                f"nnU-Net model directory not found: {self.model_dir}. Train one with "
                "nnUNetv2_train and point model.nnunet_model_dir at the result."
            )
        self.folds = folds
        self.checkpoint_name = checkpoint_name
        self._predictor: Any | None = None

    def _load(self) -> Any:
        """Lazily construct the nnU-Net predictor."""
        if self._predictor is not None:
            return self._predictor
        try:
            from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
        except ImportError as exc:
            raise BackboneError(
                "nnU-Net v2 is not installed. Install it with 'pip install nnunetv2' "
                "to use the nnunet backbone."
            ) from exc

        torch = _require_torch()
        predictor = nnUNetPredictor(
            tile_step_size=0.5,
            use_gaussian=True,
            use_mirroring=True,
            device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
            verbose=False,
            allow_tqdm=False,
        )
        predictor.initialize_from_trained_model_folder(
            str(self.model_dir),
            use_folds=self.folds,
            checkpoint_name=self.checkpoint_name,
        )
        self._predictor = predictor
        return predictor

    def predict(
        self, image: np.ndarray, spacing: tuple[float, float, float]
    ) -> np.ndarray:
        """Predict a label map for one volume.

        Parameters
        ----------
        image
            ``(X, Y, Z)`` intensity volume.
        spacing
            Voxel spacing in mm.

        Returns
        -------
        numpy.ndarray
            Integer label map with the same shape as ``image``.
        """
        predictor = self._load()
        # nnU-Net expects (channel, Z, Y, X) with matching spacing order.
        data = np.asarray(image, dtype=np.float32)[None].transpose(0, 3, 2, 1)
        properties = {"spacing": tuple(reversed(spacing))}
        prediction = predictor.predict_single_npy_array(data, properties, None, None, False)
        return np.asarray(prediction).transpose(2, 1, 0)


def sliding_window_predict(
    model: Any,
    image: np.ndarray,
    *,
    roi_size: tuple[int, int, int] = (128, 128, 128),
    sw_batch_size: int = 2,
    overlap: float = 0.5,
    amp_dtype: str = "bf16",
    device: str | None = None,
    return_probabilities: bool = True,
) -> tuple[np.ndarray, np.ndarray | None]:
    """Run MONAI sliding-window inference on one volume.

    Parameters
    ----------
    model
        Trained network.
    image
        ``(X, Y, Z)`` or ``(C, X, Y, Z)`` volume.
    roi_size
        Sliding-window size.
    sw_batch_size
        Windows evaluated per forward pass.
    overlap
        Window overlap fraction. Gaussian blending is used, so 0.5 is a good
        default; lower values leave visible seams in the probability map, which
        matters here because those probabilities feed conformal risk control.
    amp_dtype
        ``bf16``, ``fp16`` or ``fp32``.
    device
        Torch device; defaults to CUDA when available.
    return_probabilities
        Also return the per-voxel softmax probabilities as float16.

    Returns
    -------
    tuple
        ``(labels, probabilities)``. ``probabilities`` is ``(C, X, Y, Z)``
        float16, or None when not requested.
    """
    torch = _require_torch()
    from monai.inferers import sliding_window_inference

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    array = np.asarray(image, dtype=np.float32)
    if array.ndim == 3:
        array = array[None]
    tensor = torch.from_numpy(array)[None].to(device)

    dtype_map = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}
    autocast_dtype = dtype_map.get(amp_dtype, torch.float32)
    # bf16/fp16 autocast is a CUDA feature; on CPU it is either unsupported or
    # far slower than fp32, so it is disabled there rather than silently ignored.
    use_amp = device.startswith("cuda") and amp_dtype != "fp32"

    model.eval()
    with torch.inference_mode():
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=autocast_dtype):
                logits = sliding_window_inference(
                    tensor, roi_size, sw_batch_size, model, overlap=overlap, mode="gaussian"
                )
        else:
            logits = sliding_window_inference(
                tensor, roi_size, sw_batch_size, model, overlap=overlap, mode="gaussian"
            )

        probabilities = torch.softmax(logits.float(), dim=1)[0]
        labels = probabilities.argmax(dim=0).to(torch.uint8).cpu().numpy()
        probability_array = (
            probabilities.to(torch.float16).cpu().numpy() if return_probabilities else None
        )

    del tensor, logits
    if device.startswith("cuda"):
        torch.cuda.empty_cache()

    return labels, probability_array

### Training loop and segmentation metrics

`confcarti/models/train.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/models/train.py
# ==========================================================================
"""Patch-based training loop and segmentation metrics.

The training split is the *only* split this module may read. That is enforced by
:class:`confcarti.data.guards.CalibrationGuard`, not by convention: the internal
validation subset used for early stopping is carved out of the training split,
never from calibration, because any model selection informed by calibration data
voids the conformal coverage guarantee.
"""


import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)

__all__ = [
    "TrainingHistory",
    "train_segmentation",
    "dice_score",
    "hausdorff95",
    "average_surface_distance",
    "evaluate_segmentation",
]


@dataclass
class TrainingHistory:
    """Loss and validation curves from a training run.

    Attributes
    ----------
    epochs
        Epoch indices.
    train_loss
        Mean training loss per epoch.
    val_dice
        Mean internal-validation Dice per epoch.
    best_epoch, best_val_dice
        Best internal-validation checkpoint.
    stopped_early
        Whether early stopping fired.
    seconds
        Wall-clock training time.
    """

    epochs: list[int] = field(default_factory=list)
    train_loss: list[float] = field(default_factory=list)
    val_dice: list[float] = field(default_factory=list)
    best_epoch: int = -1
    best_val_dice: float = -np.inf
    stopped_early: bool = False
    seconds: float = 0.0

    def to_frame(self) -> pd.DataFrame:
        """Return the curves as a DataFrame for plotting."""
        return pd.DataFrame(
            {"epoch": self.epochs, "train_loss": self.train_loss, "val_dice": self.val_dice}
        )


def _random_patch(
    image: np.ndarray,
    label: np.ndarray,
    patch_size: tuple[int, int, int],
    rng: np.random.Generator,
    *,
    foreground_bias: float = 0.7,
) -> tuple[np.ndarray, np.ndarray]:
    """Sample one random patch, biased towards foreground.

    Cartilage occupies a few percent of a knee volume, so uniform patch sampling
    would return mostly background and the network would learn to predict
    background. ``foreground_bias`` is the probability of centring the patch on a
    randomly chosen cartilage voxel instead.
    """
    spatial = image.shape[-3:]
    starts = []

    centre_on_foreground = rng.random() < foreground_bias
    centre = None
    if centre_on_foreground:
        foreground = np.argwhere(label[0] > 0) if label.ndim == 4 else np.argwhere(label > 0)
        if foreground.size:
            centre = foreground[rng.integers(0, len(foreground))]

    for axis, size in enumerate(patch_size):
        extent = spatial[axis]
        if size >= extent:
            starts.append(0)
            continue
        if centre is not None:
            lo = int(centre[axis]) - size // 2
            lo = int(np.clip(lo, 0, extent - size))
        else:
            lo = int(rng.integers(0, extent - size + 1))
        starts.append(lo)

    slices = tuple(
        slice(s, min(s + size, extent))
        for s, size, extent in zip(starts, patch_size, spatial)
    )
    image_patch = image[(slice(None), *slices)]
    label_patch = label[(slice(None), *slices)] if label.ndim == 4 else label[slices]

    # Pad if the volume was smaller than the patch.
    pad = [(0, 0)] + [
        (0, max(0, size - actual))
        for size, actual in zip(patch_size, image_patch.shape[-3:])
    ]
    if any(p[1] for p in pad):
        image_patch = np.pad(image_patch, pad, mode="constant")
        label_pad = pad if label_patch.ndim == 4 else pad[1:]
        label_patch = np.pad(label_patch, label_pad, mode="constant")

    return image_patch, label_patch


def train_segmentation(
    dataset: Any,
    train_indices: list[int],
    *,
    out_channels: int = 6,
    backbone: str = "swinunetr",
    patch_size: tuple[int, int, int] = (96, 96, 64),
    batch_size: int = 2,
    max_epochs: int = 50,
    steps_per_epoch: int = 20,
    learning_rate: float = 1e-4,
    weight_decay: float = 1e-5,
    grad_clip_norm: float = 1.0,
    dice_ce_lambda: float = 0.5,
    internal_val_fraction: float = 0.15,
    early_stopping_patience: int = 15,
    amp_dtype: str = "bf16",
    gradient_checkpointing: bool = False,
    freeze_encoder_flag: bool = True,
    unfreeze_last_stage: bool = False,
    feature_size: int = 48,
    device: str | None = None,
    seed: int = 0,
    checkpoint_path: str | Path | None = None,
) -> tuple[Any, TrainingHistory]:
    """Train a segmentation backbone on patches from the training split.

    Parameters
    ----------
    dataset
        Indexable dataset yielding ``{"image", "label", ...}`` dicts.
    train_indices
        Dataset indices belonging to the **training** split only.
    out_channels
        Number of classes including background.
    backbone
        ``swinunetr`` or ``segresnet``.
    patch_size
        Training patch size.
    batch_size
        Patches per optimiser step.
    max_epochs, steps_per_epoch
        Schedule length.
    learning_rate, weight_decay
        AdamW hyper-parameters.
    grad_clip_norm
        Gradient-norm clip.
    dice_ce_lambda
        Weight of the Dice term; cross-entropy gets ``1 - lambda``.
    internal_val_fraction
        Fraction of ``train_indices`` held out for early stopping.
    early_stopping_patience
        Epochs without improvement before stopping.
    amp_dtype
        ``bf16``, ``fp16`` or ``fp32``.
    gradient_checkpointing
        Enable activation checkpointing where the backbone supports it.
    freeze_encoder_flag, unfreeze_last_stage, feature_size
        Backbone configuration.
    device
        Torch device.
    seed
        RNG seed.
    checkpoint_path
        Where to save the best checkpoint.

    Returns
    -------
    tuple
        ``(model, history)``.

    Raises
    ------
    ValueError
        If ``train_indices`` is empty or too small to split.
    """
    import torch
    from monai.losses import DiceCELoss

    if not train_indices:
        raise ValueError("train_indices is empty; nothing to train on")

    rng = seeded_generator(seed)
    indices = np.asarray(sorted(train_indices))
    rng.shuffle(indices)

    n_val = max(1, int(round(len(indices) * internal_val_fraction)))
    if len(indices) - n_val < 1:
        raise ValueError(
            f"{len(indices)} training case(s) cannot be split into train and internal "
            f"validation at internal_val_fraction={internal_val_fraction}"
        )
    val_idx = indices[:n_val].tolist()
    fit_idx = indices[n_val:].tolist()

    logger.info(
        "training on %d case(s), internal validation on %d (carved from TRAIN, never "
        "from calibration)",
        len(fit_idx),
        len(val_idx),
    )

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = build_backbone(
        backbone,
        out_channels=out_channels,
        img_size=patch_size,
        feature_size=feature_size,
        freeze=freeze_encoder_flag,
        unfreeze_last_stage=unfreeze_last_stage,
        device=device,
    )

    if gradient_checkpointing and hasattr(model, "swinViT"):
        for module in model.swinViT.modules():
            if hasattr(module, "use_checkpoint"):
                module.use_checkpoint = True

    loss_fn = DiceCELoss(
        to_onehot_y=True,
        softmax=True,
        include_background=False,
        lambda_dice=dice_ce_lambda,
        lambda_ce=1.0 - dice_ce_lambda,
    )
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(max_epochs, 1))

    use_amp = device.startswith("cuda") and amp_dtype != "fp32"
    autocast_dtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(
        amp_dtype, torch.float32
    )

    history = TrainingHistory()
    patience = 0
    start = time.time()

    # The guard makes an accidental read of calibration data an exception rather
    # than a silent, unrecoverable invalidation of the coverage guarantee.
    with CalibrationGuard(allow=["train"], label="train_segmentation"):
        for epoch in range(max_epochs):
            model.train()
            epoch_losses = []

            for _ in range(steps_per_epoch):
                images, labels = [], []
                for _ in range(batch_size):
                    item = dataset[int(rng.choice(fit_idx))]
                    image_patch, label_patch = _random_patch(
                        item["image"], item["label"], patch_size, rng
                    )
                    images.append(image_patch)
                    labels.append(label_patch)

                image_batch = torch.from_numpy(np.stack(images)).float().to(device)
                label_batch = torch.from_numpy(np.stack(labels)).long().to(device)

                optimizer.zero_grad(set_to_none=True)
                if use_amp:
                    with torch.autocast(device_type="cuda", dtype=autocast_dtype):
                        logits = model(image_batch)
                        loss = loss_fn(logits, label_batch)
                else:
                    logits = model(image_batch)
                    loss = loss_fn(logits, label_batch)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, grad_clip_norm)
                optimizer.step()
                epoch_losses.append(float(loss.detach().cpu()))

            scheduler.step()

            val_dice = _internal_validation(
                model, dataset, val_idx, patch_size, out_channels, device
            )
            history.epochs.append(epoch)
            history.train_loss.append(float(np.mean(epoch_losses)))
            history.val_dice.append(val_dice)

            if val_dice > history.best_val_dice:
                history.best_val_dice = val_dice
                history.best_epoch = epoch
                patience = 0
                if checkpoint_path is not None:
                    Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
                    torch.save(
                        {
                            "model_state_dict": model.state_dict(),
                            "epoch": epoch,
                            "val_dice": val_dice,
                            "backbone": backbone,
                        },
                        checkpoint_path,
                    )
            else:
                patience += 1

            logger.info(
                "epoch %3d | train loss %.4f | internal val Dice %.4f%s",
                epoch,
                history.train_loss[-1],
                val_dice,
                "  *" if history.best_epoch == epoch else "",
            )

            if patience >= early_stopping_patience:
                history.stopped_early = True
                logger.info("early stopping at epoch %d", epoch)
                break

    history.seconds = time.time() - start
    return model, history


def _internal_validation(
    model: Any,
    dataset: Any,
    val_indices: list[int],
    patch_size: tuple[int, int, int],
    out_channels: int,
    device: str,
) -> float:
    """Mean foreground Dice over the internal-validation subset."""
    import torch

    scores = []
    model.eval()
    for index in val_indices:
        item = dataset[index]
        prediction, _ = sliding_window_predict(
            model,
            item["image"],
            roi_size=patch_size,
            sw_batch_size=1,
            overlap=0.25,
            amp_dtype="fp32",
            device=device,
            return_probabilities=False,
        )
        truth = np.asarray(item["label"])[0]
        per_class = [
            dice_score(prediction == c, truth == c)
            for c in range(1, out_channels)
            if (truth == c).any()
        ]
        if per_class:
            scores.append(float(np.mean(per_class)))

    if device.startswith("cuda"):
        torch.cuda.empty_cache()
    return float(np.mean(scores)) if scores else 0.0


# --------------------------------------------------------------------------- #
# Segmentation metrics
# --------------------------------------------------------------------------- #


def dice_score(prediction: np.ndarray, truth: np.ndarray) -> float:
    """Dice similarity coefficient between two binary masks.

    .. math:: \\mathrm{DSC} = \\frac{2 |A \\cap B|}{|A| + |B|}

    Parameters
    ----------
    prediction, truth
        Boolean masks of the same shape.

    Returns
    -------
    float
        Dice in [0, 1]. Two empty masks score 1.0 (they agree perfectly);
        one empty and one not scores 0.0.

    Examples
    --------
    >>> a = np.array([1, 1, 0, 0], dtype=bool)
    >>> float(dice_score(a, a))
    1.0
    >>> float(dice_score(a, ~a))
    0.0
    """
    prediction = np.asarray(prediction).astype(bool)
    truth = np.asarray(truth).astype(bool)
    if prediction.shape != truth.shape:
        raise ValueError(f"shape mismatch: {prediction.shape} vs {truth.shape}")

    total = prediction.sum() + truth.sum()
    if total == 0:
        return 1.0
    return float(2.0 * np.logical_and(prediction, truth).sum() / total)


def _surface_distances(
    prediction: np.ndarray, truth: np.ndarray, spacing: tuple[float, float, float]
) -> tuple[np.ndarray, np.ndarray]:
    """Symmetric surface-to-surface distances between two binary masks, in mm."""
    from scipy import ndimage

    prediction = np.asarray(prediction).astype(bool)
    truth = np.asarray(truth).astype(bool)
    if not prediction.any() or not truth.any():
        return np.array([np.inf]), np.array([np.inf])

    def _border(mask: np.ndarray) -> np.ndarray:
        eroded = ndimage.binary_erosion(mask, iterations=1, border_value=0)
        return mask & ~eroded

    pred_border = _border(prediction)
    truth_border = _border(truth)

    dist_to_truth = ndimage.distance_transform_edt(~truth_border, sampling=spacing)
    dist_to_pred = ndimage.distance_transform_edt(~pred_border, sampling=spacing)

    return dist_to_truth[pred_border], dist_to_pred[truth_border]


def hausdorff95(
    prediction: np.ndarray,
    truth: np.ndarray,
    spacing: tuple[float, float, float] = (1.0, 1.0, 1.0),
) -> float:
    """95th-percentile symmetric Hausdorff distance in mm.

    The 95th percentile rather than the maximum, because a single stray voxel
    would otherwise dominate the metric.

    Parameters
    ----------
    prediction, truth
        Boolean masks.
    spacing
        Voxel spacing in mm.

    Returns
    -------
    float
        HD95 in mm, or inf when either mask is empty.
    """
    a, b = _surface_distances(prediction, truth, spacing)
    if not np.isfinite(a).all() or not np.isfinite(b).all():
        return float("inf")
    return float(max(np.percentile(a, 95), np.percentile(b, 95)))


def average_surface_distance(
    prediction: np.ndarray,
    truth: np.ndarray,
    spacing: tuple[float, float, float] = (1.0, 1.0, 1.0),
) -> float:
    """Average symmetric surface distance in mm.

    Parameters
    ----------
    prediction, truth
        Boolean masks.
    spacing
        Voxel spacing in mm.

    Returns
    -------
    float
        ASSD in mm, or inf when either mask is empty.
    """
    a, b = _surface_distances(prediction, truth, spacing)
    if not np.isfinite(a).all() or not np.isfinite(b).all():
        return float("inf")
    return float((a.sum() + b.sum()) / (a.size + b.size))


def evaluate_segmentation(
    prediction: np.ndarray,
    truth: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    subject_id: str,
    kl_grade: int | None = None,
    site: str | None = None,
    model_name: str = "unknown",
    labels: dict[int, str] | None = None,
    compute_distances: bool = True,
) -> pd.DataFrame:
    """Per-ROI Dice, HD95 and ASSD as tidy long-format rows.

    Parameters
    ----------
    prediction, truth
        Integer label volumes.
    spacing
        Voxel spacing in mm.
    subject_id, kl_grade, site, model_name
        Stratifiers carried into every row.
    labels
        Label value -> name. Defaults to the OAIZIB-CM convention.
    compute_distances
        Compute HD95 and ASSD. They cost several seconds per ROI, so batch
        evaluations may skip them.

    Returns
    -------
    pandas.DataFrame
        Long-format rows with ``metric`` in {``dice``, ``hd95_mm``, ``assd_mm``}.
    """
    labels = labels or {k: v for k, v in LABEL_NAMES.items() if k != 0}
    rows: list[dict[str, Any]] = []

    for value, name in labels.items():
        pred_mask = prediction == value
        truth_mask = truth == value
        if not truth_mask.any() and not pred_mask.any():
            continue

        base = {
            "subject_id": str(subject_id),
            "subregion": name,
            "kl_grade": kl_grade,
            "site": site,
            "split": "test",
            "model": model_name,
            "method": "segmentation",
            "seed": None,
        }
        rows.append({**base, "metric": "dice", "value": dice_score(pred_mask, truth_mask)})

        if compute_distances:
            rows.append(
                {**base, "metric": "hd95_mm", "value": hausdorff95(pred_mask, truth_mask, spacing)}
            )
            rows.append(
                {
                    **base,
                    "metric": "assd_mm",
                    "value": average_surface_distance(pred_mask, truth_mask, spacing),
                }
            )

    return pd.DataFrame(rows)

### Per-knee morphometry pipeline

`confcarti/eval/pipeline.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/eval/pipeline.py
# ==========================================================================
"""Per-knee morphometry pipeline: masks in, tidy long-format rows out.

One function, :func:`analyse_knee`, runs the whole geometric chain for a single
segmentation: surfaces, thickness, curvature, bulge, parcellation, denuded-bone
metrics. It returns both a tidy DataFrame (the single source of truth every
table and figure is built from) and the per-vertex fields needed to paint the 3D
surface maps.
"""


import logging
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)

__all__ = ["KneeAnalysis", "analyse_knee", "compartment_surfaces"]


@dataclass
class KneeAnalysis:
    """Everything computed for one knee.

    Attributes
    ----------
    rows
        Tidy long-format results.
    surfaces
        Per-compartment meshes and per-vertex fields, for visualisation.
    warnings
        Non-fatal problems encountered (empty plates, unestimable curvature...).
    """

    rows: pd.DataFrame
    surfaces: dict[str, dict[str, Any]] = field(default_factory=dict)
    warnings: list[str] = field(default_factory=list)


def compartment_surfaces(
    label: np.ndarray,
    spacing: tuple[float, float, float],
    compartment_key: str,
    *,
    gaussian_sigma_mm: float = 1.0,
    smoothing_iterations: int = 10,
    taubin_lambda: float = 0.5,
    taubin_mu: float = -0.53,
    min_component_vertices: int = 50,
    closing_radius_mm: float = 10.0,
) -> dict[str, Any]:
    """Extract the BCI and articular surfaces for one compartment.

    Parameters
    ----------
    label
        5-ROI label volume.
    spacing
        Voxel spacing in mm.
    compartment_key
        ``FC``, ``MTC`` or ``LTC``.
    gaussian_sigma_mm
        Mask pre-smoothing. Note the default of 1.0 mm rather than the 0.4 mm
        used for pure surface extraction: curvature is a second derivative and
        needs the extra smoothing to measure anatomy instead of voxel
        staircasing.
    smoothing_iterations, taubin_lambda, taubin_mu, min_component_vertices
        Mesh extraction parameters.
    closing_radius_mm
        Largest full-thickness defect bridged when forming tAB. Setting this to
        0 measures cAB instead and forces dAB to zero.

    Returns
    -------
    dict
        ``bci``, ``articular``, ``cartilage_mask``, ``bone_mask`` and reports.
    """
    comp = COMPARTMENTS[compartment_key]
    cartilage_mask = label == comp.cartilage_label
    bone_mask = label == comp.bone_label

    kwargs = {
        "gaussian_sigma_mm": gaussian_sigma_mm,
        "smoothing_iterations": smoothing_iterations,
        "taubin_lambda": taubin_lambda,
        "taubin_mu": taubin_mu,
        "min_component_vertices": min_component_vertices,
    }
    bci, bci_report = extract_bci_mesh(
        bone_mask, cartilage_mask, spacing, closing_radius_mm=closing_radius_mm, **kwargs
    )
    articular, articular_report = extract_articular_surface(
        cartilage_mask, spacing, bone_mask=bone_mask, **kwargs
    )
    return {
        "bci": bci,
        "articular": articular,
        "bci_report": bci_report,
        "articular_report": articular_report,
        "cartilage_mask": cartilage_mask,
        "bone_mask": bone_mask,
    }


def _row(
    subject_id: str,
    subregion: str,
    metric: str,
    value: float,
    context: dict[str, Any],
) -> dict[str, Any]:
    """Build one tidy result row."""
    return {
        "subject_id": str(subject_id),
        "subregion": subregion,
        "kl_grade": context.get("kl_grade"),
        "site": context.get("site"),
        "split": context.get("split"),
        "model": context.get("model"),
        "method": context.get("method"),
        "seed": context.get("seed"),
        "metric": metric,
        "value": float(value),
    }


def analyse_knee(
    label: np.ndarray,
    spacing: tuple[float, float, float],
    *,
    subject_id: str,
    kl_grade: int | None = None,
    site: str | None = None,
    split: str | None = None,
    laterality: str = "right",
    model_name: str = "manual",
    seed: int | None = None,
    thickness_method: str = "surface_normal",
    parcellation_scheme: str = "cartimorph20",
    compute_geometry: bool = True,
    denuded_enabled: bool = True,
    curvature_radius_mm: float = 3.0,
    curvature_min_radius_mm: float | None = 1.5,
    curvature_adaptive: bool = True,
    curvature_mask_boundary: bool = False,
    bulge_form_radius_mm: float = 12.0,
    closing_radius_mm: float = 10.0,
    min_thickness_mm: float = 0.35,
    min_patch_area_mm2: float = 1.0,
    prior_strength: float = 1.0,
    prior_gate_enabled: bool = True,
    central_percentage: float = 0.5,
    max_depth_mm: float = 10.0,
    normal_neighbours: int = 30,
    smoothing_neighbours: int = 10,
    keep_surfaces: bool = True,
) -> KneeAnalysis:
    """Run the full morphometry chain on one segmented knee.

    Parameters
    ----------
    label
        5-ROI label volume following the OAIZIB-CM convention.
    spacing
        Voxel spacing in mm.
    subject_id, kl_grade, site, split, model_name, seed
        Stratifiers carried into every output row.
    laterality
        ``left`` or ``right``; controls parcellation mirroring.
    thickness_method
        ``surface_normal``, ``distance_transform`` or ``nearest_neighbour``
        (ablation A1).
    parcellation_scheme
        ``cartimorph20`` or ``compartment5``.
    compute_geometry
        Compute curvature and bulge (ablation A8).
    denuded_enabled
        Compute the denuded branch (ablation A2).
    curvature_radius_mm
        Nominal physical radius of the curvature fit.
    curvature_min_radius_mm, curvature_adaptive
        Adaptive-scale fallback near a plate rim; see
        :func:`confcarti.thickness.curvature.compute_curvature`.
    curvature_mask_boundary
        Additionally discard a geometric margin around the plate rim.
        Off by default -- the old ``2 * radius_mm`` margin removed
        76-87 per cent of every surface.
    bulge_form_radius_mm
        Radius of the bulge form fit.
    closing_radius_mm
        Largest full-thickness defect bridged into tAB.
    min_thickness_mm, min_patch_area_mm2
        Denuded-labelling parameters.
    prior_strength, prior_gate_enabled
        Continuity-prior parameters (ablations A6, A2).
    central_percentage
        CartiMorph ``cc_percentage`` for femoral parcellation.
    max_depth_mm, normal_neighbours, smoothing_neighbours
        Thickness parameters.
    keep_surfaces
        Retain meshes and per-vertex fields for visualisation. Set False in
        batch runs to keep memory flat.

    Returns
    -------
    KneeAnalysis

    Notes
    -----
    A compartment whose cartilage or bone mask is empty is *skipped with a
    recorded warning*, not silently dropped: an absent plate is a finding (or a
    segmentation failure), and either way the downstream tables must show a gap
    rather than an implicit zero.
    """
    context = {
        "kl_grade": kl_grade,
        "site": site,
        "split": split,
        "model": model_name,
        "method": thickness_method,
        "seed": seed,
    }
    rows: list[dict[str, Any]] = []
    surfaces: dict[str, dict[str, Any]] = {}
    warnings: list[str] = []
    label = np.asarray(label)

    for key in ("FC", "MTC", "LTC"):
        comp = COMPARTMENTS[key]
        if not (label == comp.cartilage_label).any():
            warnings.append(f"{subject_id}: compartment {key} has no cartilage voxels")
            logger.warning("%s: compartment %s has no cartilage; skipping", subject_id, key)
            continue
        if not (label == comp.bone_label).any():
            warnings.append(f"{subject_id}: compartment {key} has no bone voxels")
            continue

        try:
            surface = compartment_surfaces(
                label, spacing, key, closing_radius_mm=closing_radius_mm
            )
        except ValueError as exc:
            warnings.append(f"{subject_id}: {key} surface extraction failed: {exc}")
            logger.warning("%s: %s surface extraction failed: %s", subject_id, key, exc)
            continue

        bci = surface["bci"]
        articular = surface["articular"]
        cartilage_mask = surface["cartilage_mask"]

        # ---- thickness --------------------------------------------------- #
        thickness_result = compute_thickness(
            bci,
            articular,
            thickness_method,
            cartilage_mask=cartilage_mask,
            spacing=spacing,
            n_neighbours=normal_neighbours,
            smoothing_neighbours=smoothing_neighbours,
            max_depth_mm=max_depth_mm,
        )
        thickness = thickness_result.thickness_mm

        # ---- denuded ----------------------------------------------------- #
        probe = None
        if denuded_enabled and thickness_result.normals is not None:
            probe = denuded_from_cartilage_probe(
                bci, thickness_result.normals, cartilage_mask, spacing
            )
        denuded_result = derive_denuded_labels(
            thickness,
            bci,
            min_thickness_mm=min_thickness_mm if denuded_enabled else -1.0,
            min_patch_area_mm2=min_patch_area_mm2,
            denuded_probe=probe,
        )
        gate = prior_gate(
            denuded_result.denuded, strength=prior_strength, enabled=prior_gate_enabled
        )

        # ---- geometry ---------------------------------------------------- #
        curvature_result = None
        bulge_result = None
        thickness_bulge = None
        if compute_geometry:
            try:
                curvature_result = compute_curvature(
                    bci,
                    radius_mm=curvature_radius_mm,
                    min_radius_mm=curvature_min_radius_mm,
                    adaptive_radius=curvature_adaptive,
                    mask_boundary=curvature_mask_boundary,
                    smoothing_iterations=3,
                )
                if curvature_result.estimable_fraction < 0.5:
                    warnings.append(
                        f"{subject_id}: {key} curvature estimable on only "
                        f"{100 * curvature_result.estimable_fraction:.0f}% of the "
                        f"surface at radius {curvature_radius_mm} mm"
                    )
            except (ValueError, np.linalg.LinAlgError) as exc:
                warnings.append(f"{subject_id}: {key} curvature failed: {exc}")
            try:
                bulge_result = compute_bulge(
                    articular, form_radius_mm=bulge_form_radius_mm
                )
            except ValueError as exc:
                warnings.append(f"{subject_id}: {key} bulge failed: {exc}")
            thickness_bulge = thickness_residual_bulge(bci, thickness, smoothing_iterations=60)

        # ---- parcellation ------------------------------------------------ #
        parcels = parcellate(
            bci,
            key,
            knee_side=laterality if laterality in ("left", "right") else "right",
            scheme=parcellation_scheme,
            central_percentage=central_percentage,
        )

        # ---- compartment-level rows -------------------------------------- #
        metrics = denuded_metrics(denuded_result)
        for name, value in metrics.items():
            rows.append(_row(subject_id, key, name, value, context))

        summary = thickness_result.summary()
        rows.append(_row(subject_id, key, "thickness_mean_mm", summary["mean_mm"], context))
        rows.append(_row(subject_id, key, "thickness_median_mm", summary["median_mm"], context))
        rows.append(
            _row(subject_id, key, "thickness_defined_fraction", summary["defined_fraction"], context)
        )
        rows.append(
            _row(
                subject_id,
                key,
                "cartilage_volume_mm3",
                float(np.count_nonzero(cartilage_mask) * float(np.prod(spacing))),
                context,
            )
        )
        rows.append(
            _row(subject_id, key, "articular_area_mm2", float(articular.area), context)
        )
        rows.append(_row(subject_id, key, "prior_gate_mean", float(gate.mean()), context))

        if curvature_result is not None:
            for name, value in curvature_summary(curvature_result).items():
                rows.append(_row(subject_id, key, f"curv_{name}", value, context))
        if bulge_result is not None:
            for name, value in bulge_summary(bulge_result).items():
                rows.append(_row(subject_id, key, name, value, context))
        if thickness_bulge is not None:
            finite = thickness_bulge[np.isfinite(thickness_bulge)]
            if finite.size:
                rows.append(
                    _row(subject_id, key, "thickness_bulge_max_mm", float(finite.max()), context)
                )
                rows.append(
                    _row(
                        subject_id,
                        key,
                        "thickness_bulge_p99_mm",
                        float(np.percentile(finite, 99)),
                        context,
                    )
                )
                rows.append(
                    _row(
                        subject_id,
                        key,
                        "thickness_bulge_rms_mm",
                        float(np.sqrt(np.mean(finite**2))),
                        context,
                    )
                )

        # ---- subregion-level rows ---------------------------------------- #
        for subregion in parcels.subregions:
            member = parcels.mask(subregion)
            if not member.any():
                continue
            rows.extend(
                _subregion_rows(
                    subject_id,
                    subregion,
                    member,
                    thickness,
                    denuded_result,
                    curvature_result,
                    context,
                )
            )

        n_unassigned = int(np.sum(parcels.labels == UNASSIGNED))
        if n_unassigned:
            rows.append(
                _row(
                    subject_id,
                    key,
                    "parcellation_unassigned_fraction",
                    n_unassigned / max(len(parcels.labels), 1),
                    context,
                )
            )

        if keep_surfaces:
            surfaces[key] = {
                "bci": bci,
                "articular": articular,
                "thickness_mm": thickness,
                "denuded": denuded_result.denuded,
                "vertex_area_mm2": denuded_result.vertex_area_mm2,
                "parcellation": parcels.labels,
                "prior_gate": gate,
                "curvature": curvature_result,
                "bulge": bulge_result,
                "thickness_bulge_mm": thickness_bulge,
                "reports": {
                    "bci": surface["bci_report"],
                    "articular": surface["articular_report"],
                },
            }

    return KneeAnalysis(rows=pd.DataFrame(rows), surfaces=surfaces, warnings=warnings)


def _subregion_rows(
    subject_id: str,
    subregion: str,
    member: np.ndarray,
    thickness: np.ndarray,
    denuded_result: Any,
    curvature_result: Any,
    context: dict[str, Any],
) -> list[dict[str, Any]]:
    """Per-subregion metrics restricted to the member vertices."""
    areas = denuded_result.vertex_area_mm2
    denuded = denuded_result.denuded

    sub_thickness = thickness[member]
    sub_areas = areas[member]
    sub_denuded = denuded[member]

    finite = sub_thickness[np.isfinite(sub_thickness)]
    rows = [
        _row(subject_id, subregion, "n_vertices", float(member.sum()), context),
        _row(subject_id, subregion, "tAB_mm2", float(sub_areas.sum()), context),
        _row(subject_id, subregion, "dAB_mm2", float(sub_areas[sub_denuded].sum()), context),
        _row(
            subject_id,
            subregion,
            "dAB_percent",
            100.0 * float(sub_areas[sub_denuded].sum()) / max(float(sub_areas.sum()), 1e-9),
            context,
        ),
        _row(
            subject_id,
            subregion,
            "ThCtAB_mm",
            compute_thctab(sub_thickness, sub_denuded, sub_areas),
            context,
        ),
        _row(
            subject_id,
            subregion,
            "ThCcAB_mm",
            compute_thccab(sub_thickness, sub_denuded, sub_areas),
            context,
        ),
    ]
    if finite.size:
        rows.append(
            _row(subject_id, subregion, "thickness_mean_mm", float(finite.mean()), context)
        )
        rows.append(
            _row(subject_id, subregion, "thickness_median_mm", float(np.median(finite)), context)
        )

    if curvature_result is not None:
        for name, values in (
            ("curv_mean_curvature_median", curvature_result.mean_curvature),
            ("curv_gaussian_curvature_median", curvature_result.gaussian_curvature),
            ("curv_curvedness_median", curvature_result.curvedness),
            ("curv_shape_index_median", curvature_result.shape_index),
        ):
            sub = values[member]
            sub = sub[np.isfinite(sub)]
            if sub.size:
                rows.append(_row(subject_id, subregion, name, float(np.median(sub)), context))

    return rows

### Plot style and palettes

`confcarti/viz/style.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/viz/style.py
# ==========================================================================
"""Shared plotting style: palettes, helpers and figure saving.

Colour follows the job the data does, not taste:

* **categorical** (compartment, method, scheme) -- a fixed hue order, assigned by
  entity and never cycled, so a filtered chart does not repaint the survivors;
* **sequential** (thickness, curvedness -- magnitudes) -- one hue, light to dark;
* **diverging** (bulge, bias, coverage gap -- signed quantities) -- two hues
  around a neutral grey midpoint, so zero is visually neutral and the sign is
  readable at a glance.

The categorical hues are the validated default palette shipped with the
``dataviz`` skill, which passes the colour-vision-deficiency separation checks.
"""


from pathlib import Path
from typing import Any

import matplotlib as mpl
import numpy as np

__all__ = [
    "CATEGORICAL",
    "KL_COLORS",
    "SEQUENTIAL",
    "DIVERGING",
    "apply_style",
    "save_figure",
    "compartment_color",
    "annotate_n",
]

#: Fixed categorical hue order. Assign by entity, in this order, never cycled.
CATEGORICAL: tuple[str, ...] = (
    "#2a78d6",  # blue
    "#eb6834",  # orange
    "#1baf7a",  # aqua
    "#eda100",  # yellow
    "#e87ba4",  # magenta
    "#008300",  # green
    "#4a3aa7",  # violet
    "#8a8a8a",  # grey ("other")
)

#: KL grade is *ordinal severity*, so it gets a sequential ramp rather than
#: categorical hues -- KL 4 should look further from KL 0 than KL 1 does.
KL_COLORS: tuple[str, ...] = ("#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c")

#: Single-hue ramp for magnitudes (thickness, area, curvedness).
SEQUENTIAL = "viridis"

#: Two-hue ramp with a neutral midpoint, for signed quantities (bulge, bias).
DIVERGING = "RdBu_r"

_COMPARTMENT_ORDER = ("FC", "MTC", "LTC")


def apply_style() -> None:
    """Apply the shared matplotlib style: recessive grid, thin marks, no chartjunk."""
    mpl.rcParams.update(
        {
            "figure.dpi": 110,
            "savefig.dpi": 200,
            "savefig.bbox": "tight",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.titleweight": "bold",
            "axes.labelsize": 9,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": True,
            "axes.axisbelow": True,
            "grid.color": "#e6e6e6",
            "grid.linewidth": 0.6,
            "legend.frameon": False,
            "legend.fontsize": 8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "figure.facecolor": "white",
            "axes.prop_cycle": mpl.cycler(color=list(CATEGORICAL)),
        }
    )


def compartment_color(key: str) -> str:
    """Stable colour for a compartment, assigned by identity not by position."""
    if key in _COMPARTMENT_ORDER:
        return CATEGORICAL[_COMPARTMENT_ORDER.index(key)]
    return CATEGORICAL[-1]


def kl_color(grade: int) -> str:
    """Colour for a KL grade on the ordinal severity ramp."""
    return KL_COLORS[int(np.clip(grade, 0, 4))]


def annotate_n(ax: Any, counts: dict[str, int], *, y: float = 0.02) -> None:
    """Write the group size under each categorical tick.

    A coverage or thickness bar is uninterpretable without knowing how many
    subjects it rests on, so ``n`` is printed rather than left to a caption.
    """
    labels = [t.get_text() for t in ax.get_xticklabels()]
    for i, label in enumerate(labels):
        if label in counts:
            ax.text(
                i,
                y,
                f"n={counts[label]}",
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=7,
                color="#52514e",
            )


def save_figure(
    fig: Any,
    name: str,
    output_dir: str | Path = "results/figures",
    formats: tuple[str, ...] = ("png",),
) -> list[Path]:
    """Save a figure in every requested format.

    Parameters
    ----------
    fig
        Matplotlib figure.
    name
        Base filename without extension.
    output_dir
        Destination directory, created if absent.
    formats
        File formats.

    Returns
    -------
    list of pathlib.Path
        Written paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for fmt in formats:
        path = output_dir / f"{name}.{fmt}"
        fig.savefig(path, format=fmt)
        written.append(path)
    return written

### Figure library

`confcarti/viz/figures.py`

In [ ]:
from __future__ import annotations
# ==========================================================================
# confcarti/viz/figures.py
# ==========================================================================
"""Figure library: dataset analysis, 3D surface maps, conformal and comparison plots.

Every figure is built from the tidy results CSV (or a metadata table), never from
model outputs directly, so a figure can always be regenerated from the archived
results alone.
"""


import logging
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.figure import Figure


logger = logging.getLogger(__name__)

__all__ = [
    "plot_cohort_overview",
    "plot_metric_by_kl",
    "plot_surface_3d",
    "plot_surface_panel",
    "plot_calibration_curve",
    "plot_conditional_coverage",
    "plot_width_distribution",
    "plot_bland_altman",
    "plot_method_comparison",
    "plot_calibration_size_sweep",
    "plot_denuded_by_kl",
    "plot_training_curves",
    "surface_plotly",
]


def _pivot(results: pd.DataFrame, metric: str, subregion: str | None = None) -> pd.DataFrame:
    """Select one metric (optionally one subregion) from the tidy results."""
    frame = results[results["metric"] == metric]
    if subregion is not None:
        frame = frame[frame["subregion"] == subregion]
    return frame


# --------------------------------------------------------------------------- #
# Dataset analysis
# --------------------------------------------------------------------------- #


def plot_cohort_overview(metadata: pd.DataFrame) -> Figure:
    """Four-panel description of the cohort: KL, site, age and BMI.

    Parameters
    ----------
    metadata
        Canonical metadata table.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_style()
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))

    # KL distribution -- ordinal severity, so the sequential KL ramp.
    ax = axes[0, 0]
    counts = metadata["kl_grade"].value_counts().sort_index()
    ax.bar(
        counts.index.astype(int),
        counts.to_numpy(),
        color=[KL_COLORS[int(k)] for k in counts.index],
        width=0.7,
    )
    for x, y in zip(counts.index.astype(int), counts.to_numpy()):
        ax.text(x, y, f"{y}", ha="center", va="bottom", fontsize=8)
    ax.set_xlabel("Kellgren-Lawrence grade")
    ax.set_ylabel("subjects")
    ax.set_title("Radiographic severity")
    ax.set_xticks([0, 1, 2, 3, 4])

    # KL x site -- the joint stratification the splits must preserve.
    ax = axes[0, 1]
    table = pd.crosstab(metadata["kl_grade"], metadata["site"])
    bottom = np.zeros(len(table))
    for i, site in enumerate(table.columns):
        ax.bar(
            table.index.astype(int),
            table[site].to_numpy(),
            bottom=bottom,
            label=str(site),
            color=CATEGORICAL[i % len(CATEGORICAL)],
            width=0.7,
            edgecolor="white",
            linewidth=2,
        )
        bottom += table[site].to_numpy()
    ax.set_xlabel("KL grade")
    ax.set_ylabel("subjects")
    ax.set_title("KL x site (stratification cells)")
    ax.legend(title="site", ncol=2)
    ax.set_xticks([0, 1, 2, 3, 4])

    # Age by KL.
    ax = axes[1, 0]
    if "age" in metadata.columns:
        data = [
            metadata.loc[metadata["kl_grade"] == k, "age"].dropna().to_numpy()
            for k in sorted(metadata["kl_grade"].unique())
        ]
        parts = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False)
        for patch, grade in zip(parts["boxes"], sorted(metadata["kl_grade"].unique())):
            patch.set_facecolor(KL_COLORS[int(grade)])
            patch.set_edgecolor("#52514e")
        for element in ("medians", "whiskers", "caps"):
            for line in parts[element]:
                line.set_color("#52514e")
        ax.set_xticklabels([str(int(k)) for k in sorted(metadata["kl_grade"].unique())])
        ax.set_xlabel("KL grade")
        ax.set_ylabel("age (years)")
        ax.set_title("Age by severity")
    else:
        ax.set_visible(False)

    # BMI -- missingness matters here, so it is stated on the plot.
    ax = axes[1, 1]
    if "bmi" in metadata.columns:
        bmi = pd.to_numeric(metadata["bmi"], errors="coerce")
        ax.hist(bmi.dropna(), bins=20, color=CATEGORICAL[0], edgecolor="white")
        ax.set_xlabel("BMI (kg/m$^2$)")
        ax.set_ylabel("subjects")
        missing = int(bmi.isna().sum())
        ax.set_title(f"BMI  ({missing} missing of {len(bmi)})")
    else:
        ax.set_visible(False)

    fig.suptitle(
        f"Cohort overview -- {metadata['subject_id'].nunique()} subjects", fontweight="bold"
    )
    fig.tight_layout()
    return fig


def plot_metric_by_kl(
    results: pd.DataFrame,
    metric: str,
    *,
    subregions: tuple[str, ...] = ("FC", "MTC", "LTC"),
    ylabel: str | None = None,
    title: str | None = None,
) -> Figure:
    """Distribution of one metric across KL grades, one panel per compartment.

    Parameters
    ----------
    results
        Tidy results table.
    metric
        Metric name to plot.
    subregions
        Which subregions get a panel.
    ylabel, title
        Axis label and figure title.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_style()
    fig, axes = plt.subplots(1, len(subregions), figsize=(4 * len(subregions), 3.6), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, subregion in zip(axes, subregions):
        frame = _pivot(results, metric, subregion).dropna(subset=["value"])
        if frame.empty:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(subregion)
            continue

        grades = sorted(frame["kl_grade"].dropna().unique())
        data = [frame.loc[frame["kl_grade"] == g, "value"].to_numpy() for g in grades]
        parts = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False)
        for patch, grade in zip(parts["boxes"], grades):
            patch.set_facecolor(KL_COLORS[int(grade)])
            patch.set_edgecolor("#52514e")
        for element in ("medians", "whiskers", "caps"):
            for line in parts[element]:
                line.set_color("#52514e")

        # Individual points: with a few dozen knees the box hides the sample size.
        for i, values in enumerate(data, start=1):
            jitter = np.random.default_rng(0).normal(0, 0.05, len(values))
            ax.plot(i + jitter, values, "o", ms=2.5, color="#52514e", alpha=0.45, mew=0)

        ax.set_xticklabels([str(int(g)) for g in grades])
        ax.set_xlabel("KL grade")
        ax.set_title(subregion)

    axes[0].set_ylabel(ylabel or metric)
    fig.suptitle(title or f"{metric} by KL grade", fontweight="bold")
    fig.tight_layout()
    return fig


def plot_denuded_by_kl(results: pd.DataFrame) -> Figure:
    """dAB% and the ThCtAB / ThCcAB divergence across KL grades.

    The second panel is the point of the denuded branch: ThCcAB (covered bone
    only) flattens out in advanced disease while ThCtAB (total bone, denuded
    counted as 0 mm) keeps falling. Reporting only the former hides the loss.
    """
    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

    ax = axes[0]
    frame = _pivot(results, "dAB_percent")
    frame = frame[frame["subregion"].isin(["FC", "MTC", "LTC"])]
    for i, subregion in enumerate(["FC", "MTC", "LTC"]):
        sub = frame[frame["subregion"] == subregion]
        if sub.empty:
            continue
        grouped = sub.groupby("kl_grade")["value"].agg(["mean", "sem"])
        ax.errorbar(
            grouped.index.astype(int),
            grouped["mean"],
            yerr=grouped["sem"].fillna(0),
            marker="o",
            capsize=3,
            color=compartment_color(subregion),
            label=subregion,
        )
    ax.set_xlabel("KL grade")
    ax.set_ylabel("dAB (% of tAB)")
    ax.set_title("Full-thickness cartilage loss")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.legend()

    ax = axes[1]
    for metric, style, label in (
        ("ThCtAB_mm", "-", "ThCtAB (total bone, denuded = 0 mm)"),
        ("ThCcAB_mm", "--", "ThCcAB (covered bone only)"),
    ):
        sub = _pivot(results, metric, "FC")
        if sub.empty:
            continue
        grouped = sub.groupby("kl_grade")["value"].agg(["mean", "sem"])
        ax.errorbar(
            grouped.index.astype(int),
            grouped["mean"],
            yerr=grouped["sem"].fillna(0),
            marker="o",
            linestyle=style,
            capsize=3,
            color=CATEGORICAL[0] if metric == "ThCtAB_mm" else CATEGORICAL[1],
            label=label,
        )
    ax.set_xlabel("KL grade")
    ax.set_ylabel("mean thickness (mm)")
    ax.set_title("Femoral cartilage: why both are reported")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.legend()

    fig.tight_layout()
    return fig


# --------------------------------------------------------------------------- #
# 3D surfaces
# --------------------------------------------------------------------------- #


def plot_surface_3d(
    mesh: Any,
    field: np.ndarray,
    *,
    title: str = "",
    cmap: str = SEQUENTIAL,
    label: str = "",
    vmin: float | None = None,
    vmax: float | None = None,
    symmetric: bool = False,
    elev: float = 22.0,
    azim: float = -60.0,
    ax: Any = None,
) -> Figure:
    """Paint a per-vertex scalar field on a 3D surface.

    Parameters
    ----------
    mesh
        A trimesh surface.
    field
        Per-vertex scalar. NaN vertices are rendered in neutral grey, which is
        deliberate: "not measurable here" must look different from a low value.
    title, label
        Panel title and colourbar label.
    cmap
        Colormap. Use a sequential map for magnitudes and a diverging map for
        signed fields.
    vmin, vmax
        Colour limits; taken from robust percentiles when omitted.
    symmetric
        Force limits symmetric about zero -- required for signed fields so that
        the neutral midpoint really sits at zero.
    elev, azim
        View angles.
    ax
        Existing 3D axes to draw into.

    Returns
    -------
    matplotlib.figure.Figure
    """
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection

    apply_style()
    if ax is None:
        fig = plt.figure(figsize=(5.2, 4.4))
        ax = fig.add_subplot(111, projection="3d")
    else:
        fig = ax.get_figure()

    vertices = np.asarray(mesh.vertices, dtype=float)
    faces = np.asarray(mesh.faces)
    values = np.asarray(field, dtype=float)

    # Face value = mean of its vertices, ignoring NaN.
    face_values = np.nanmean(values[faces], axis=1)
    finite = face_values[np.isfinite(face_values)]

    if finite.size == 0:
        ax.text2D(0.5, 0.5, "no measurable values", transform=ax.transAxes, ha="center")
        ax.set_title(title)
        return fig

    if vmin is None or vmax is None:
        lo, hi = np.percentile(finite, [2, 98])
        if symmetric:
            span = max(abs(lo), abs(hi))
            lo, hi = -span, span
        vmin = lo if vmin is None else vmin
        vmax = hi if vmax is None else vmax
    if vmax - vmin < 1e-12:
        vmax = vmin + 1e-12

    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    colormap = plt.get_cmap(cmap)
    colors = colormap(norm(face_values))
    colors[~np.isfinite(face_values)] = (0.82, 0.82, 0.82, 1.0)

    collection = Poly3DCollection(
        vertices[faces], facecolors=colors, edgecolor="none", linewidths=0
    )
    ax.add_collection3d(collection)

    lower = vertices.min(axis=0)
    upper = vertices.max(axis=0)
    centre = 0.5 * (lower + upper)
    radius = 0.5 * float((upper - lower).max())
    for setter, c in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), centre):
        setter(c - radius, c + radius)

    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("x (mm)", labelpad=-6)
    ax.set_ylabel("y (mm)", labelpad=-6)
    ax.set_zlabel("z (mm)", labelpad=-6)
    ax.tick_params(labelsize=6, pad=-2)
    ax.set_title(title)
    ax.grid(False)

    mappable = plt.cm.ScalarMappable(norm=norm, cmap=colormap)
    mappable.set_array([])
    bar = fig.colorbar(mappable, ax=ax, shrink=0.62, pad=0.02)
    bar.set_label(label or "value", fontsize=8)
    bar.ax.tick_params(labelsize=7)
    return fig


def plot_surface_panel(surfaces: dict[str, Any], compartment: str = "FC") -> Figure:
    """Thickness, curvature and bulge painted on one compartment's surfaces.

    Parameters
    ----------
    surfaces
        The ``surfaces`` dict from :class:`confcarti.eval.pipeline.KneeAnalysis`.
    compartment
        Which compartment to render.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_style()
    data = surfaces[compartment]
    bci = data["bci"]

    fig = plt.figure(figsize=(15, 4.2))
    panels = []

    thickness = data["thickness_mm"].copy()
    panels.append((bci, thickness, "Thickness", SEQUENTIAL, "mm", False))

    denuded = data["denuded"].astype(float)
    panels.append((bci, denuded, "Denuded bone (dAB)", "Reds", "1 = denuded", False))

    if data.get("curvature") is not None:
        panels.append(
            (
                bci,
                data["curvature"].mean_curvature,
                "Mean curvature $H$",
                DIVERGING,
                "mm$^{-1}$",
                True,
            )
        )

    if data.get("thickness_bulge_mm") is not None:
        panels.append(
            (
                bci,
                data["thickness_bulge_mm"],
                "Thickness bulge residual",
                DIVERGING,
                "mm",
                True,
            )
        )

    for i, (mesh, field, title, cmap, label, symmetric) in enumerate(panels):
        ax = fig.add_subplot(1, len(panels), i + 1, projection="3d")
        plot_surface_3d(
            mesh, field, title=title, cmap=cmap, label=label, symmetric=symmetric, ax=ax
        )

    fig.suptitle(f"{compartment} bone-cartilage interface", fontweight="bold")
    fig.tight_layout()
    return fig


def surface_plotly(
    mesh: Any,
    field: np.ndarray,
    *,
    title: str = "",
    colorscale: str = "Viridis",
    label: str = "value",
) -> Any:
    """Interactive 3D surface via plotly, for notebook exploration.

    Parameters
    ----------
    mesh
        A trimesh surface.
    field
        Per-vertex scalar.
    title
        Figure title.
    colorscale
        Plotly colorscale name.
    label
        Colourbar label.

    Returns
    -------
    plotly.graph_objects.Figure

    Raises
    ------
    ImportError
        If plotly is not installed.
    """
    try:
        import plotly.graph_objects as go
    except ImportError as exc:  # pragma: no cover
        raise ImportError(
            "interactive 3D needs plotly: pip install 'confcarti[viz3d]'"
        ) from exc

    vertices = np.asarray(mesh.vertices, dtype=float)
    faces = np.asarray(mesh.faces)
    values = np.asarray(field, dtype=float)
    finite = values[np.isfinite(values)]
    lo, hi = (np.percentile(finite, [2, 98]) if finite.size else (0.0, 1.0))

    figure = go.Figure(
        data=[
            go.Mesh3d(
                x=vertices[:, 0],
                y=vertices[:, 1],
                z=vertices[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                intensity=np.nan_to_num(values, nan=float(lo)),
                colorscale=colorscale,
                cmin=float(lo),
                cmax=float(hi),
                colorbar={"title": label},
                showscale=True,
                flatshading=False,
            )
        ]
    )
    figure.update_layout(
        title=title,
        scene={
            "xaxis_title": "x (mm)",
            "yaxis_title": "y (mm)",
            "zaxis_title": "z (mm)",
            "aspectmode": "data",
        },
        margin={"l": 0, "r": 0, "t": 40, "b": 0},
        height=520,
    )
    return figure


# --------------------------------------------------------------------------- #
# Conformal
# --------------------------------------------------------------------------- #


def plot_calibration_curve(curves: dict[str, pd.DataFrame]) -> Figure:
    """Nominal versus empirical coverage, one line per scheme.

    Parameters
    ----------
    curves
        Scheme name -> DataFrame from
        :func:`confcarti.conformal.coverage.calibration_curve`.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    ax = axes[0]
    ax.plot([0, 1], [0, 1], color="#8a8a8a", lw=1.2, ls="--", label="nominal", zorder=1)
    for i, (name, frame) in enumerate(curves.items()):
        ax.plot(
            frame["nominal"],
            frame["empirical"],
            marker="o",
            ms=3.5,
            color=CATEGORICAL[i % len(CATEGORICAL)],
            label=name,
            zorder=2,
        )
    ax.set_xlabel("nominal coverage $1-\\alpha$")
    ax.set_ylabel("empirical coverage")
    ax.set_title("Calibration")
    ax.legend()

    ax = axes[1]
    ax.axhline(0.0, color="#8a8a8a", lw=1.2, ls="--")
    for i, (name, frame) in enumerate(curves.items()):
        ax.plot(
            frame["nominal"],
            frame["gap"],
            marker="o",
            ms=3.5,
            color=CATEGORICAL[i % len(CATEGORICAL)],
            label=name,
        )
    ax.set_xlabel("nominal coverage $1-\\alpha$")
    ax.set_ylabel("empirical $-$ nominal")
    ax.set_title("Coverage gap (above 0 is valid)")
    ax.legend()

    fig.tight_layout()
    return fig


def plot_conditional_coverage(
    tables: dict[str, pd.DataFrame], nominal: float = 0.9, *, group_label: str = "KL grade"
) -> Figure:
    """Per-group coverage with Wilson intervals and the nominal line.

    Parameters
    ----------
    tables
        Scheme name -> conditional-coverage table.
    nominal
        Target coverage, drawn as a reference line.
    group_label
        Axis label for the grouping variable.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_style()
    fig, ax = plt.subplots(figsize=(7.5, 4))

    names = list(tables)
    groups = sorted({g for t in tables.values() for g in t["group"]})
    width = 0.8 / max(len(names), 1)

    for i, name in enumerate(names):
        table = tables[name].set_index("group").reindex(groups)
        x = np.arange(len(groups)) + i * width - 0.4 + width / 2
        coverage = table["coverage"].to_numpy(dtype=float)
        lower = coverage - table["ci_low"].to_numpy(dtype=float)
        upper = table["ci_high"].to_numpy(dtype=float) - coverage
        ax.bar(
            x,
            coverage,
            width=width * 0.9,
            color=CATEGORICAL[i % len(CATEGORICAL)],
            label=name,
            edgecolor="white",
            linewidth=1.5,
        )
        ax.errorbar(
            x, coverage, yerr=[np.abs(lower), np.abs(upper)], fmt="none", ecolor="#52514e", capsize=2.5, lw=1
        )

    ax.axhline(nominal, color="#0b0b0b", ls="--", lw=1.4, zorder=5)
    ax.text(
        len(groups) - 0.5,
        nominal,
        f"  nominal {nominal:.2f}",
        va="bottom",
        ha="right",
        fontsize=8,
    )
    ax.set_xticks(np.arange(len(groups)))
    ax.set_xticklabels([str(g) for g in groups])
    ax.set_xlabel(group_label)
    ax.set_ylabel("empirical coverage")
    ax.set_ylim(0, 1.05)
    ax.set_title("Conditional coverage (error bars: 95% Wilson)")
    ax.legend(ncol=len(names))

    first = tables[names[0]].set_index("group").reindex(groups)
    annotate_n(ax, {str(g): int(n) for g, n in first["n"].items() if pd.notna(n)})
    fig.tight_layout()
    return fig


def plot_width_distribution(
    widths_by_group: dict[str, np.ndarray], *, xlabel: str = "interval width (mm)"
) -> Figure:
    """Interval-width distributions per group."""
    apply_style()
    fig, ax = plt.subplots(figsize=(7, 4))
    groups = list(widths_by_group)
    data = [np.asarray(widths_by_group[g], dtype=float) for g in groups]
    data = [d[np.isfinite(d)] for d in data]

    parts = ax.violinplot(data, showmedians=True, widths=0.8)
    for i, body in enumerate(parts["bodies"]):
        body.set_facecolor(KL_COLORS[i % len(KL_COLORS)])
        body.set_alpha(0.85)
        body.set_edgecolor("#52514e")
    for key in ("cmedians", "cbars", "cmins", "cmaxes"):
        if key in parts:
            parts[key].set_color("#52514e")

    ax.set_xticks(np.arange(1, len(groups) + 1))
    ax.set_xticklabels([str(g) for g in groups])
    ax.set_xlabel("KL grade")
    ax.set_ylabel(xlabel)
    ax.set_title("Interval width by severity")
    fig.tight_layout()
    return fig


def plot_calibration_size_sweep(sweep: pd.DataFrame) -> Figure:
    """Coverage stability as the calibration set shrinks (ablation A5)."""
    apply_style()
    fig, ax = plt.subplots(figsize=(6.5, 4))
    nominal = float(sweep["nominal"].iloc[0]) if not sweep.empty else 0.9

    ax.errorbar(
        sweep["calibration_size"],
        sweep["mean_coverage"],
        yerr=sweep["sd_coverage"],
        marker="o",
        capsize=3,
        color=CATEGORICAL[0],
        label="mean $\\pm$ SD over resamples",
    )
    ax.fill_between(
        sweep["calibration_size"],
        sweep["min_coverage"],
        sweep["max_coverage"],
        color=CATEGORICAL[0],
        alpha=0.15,
        label="min-max range",
    )
    ax.axhline(nominal, color="#0b0b0b", ls="--", lw=1.4, label=f"nominal {nominal:.2f}")
    ax.set_xlabel("calibration-set size $n$")
    ax.set_ylabel("empirical coverage")
    ax.set_title("Validity holds in expectation; stability is what $n$ buys")
    ax.set_xscale("log")
    ax.set_xticks(sweep["calibration_size"])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.legend()
    fig.tight_layout()
    return fig


# --------------------------------------------------------------------------- #
# Agreement and comparison
# --------------------------------------------------------------------------- #


def plot_bland_altman(result: Any, *, title: str = "", unit: str = "mm") -> Figure:
    """Bland-Altman plot with bias, limits of agreement and the trend line."""
    apply_style()
    fig, ax = plt.subplots(figsize=(6.2, 4.2))

    ax.plot(result.means, result.differences, "o", ms=4, color=CATEGORICAL[0], alpha=0.7, mew=0)
    ax.axhline(result.bias, color=CATEGORICAL[1], lw=1.8, label=f"bias {result.bias:+.3f} {unit}")
    for loa, name in ((result.loa_high, "+1.96 SD"), (result.loa_low, "-1.96 SD")):
        ax.axhline(loa, color="#8a8a8a", ls="--", lw=1.2)
        ax.text(ax.get_xlim()[1], loa, f" {name}\n {loa:+.3f}", va="center", fontsize=7)
    ax.axhline(0, color="#c8c8c8", lw=0.8, zorder=0)

    if abs(result.proportional_bias_slope) > 1e-9 and result.proportional_bias_p < 0.05:
        xs = np.linspace(result.means.min(), result.means.max(), 50)
        ax.plot(
            xs,
            result.bias + result.proportional_bias_slope * (xs - result.means.mean()),
            color=CATEGORICAL[2],
            ls=":",
            lw=1.6,
            label=f"proportional bias (p={result.proportional_bias_p:.3g})",
        )

    ax.set_xlabel(f"mean of the two methods ({unit})")
    ax.set_ylabel(f"difference ({unit})")
    ax.set_title(title or f"Bland-Altman (n={result.n})")
    ax.legend(loc="upper left")
    fig.tight_layout()
    return fig


def plot_method_comparison(
    results: pd.DataFrame, metric: str = "thickness_mean_mm", subregion: str = "FC"
) -> Figure:
    """Compare thickness methods (ablation A1) -- disagreement is a result.

    Parameters
    ----------
    results
        Tidy results carrying several ``method`` values.
    metric
        Metric to compare.
    subregion
        Subregion to restrict to.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_style()
    frame = _pivot(results, metric, subregion).dropna(subset=["value"])
    methods = sorted(frame["method"].dropna().unique())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    ax = axes[0]
    for i, method in enumerate(methods):
        sub = frame[frame["method"] == method]
        grouped = sub.groupby("kl_grade")["value"].agg(["mean", "sem"])
        ax.errorbar(
            grouped.index.astype(int),
            grouped["mean"],
            yerr=grouped["sem"].fillna(0),
            marker="o",
            capsize=3,
            color=CATEGORICAL[i % len(CATEGORICAL)],
            label=method,
        )
    ax.set_xlabel("KL grade")
    ax.set_ylabel(metric)
    ax.set_title(f"{subregion}: {metric} by method")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.legend()

    ax = axes[1]
    wide = frame.pivot_table(index="subject_id", columns="method", values="value")
    if len(methods) >= 2 and not wide.empty:
        reference = methods[0]
        for i, method in enumerate(methods[1:], start=1):
            pair = wide[[reference, method]].dropna()
            if pair.empty:
                continue
            ax.plot(
                pair[reference],
                pair[method] - pair[reference],
                "o",
                ms=4,
                color=CATEGORICAL[i % len(CATEGORICAL)],
                alpha=0.75,
                mew=0,
                label=f"{method} $-$ {reference}",
            )
        ax.axhline(0, color="#8a8a8a", lw=1.2, ls="--")
        ax.set_xlabel(f"{reference} ({metric})")
        ax.set_ylabel("difference from reference")
        ax.set_title("Between-method disagreement (a result, not a bug)")
        ax.legend()
    fig.tight_layout()
    return fig


def plot_training_curves(history: Any) -> Figure:
    """Training loss and internal-validation Dice.

    Two measures on different scales, so two panels rather than a dual axis.
    """
    apply_style()
    frame = history.to_frame() if hasattr(history, "to_frame") else pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

    axes[0].plot(frame["epoch"], frame["train_loss"], color=CATEGORICAL[0])
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("Dice + CE loss")
    axes[0].set_title("Training loss")

    axes[1].plot(frame["epoch"], frame["val_dice"], color=CATEGORICAL[2])
    if hasattr(history, "best_epoch") and history.best_epoch >= 0:
        axes[1].axvline(history.best_epoch, color="#8a8a8a", ls="--", lw=1.2)
        axes[1].text(
            history.best_epoch,
            axes[1].get_ylim()[0],
            f" best epoch {history.best_epoch}",
            fontsize=7,
            va="bottom",
        )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("mean foreground Dice")
    axes[1].set_title("Internal validation (carved from TRAIN)")

    fig.tight_layout()
    return fig

## 2. Data

Two sources, one interface. If the OAI-ZIB subject table can be fetched, it is used for
the cohort description (it is public metadata published with CartiMorph: subject id,
KL grade, image release, knee side, age, sex, BMI). Images and masks are **not**
redistributed, so the morphometry runs on analytic phantoms whose ground truth is known
in closed form — which is also what makes §4's validation possible.

Set `USE_REAL_DATA = True` and point `OAIZIB_ROOT` at your own OAI-ZIB export to run the
identical pipeline on real volumes.

In [ ]:
# --- configuration for this run ------------------------------------------------
USE_REAL_DATA = True                  # True if you hold OAI-ZIB / OAIZIB-CM locally
OAIZIB_ROOT   = r"D:\ishtiaque\Knee segementation\OAI-ZIB\OAIZIB-CM\OAIZIB-CM"
N_SUBJECTS    = None                  # None = every case found; set an int for a smoke test
RESAMPLE_TO_MM = 0.5                  # isotropic resample before meshing; None = acquisition grid
N_PHANTOM     = 32                    # synthetic cohort size when USE_REAL_DATA is False
N_RELIABILITY = 8                     # subjects re-measured for precision/agreement
REPEATS       = 3                     # repeat measurements per subject
SEED          = 20240617
ALPHA         = 0.10                  # target miscoverage -> 90% intervals

# --- curvature ------------------------------------------------------------------
# CURVATURE_RADIUS_MM is the spatial scale the curvature is measured at. The
# original run also carried an implicit `2 * radius` = 6.0 mm boundary margin
# that discarded 76-87% of every cartilage plate; that margin is gone, replaced
# by a per-vertex support test (see confcarti/thickness/curvature.py). What
# remains here is the scale itself, plus the floor the adaptive fallback may
# shrink to near a rim.
CURVATURE_RADIUS_MM      = 3.0
CURVATURE_MIN_RADIUS_MM  = 1.5        # ~3 marching-cubes edge lengths at 0.5 mm
CURVATURE_ADAPTIVE       = True       # False = strict single-scale field

# --- segmentation ----------------------------------------------------------------
# 64 x 64 x 48 at 0.5 mm is a 32 x 32 x 24 mm box: it clipped ~50% of the
# cartilage plate on every case in the first run ("joint-centred crop ... clipped
# N cartilage voxel(s) (52.00% of the plate)"). A knee needs ~80 x 80 x 48 mm.
CROP_SIZE     = (160, 160, 96) if USE_REAL_DATA else (64, 64, 48)
MAX_EPOCHS    = 500 if USE_REAL_DATA else 6

seed_report = set_all_seeds(SEED)
print(seed_report)
print(f"cohort: {'OAIZIB-CM at ' + str(OAIZIB_ROOT) if USE_REAL_DATA else str(N_PHANTOM) + ' phantoms'}")


In [ ]:
# --- cohort: real OAIZIB-CM when USE_REAL_DATA, phantoms otherwise -------------
#
# THE BUG THIS REPLACES
# ---------------------
# The original cell declared USE_REAL_DATA and OAIZIB_ROOT and then ignored both:
# it fetched the real *metadata* over the network and immediately overwrote the
# cohort with `make_phantom_cohort(N_PHANTOM, seed=SEED)`. Every number in the
# notebook -- thickness, curvature, bulge, Dice, coverage -- came from 32
# analytic phantoms, while the run manifest recorded "used_real_data": true
# because it read the flag rather than what the flag did. That is the single
# most consequential defect in the run: not a crash, a silent substitution.
#
# Everything downstream is untouched. `knees` still yields objects with
# .image / .label / .spacing and still lines up row-for-row with `metadata`, so
# sections 3-10 need no changes beyond the curvature parameters.
import io
import urllib.request
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path


def fetch_oaizib_metadata(timeout=30):
    """Fetch the public OAI-ZIB subject tables published with CartiMorph."""
    base = "https://raw.githubusercontent.com/YongchengYAO/CartiMorph/main/Dataset/OAIZIB"
    tables = {}
    for name in ("CartiMorph_dataset2", "CartiMorph_dataset3"):   # train + test = 481 subjects
        with urllib.request.urlopen(f"{base}/{name}.xlsx", timeout=timeout) as fh:
            tables[name] = pd.read_excel(io.BytesIO(fh.read()))
    return from_cartimorph_tables(tables)


real_metadata = None
try:
    real_metadata = fetch_oaizib_metadata()
    print(f"fetched real OAI-ZIB metadata: {len(real_metadata)} subjects")
    print("KL distribution:", real_metadata.kl_grade.value_counts().sort_index().to_dict())
    print("sites (OAI image release):", real_metadata.site.value_counts().to_dict())
except Exception as exc:
    print(f"could not fetch OAI-ZIB metadata ({type(exc).__name__}: {exc}).")
    print("Continuing with the phantom cohort — every later section still runs.")


# --------------------------------------------------------------------------- #
# A lazy cohort. 500 knees at 0.5 mm isotropic is ~15 GB of image+label if held
# in RAM; volumes are therefore read on access and one is kept.
# --------------------------------------------------------------------------- #
@dataclass(slots=True)
class KneeCase:
    """One knee, with the attribute names the rest of the notebook expects."""

    subject_id: str
    image: np.ndarray
    label: np.ndarray
    spacing: tuple


class OAIZIBCohort(Sequence):
    """Lazily-loaded OAIZIB-CM cohort that quacks like the phantom cohort.

    Parameters
    ----------
    cases
        ``CaseRecord``s from :func:`discover_cases`.
    resample_mm
        Resample every volume to this isotropic spacing before analysis, or
        None to analyse on the acquisition grid. OAI DESS is acquired at
        roughly 0.36 x 0.36 x 0.7 mm; marching cubes on a grid that anisotropic
        produces triangles twice as long in z as in x, and the resulting
        curvature carries the grid's anisotropy rather than the anatomy's. This
        is the same isotropic resampling CartiMorph applies before meshing.
    """

    def __init__(self, cases, resample_mm: float | None = 0.5) -> None:
        self._cases = list(cases)
        self._resample_mm = resample_mm
        self._cached_index: int | None = None
        self._cached: KneeCase | None = None

    def __len__(self) -> int:
        return len(self._cases)

    @property
    def subject_ids(self) -> list[str]:
        return [c.subject_id for c in self._cases]

    def __getitem__(self, index):
        if isinstance(index, slice):
            return [self[i] for i in range(*index.indices(len(self)))]
        index = int(index)
        if index < 0:
            index += len(self)
        if self._cached_index == index and self._cached is not None:
            return self._cached

        record = self._cases[index]
        image, spacing = load_volume(record.image_path)
        label, label_spacing = load_volume(record.label_path)
        if not np.allclose(label_spacing, spacing, atol=1e-3):
            raise ValueError(
                f"{record.subject_id}: image spacing {spacing} != label spacing "
                f"{label_spacing}. Resampling them independently would misalign the "
                "cartilage plate from the bone it sits on."
            )

        if self._resample_mm is not None and not np.allclose(
            spacing, self._resample_mm, atol=1e-3
        ):
            target = (self._resample_mm,) * 3
            image = resample_volume(image, spacing, target, is_label=False)
            label = resample_volume(label, spacing, target, is_label=True)
            spacing = target

        case = KneeCase(
            subject_id=record.subject_id,
            image=np.asarray(image, dtype=np.float32),
            label=np.asarray(label, dtype=np.int16),
            spacing=tuple(float(s) for s in spacing),
        )
        self._cached_index, self._cached = index, case
        return case


def _find_oaizib_layout(root):
    """Locate the image/label directories, whatever the release called them.

    OAIZIB-CM ships nnU-Net style, but different mirrors split it differently
    (imagesTr/labelsTr only, or plus imagesTs/labelsTs). Rather than fail on a
    naming mismatch, look for every pair that exists and say what was found.
    """
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(
            f"OAIZIB_ROOT does not exist: {root}\n"
            "Set it to the directory that contains imagesTr/ and labelsTr/."
        )
    pairs = [
        (i, l)
        for i, l in (("imagesTr", "labelsTr"), ("imagesTs", "labelsTs"),
                     ("images", "labels"))
        if (root / i).is_dir() and (root / l).is_dir()
    ]
    if not pairs:
        found = sorted(p.name for p in root.iterdir() if p.is_dir())
        raise FileNotFoundError(
            f"no image/label directory pair under {root}. Sub-directories present: "
            f"{found}. Expected imagesTr/ + labelsTr/ (nnU-Net layout)."
        )
    return pairs


USE_REAL_DATA_EFFECTIVE = False
knees = metadata = None

if USE_REAL_DATA:
    if real_metadata is None:
        raise RuntimeError(
            "USE_REAL_DATA=True but the OAI-ZIB metadata could not be fetched. Every "
            "subject needs a KL grade and a site or it cannot be stratified, split or "
            "used for conditional coverage. Download CartiMorph_dataset2.xlsx and "
            "CartiMorph_dataset3.xlsx by hand and load them with "
            "from_cartimorph_tables({'d2': pd.read_excel(...), 'd3': pd.read_excel(...)})."
        )

    records = []
    for images_dir, labels_dir in _find_oaizib_layout(OAIZIB_ROOT):
        found = discover_cases(
            OAIZIB_ROOT, images_dir=images_dir, labels_dir=labels_dir,
            require_labels=True,
        )
        print(f"  {images_dir}/ + {labels_dir}/: {len(found)} case(s)")
        records.extend(found)

    # Join on subject id. The image set and the published subject tables do not
    # have to agree exactly (OAI-ZIB is 507 scans, the CartiMorph tables cover
    # 481), and a case without a KL grade cannot enter any split -- so the
    # intersection is the cohort, and the losses are reported rather than
    # dropped in silence.
    have_metadata = set(real_metadata.subject_id.astype(str))
    kept = [r for r in records if r.subject_id in have_metadata]
    dropped = sorted({r.subject_id for r in records} - have_metadata)
    if dropped:
        print(f"  {len(dropped)} case(s) on disk have no metadata row and are excluded "
              f"(e.g. {dropped[:5]})")
    missing_images = sorted(have_metadata - {r.subject_id for r in kept})
    if missing_images:
        print(f"  {len(missing_images)} metadata subject(s) have no image on disk "
              f"(e.g. {missing_images[:5]})")
    if not kept:
        raise RuntimeError(
            "no case on disk matched a metadata subject id. discover_cases parses the "
            "7-digit OAI id out of the filename; check that your files look like "
            "'9001104_V00.nii.gz' or 'sub-9001104_....nii.gz'."
        )

    if N_SUBJECTS is not None:
        kept = kept[:N_SUBJECTS]
        print(f"  restricted to the first {len(kept)} case(s) (N_SUBJECTS)")

    knees = OAIZIBCohort(kept, resample_mm=RESAMPLE_TO_MM)
    # metadata must be row-for-row aligned with `knees`: every later cell pairs
    # them with zip(knees, metadata.iterrows()).
    metadata = (
        real_metadata.set_index(real_metadata.subject_id.astype(str))
        .loc[[c.subject_id for c in kept]]
        .reset_index(drop=True)
    )
    USE_REAL_DATA_EFFECTIVE = True

    probe = knees[0]
    print(f"\nOAIZIB-CM cohort: {len(knees)} knees, volume {probe.label.shape} @ "
          f"{probe.spacing[0]:.3f} mm"
          + ("" if RESAMPLE_TO_MM is None else f" (resampled to {RESAMPLE_TO_MM} mm isotropic)"))
    print("labels present in the first case:", np.unique(probe.label).tolist())
else:
    knees, metadata = make_phantom_cohort(N_PHANTOM, seed=SEED)
    print(f"\nphantom cohort: {len(knees)} knees, volume {knees[0].label.shape} @ "
          f"{knees[0].spacing[0]} mm")
    print("NOTE: these are analytic phantoms. Nothing below is a statement about "
          "real knees until USE_REAL_DATA is True and OAIZIB_ROOT points at data.")

report = validate_metadata(metadata)
print("\nmetadata valid:", report.ok, "| KL:", report.kl_counts, "| sites:", report.site_counts)
for w in report.warnings[:3]:
    print("  warning:", w)


In [ ]:
# --- subject-level splits, stratified jointly on KL grade and site -------------
splits = make_splits(metadata, fractions=(0.6, 0.2, 0.2), seed=SEED,
                     stratify_on=("kl_grade", "site"))
split_report = stratification_report(metadata, splits)

print("split sizes:", split_report.sizes)
print("realised fractions:", {k: round(v, 3) for k, v in split_report.fractions.items()})
print(f"max |realised - population| proportion deviation: {split_report.max_abs_deviation:.4f}")
print("\nKL x split:\n", split_report.kl_by_split)

assert_disjoint(splits)   # raises on any subject appearing in two splits
SUBJECT_SPLIT = {s: name for name, ids in splits.items() for s in ids}
print("\nno subject leakage between splits ✓")

## 3. Dataset analysis and visualisation

The cohort description drives every stratified result later on. Note the KL imbalance —
it is the reason marginal averages are not trusted anywhere in this notebook.

In [ ]:
apply_style()
describe = real_metadata if real_metadata is not None else metadata
fig = plot_cohort_overview(describe)
plt.show()

summary = summarise_metadata(describe)
print("KL x site contingency:\n", summary["kl_by_site"])
print("\nper-KL continuous variables:\n", summary.get("continuous"))

In [ ]:
# --- a look at the raw volumes -------------------------------------------------
from matplotlib.colors import ListedColormap, BoundaryNorm

label_cmap = ListedColormap(["#00000000", "#c9c9c9", "#2a78d6", "#a0a0a0", "#eb6834", "#1baf7a"])
norm = BoundaryNorm(np.arange(-0.5, 6.5), label_cmap.N)

examples = [next(i for i, (_, m) in enumerate(metadata.iterrows()) if m.kl_grade == g)
            for g in sorted(metadata.kl_grade.unique())]

fig, axes = plt.subplots(2, len(examples), figsize=(3.0 * len(examples), 6))
for col, idx in enumerate(examples):
    knee = knees[idx]
    sl = knee.image.shape[0] // 3
    axes[0, col].imshow(knee.image[sl].T, cmap="gray", origin="lower")
    axes[0, col].set_title(f"KL {metadata.iloc[idx].kl_grade}  ({metadata.iloc[idx].subject_id})")
    axes[1, col].imshow(knee.image[sl].T, cmap="gray", origin="lower")
    axes[1, col].imshow(knee.label[sl].T, cmap=label_cmap, norm=norm, origin="lower", alpha=0.65)
    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
axes[0, 0].set_ylabel("image", fontsize=9)
axes[1, 0].set_ylabel("5-ROI mask", fontsize=9)
fig.suptitle("Sagittal slice by KL grade — blue = femoral cartilage, orange/green = tibial plates",
             fontweight="bold")
fig.tight_layout(); plt.show()

N_CONVENTION_CHECK = min(len(metadata), 20)   # reading 481 volumes to check a
                                             # label convention is not worth it
print(f"label convention check on {N_CONVENTION_CHECK} case(s):")
bad = [metadata.subject_id.iloc[j] for j in range(N_CONVENTION_CHECK)
       if not validate_label_convention(knees[j].label).ok]
print("  subjects failing the convention check:", bad or "none ✓")

In [ ]:
# pip install rtree

## 4. Geometry validation against closed-form answers

Before measuring anything anatomical, each geometric routine is checked on a shape whose
answer is known exactly. These are the acceptance criteria; if any fails, nothing
downstream is trustworthy.

In [ ]:
import trimesh

checks = []

# --- thickness: two concentric spheres of known gap, and a flat slab -----------
gap = 2.0
inner = trimesh.creation.icosphere(subdivisions=4, radius=10.0)
outer = trimesh.creation.icosphere(subdivisions=4, radius=10.0 + gap)
radial = np.asarray(inner.vertices) / np.linalg.norm(np.asarray(inner.vertices), axis=1, keepdims=True)
t_sphere, _ = thickness_surface_normal(inner, outer, normals=radial, max_depth_mm=10)
checks.append(("thickness, concentric spheres (gap 2.00 mm)", float(np.nanmedian(t_sphere)), gap, 0.02))

low = trimesh.creation.box(extents=(20, 20, 0.01)); high = low.copy(); high.apply_translation([0, 0, 2.0])
flat_n = np.tile([0.0, 0.0, 1.0], (len(low.vertices), 1))
t_slab, _ = thickness_surface_normal(low, high, normals=flat_n, max_depth_mm=10)
checks.append(("thickness, flat slab (2.00 mm)", float(np.nanmedian(t_slab)), 2.0, 0.02))

# --- curvature: sphere H=1/R, K=1/R^2 ; plane H=K=0 ----------------------------
for R in (5.0, 10.0):
    res = compute_curvature(trimesh.creation.icosphere(subdivisions=4, radius=R),
                            radius_mm=None, smoothing_iterations=0, mask_boundary=False)
    checks.append((f"curvature H, sphere R={R:g}", float(np.nanmedian(res.mean_curvature)), 1/R, 0.05))
    checks.append((f"curvature K, sphere R={R:g}", float(np.nanmedian(res.gaussian_curvature)), 1/R**2, 0.10))

box = trimesh.creation.box(extents=(20, 20, 1))
plane = trimesh.Trimesh(vertices=box.vertices, faces=box.faces[box.triangles_center[:, 2] > 0.4],
                        process=True).subdivide().subdivide().subdivide()
pl = compute_curvature(plane, radius_mm=None, smoothing_iterations=0, mask_boundary=False)
checks.append(("curvature H, plane (0)", float(np.nanmedian(pl.mean_curvature)), 0.0, None))


# --- curvature on an OPEN patch, which is what a cartilage plate is -----------
# The checks above are all on closed or unbounded surfaces -- a sphere, a plane --
# and that is exactly why the boundary bug survived them. A cartilage plate is an
# open sheet ~20-25 mm across; every point of it is close to a rim. This builds
# a strip cut from a cylinder of tibial-plate size, where H = 1/(2R) and K = 0
# exactly, and checks both the accuracy and how much of the patch survives.
def _cylinder_strip(radius_mm=22.0, width_mm=22.0, arc_mm=24.0, edge_mm=0.5):
    n_ax = int(round(width_mm / edge_mm)) + 1
    n_ci = int(round(arc_mm / edge_mm)) + 1
    axial = np.linspace(-width_mm / 2, width_mm / 2, n_ax)
    theta = np.linspace(-arc_mm / (2 * radius_mm), arc_mm / (2 * radius_mm), n_ci)
    tt, aa = np.meshgrid(theta, axial, indexing="ij")
    verts = np.stack([radius_mm * np.cos(tt), radius_mm * np.sin(tt), aa], -1).reshape(-1, 3)
    f = []
    for i_ in range(n_ci - 1):
        for j_ in range(n_ax - 1):
            a_ = i_ * n_ax + j_
            f += [[a_, a_ + n_ax, a_ + 1], [a_ + 1, a_ + n_ax, a_ + n_ax + 1]]
    m = trimesh.Trimesh(vertices=verts, faces=np.asarray(f), process=False)
    if np.dot(m.vertex_normals[0], m.vertices[0] * [1, 1, 0]) < 0:
        m.invert()
    return m

_R = 22.0
_strip = _cylinder_strip(radius_mm=_R)
_new = compute_curvature(_strip, radius_mm=CURVATURE_RADIUS_MM,
                         min_radius_mm=CURVATURE_MIN_RADIUS_MM,
                         adaptive_radius=CURVATURE_ADAPTIVE, smoothing_iterations=3)
_old = compute_curvature(_strip, radius_mm=CURVATURE_RADIUS_MM, smoothing_iterations=3,
                         mask_boundary=True, boundary_margin_mm=2 * CURVATURE_RADIUS_MM,
                         boundary_metric="euclidean")
checks.append(("curvature H, open strip R=22 (plate-sized)",
               float(np.nanmedian(_new.mean_curvature)), 1 / (2 * _R), 0.10))
checks.append(("curvature K, open strip (0)",
               float(np.nanmedian(_new.gaussian_curvature)), 0.0, None))
checks.append(("open strip: fraction of plate estimable",
               _new.estimable_fraction, 1.0, 0.20))

print(f"open-patch coverage: {100 * _new.estimable_fraction:5.1f}% with the support test, "
      f"{100 * _old.estimable_fraction:5.1f}% with the old {2 * CURVATURE_RADIUS_MM:.0f} mm "
      f"Euclidean margin")
print("The second number is the defect: on a plate-sized open patch the old rule keeps "
      "an island\nin the middle and calls it the plate's curvature.\n")

# --- Taubin smoothing preserves volume ----------------------------------------
sph = trimesh.creation.icosphere(subdivisions=3, radius=10.0); noisy = sph.copy()
noisy.vertices = np.asarray(noisy.vertices) + np.random.default_rng(0).normal(0, 0.15, (len(sph.vertices), 3))
checks.append(("Taubin volume drift", float(abs(taubin_smooth(noisy, iterations=20).volume - sph.volume) / sph.volume), 0.0, None))

print(f"{'check':46s} {'measured':>12s} {'expected':>10s}   result")
print("-" * 88)
ok = True
for name, measured, expected, rel_tol in checks:
    if rel_tol is None:
        passed = abs(measured - expected) < 0.05
    else:
        passed = abs(measured - expected) <= rel_tol * abs(expected)
    ok &= passed
    print(f"{name:46s} {measured:12.5f} {expected:10.5f}   {'PASS' if passed else 'FAIL'}")
print("-" * 88)
print("all geometry checks passed ✓" if ok else "SOME CHECKS FAILED — do not trust downstream results")

In [ ]:
# --- bulge: a 1.5 mm Gaussian bump on a 20 mm sphere, and a clean control ------
sphere = trimesh.creation.icosphere(subdivisions=4, radius=20.0)
v = np.asarray(sphere.vertices); u = v / np.linalg.norm(v, axis=1, keepdims=True)
w = np.exp(-((u - np.array([0, 0, 1.0])) ** 2).sum(1) / (2 * 0.05))
bumped = sphere.copy(); bumped.vertices = v + u * (1.5 * w)[:, None]

rows = []
for R in (8.0, 12.0, 15.0, 20.0):
    s = bulge_summary(compute_bulge(bumped, form_radius_mm=R))
    rows.append({"form_radius_mm": R, "recovered_height_mm": round(s["bulge_max_height_mm"], 3),
                 "recovered_fraction": round(s["bulge_max_height_mm"] / 1.5, 3),
                 "bulge_area_mm2": round(s["bulge_area_mm2"], 1)})
control = bulge_summary(compute_bulge(sphere, form_radius_mm=12.0))

print("Bulge detection on a known 1.5 mm bump (~4.5 mm wide):")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nclean-sphere control at 12 mm: RMS {control['bulge_rms_mm']:.4f} mm, "
      f"bulge area {control['bulge_area_mm2']:.2f} mm²")
print("""
Read this honestly: a local form fit always absorbs part of the feature it isolates, so
amplitudes are UNDER-estimated and the bulge field is a relative index — good for ranking
subregions and tracking change, not an absolute height. There is also a false-positive
floor on strongly curved surfaces that grows with form_radius / radius_of_curvature.""")

## 5. Per-knee morphometry

For every knee: extract the bone–cartilage interface (**including** denuded bone — the
surface-closing step is what makes `dAB` measurable at all) and the articular surface,
then measure thickness, curvature, bulge, the 20-region parcellation and the
denuded-bone metrics.

In [ ]:
import time

all_rows, kept_surfaces, all_warnings = [], {}, []
keep_ids = set(metadata.subject_id.iloc[:4])          # keep a few for 3D rendering

start = time.time()
for i, (_, meta_row) in enumerate(metadata.iterrows()):
    knee = knees[i]
    analysis = analyse_knee(
        knee.label, knee.spacing,
        subject_id=meta_row.subject_id,
        kl_grade=int(meta_row.kl_grade),
        site=meta_row.site,
        split=SUBJECT_SPLIT.get(meta_row.subject_id),
        laterality=meta_row.laterality,
        model_name="manual",
        seed=SEED,
        thickness_method="surface_normal",
        curvature_radius_mm=CURVATURE_RADIUS_MM,
        curvature_min_radius_mm=CURVATURE_MIN_RADIUS_MM,
        curvature_adaptive=CURVATURE_ADAPTIVE,
        keep_surfaces=meta_row.subject_id in keep_ids,
    )
    all_rows.append(analysis.rows)
    all_warnings.extend(analysis.warnings)
    if analysis.surfaces:
        kept_surfaces[meta_row.subject_id] = analysis.surfaces
    # A real cohort is 400+ knees at tens of seconds each. Silence for two hours
    # is indistinguishable from a hang, so say where we are.
    if (i + 1) % 10 == 0 or i + 1 == len(metadata):
        rate = (time.time() - start) / (i + 1)
        print(f"  {i + 1}/{len(metadata)} knees  ({rate:.1f}s each, "
              f"~{rate * (len(metadata) - i - 1) / 60:.0f} min left)", flush=True)

results = pd.concat(all_rows, ignore_index=True)
print(f"analysed {len(metadata)} knees in {time.time()-start:.0f}s")
print(f"tidy results: {len(results)} rows, {results.metric.nunique()} distinct metrics, "
      f"{results.subregion.nunique()} subregions")
print("warnings:", len(all_warnings))
for w in all_warnings[:3]:
    print("  ", w)

# How much of each plate the curvature actually describes. Without this the
# curvature columns are uninterpretable: a compartment mean says nothing about
# whether it covers the plate or an island in the middle of it.
est = results[results.metric == "curv_estimable_fraction"]
if len(est):
    print("\ncurvature estimable fraction by compartment:")
    print(est.pivot_table(index="subregion", values="value",
                          aggfunc=["mean", "min"]).round(3).to_string())

results.to_csv("confcarti_results.csv", index=False)
print("\nwrote confcarti_results.csv — every table and figure below is built from this file alone")
results.head(8)

In [ ]:
# --- 3D surface maps ------------------------------------------------------------
subject = sorted(kept_surfaces)[0]
fig = plot_surface_panel(kept_surfaces[subject], "FC")
fig.suptitle(f"{subject} — femoral BCI  (grey = not measurable: curvature is undefined "
             f"within a fit radius of an open edge)", fontweight="bold")
plt.show()

for comp in ("MTC", "LTC"):
    if comp in kept_surfaces[subject]:
        fig = plot_surface_panel(kept_surfaces[subject], comp)
        plt.show()

In [ ]:
# --- interactive 3D (plotly) ---------------------------------------------------
if HAS_PLOTLY:
    data = kept_surfaces[subject]["FC"]
    fig3d = surface_plotly(data["bci"], data["thickness_mm"],
                           title=f"{subject} — femoral cartilage thickness (mm)",
                           colorscale="Viridis", label="mm")
    fig3d.show()
else:
    print("plotly unavailable — static 3D panels above cover the same fields")

In [ ]:
# --- morphometry by severity ---------------------------------------------------
for metric, ylabel in [("ThCtAB_mm", "ThCtAB (mm)"),
                       ("curv_curvedness_median", "curvedness (mm⁻¹)"),
                       ("bulge_rms_mm", "bulge RMS (mm)")]:
    if (results.metric == metric).any():
        fig = plot_metric_by_kl(results, metric, ylabel=ylabel)
        plt.show()

fig = plot_denuded_by_kl(results)
plt.show()

pivot = (results[results.metric.isin(["ThCtAB_mm", "ThCcAB_mm", "dAB_percent",
                                      "curv_curvedness_median", "bulge_rms_mm"])]
         .pivot_table(index="kl_grade", columns="metric", values="value", aggfunc="mean")
         .round(4))
print("Compartment-level means by KL grade:\n")
print(pivot.to_string())

## 6. Segmentation model

`FrozenSwinUNETR` — a MONAI SwinUNETR v2 with the encoder frozen and only the decoder and
head trained. The internal validation subset used for early stopping is carved out of the
**training** split; `CalibrationGuard` turns any read of the calibration split into an
exception rather than a silent invalidation of the coverage guarantee.

This section is a short demonstration run on phantoms. Skip it (or leave `HAS_TORCH`
False) and everything after still works — the conformal layer operates on measurements,
not on the network.

In [ ]:
RUN_TRAINING = HAS_TORCH and True     # set False to skip
# MAX_EPOCHS and CROP_SIZE come from the configuration cell: 6 epochs on a
# 32 x 32 x 24 mm crop is a smoke test, not a model.

model = history = None
if RUN_TRAINING:
    import torch
    dataset = PhantomDataset(knees, metadata, crop_size=CROP_SIZE)
    train_idx = [i for i, s in enumerate(metadata.subject_id) if SUBJECT_SPLIT.get(s) == "train"]
    print(f"training on {len(train_idx)} subjects | device: "
          f"{'cuda' if torch.cuda.is_available() else 'cpu'}")

    model, history = train_segmentation(
        dataset, train_idx,
        out_channels=6, backbone="segresnet",     # cheap CNN for the demo; use 'swinunetr' for real runs
        patch_size=CROP_SIZE, batch_size=1,
        max_epochs=MAX_EPOCHS, steps_per_epoch=4,
        internal_val_fraction=0.25, early_stopping_patience=MAX_EPOCHS,
        amp_dtype="fp32", freeze_encoder_flag=False, seed=SEED,
    )
    print(f"\nbest internal-validation Dice {history.best_val_dice:.4f} at epoch {history.best_epoch}")
    fig = plot_training_curves(history); plt.show()
else:
    print("training skipped (torch/MONAI unavailable or RUN_TRAINING=False)")

In [ ]:
# --- the calibration guard, demonstrated ---------------------------------------
try:
    with CalibrationGuard(allow=["train"], label="demo_training_loop"):
        check_subject_access(splits["calibration"][:3], splits)
    print("NO GUARD FIRED — this would be a silent invalidation of the guarantee")
except CalibrationLeakageError as exc:
    print("CalibrationGuard correctly refused the read:\n")
    print(" ", exc)

In [ ]:
# --- inference + segmentation metrics ------------------------------------------
seg_rows = []
if RUN_TRAINING and model is not None:
    test_idx = [i for i, s in enumerate(metadata.subject_id) if SUBJECT_SPLIT.get(s) == "test"]
    for i in test_idx[:6]:
        item = dataset[i]
        pred, prob = sliding_window_predict(model, item["image"], roi_size=CROP_SIZE,
                                            sw_batch_size=1, overlap=0.25, amp_dtype="fp32")
        seg_rows.append(evaluate_segmentation(
            pred, np.asarray(item["label"])[0], item["spacing"],
            subject_id=item["subject_id"], kl_grade=item["kl_grade"], site=item["site"],
            model_name="segresnet", compute_distances=False))
    if seg_rows:
        seg = pd.concat(seg_rows, ignore_index=True)
        print("Dice by ROI (demo model, 6 epochs — not a performance claim):\n")
        print(seg[seg.metric == "dice"].pivot_table(index="subregion", values="value",
                                                    aggfunc=["mean", "std", "count"]).round(3))
        print("\nKL-stratified Dice:\n")
        print(seg[seg.metric == "dice"].pivot_table(index="kl_grade", columns="subregion",
                                                    values="value").round(3))
else:
    print("inference skipped")
print("""
For reference, published OAIZIB-CM nnU-Net Dice is roughly 0.86–0.90 femoral and
0.83–0.87 tibial cartilage. A 6-epoch phantom demo will not reach that and is not
intended to; point OAIZIB_ROOT at real data and train properly to compare.""")

## 7. The conformal layer

The target is per-subregion mean thickness. A predictor is fitted on the **training**
split only; the **calibration** split supplies conformity scores; the **test** split is
where coverage is measured. Nothing but the conformal layer ever reads calibration.

Four questions:

1. Does marginal coverage hit its nominal level?
2. Does it hold **conditionally on KL grade** — or does the average hide a failure?
3. How wide are the intervals, and do they adapt to difficulty?
4. What happens under distribution shift, and does reweighting help?

In [ ]:
# --- build the prediction task from the tidy results ---------------------------
TARGET_METRIC = "ThCtAB_mm"
COMPARTMENTS_USED = ["FC", "MTC", "LTC"]

table = (results[(results.metric == TARGET_METRIC) & results.subregion.isin(COMPARTMENTS_USED)]
         .dropna(subset=["value"])
         .pivot_table(index=["subject_id", "kl_grade", "site", "split"],
                      columns="subregion", values="value")
         .reset_index())

# Features available WITHOUT the target: geometry of the same knee.
feature_metrics = ["curv_curvedness_median", "curv_mean_curvature_median",
                   "bulge_rms_mm", "tAB_mm2", "articular_area_mm2"]
features = (results[results.metric.isin(feature_metrics) & results.subregion.isin(COMPARTMENTS_USED)]
            .pivot_table(index="subject_id", columns=["subregion", "metric"], values="value"))
features.columns = [f"{a}_{b}" for a, b in features.columns]
task = table.merge(features.reset_index(), on="subject_id", how="left")
task = task.dropna(subset=COMPARTMENTS_USED)

long = task.melt(id_vars=["subject_id", "kl_grade", "site", "split"] + list(features.columns),
                 value_vars=COMPARTMENTS_USED, var_name="subregion", value_name="target")
long = long.dropna(subset=["target"]).reset_index(drop=True)
print(f"prediction task: {len(long)} (subject, subregion) rows")
print(long.split.value_counts().to_dict())

In [ ]:
# --- fit the point predictor and a heteroscedastic scale, on TRAIN only --------
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

FEATURES = list(features.columns) + ["subregion", "site"]
numeric = list(features.columns)

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["subregion", "site"]),
])

def fit_on_train(target_col, **kw):
    with CalibrationGuard(allow=["train"], label="fit_predictor"):
        check_split_access("train")
    tr = long[long.split == "train"]
    model_ = make_pipeline(pre, GradientBoostingRegressor(random_state=SEED, **kw))
    model_.fit(tr[FEATURES], tr[target_col])
    return model_

point = fit_on_train("target")
long["prediction"] = point.predict(long[FEATURES])

# sigma(x): a second model on |residual|, again TRAIN only.
tr = long[long.split == "train"].copy()
tr["abs_res"] = np.abs(tr.target - tr.prediction)
scale_model = make_pipeline(pre, GradientBoostingRegressor(random_state=SEED))
scale_model.fit(tr[FEATURES], tr["abs_res"])
long["sigma"] = np.maximum(scale_model.predict(long[FEATURES]), 1e-3)

# quantile models for CQR, TRAIN only.
q_lo_m = fit_on_train("target", loss="quantile", alpha=ALPHA / 2)
q_hi_m = fit_on_train("target", loss="quantile", alpha=1 - ALPHA / 2)
long["q_lo"] = q_lo_m.predict(long[FEATURES])
long["q_hi"] = q_hi_m.predict(long[FEATURES])

cal = long[long.split == "calibration"].reset_index(drop=True)
tst = long[long.split == "test"].reset_index(drop=True)
print(f"calibration n={len(cal)}  test n={len(tst)}")

# The conformal quantile is the ceil((n+1)(1-alpha))-th order statistic, so it simply
# does not exist unless n >= ceil(1/alpha) - 1. Rather than crash on a small cohort,
# relax alpha to the tightest level this calibration set can actually support -- and
# say so, because a silently different alpha would make every later number wrong.
needed = min_calibration_size(ALPHA)
print(f"minimum calibration size for alpha={ALPHA}: {needed}")
if len(cal) < needed:
    relaxed = float(np.ceil(100 / (len(cal) + 1)) / 100)
    print(f"\n  !! only {len(cal)} calibration points: the alpha={ALPHA} quantile does not exist.")
    print(f"  !! relaxing to alpha={relaxed} (nominal coverage {1-relaxed:.0%}) for this run.")
    print(f"  !! increase N_PHANTOM (or use real data) to hold alpha={ALPHA}.")
    ALPHA = relaxed
print(f"train MAE {np.abs(tr.target - tr.prediction).mean():.4f} mm | "
      f"test MAE {np.abs(tst.target - tst.prediction).mean():.4f} mm")

In [ ]:
# --- the four schemes ----------------------------------------------------------
groups_cal = cal.kl_grade.astype(int).astype(str).to_numpy()
groups_tst = tst.kl_grade.astype(int).astype(str).to_numpy()

intervals = {}
intervals["split (absolute)"] = split_conformal(
    cal.target.to_numpy(), cal.prediction.to_numpy(), tst.prediction.to_numpy(), alpha=ALPHA)
intervals["normalized"] = normalized_conformal(
    cal.target.to_numpy(), cal.prediction.to_numpy(), cal.sigma.to_numpy(),
    tst.prediction.to_numpy(), tst.sigma.to_numpy(), alpha=ALPHA)
intervals["CQR"] = cqr_conformal(
    cal.target.to_numpy(), cal.q_lo.to_numpy(), cal.q_hi.to_numpy(),
    tst.q_lo.to_numpy(), tst.q_hi.to_numpy(), alpha=ALPHA)
intervals["Mondrian (KL)"] = mondrian_conformal(
    cal.target.to_numpy(), cal.prediction.to_numpy(), groups_cal,
    tst.prediction.to_numpy(), groups_tst, alpha=ALPHA,
    min_group_size=min_calibration_size(ALPHA), merge_small_groups=True)

truth = tst.target.to_numpy()
rows = []
for name, iv in intervals.items():
    rep = coverage_report(iv, truth, groups_tst)
    rows.append({"scheme": name, "coverage": round(rep.marginal, 4),
                 "nominal": rep.nominal,
                 "worst KL group": round(rep.worst_group_coverage, 4),
                 "gap": round(rep.coverage_gap, 4),
                 "mean width (mm)": round(rep.mean_width, 4),
                 "width~|err| corr": round(rep.width_error_correlation, 3)})
summary_conformal = pd.DataFrame(rows)
print(summary_conformal.to_string(index=False))
print("""
'worst KL group' is the number a clinical claim actually rests on. A scheme with good
marginal coverage and a poor worst-group value is covering the easy knees and missing
the diseased ones.""")

In [ ]:
# --- conditional coverage and width -------------------------------------------
tables = {name: conditional_coverage(iv, truth, groups_tst) for name, iv in intervals.items()}
fig = plot_conditional_coverage(tables, nominal=1 - ALPHA, group_label="KL grade")
plt.show()

best = "Mondrian (KL)"
widths = {g: intervals[best].width[groups_tst == g] for g in sorted(set(groups_tst))}
widths = {g: w for g, w in widths.items() if np.isfinite(w).sum() >= 2}
if widths:
    fig = plot_width_distribution(widths); plt.show()
else:
    print("too few test points per KL grade to draw a width distribution")

print(f"per-group detail — {best}:\n")
print(tables[best].round(4).to_string(index=False))

In [ ]:
# --- calibration curves: nominal vs empirical across alpha ---------------------
# The sweep cannot go below the tightest alpha this calibration set supports:
# the quantile is an order statistic, so alpha < 1/(n+1) simply has no estimator.
# Sweeping into that region raises rather than silently returning something wrong.
alpha_floor = 1.0 / (len(cal) + 1)
alpha_grid = np.linspace(max(0.01, alpha_floor * 1.05), 0.5, 20)
print(f"sweeping alpha over [{alpha_grid[0]:.3f}, {alpha_grid[-1]:.2f}] "
      f"(floor {alpha_floor:.3f} set by n_cal={len(cal)})")

curves = {}
curves["split (absolute)"] = calibration_curve(
    lambda a: split_conformal(cal.target.to_numpy(), cal.prediction.to_numpy(),
                              tst.prediction.to_numpy(), alpha=a), truth, alphas=alpha_grid)
curves["normalized"] = calibration_curve(
    lambda a: normalized_conformal(cal.target.to_numpy(), cal.prediction.to_numpy(),
                                   cal.sigma.to_numpy(), tst.prediction.to_numpy(),
                                   tst.sigma.to_numpy(), alpha=a), truth, alphas=alpha_grid)
curves["Mondrian (KL)"] = calibration_curve(
    lambda a: mondrian_conformal(cal.target.to_numpy(), cal.prediction.to_numpy(), groups_cal,
                                 tst.prediction.to_numpy(), groups_tst, alpha=a,
                                 min_group_size=2, merge_small_groups=True),
    truth, alphas=alpha_grid)
fig = plot_calibration_curve(curves); plt.show()

In [ ]:
# --- A5: how much calibration data do we need? --------------------------------
sizes = tuple(s for s in (10, 25, 50, 100, 200) if s <= len(cal))
sweep = calibration_size_sweep(cal.target.to_numpy(), cal.prediction.to_numpy(),
                               truth, tst.prediction.to_numpy(),
                               sizes=sizes, alpha=ALPHA, n_repeats=200, seed=SEED)
print(sweep.round(4).to_string(index=False))
if not sweep.empty:
    fig = plot_calibration_size_sweep(sweep); plt.show()
print("""
Validity holds in expectation at every admissible n — that is the theorem. What grows
with n is STABILITY: the scatter of any single calibration draw. That scatter, not
validity, is what determines how large a calibration split a study needs.""")

In [ ]:
# --- conformal risk control on the segmentation masks -------------------------
# Voxel subsampling: a real knee volume is ~2.5 M voxels, and holding one float64
# probability map per calibration subject is gigabytes. The risk is an expectation
# over voxels, so a fixed random subsample estimates it without bias; the seed
# keeps it reproducible.
RISK_VOXELS = 200_000
rng = np.random.default_rng(SEED)
prob_maps, truth_masks = [], []
for i, (_, m) in enumerate(metadata.iterrows()):
    if SUBJECT_SPLIT.get(m.subject_id) not in ("calibration", "test"):
        continue
    knee = knees[i]
    truth_mask = np.isin(knee.label, CARTILAGE_LABELS).reshape(-1)
    if truth_mask.size > RISK_VOXELS:
        pick = rng.choice(truth_mask.size, RISK_VOXELS, replace=False)
        truth_mask = truth_mask[pick]
    # Stand-in for a network's softmax when §6 was skipped: a noisy but informative score.
    prob = np.clip(np.where(truth_mask, rng.beta(6, 2, truth_mask.shape),
                                        rng.beta(2, 6, truth_mask.shape)), 0, 1)
    prob_maps.append(prob); truth_masks.append(truth_mask)

n_cal = sum(1 for _, m in metadata.iterrows() if SUBJECT_SPLIT.get(m.subject_id) == "calibration")
crc = calibrate_segmentation_threshold(prob_maps[:n_cal], truth_masks[:n_cal],
                                       alpha=ALPHA, n_lambdas=200)
lambdas = np.linspace(0, 1, 200)
idx = int(np.argmin(np.abs(lambdas - crc.lambda_hat)))
held_out = [false_negative_rate_curve(p, t, lambdas)[idx]
            for p, t in zip(prob_maps[n_cal:], truth_masks[n_cal:])]

print(f"lambda_hat = {crc.lambda_hat:.4f}  ->  keep voxels with p >= {1 - crc.lambda_hat:.4f}")
print(f"calibration risk {crc.risk_at_lambda_hat:.4f} | feasible: {crc.feasible}")
print(f"held-out expected false-negative rate: {np.mean(held_out):.4f}  (target alpha = {ALPHA})")
if not crc.feasible:
    print(f"""
  !! NO lambda met the target on the calibration set, so lambda_hat fell back to the
  !! most permissive value ({crc.lambda_hat:.3f}). The held-out loss below target is
  !! then trivial -- an all-inclusive prediction set misses nothing -- and the risk
  !! guarantee at alpha={ALPHA} does NOT hold. Lower alpha's ambition or improve the
  !! probability maps.""")
elif np.mean(held_out) <= ALPHA:
    print("guarantee satisfied on held-out data ✓")
else:
    print("guarantee VIOLATED on held-out data -- investigate exchangeability")

fig, ax = plt.subplots(figsize=(6.4, 4))
ax.plot(crc.lambdas, crc.empirical_risk, label="empirical risk $\\hat{R}_n(\\lambda)$", color=CATEGORICAL[0])
ax.plot(crc.lambdas, crc.upper_bound, label="$\\frac{n}{n+1}\\hat{R}_n + \\frac{B}{n+1}$", color=CATEGORICAL[1])
ax.axhline(ALPHA, ls="--", color="#0b0b0b", lw=1.3, label=f"$\\alpha$ = {ALPHA}")
ax.axvline(crc.lambda_hat, ls=":", color="#52514e", lw=1.5, label=f"$\\hat{{\\lambda}}$ = {crc.lambda_hat:.3f}")
ax.set_xlabel("$\\lambda$"); ax.set_ylabel("false-negative rate")
ax.set_title("Conformal risk control"); ax.legend()
fig.tight_layout(); plt.show()

In [ ]:
# --- A4: covariate shift, and whether reweighting recovers coverage -----------
# Hold out one site entirely: calibrate on the other, test on the unseen one.
site_values = sorted(long.site.unique())
if len(site_values) >= 2:
    held, other = site_values[0], site_values[1]
    cal_s = long[(long.split.isin(["calibration", "train"])) & (long.site == other)]
    tst_s = long[long.site == held]

    plain = split_conformal(cal_s.target.to_numpy(), cal_s.prediction.to_numpy(),
                            tst_s.prediction.to_numpy(), alpha=ALPHA)
    w_cal, w_tst = estimate_likelihood_ratio(
        cal_s[numeric].fillna(cal_s[numeric].median()).to_numpy(),
        tst_s[numeric].fillna(cal_s[numeric].median()).to_numpy(), seed=SEED)
    weighted = weighted_conformal(cal_s.target.to_numpy(), cal_s.prediction.to_numpy(),
                                  tst_s.prediction.to_numpy(), w_cal, w_tst, alpha=ALPHA)

    y_s = tst_s.target.to_numpy()
    print(f"held-out site: {held}   (calibrated on {other})")
    print(f"  unweighted coverage : {plain.coverage(y_s):.4f}")
    print(f"  weighted coverage   : {weighted.coverage(y_s):.4f}")
    print(f"  effective calibration size: {weighted.extras['effective_sample_size']:.0f} of {len(cal_s)}")
    print("""
  Weighted conformal assumes COVARIATE shift: P(Y|X) unchanged, only P(X) moved. If the
  image-to-thickness relationship itself differs at the new site — different sequence,
  different rater convention — no reweighting of X can repair it.""")
else:
    print("only one site present; the shift experiment needs at least two")

## 8. Reliability, agreement and detectability

**Not** "MAE against ground truth". The reference thickness is derived from a manual mask
by the *same* meshing and ray-casting code, so it shares its preprocessing and
discretisation error with the measurement — the difference between them understates the
true error and has no interpretation as accuracy. What can be said honestly is how
reproducible the measurement is, how well two measurements agree, and whether the
measurement can detect the change it needs to.

In [ ]:
# --- test-retest precision: re-measure with perturbed segmentations ------------
# The pipeline is run ONCE per (subject, repeat) and cached; precision, agreement
# and detectability are all derived from that one table. Recomputing per statistic
# would triple the runtime for identical numbers.
from scipy import ndimage

def jitter_label(label, rng, z=0.8, smooth_vox=1.5):
    """Simulate rater/segmentation variability by perturbing mask boundaries.

    The noise field is renormalised to unit variance AFTER smoothing. Without
    that step this function is a silent no-op: Gaussian-filtering white noise
    with sigma=1.5 in 3D cuts its standard deviation by more than an order of
    magnitude, so a fixed threshold of ~0.6 is never crossed, every "repeat"
    returns the original mask, and the test-retest SEM comes out as exactly
    0.0000 -- which in turn makes SDD zero and every subregion look trivially
    'detectable'. A reliability analysis that reports perfect reproducibility
    is the most dangerous kind of wrong.
    """
    out = label.copy()
    for value in CARTILAGE_LABELS:
        mask = label == value
        if not mask.any():
            continue
        noise = ndimage.gaussian_filter(rng.normal(0, 1, mask.shape), smooth_vox)
        spread = noise.std()
        if spread < 1e-12:
            continue
        noise = noise / spread                      # unit variance -> z is a real z-score
        grown = ndimage.binary_dilation(mask) & (noise > z)
        shrunk = mask & ~(noise < -z)
        out[mask] = 0
        out[shrunk | grown] = value
    return out


# Verify the perturbation actually perturbs, rather than trusting that it does.
_probe = knees[0].label
_jittered = jitter_label(_probe, np.random.default_rng(0))
_changed = int((_probe != _jittered).sum())
print(f"segmentation jitter changes {_changed} voxel(s) per repeat "
      f"({100 * _changed / max((_probe > 0).sum(), 1):.2f}% of the foreground)")
assert _changed > 0, "jitter_label is a no-op; test-retest precision would be meaningless"

repeat_subjects = list(metadata.subject_id[:N_RELIABILITY])
repeat_rows = []
rng = np.random.default_rng(SEED)

for j, (_, m) in enumerate(metadata.iterrows()):
    if m.subject_id not in repeat_subjects:
        continue
    knee = knees[j]
    for r in range(REPEATS):
        lab = knee.label if r == 0 else jitter_label(knee.label, rng)
        rows_ = analyse_knee(lab, knee.spacing, subject_id=m.subject_id,
                             kl_grade=int(m.kl_grade), site=m.site,
                             compute_geometry=False, keep_surfaces=False).rows
        rows_["repeat"] = r
        repeat_rows.append(rows_)

repeats_table = pd.concat(repeat_rows, ignore_index=True)
print(f"cached {len(repeat_subjects)} subjects x {REPEATS} repeats "
      f"= {len(repeat_subjects) * REPEATS} pipeline runs")

def repeat_matrix(compartment, metric="ThCtAB_mm"):
    """(n_subjects, n_repeats) matrix of one metric, from the cached table."""
    wide = (repeats_table[(repeats_table.subregion == compartment)
                          & (repeats_table.metric == metric)]
            .pivot_table(index="subject_id", columns="repeat", values="value"))
    mat = wide.to_numpy(dtype=float)
    return mat[np.isfinite(mat).all(axis=1)]

floor = error_floor_from_testretest(repeat_matrix("FC"))
print("\nMeasurement-error floor for FC ThCtAB "
      "(every accuracy claim is relative to this):")
for k, v in floor.items():
    print(f"  {k:16s} {v:.4f}" if isinstance(v, float) else f"  {k:16s} {v}")

# --- agreement against a repeated measurement (not "accuracy") ----------------
# Repeat 0 (the original mask) vs repeat 1 (a perturbed segmentation): this is an
# AGREEMENT comparison between two measurements of the same knee, which is the
# honest question. "MAE vs ground truth" is not, because the reference is derived
# from the same meshing and ray-casting code and is not independent of it.
pair = (repeats_table[(repeats_table.subregion == "FC")
                      & (repeats_table.metric == "ThCtAB_mm")
                      & repeats_table.repeat.isin([0, 1])]
        .pivot_table(index="subject_id", columns="repeat", values="value")
        .dropna())

a, b = pair[0].to_numpy(), pair[1].to_numpy()
ba = bland_altman(a, b)
icc_res = icc(a, b, "ICC(2,1)")

print(f"Bland-Altman  bias {ba.bias:+.4f} mm  (95% CI {ba.bias_ci[0]:+.4f} to {ba.bias_ci[1]:+.4f})")
print(f"              limits of agreement {ba.loa_low:+.4f} to {ba.loa_high:+.4f} mm")
print(f"              proportional bias slope {ba.proportional_bias_slope:+.4f} "
      f"(p={ba.proportional_bias_p:.3g})")
print(f"ICC(2,1) {icc_res['icc']:.4f}  "
      f"(95% CI {icc_res['ci_low']:.4f}-{icc_res['ci_high']:.4f}, n={icc_res['n']})")
print("""
ICC(2,1) is used rather than ICC(3,1) because it charges any systematic difference
between the two measurements against the agreement, which is what we want when
comparing a pipeline against a reference rather than checking mere consistency.""")

fig = plot_bland_altman(ba, title="FC ThCtAB: original vs perturbed segmentation")
plt.show()

# --- RQ4: which subregions can detect their own annual change? ----------------
# Literature annual thickness change (Eckstein et al.); replace with your own
# longitudinal data when you have it.
ANNUAL_CHANGE_MM = {"FC": -0.035, "MTC": -0.045, "LTC": -0.020}

sdd_by_region = {}
for comp in COMPARTMENTS_USED:
    mat = repeat_matrix(comp)
    if len(mat):
        sdd_by_region[comp] = sdd(sem(mat))

detect = detectability_table(sdd_by_region, ANNUAL_CHANGE_MM)
print("Can one year of change be detected in an INDIVIDUAL knee?\n")
print(detect.round(4).to_string(index=False))
print("""
'detectable = False' does not make a subregion useless: group-level change is still
detectable with enough subjects, because the standard error of a mean shrinks as
1/sqrt(n). It means change in an individual knee over one year cannot be
distinguished from measurement noise there.""")

## 9. Comparative and ablation analysis

Seven ablations, each a single flag. The disagreement between thickness methods (A1) is a
**result**, not something to tune away — the three estimators measure genuinely different
things and their spread bounds how much of any reported effect is method choice.

In [ ]:
# --- A1: thickness method ------------------------------------------------------
ABLATION_N = min(12, len(knees))   # lower to shorten the sweep
method_rows = []
for method in ("surface_normal", "distance_transform", "nearest_neighbour"):
    for j in range(ABLATION_N):
        m = metadata.iloc[j]
        knee = knees[j]
        r = analyse_knee(knee.label, knee.spacing, subject_id=m.subject_id,
                         kl_grade=int(m.kl_grade), site=m.site,
                         split=SUBJECT_SPLIT.get(m.subject_id),
                         thickness_method=method, compute_geometry=False,
                         keep_surfaces=False).rows
        method_rows.append(r)
ablation_a1 = pd.concat(method_rows, ignore_index=True)

fig = plot_method_comparison(ablation_a1, metric="ThCtAB_mm", subregion="FC")
plt.show()

wide = (ablation_a1[(ablation_a1.metric == "ThCtAB_mm") & (ablation_a1.subregion == "FC")]
        .pivot_table(index="subject_id", columns="method", values="value"))
print("A1 — FC ThCtAB by method (mm):\n")
print(wide.describe().round(4).to_string())
print("\npairwise mean difference (mm):")
for i, a_ in enumerate(wide.columns):
    for b_ in wide.columns[i+1:]:
        d = (wide[b_] - wide[a_]).dropna()
        print(f"  {b_:20s} - {a_:20s}: {d.mean():+.4f}  (SD {d.std():.4f}, "
              f"max |diff| {d.abs().max():.4f})")

In [ ]:
# --- A2 (denuded branch), A6 (prior strength), A8 (geometry features) ---------
variant_rows = []
VARIANTS = [
    ("A2: denuded on",   dict(denuded_enabled=True)),
    ("A2: denuded off",  dict(denuded_enabled=False)),
    ("A6: prior 0.0",    dict(prior_strength=0.0)),
    ("A6: prior 1.0",    dict(prior_strength=1.0)),
    ("A6: prior gate off", dict(prior_gate_enabled=False)),
    ("A8: geometry off", dict(compute_geometry=False)),
]
for name, kwargs in VARIANTS:
    for j in range(ABLATION_N):
        m = metadata.iloc[j]
        knee = knees[j]
        r = analyse_knee(knee.label, knee.spacing, subject_id=m.subject_id,
                         kl_grade=int(m.kl_grade), keep_surfaces=False, **kwargs).rows
        r["variant"] = name
        variant_rows.append(r)
ablations = pd.concat(variant_rows, ignore_index=True)

table = (ablations[(ablations.subregion == "FC") &
                   ablations.metric.isin(["ThCtAB_mm", "ThCcAB_mm", "dAB_percent", "prior_gate_mean"])]
         .pivot_table(index="variant", columns="metric", values="value", aggfunc="mean").round(4))
print("Ablations A2 / A6 / A8 — FC compartment means:\n")
print(table.to_string())
print("""
Read the A2 rows together: with the denuded branch off, ThCtAB and ThCcAB coincide, because
'denuded' is what makes them differ. Any thinning that has progressed to full-thickness loss
then disappears from the reported thickness entirely.""")

In [ ]:
# --- A3 (score) and A4 (scheme), scored on coverage ---------------------------
def summarise(name, iv):
    rep = coverage_report(iv, truth, groups_tst)
    return {"variant": name, "coverage": round(rep.marginal, 4),
            "worst KL group": round(rep.worst_group_coverage, 4),
            "mean width (mm)": round(rep.mean_width, 4),
            "median width (mm)": round(rep.median_width, 4),
            "adaptivity (width~|err|)": round(rep.width_error_correlation, 3)}

grid = [summarise(f"A3 {k}", v) for k, v in intervals.items()]
grid.append(summarise("A4 marginal", intervals["split (absolute)"]))
grid.append(summarise("A4 mondrian", intervals["Mondrian (KL)"]))
ablation_conformal = pd.DataFrame(grid).drop_duplicates("variant")
print("Ablations A3 (score) and A4 (scheme):\n")
print(ablation_conformal.to_string(index=False))

## 10. Results summary

Everything below is regenerated from `confcarti_results.csv` plus the conformal tables —
no analysis reads model outputs directly, so re-running this section reproduces every
number from the archived results alone.

In [ ]:
# --- publication-ready tables --------------------------------------------------
def with_ci(series, confidence=0.95):
    values = np.asarray(series.dropna(), dtype=float)
    if values.size < 2:
        return "—"
    from scipy import stats as st
    mean = values.mean()
    half = st.t.ppf(0.5 + confidence / 2, values.size - 1) * values.std(ddof=1) / np.sqrt(values.size)
    return f"{mean:.3f} ({mean-half:.3f}–{mean+half:.3f})"

print("=" * 84)
print("TABLE 1 — Morphometry by KL grade, mean (95% CI)".center(84))
print("=" * 84)
metrics_t1 = ["ThCtAB_mm", "ThCcAB_mm", "dAB_percent", "curv_curvedness_median", "bulge_rms_mm"]
t1 = []
for grade in sorted(results.kl_grade.dropna().unique()):
    row = {"KL": int(grade),
           "n": results[(results.kl_grade == grade) & (results.subregion == "FC")].subject_id.nunique()}
    for metric in metrics_t1:
        sel = results[(results.kl_grade == grade) & (results.subregion == "FC")
                      & (results.metric == metric)]
        row[metric] = with_ci(sel.value)
    t1.append(row)
table1 = pd.DataFrame(t1)
print(table1.to_string(index=False))

print("\n" + "=" * 84)
print("TABLE 2 — Conformal coverage at nominal {:.0%}".format(1 - ALPHA).center(84))
print("=" * 84)
print(summary_conformal.to_string(index=False))

print("\n" + "=" * 84)
print("TABLE 3 — Conditional coverage by KL grade (Mondrian)".center(84))
print("=" * 84)
print(tables["Mondrian (KL)"].round(4).to_string(index=False))

table1.to_csv("table1_morphometry_by_kl.csv", index=False)
summary_conformal.to_csv("table2_coverage.csv", index=False)
tables["Mondrian (KL)"].to_csv("table3_conditional_coverage.csv", index=False)
detect.to_csv("table4_detectability.csv", index=False)
print("\nwrote table1–table4 CSVs")

In [ ]:
# --- LaTeX for the paper -------------------------------------------------------
print(table1.to_latex(index=False, escape=False,
                      caption="Cartilage morphometry by Kellgren-Lawrence grade, "
                              "femoral compartment. Mean (95\\% CI).",
                      label="tab:morphometry"))

In [ ]:
# --- run manifest: what produced these numbers --------------------------------
import json, platform, sys as _sys
manifest = {
    "seed": SEED,
    "alpha": ALPHA,
    "n_subjects": int(metadata.subject_id.nunique()),
    "splits": {k: len(v) for k, v in splits.items()},
    # The *effective* value. The original reported the flag, which is how a
    # phantom run came to be recorded as a real-data run.
    "used_real_data": bool(USE_REAL_DATA_EFFECTIVE),
    "curvature_radius_mm": CURVATURE_RADIUS_MM,
    "curvature_min_radius_mm": CURVATURE_MIN_RADIUS_MM,
    "curvature_adaptive": bool(CURVATURE_ADAPTIVE),
    "fetched_real_metadata": real_metadata is not None,
    "n_result_rows": int(len(results)),
    "metrics": sorted(results.metric.unique().tolist()),
    "python": _sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch_available": HAS_TORCH,
}
with open("confcarti_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
print(json.dumps(manifest, indent=2)[:900])
print("\nwrote confcarti_manifest.json")

## What this notebook does and does not establish

**Validated against closed-form answers** (§4, and `tests/test_confcarti.py`):
surface-normal thickness to within 2% on concentric spheres and a slab; curvature to
within 5% of `1/R` on spheres and 0 on a plane; Taubin smoothing volume-preserving;
split-conformal coverage matching the exact finite-sample expectation
`⌈(n+1)(1−α)⌉/(n+1)` over hundreds of replications, *including with a deliberately
miscalibrated predictor*; Mondrian repairing group-conditional under-coverage; conformal
risk control holding `E[L] ≤ α` on held-out data; RMS CV%, SEM, SDD, SRM, ICC and
Bland–Altman against closed forms.

**Known limits, stated rather than hidden:**

* **Curvature is undefined near an open boundary.** The bone–cartilage interface is an
  open patch, and a quadric fit within one fit-radius of its edge is one-sided and badly
  biased — on a phantom femoral interface, masking only the boundary vertices leaves the
  median ~100% too high. Curvature is returned as `NaN` inside a two-radius margin, which
  on a small phantom leaves only ~20% of vertices estimable. Real OAI-ZIB volumes are
  much larger and the estimable fraction is correspondingly higher.
* **Bulge amplitudes are under-estimated** (~37–54% of a known 1.5 mm bump, depending on
  form radius) and have a false-positive floor on strongly curved surfaces. Treat the
  bulge field as a relative index; do not compare condyles to plateaus directly.
* **dAB is deliberately conservative.** The lateral tolerance that prevents a false
  denuded rim around every plate also erodes each real defect by that tolerance: defects
  with radius ≤ 1.6 mm vanish, 3 mm defects recover ~35% of their area, 4.5 mm ~30%. dAB
  under-reports full-thickness loss rather than inventing it — zero false-positive dAB on
  healthy phantoms.
* **The site variable is an acquisition-batch surrogate**, the OAI image release, not the
  clinical site (`V00SITE`, which lives in the OAI enrollees table and is not
  redistributed here). "Site-conditional coverage" should name which variable it means.
* **Phantom numbers are not clinical results.** They exercise the pipeline and validate
  the mathematics. Point `OAIZIB_ROOT` at real data for anything else.
* **The segmentation demo is 6 epochs on phantoms** and is not a performance claim.

**The coverage guarantee, precisely.** Split conformal gives
`P(Y ∈ C(X)) ≥ 1 − α` in finite samples, distribution-free, *under exchangeability of
calibration and test data*. It is marginal unless Mondrian grouping is used, in which case
it is conditional on the grouping variable for groups with enough calibration points —
groups that fell back to the pooled threshold are named in `extras["merged"]` and are
**not** guaranteed. The guarantee says nothing about whether the point prediction is
clinically useful, and it is void the moment any calibration subject informs training,
tuning, thresholding or model selection. That is why `CalibrationGuard` exists.